# Results figures — Shared setup

## Mount

In [ ]:
# ── 0. Mount Google Drive & install fonts ─────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import subprocess
subprocess.run(['apt-get', 'install', '-y', '-qq', 'fonts-liberation'],
               capture_output=True)

import matplotlib
matplotlib.font_manager.fontManager.addfont('/usr/share/fonts/truetype/liberation/LiberationSans-Regular.ttf')
matplotlib.font_manager.fontManager.addfont('/usr/share/fonts/truetype/liberation/LiberationSans-Bold.ttf')
matplotlib.font_manager.fontManager.addfont('/usr/share/fonts/truetype/liberation/LiberationSans-Italic.ttf')
matplotlib.font_manager._load_fontmanager(try_read_cache=False)
print("Fonts installed.")

## Packages

In [ ]:
!pip install adjustText
!pip install gseapy
!pip install scanpy

In [ ]:
# ── 1. Imports (single block for entire notebook) ─────────────────
import csv
import os
import numpy as np
import pandas as pd
import textwrap
import io
import csv as csvmod
from collections import defaultdict
import gseapy as gp
import scanpy as sc

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
from matplotlib.path import Path
from adjustText import adjust_text
from matplotlib.patches import Ellipse
import matplotlib.gridspec as gridspec
import warnings
print("Imports done.")

## Setup

In [ ]:
# ── 2. Shared configuration ──────────────────────────────────────
DPI = 300
FONT_FAMILY = 'Liberation Sans'   # metrically identical to Arial; available in Colab
SAVE_PDF = True
SAVE_PNG = True

# Paths
DATA_BASE  = "/content/drive/MyDrive/IMSB/projects/Hubers lab/Aging_DEG_Enrichment_Cellchat/2025_wrapup"
OUTPUT_DIR = "/content/drive/MyDrive/Promotion/Thesis/Thesis_writing/Results/scripts/figures"
OUTPUT_DIR_DIS = "/content/drive/MyDrive/Promotion/Thesis/Thesis_writing/Results/scripts/figures_discussion/"

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR_DIS, exist_ok=True)
# ── v2 Config ────────────────────────────────────────────────────
MUSCLE_V2 = f"{DATA_BASE}/Muscle_v2"


# Input CSVs
KIDNEY_CSV = f"{DATA_BASE}/Kidney/1_DEGs_tables/kidney_res_age_results_with_symbols.csv"
MUSCLE_CSV = f"{DATA_BASE}/Muscle_old/1_DEGs_tables/muscle_res_age_results_with_symbols.csv"

# ── Shared color palette (muted academic) ─────────────────────────
#                Light fill      Dark (edge / text)
C_RED     = '#E8D5D5';  C_RED_D    = '#A32D2D'
C_BLUE    = '#D5E3F0';  C_BLUE_D   = '#185FA5'
C_TEAL    = '#D0EDE2';  C_TEAL_D   = '#0F6E56'
C_GREEN   = '#C8DEB5';  C_GREEN_D  = '#3B6D11'
C_PURPLE  = '#D5D0E8';  C_PURP_D   = '#534AB7'
C_AMBER   = '#F0E2C8';  C_AMBER_D  = '#854F0B'
C_CORAL   = '#EBCFC5';  C_CORAL_D  = '#993C1D'
C_GRAY    = '#E0DFDB';  C_GRAY_D   = '#73726c'
C_WHITE   = '#FFFFFF'
C_BG_ALT  = '#F7F7F5'
C_TEXT    = '#1a1a1a'
C_TEXT2   = '#555555'

# Global matplotlib defaults
plt.rcParams.update({
    'font.family': FONT_FAMILY,
    'font.size': 9,
    'axes.linewidth': 0,
    'figure.facecolor': 'white',
})

# ── Shared save helper ────────────────────────────────────────────
def save_fig(fig, filename):
    """Save figure as PDF and/or PNG to OUTPUT_DIR."""
    if SAVE_PDF:
        p = os.path.join(OUTPUT_DIR, f'{filename}.pdf')
        fig.savefig(p, dpi=DPI, bbox_inches='tight', facecolor='white', edgecolor='none')
        print(f'  Saved {p}')
    if SAVE_PNG:
        p = os.path.join(OUTPUT_DIR, f'{filename}.png')
        fig.savefig(p, dpi=DPI, bbox_inches='tight', facecolor='white', edgecolor='none')
        print(f'  Saved {p}')
    plt.close(fig)

print("Config loaded.")

In [ ]:
# ── Cell type name cleaning ───────────────────────────────────────
# Use clean_name(ct) anywhere a cell type name appears as a label.
# Data files keep original names — this only affects display.

CLEAN_NAMES = {
    # Kidney
    'EC-1(gEC)':       'EC-1/gEC',
    'PECs?':           'PECs',
    'Undef':           'Undefined',
    # Muscle
    'Immune cells?':   'Immune cells',
    'Myonuclei (Myh4+?)':                      'Myh4+ myonuclei',
    'Myonuclei (Myh1+)':                       'Myh1+ myonuclei',
    'Myonuclei (differentiating myocytes?)':    'Differentiating myocytes',
    'Myonuclei (myotendinous junction nuclei)': 'MTJ nuclei',
    'Myonuclei (neuromuscular junction nuclei)':'NMJ nuclei',
    'Myonuclei':                                'General myonuclei',
    'Muscle stem cell':                         'Muscle stem cells',
}

# Short names for circular network plots where space is limited
SHORT_NAMES = {
    **CLEAN_NAMES,
    'Endothelial cells': 'ECs',
    'EC-1/gEC':          'EC-1',
    'Lymphatic vessels':  'Lymph.',
    'Macrophages':        'Macro.',
    'Fibroblasts':        'Fibro.',
    'Muscle stem cells':  'MuSC',
    'Tenocytes':          'Teno.',
    'FAPs':               'FAPs',
    'Differentiating myocytes': 'Diff. myo.',
    'General myonuclei':  'Myo.',
}

def clean_name(name):
    """Full clean name for axis labels, tables, text."""
    return CLEAN_NAMES.get(name, name)

def short_name(name):
    """Short name for circular network plots."""
    # First clean, then shorten
    cleaned = CLEAN_NAMES.get(name, name)
    return SHORT_NAMES.get(cleaned, SHORT_NAMES.get(name, cleaned))

# 3.1 Characterization of Aging Transcriptomes in Kidney and Muscle

## Fig 3.1A — Volcano plots aging

In [ ]:
FILENAME = 'Fig_3_1A_volcano_aging_kidney_muscle'
PADJ_THR = 0.05

def load_degs(filepath):
    lfcs, padjs, syms = [], [], []
    with open(filepath) as f:
        for r in csv.DictReader(f):
            p, l = r.get('padj','NA'), r.get('log2FoldChange','NA')
            if p in ('NA','','NaN') or l in ('NA','','NaN'): continue
            padj = max(float(p), 1e-310)
            lfcs.append(float(l)); padjs.append(padj); syms.append(r.get('Symbol',''))
    return np.array(lfcs), np.array(padjs), syms

k_lfc, k_padj, k_sym = load_degs(f"{DATA_BASE}/Kidney/1_DEGs_tables/kidney_res_age_results_with_symbols.csv")
m_lfc, m_padj, m_sym = load_degs(f"{DATA_BASE}/Muscle_old/1_DEGs_tables/muscle_res_age_results_with_symbols.csv")

print(f"Kidney: {len(k_lfc)} genes | Muscle: {len(m_lfc)} genes")

def plot_volcano(ax, lfc, padj, syms, title, label_genes):
    nlog10p = -np.log10(padj)
    sig = padj < PADJ_THR
    up = sig & (lfc > 0)
    down = sig & (lfc < 0)
    ns = ~sig

    # Plot ns first (background), then sig
    ax.scatter(lfc[ns], nlog10p[ns], c='#CCCCCC', s=2, alpha=0.15, edgecolors='none', rasterized=True, zorder=1)
    ax.scatter(lfc[up], nlog10p[up], c=C_RED_D, s=6, alpha=0.45, edgecolors='none', rasterized=True, zorder=2)
    ax.scatter(lfc[down], nlog10p[down], c=C_BLUE_D, s=6, alpha=0.45, edgecolors='none', rasterized=True, zorder=2)

    # Reference lines
    ax.axhline(-np.log10(PADJ_THR), color=C_GRAY_D, lw=0.6, ls='--', zorder=0)
    ax.axvline(0, color=C_GRAY_D, lw=0.4, zorder=0)

    # Label genes
    sym_arr = np.array(syms)
    for gene in label_genes:
        if gene.startswith('ENSMUSG') or gene == '': continue  # skip Ensembl IDs and empty
        idx = np.where(sym_arr == gene)[0]
        if len(idx) == 0: continue
        i = idx[0]
        if not sig[i]: continue
        x_off = 8 if lfc[i] > 0 else -8
        color = C_RED_D if lfc[i] > 0 else C_BLUE_D
        ax.annotate(gene, xy=(lfc[i], nlog10p[i]), xytext=(x_off, 6),
                    textcoords='offset points', fontsize=5.5, fontstyle='italic',
                    fontweight='bold', color=color, alpha=0.85,
                    arrowprops=dict(arrowstyle='-', color=color, lw=0.4, alpha=0.4), zorder=5)

    n_up = int(up.sum()); n_down = int(down.sum())
    ax.text(0.98, 0.97, f'{n_up:,} up', transform=ax.transAxes, ha='right', va='top',
            fontsize=9, fontweight='bold', color=C_RED_D)
    ax.text(0.02, 0.97, f'{n_down:,} down', transform=ax.transAxes, ha='left', va='top',
            fontsize=9, fontweight='bold', color=C_BLUE_D)
    ax.text(0.98, 0.04, 'padj = 0.05', transform=ax.transAxes, ha='right', va='bottom',
            fontsize=6, color=C_GRAY_D)

    ax.set_xlabel('log$_2$ fold change (age vs. ctrl)', fontsize=10, fontweight='bold')
    ax.set_ylabel('$-$log$_{10}$(padj)', fontsize=10, fontweight='bold')
    ax.set_title(title, fontsize=12, fontweight='bold', color=C_TEXT, pad=10)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_linewidth(0.5)
    ax.spines['bottom'].set_linewidth(0.5)
    ax.tick_params(labelsize=8)

# Top genes by significance x effect size
def pick_labels(lfc, padj, syms, n=12, extras=[]):
    nlog10p = -np.log10(padj)
    sig = padj < PADJ_THR
    score = nlog10p * np.abs(lfc) * sig
    top_idx = np.argsort(score)[::-1]
    # Mix of up and down
    up_labels, down_labels = [], []
    for i in top_idx:
        if len(up_labels) + len(down_labels) >= n: break
        if lfc[i] > 0 and len(up_labels) < n//2 + 1 and not syms[i].startswith('ENSMUSG') and syms[i] != '':
            up_labels.append(syms[i])
        elif lfc[i] < 0 and len(down_labels) < n//2 + 1 and not syms[i].startswith('ENSMUSG') and syms[i] != '':
            down_labels.append(syms[i])
    return set(up_labels + down_labels) | set(extras)

k_labels = pick_labels(k_lfc, k_padj, k_sym, 14, ['Tgfb1','Flt1','B2m','Gstm1','Prkd1'])
m_labels = pick_labels(m_lfc, m_padj, m_sym, 14, ['Tgfb1','Ryr3','Kcnq5','Wwox','Ptprm'])

fig, (ax_k, ax_m) = plt.subplots(1, 2, figsize=(13, 6.5))

plot_volcano(ax_k, k_lfc, k_padj, k_sym, 'Kidney', k_labels)
plot_volcano(ax_m, m_lfc, m_padj, m_sym, 'Muscle', m_labels)

# Match y-axes
y_max = max(ax_k.get_ylim()[1], ax_m.get_ylim()[1])
ax_k.set_ylim(-2, y_max * 1.02)
ax_m.set_ylim(-2, y_max * 1.02)

fig.suptitle('Fig. 3.1A \u2014 Aging transcriptional signatures in kidney and muscle',
             fontsize=11, fontweight='bold', color=C_TEXT, y=0.99)

legend_elements = [
    mpatches.Patch(facecolor=C_RED_D, alpha=0.6, label='Upregulated (padj < 0.05)'),
    mpatches.Patch(facecolor=C_BLUE_D, alpha=0.6, label='Downregulated (padj < 0.05)'),
    mpatches.Patch(facecolor='#CCCCCC', alpha=0.4, label='Not significant'),
]
fig.legend(handles=legend_elements, loc='lower center', ncol=3, fontsize=8,
           frameon=False, bbox_to_anchor=(0.5, 0.02))

fig.text(0.5, -0.01,
         'Bulk RNA-seq, DESeq2 age vs. ctrl contrast. Dashed line = padj 0.05 threshold. '
         'Top genes labelled by significance \u00d7 effect size.',
         ha='center', fontsize=6.5, color=C_TEXT2)

plt.tight_layout(rect=[0, 0.06, 1, 0.97])
save_fig(fig, FILENAME)
print('Done \u2014 Fig 3.1A')

## Fig 3.1C — GOBP aging enrichment

In [ ]:
FILENAME = 'Fig_3_1CD_GOBP_aging_kidney_muscle'

# ── Load enrichment data ─────────────────────────────────────────
def load_enrich(filepath, n=18):
    """Load top n GOBP terms, removing near-duplicates."""
    with open(filepath) as f:
        rows = list(csv.DictReader(f))

    terms = []
    seen_words = []
    for r in rows:
        desc = r['Description']
        # Simple dedup: skip if >60% word overlap with an already-selected term
        words = set(desc.lower().split())
        overlap = any(len(words & s) / max(len(words), 1) > 0.6 for s in seen_words)
        if overlap and len(terms) > 3:
            continue

        num, denom = r['GeneRatio'].split('/')
        terms.append({
            'desc': desc,
            'count': int(num),
            'bg_count': int(denom),
            'gene_ratio': int(num) / int(denom),
            'padj': float(r['p.adjust']),
            'nlog10p': -np.log10(max(float(r['p.adjust']), 1e-310)),
        })
        seen_words.append(words)
        if len(terms) >= n:
            break
    return terms

k_terms = load_enrich(f"{DATA_BASE}/Kidney/2_Enrichment_bulk&sn/GOBP bulk/kidney_enrichment_results_aging.csv", 15)
m_terms = load_enrich(f"{DATA_BASE}/Muscle_old/2_Enrichment_tables/GOBP bulk/muscle_enrichment_results_aging.csv", 15)

# ── Plot ─────────────────────────────────────────────────────────
fig, (ax_k, ax_m) = plt.subplots(1, 2, figsize=(14, 8),
                                  gridspec_kw={'width_ratios': [1.1, 1]})

def plot_dotplot(ax, terms, title, color_accent, total_degs):
    n = len(terms)
    y_pos = np.arange(n)[::-1]

    counts = [t['count'] for t in terms]
    nlog10p = [t['nlog10p'] for t in terms]
    gene_ratio = [t['gene_ratio'] for t in terms]
    descs = [t['desc'] for t in terms]

    # Wrap long descriptions
    descs_wrapped = []
    for d in descs:
        if len(d) > 45:
            d = textwrap.fill(d, width=40)
        descs_wrapped.append(d)

    # Dot size proportional to gene count
    min_count, max_count = min(counts), max(counts)
    sizes = [30 + (c - min_count) / max(max_count - min_count, 1) * 200 for c in counts]

    # Color by -log10(padj)
    scatter = ax.scatter(
        gene_ratio, y_pos,
        s=sizes, c=nlog10p,
        cmap='YlOrRd', vmin=min(nlog10p)*0.8, vmax=max(nlog10p)*1.05,
        edgecolors=color_accent, linewidths=0.4,
        zorder=3, alpha=0.85
    )

    ax.set_yticks(y_pos)
    ax.set_yticklabels(descs_wrapped, fontsize=10)
    ax.set_xlabel('Gene ratio', fontsize=9.5, fontweight='bold')
    ax.set_title(title, fontsize=11, fontweight='bold', color=C_TEXT, pad=10)

    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_linewidth(0.5)
    ax.spines['bottom'].set_linewidth(0.5)
    ax.tick_params(axis='y', length=0)
    ax.grid(axis='x', alpha=0.15, lw=0.5)

    # DEG count annotation
    ax.text(0.98, 0.02, f'{total_degs:,} aging DEGs\n{len(terms)} of {2757 if "Kidney" in title else 264} terms shown',
            transform=ax.transAxes, ha='right', va='bottom',
            fontsize=6.5, color=C_TEXT2, fontstyle='italic')

    return scatter

sc_k = plot_dotplot(ax_k, k_terms, 'Kidney — aging GOBP enrichment', C_RED_D, 7499)
sc_m = plot_dotplot(ax_m, m_terms, 'Muscle — aging GOBP enrichment', C_BLUE_D, 1739)

# Inset colorbars (inside plot area, bottom-left)
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
for ax_ref, sc in [(ax_k, sc_k), (ax_m, sc_m)]:
    cbax = inset_axes(ax_ref, width="30%", height="3%", loc='lower left',
                       bbox_to_anchor=(0.02, 0.02, 1, 1), bbox_transform=ax_ref.transAxes)
    cbar = fig.colorbar(sc, cax=cbax, orientation='horizontal')
    cbar.set_label('$-$log$_{10}$(padj)', fontsize=6, labelpad=2)
    cbar.ax.tick_params(labelsize=5)

# Size legend
for n_leg, lab in [(50, '50'), (150, '150'), (250, '250')]:
    sz = 30 + (n_leg - 20) / 250 * 200
    ax_m.scatter([], [], s=sz, c='gray', alpha=0.5, edgecolors=C_GRAY_D,
                  linewidths=0.4, label=f'{n_leg} genes')
ax_m.legend(loc='lower right', fontsize=6.5, frameon=True, framealpha=0.9,
            edgecolor='#E0E0E0', title='Gene count', title_fontsize=6.5,
            handletextpad=0.3, borderpad=0.5, labelspacing=0.8)

# Suptitle
fig.suptitle('Fig. 3.1C/D \u2014 GO Biological Process enrichment of aging DEGs',
             fontsize=11, fontweight='bold', color=C_TEXT, y=1.01)

# Footnote
fig.text(0.5, 0.005,
         'enrichGO (clusterProfiler), padj < 0.05. Near-duplicate terms removed for clarity.\n'
         'Kidney: 2,757 significant GOBP terms from 7,499 aging DEGs. Muscle: 264 terms from 1,739 DEGs.',
         ha='center', fontsize=8, color=C_TEXT2)

plt.tight_layout(rect=[0, 0.03, 1, 0.97])
save_fig(fig, FILENAME)
print('Done \u2014 Fig 3.1C/D')

## Table 3.2A — Rescue framework summary

In [ ]:
# ── Table 3.2A ────────────────────────────────────────────────────
FILENAME = 'Table_3_2A_rescue_summary'

rows = [
    (0, 'Genes tested',                          '22,109',                      '21,055',                      True,  C_BG_ALT, False),
    (0, 'Aging DEGs (padj < 0.05)',               '7,499',                       '1,739',                       True,  C_BG_ALT, False),
    (1, 'Upregulated',                            '3,704 (49.4%)',               '881 (50.7%)',                 False, C_WHITE,  False),
    (1, 'Downregulated',                          '3,795 (50.6%)',               '858 (49.3%)',                 False, C_WHITE,  False),
    (0, '',                                       '',                            '',                            False, C_WHITE,  False),
    (0, 'Stage 1 \u2014 Rescued by combined intervention', '',                  '',                            True,  '#EEF4FB', False),
    (1, 'Sign reversal (combi vs. aging)',         '4,645 (61.9% of aging DEGs)','926 (53.2%)',                 False, C_WHITE,  False),
    (1, 'Not reversed by combined intervention',   '2,854 (38.1%)',              '813 (46.8%)',                 False, C_WHITE,  True),
    (0, '',                                       '',                            '',                            False, C_WHITE,  False),
    (0, 'Stage 2 \u2014 Rescued-to-normal',      '',                            '',                            True,  '#EDF8F2', False),
    (1, 'Rescued + restored (|LFC combi vs ctrl| < 0.5)', '1,961 (42.2% of rescued)', '511 (55.2%)',           False, C_WHITE,  False),
    (1, 'Rescued but not normalised',             '2,684 (57.8%)',               '415 (44.8%)',                 False, C_WHITE,  True),
    (0, '',                                       '',                            '',                            False, C_WHITE,  False),
    (0, 'Stage 3 \u2014 Intervention driver classification', '',                '',                            True,  '#FDF8EE', False),
    (0, '(within rescued-to-normal set)',          '',                            '',                            False, C_WHITE,  True),
    (1, 'DR-exclusive',                           '546 (27.8%)',                 '241 (47.2%)',                 False, C_WHITE,  False),
    (1, 'sActRIIB-exclusive',                     '108 (5.5%)',                  '23 (4.5%)',                   False, C_WHITE,  False),
    (1, 'Combination-only (synergistic)',          '87 (4.4%)',                   '18 (3.5%)',                   False, C_WHITE,  False),
    (1, 'Both DR + sActRIIB (convergent)',         '1,220 (62.2%)',              '229 (44.8%)',                 False, C_WHITE,  False),
    (0, '',                                       '',                            '',                            False, C_WHITE,  False),
    (0, 'Non-rescued aging DEGs',                 '',                            '',                            True,  '#FDF2F2', False),
    (1, 'Not rescued by any intervention',        '1,169 (15.6% of aging DEGs)','300 (17.3%)',            False, C_WHITE,  False),
    (1, 'Rescued by DR/sAct but not by combi',    '1,685 (22.5%)',              '\u2014',                     False, C_WHITE,  True),
]

fig, ax = plt.subplots(1, 1, figsize=(10, 8.5))
ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.axis('off')

table_left, table_right = 0.04, 0.96
table_width = table_right - table_left
col_widths = [0.48, 0.26, 0.26]
col_xs = [table_left]
for w in col_widths[:-1]:
    col_xs.append(col_xs[-1] + w * table_width)
row_height, header_height = 0.028, 0.032
y_top = 0.89

# Caption
ax.text(table_left, 0.96, 'Table 3.2A.', fontsize=9, fontweight='bold', color=C_TEXT, va='top')
ax.text(table_left + 0.065, 0.96,
        'Summary of rescue framework categories for kidney and muscle bulk RNA-seq datasets.',
        fontsize=8.5, color=C_TEXT, va='top')
ax.text(table_left, 0.935,
        'The rescue framework is applied in three sequential stages: (1) identification of aging DEGs showing\n'
        'directional reversal under the combined intervention, (2) filtering to genes restored to near-control\n'
        'expression levels, and (3) classification by individual intervention driver.',
        fontsize=7.5, color=C_TEXT2, va='top', linespacing=1.4)

# Header
ax.plot([table_left, table_right], [y_top, y_top], color=C_TEXT, lw=1.5, clip_on=False)
for i, (lab, align, pad) in enumerate(zip(
        ['Category', 'Kidney', 'Muscle'], ['left', 'right', 'right'], [0.01, -0.01, -0.01])):
    hx = col_xs[i] + pad if align == 'left' else col_xs[i] + col_widths[i] * table_width + pad
    ax.text(hx, y_top - header_height/2, lab, fontsize=8.5, fontweight='bold', color=C_TEXT, va='center', ha=align)
y_hdr_bot = y_top - header_height
ax.plot([table_left, table_right], [y_hdr_bot, y_hdr_bot], color=C_TEXT, lw=0.8, clip_on=False)

# Data rows
y_cur = y_hdr_bot
for indent, cat, kidney, muscle, is_sec, bg, is_gray in rows:
    if cat == '' and kidney == '' and muscle == '':
        y_cur -= row_height * 0.4; continue
    rh = row_height * 1.05 if is_sec else row_height
    if bg != C_WHITE:
        ax.add_patch(mpatches.Rectangle((table_left, y_cur - rh), table_width, rh, fc=bg, ec='none', zorder=0))
    ax.plot([table_left, table_right], [y_cur - rh, y_cur - rh], color='#E5E5E5', lw=0.3, clip_on=False, zorder=1)
    tc = C_TEXT2 if is_gray else C_TEXT
    cat_x = col_xs[0] + 0.01 + indent * 0.025
    cat_fs = 6.5 if '(within' in cat else (8 if is_sec else 7.5)
    cat_sty = 'italic' if '(within' in cat else 'normal'
    ax.text(cat_x, y_cur - rh/2, cat, fontsize=cat_fs, fontweight='bold' if is_sec else 'normal',
            fontstyle=cat_sty, color=tc, va='center', ha='left', zorder=2)
    if kidney:
        ax.text(col_xs[1] + col_widths[1]*table_width - 0.01, y_cur - rh/2, kidney,
                fontsize=7.5, color=tc, va='center', ha='right', zorder=2)
    if muscle:
        ax.text(col_xs[2] + col_widths[2]*table_width - 0.01, y_cur - rh/2, muscle,
                fontsize=7.5, color=tc, va='center', ha='right', zorder=2)
    y_cur -= rh
ax.plot([table_left, table_right], [y_cur, y_cur], color=C_TEXT, lw=1.5, clip_on=False)

# Footnotes
fn_y = y_cur - 0.025
for fn in [
    'Aging DEGs defined as padj < 0.05 and |log2FC| > 0 (age vs. ctrl contrast, DESeq2). Rescue defined as sign(LFC_aging) \u2260 sign(LFC_intervention).',
    'Rescued-to-normal requires additionally |LFC combi vs. ctrl| < 0.5. Percentages in Stages 1\u20132 relative to preceding category; Stage 3 relative to rescued-to-normal set.',
    'Pipeline note: kidney re-processed from raw reads (STAR + Salmon + DESeq2, GRCm39); muscle used pre-processed FeatureCounts counts (GRCm38).',
]:
    ax.text(table_left + 0.01, fn_y, fn, fontsize=6, color=C_TEXT2, va='top', wrap=True)
    fn_y -= 0.025

plt.tight_layout(pad=0.3)
save_fig(fig, FILENAME)
print('Done \u2014 Table 3.2A')

## Fig 3.1B — DEG counts per tissue (up/down)

In [ ]:
# ── Fig 3.1B ─────────────────────────────────────────────────────
FILENAME = 'Fig_3_1B_DEG_counts_per_tissue'

def count_degs(filepath):
    with open(filepath) as f:
        rows = list(csv.DictReader(f))
    sig = [r for r in rows if r.get('padj', 'NA') not in ('NA', '') and float(r['padj']) < 0.05]
    up   = sum(1 for r in sig if float(r['log2FoldChange']) > 0)
    down = sum(1 for r in sig if float(r['log2FoldChange']) < 0)
    return up, down, len(rows)

k_up, k_down, k_total = count_degs(KIDNEY_CSV)
m_up, m_down, m_total = count_degs(MUSCLE_CSV)
print(f"Kidney  \u2014 tested: {k_total:,}  up: {k_up:,}  down: {k_down:,}  total DEGs: {k_up+k_down:,}")
print(f"Muscle  \u2014 tested: {m_total:,}  up: {m_up:,}  down: {m_down:,}  total DEGs: {m_up+m_down:,}")

tissues   = ['Kidney', 'Muscle']
up_vals   = [k_up, m_up]
down_vals = [k_down, m_down]
x = np.arange(len(tissues))
width = 0.32

fig, ax = plt.subplots(figsize=(5, 4.5))

bars_up   = ax.bar(x - width/2, up_vals,   width, label='Upregulated',
                    color=C_RED, edgecolor=C_RED_D, linewidth=0.5)
bars_down = ax.bar(x + width/2, down_vals, width, label='Downregulated',
                    color=C_BLUE, edgecolor=C_BLUE_D, linewidth=0.5)

for bars in [bars_up, bars_down]:
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + 80,
                f'{int(h):,}', ha='center', va='bottom', fontsize=9, fontweight='bold')

ax.set_ylabel('Number of DEGs (padj < 0.05)', fontsize=12)
ax.set_xticks(x)
ax.set_xticklabels(tissues, fontsize=13, fontweight='bold')
ax.set_ylim(0, max(up_vals + down_vals) * 1.15)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{int(v):,}'))
ax.legend(frameon=True, framealpha=1, edgecolor='none', fontsize=10, loc='upper right')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
save_fig(fig, FILENAME)
print('Done \u2014 Fig 3.1B')

## Fig 3.1E/F ─ Violin plots of aging module scores by cell type

In [ ]:
# ── Fig 3.1E/F ───────────────────────────────────────────────────
# Violin plots of aging module scores by cell type
FILENAME_E = 'Fig_3_1E_violin_aging_module_kidney'
FILENAME_F = 'Fig_3_1F_violin_aging_module_muscle'

KIDNEY_MODULE_CSV = f"{DATA_BASE}/Kidney/kidney_AgingGeneScores_perCell_seurat.csv"
MUSCLE_MODULE_CSV = f"{DATA_BASE}/Muscle_old/muscle_AgingGeneScores_perCell_seurat.csv"

def load_module_scores(filepath):
    ct_scores = defaultdict(list)
    with open(filepath) as f:
        for r in csv.DictReader(f):
            ct_scores[r['celltype']].append(float(r['Aged1']))
    return ct_scores

def plot_violin(ct_scores, title, filename, figsize=(12, 6)):
    cts_sorted = sorted(ct_scores.keys(),
                        key=lambda ct: np.median(ct_scores[ct]), reverse=True)
    data = [ct_scores[ct] for ct in cts_sorted]
    n = len(cts_sorted)

    fig, ax = plt.subplots(figsize=figsize)

    palette = ['#E8918D', '#E8A86B', '#D4B56A', '#A3BE8C', '#7DB89E',
               '#6BBFB0', '#5EAAA8', '#6BA3C7', '#7E94C0', '#9B8EC2',
               '#B68CB5', '#C98BA3', '#D4A19A', '#BDB5A0', '#A0C4A8']
    colors = [palette[i % len(palette)] for i in range(n)]

    parts = ax.violinplot(data, positions=range(n), showmeans=False,
                           showmedians=False, showextrema=False)
    for i, body in enumerate(parts['bodies']):
        body.set_facecolor(colors[i])
        body.set_edgecolor(colors[i])
        body.set_alpha(0.75)
        body.set_linewidth(0.5)

    for i, d in enumerate(data):
        med = np.median(d)
        q1, q3 = np.percentile(d, [25, 75])
        ax.vlines(i, q1, q3, color='#333333', linewidth=0.8, zorder=3)
        ax.scatter(i, med, color='white', s=8, edgecolors='#333333',
                   linewidths=0.5, zorder=4)

    # ── CHANGED: apply clean_name before label formatting ─────────
    labels = [textwrap.fill(clean_name(ct), width=15) if len(clean_name(ct)) > 12 else clean_name(ct) for ct in cts_sorted]
    ax.set_xticks(range(n))
    ax.set_xticklabels(labels, fontsize=7, rotation=45, ha='right')
    ax.set_ylabel('Module score (aged gene set)', fontsize=10, fontweight='bold')
    ax.set_title(title, fontsize=12, fontweight='bold', color=C_TEXT, pad=12)
    ax.axhline(0, color=C_GRAY_D, lw=0.5, ls='--', zorder=0)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_linewidth(0.5)
    ax.spines['bottom'].set_linewidth(0.5)
    ax.tick_params(axis='y', labelsize=8)
    ax.grid(axis='y', alpha=0.1, lw=0.5)

    # ── CHANGED: clean names in top 3 annotation ──────────────────
    top3 = [clean_name(ct) for ct in cts_sorted[:3]]
    ax.text(0.98, 0.97, f'Top 3: {", ".join(top3)}',
            transform=ax.transAxes, ha='right', va='top', fontsize=7,
            color=C_TEXT2, fontstyle='italic',
            bbox=dict(boxstyle='round,pad=0.3', fc=C_WHITE, ec='#E0E0E0', lw=0.5))

    total_cells = sum(len(d) for d in data)
    ax.text(0.02, 0.97, f'{total_cells:,} nuclei across {n} cell types',
            transform=ax.transAxes, ha='left', va='top', fontsize=7, color=C_TEXT2)

    plt.tight_layout()
    save_fig(fig, filename)
    print(f'Done — {filename}')

k_scores = load_module_scores(KIDNEY_MODULE_CSV)
plot_violin(k_scores, 'Fig. 3.1E — Kidney aging module scores by cell type',
            FILENAME_E, figsize=(13, 6))

m_scores = load_module_scores(MUSCLE_MODULE_CSV)
plot_violin(m_scores, 'Fig. 3.1F — Muscle aging module scores by cell type',
            FILENAME_F, figsize=(11, 6))

## Fig 3.1G — Long-gene bias in aging transcriptomes (kidney + muscle)

In [ ]:
# ── Fig 3.1G ─────────────────────────────────────────────────────
# Long-gene bias in aging: are longer genes more affected?
# Tests the transcription stress / DNA damage hypothesis genome-wide.
FILENAME = 'Fig_3_1G_long_gene_bias_aging'

# ── 1. Fetch gene lengths from Ensembl BioMart (runs on Colab) ───
import subprocess
subprocess.run(['pip', 'install', 'pybiomart', '-q'], capture_output=True)

from pybiomart import Dataset
from scipy import stats as sp_stats

print("Fetching gene lengths from Ensembl BioMart...")
dataset = Dataset(name='mmusculus_gene_ensembl', host='http://oct2024.archive.ensembl.org')
#dataset = Dataset(name='mmusculus_gene_ensembl', host='http://www.ensembl.org')
result = dataset.query(attributes=['ensembl_gene_id', 'external_gene_name',
                                    'start_position', 'end_position',
                                    'chromosome_name'])
result.columns = ['ensembl_id', 'symbol', 'start', 'end', 'chr']
result['gene_length_kb'] = (result['end'] - result['start']) / 1000
gene_len = dict(zip(result['ensembl_id'], result['gene_length_kb']))
print(f"Got {len(gene_len)} gene lengths from Ensembl")

In [ ]:
import csv, numpy as np
from scipy import stats as sp_stats

def load_with_lengths(deg_csv, gene_len_dict):
    lfcs, lengths, padjs = [], [], []
    with open(deg_csv) as f:
        for r in csv.DictReader(f):
            eid = list(r.values())[0].split('.')[0]
            p, l = r.get('padj','NA'), r.get('log2FoldChange','NA')
            if p in ('NA','','NaN') or l in ('NA','','NaN'): continue
            if eid not in gene_len_dict: continue
            lfcs.append(float(l)); lengths.append(gene_len_dict[eid])
            padjs.append(max(float(p),1e-310))
    return np.array(lfcs), np.array(lengths), np.array(padjs)

k_lfc, k_len, k_padj = load_with_lengths(
    f"{DATA_BASE}/Kidney/1_DEGs_tables/kidney_res_age_results_with_symbols.csv", gene_len)
m_lfc_v2, m_len_v2, m_padj_v2 = load_with_lengths(
    f"{MUSCLE_V2}/muscle_v2_res_age_results_with_symbols.csv", gene_len)

print("gene_len entries:", len(gene_len))
print("kidney matched:", len(k_lfc), "| muscle v2 matched:", len(m_lfc_v2))

ksig, msig = k_padj < 0.05, m_padj_v2 < 0.05
print("\n-- CONDITIONED on padj<0.05 (what Fig 3.1H reports) --")
print("kidney    signed-LFC rho:", sp_stats.spearmanr(k_len[ksig], k_lfc[ksig]), "n=", int(ksig.sum()))
print("muscle v2 signed-LFC rho:", sp_stats.spearmanr(m_len_v2[msig], m_lfc_v2[msig]), "n=", int(msig.sum()))

print("\n-- UNCONDITIONED, all matched genes --")
print("kidney    signed-LFC rho:", sp_stats.spearmanr(k_len, k_lfc))
print("muscle v2 signed-LFC rho:", sp_stats.spearmanr(m_len_v2, m_lfc_v2))

print("\n-- |LFC| version (what Fig 3.1G reports) --")
print("kidney    |LFC| rho:", sp_stats.spearmanr(k_len, np.abs(k_lfc)))
print("muscle v2 |LFC| rho:", sp_stats.spearmanr(m_len_v2, np.abs(m_lfc_v2)))

In [ ]:
# ── 2. Load DEGs and merge with lengths ──────────────────────────
def load_with_lengths(deg_csv, gene_len_dict):
    lfcs, lengths, padjs, syms = [], [], [], []
    with open(deg_csv) as f:
        for r in csv.DictReader(f):
            eid = list(r.values())[0].split('.')[0]  # first col = Ensembl ID
            p = r.get('padj', 'NA')
            l = r.get('log2FoldChange', 'NA')
            if p in ('NA', '', 'NaN') or l in ('NA', '', 'NaN'):
                continue
            if eid not in gene_len_dict:
                continue
            lfcs.append(float(l))
            lengths.append(gene_len_dict[eid])
            padjs.append(max(float(p), 1e-310))
            syms.append(r.get('Symbol', ''))
    return np.array(lfcs), np.array(lengths), np.array(padjs), syms

k_lfc, k_len, k_padj, k_sym = load_with_lengths(
    f"{DATA_BASE}/Kidney/1_DEGs_tables/kidney_res_age_results_with_symbols.csv", gene_len)
m_lfc, m_len, m_padj, m_sym = load_with_lengths(
    f"{DATA_BASE}/Muscle_old/1_DEGs_tables/muscle_res_age_results_with_symbols.csv", gene_len)

print(f"Kidney: {len(k_lfc)} genes matched | Muscle: {len(m_lfc)} genes matched")

# ── 3. Bin genes by length ───────────────────────────────────────
def bin_analysis(lengths, lfcs, padjs, bins_kb=[0, 25, 50, 100, 200, 500, 1000, 5000]):
    results = []
    for i in range(len(bins_kb) - 1):
        lo, hi = bins_kb[i], bins_kb[i+1]
        mask = (lengths >= lo) & (lengths < hi)
        if mask.sum() < 5:
            continue
        bin_lfc = lfcs[mask]
        bin_padj = padjs[mask]
        results.append({
            'bin_label': f'{lo}-{hi}' if hi < 5000 else f'{lo}+',
            'n_genes': int(mask.sum()),
            'mean_abs_lfc': float(np.mean(np.abs(bin_lfc))),
            'median_abs_lfc': float(np.median(np.abs(bin_lfc))),
            'frac_sig': float(np.mean(bin_padj < 0.05)),
            'mean_lfc': float(np.mean(bin_lfc)),
        })
    return results

bins_kb = [0, 25, 50, 100, 200, 500, 1000, 5000]
k_bins = bin_analysis(k_len, k_lfc, k_padj, bins_kb)
m_bins = bin_analysis(m_len, m_lfc, m_padj, bins_kb)

k_corr, k_pval = sp_stats.spearmanr(k_len, np.abs(k_lfc))
m_corr, m_pval = sp_stats.spearmanr(m_len, np.abs(m_lfc))
print(f"Kidney: Spearman rho(length, |LFC|) = {k_corr:.3f}, p = {k_pval:.2e}")
print(f"Muscle: Spearman rho(length, |LFC|) = {m_corr:.3f}, p = {m_pval:.2e}")

# ── 4. Plot ──────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(12, 9))

def plot_tissue(ax_top, ax_bot, bins, tissue, corr, pval, color_d):
    labels = [b['bin_label'] for b in bins]
    x = np.arange(len(bins))

    # Top: mean |LFC| per bin
    vals = [b['mean_abs_lfc'] for b in bins]
    n_genes = [b['n_genes'] for b in bins]
    ax_top.bar(x, vals, color=color_d, alpha=0.6, edgecolor=color_d, linewidth=0.5)
    for i, (v, n) in enumerate(zip(vals, n_genes)):
        ax_top.text(i, v + 0.003, f'n={n:,}', ha='center', va='bottom',
                    fontsize=5.5, color=C_TEXT2)
    ax_top.set_ylabel('Mean |log2FC|\n(aging)', fontsize=9, fontweight='bold')
    ax_top.set_title(tissue, fontsize=11, fontweight='bold', color=C_TEXT, pad=8)
    ax_top.set_xticks(x)
    ax_top.set_xticklabels([])
    ax_top.spines['top'].set_visible(False)
    ax_top.spines['right'].set_visible(False)
    ax_top.text(0.25, 0.99, f'Spearman \u03c1 = {corr:.3f}\np = {pval:.1e}',
                transform=ax_top.transAxes, ha='right', va='top', fontsize=7.5,
                bbox=dict(boxstyle='round,pad=0.3', fc=C_WHITE, ec='#E0E0E0', lw=0.5))

    # Bottom: fraction significant per bin
    frac = [b['frac_sig'] for b in bins]
    ax_bot.bar(x, frac, color=color_d, alpha=0.4, edgecolor=color_d, linewidth=0.5)
    ax_bot.set_ylabel('Fraction significant\n(padj < 0.05)', fontsize=9, fontweight='bold')
    ax_bot.set_xlabel('Gene body length (kb)', fontsize=9, fontweight='bold')
    ax_bot.set_xticks(x)
    ax_bot.set_xticklabels(labels, fontsize=7.5, rotation=30, ha='right')
    ax_bot.spines['top'].set_visible(False)
    ax_bot.spines['right'].set_visible(False)
    ax_bot.axhline(0.5, color=C_GRAY_D, lw=0.5, ls='--', zorder=0)

plot_tissue(axes[0, 0], axes[1, 0], k_bins, 'Kidney', k_corr, k_pval, C_RED_D)
plot_tissue(axes[0, 1], axes[1, 1], m_bins, 'Muscle', m_corr, m_pval, C_BLUE_D)

fig.suptitle('Fig. 3.1G \u2014 Gene length bias in aging transcriptional dysregulation',
             fontsize=11, fontweight='bold', color=C_TEXT, y=1.01)

fig.text(0.5, -0.01,
         'Genes binned by genomic span (start to end, Ensembl). Top: mean |log2FC| per length bin.\n'
         'Bottom: fraction reaching significance per bin. Spearman correlation on unbinned data.\n'
         'A positive correlation supports the transcription stress hypothesis (DNA damage blocks long-gene transcription).',
         ha='center', fontsize=6.5, color=C_TEXT2, linespacing=1.4)

plt.tight_layout(rect=[0, 0.04, 1, 0.98])
save_fig(fig, FILENAME)
print('Done \u2014 Fig 3.1G')

## Fig 3.1H — Directional long-gene bias: do long genes go DOWN more than short genes?

In [ ]:
# ── Fig 3.1H ─────────────────────────────────────────────────────
# Directional long-gene bias: do long genes go DOWN more than short genes?
# Tests whether the transcription stress hypothesis (polymerase stalling
# → reduced transcription of long genes) holds genome-wide.
#
# Requires: gene_len dict from Fig 3.1G cell (run that cell first).
# Also requires: k_lfc, k_len, k_padj, m_lfc, m_len, m_padj from Fig 3.1G.
FILENAME = 'Fig_3_1H_long_gene_directional_bias'

# ── 1. Directional bin analysis ──────────────────────────────────
def directional_bin_analysis(lengths, lfcs, padjs, bins_kb=[0, 25, 50, 100, 200, 500, 1000, 5000]):
    """For each length bin, compute fraction of sig DEGs that are DOWN."""
    results = []
    for i in range(len(bins_kb) - 1):
        lo, hi = bins_kb[i], bins_kb[i+1]
        mask = (lengths >= lo) & (lengths < hi)
        sig_mask = mask & (padjs < 0.05)
        n_sig = int(sig_mask.sum())
        if n_sig < 5:
            continue
        sig_lfc = lfcs[sig_mask]
        n_down = int((sig_lfc < 0).sum())
        n_up = int((sig_lfc > 0).sum())
        results.append({
            'bin_label': f'{lo}-{hi}' if hi < 5000 else f'{lo}+',
            'lo': lo, 'hi': hi,
            'n_total': int(mask.sum()),
            'n_sig': n_sig,
            'n_down': n_down,
            'n_up': n_up,
            'frac_down': n_down / n_sig if n_sig > 0 else 0.5,
            'mean_lfc': float(np.mean(sig_lfc)),
        })
    return results

bins_kb = [0, 25, 50, 100, 200, 500, 1000, 5000]
k_dir = directional_bin_analysis(k_len, k_lfc, k_padj, bins_kb)
m_dir = directional_bin_analysis(m_len, m_lfc, m_padj, bins_kb)

# Print summary
print("=== Kidney: fraction of sig DEGs that are DOWNREGULATED ===")
for b in k_dir:
    print(f"  {b['bin_label']:>10s} kb:  {b['frac_down']:.1%}  ({b['n_down']} down / {b['n_up']} up of {b['n_sig']} sig)")
print("\n=== Muscle: fraction of sig DEGs that are DOWNREGULATED ===")
for b in m_dir:
    print(f"  {b['bin_label']:>10s} kb:  {b['frac_down']:.1%}  ({b['n_down']} down / {b['n_up']} up of {b['n_sig']} sig)")

# Spearman correlation: gene length vs LFC (signed, not absolute)
# Negative correlation = long genes tend to go DOWN
k_sig_mask = k_padj < 0.05
m_sig_mask = m_padj < 0.05
k_dir_corr, k_dir_p = sp_stats.spearmanr(k_len[k_sig_mask], k_lfc[k_sig_mask])
m_dir_corr, m_dir_p = sp_stats.spearmanr(m_len[m_sig_mask], m_lfc[m_sig_mask])
print(f"\nKidney sig DEGs: Spearman rho(length, signed LFC) = {k_dir_corr:.3f}, p = {k_dir_p:.2e}")
print(f"Muscle sig DEGs: Spearman rho(length, signed LFC) = {m_dir_corr:.3f}, p = {m_dir_p:.2e}")
print("(Negative rho = long genes tend to be downregulated)")

# ── 2. Plot ──────────────────────────────────────────────────────
fig, (ax_k, ax_m) = plt.subplots(1, 2, figsize=(13, 5.5))

def plot_directional(ax, bins, tissue, corr, pval, color_up, color_down):
    labels = [b['bin_label'] for b in bins]
    x = np.arange(len(bins))
    frac_down = [b['frac_down'] for b in bins]
    frac_up = [1 - b['frac_down'] for b in bins]
    n_sig = [b['n_sig'] for b in bins]

    # Stacked bars: down (bottom) + up (top)
    bars_down = ax.bar(x, frac_down, color=C_BLUE_D, alpha=0.6,
                        edgecolor=C_BLUE_D, linewidth=0.5, label='Downregulated')
    bars_up = ax.bar(x, frac_up, bottom=frac_down, color=C_RED_D, alpha=0.6,
                      edgecolor=C_RED_D, linewidth=0.5, label='Upregulated')

    # 50% reference line
    ax.axhline(0.5, color=C_GRAY_D, lw=0.8, ls='--', zorder=0)
    ax.text(len(bins) - 0.5, 0.51, '50%', fontsize=6.5, color=C_GRAY_D, ha='right')

    # Annotate n per bin
    for i, (fd, n) in enumerate(zip(frac_down, n_sig)):
        ax.text(i, 0.02, f'n={n:,}', ha='center', va='bottom',
                fontsize=5.5, color='white', fontweight='bold')
        # Percentage label
        ax.text(i, fd - 0.03, f'{fd:.0%}', ha='center', va='top',
                fontsize=7, color='white', fontweight='bold')

    ax.set_ylabel('Fraction of significant DEGs', fontsize=9.5, fontweight='bold')
    ax.set_xlabel('Gene body length (kb)', fontsize=9.5, fontweight='bold')
    ax.set_title(tissue, fontsize=11, fontweight='bold', color=C_TEXT, pad=10)
    ax.set_xticks(x)
    ax.set_xticklabels(labels, fontsize=7.5, rotation=30, ha='right')
    ax.set_ylim(0, 1.05)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_linewidth(0.5)
    ax.spines['bottom'].set_linewidth(0.5)

    # Stats
    direction = "long genes tend DOWN" if corr < 0 else "no down-bias"
    ax.text(0.97, 0.97,
            f'Spearman \u03c1 = {corr:.3f}\np = {pval:.1e}\n({direction})',
            transform=ax.transAxes, ha='right', va='top', fontsize=7,
            bbox=dict(boxstyle='round,pad=0.3', fc=C_WHITE, ec='#E0E0E0', lw=0.5))

    ax.legend(loc='upper left', fontsize=7, frameon=True, framealpha=0.9,
              edgecolor='#E0E0E0')

plot_directional(ax_k, k_dir, 'Kidney', k_dir_corr, k_dir_p, C_RED_D, C_BLUE_D)
plot_directional(ax_m, m_dir, 'Muscle', m_dir_corr, m_dir_p, C_RED_D, C_BLUE_D)

fig.suptitle('Fig. 3.1H \u2014 Do long genes preferentially go DOWN with aging?',
             fontsize=11, fontweight='bold', color=C_TEXT, y=1.02)

fig.text(0.5, -0.02,
         'Stacked bars show the fraction of significant aging DEGs (padj < 0.05) that are down- vs upregulated per length bin.\n'
         'If transcription stress from DNA damage preferentially blocks long-gene transcription, the blue fraction should\n'
         'increase with gene length (negative Spearman \u03c1 between length and signed LFC among sig DEGs).',
         ha='center', fontsize=6.5, color=C_TEXT2, linespacing=1.4)

plt.tight_layout(rect=[0, 0.05, 1, 0.98])
save_fig(fig, FILENAME)
print('Done \u2014 Fig 3.1H')

## Fig 3.1I — Composition shifts vs transcription stress

In [ ]:
from collections import Counter

FILENAME = 'Fig_3_1I_composition_vs_transcription_stress'

# ── Load data ────────────────────────────────────────────────────
def load_degs(filepath):
    degs = {}
    with open(filepath) as f:
        for r in csv.DictReader(f):
            if r.get('padj','NA') not in ('NA','') and float(r['padj']) < 0.05:
                degs[r['Symbol']] = float(r['log2FoldChange'])
    return degs

def load_markers(filepath):
    markers = defaultdict(set)  # gene -> set of cell types
    all_genes = set()
    with open(filepath) as f:
        for r in csv.DictReader(f):
            markers[r['gene']].add(r['cluster'])
            all_genes.add(r['gene'])
    return markers, all_genes

k_degs = load_degs(f"{DATA_BASE}/Kidney/1_DEGs_tables/kidney_res_age_results_with_symbols.csv")
m_degs = load_degs(f"{DATA_BASE}/Muscle_old/1_DEGs_tables/muscle_res_age_results_with_symbols.csv")
k_markers, k_marker_genes = load_markers(f"{DATA_BASE}/Kidney/4_Aged-Rescued_genes_celltype_breakdown/kidney_markers_aged_genes_celltypes.csv")
m_markers, m_marker_genes = load_markers(f"{DATA_BASE}/Muscle_old/4_Aged-Rescued_genes_celltype_breakdown/muscle_markers_aged_genes_celltypes.csv")
# ── Per-cell-type direction breakdown ────────────────────────────
def celltype_direction(markers_dict, degs):
    """For each cell type, count how many of its marker DEGs go up vs down."""
    ct_up = Counter()
    ct_down = Counter()
    for gene, celltypes in markers_dict.items():
        if gene in degs:
            for ct in celltypes:
                if degs[gene] > 0:
                    ct_up[ct] += 1
                else:
                    ct_down[ct] += 1
    # All cell types
    all_cts = sorted(set(ct_up.keys()) | set(ct_down.keys()),
                      key=lambda ct: ct_up[ct] + ct_down[ct], reverse=True)
    return all_cts, ct_up, ct_down

k_cts, k_ct_up, k_ct_down = celltype_direction(k_markers, k_degs)
m_cts, m_ct_up, m_ct_down = celltype_direction(m_markers, m_degs)

# ── Figure: 3 panels ─────────────────────────────────────────────
fig = plt.figure(figsize=(15, 9))
gs = fig.add_gridspec(2, 2, height_ratios=[1, 1.2], hspace=0.35, wspace=0.5)

# ── Top left: Marker vs non-marker summary (both tissues) ────────
ax_sum = fig.add_subplot(gs[0, 0])

categories = ['Kidney\nmarkers', 'Kidney\nnon-markers', 'Muscle\nmarkers', 'Muscle\nnon-markers']

k_m_degs = set(k_degs.keys()) & k_marker_genes
k_nm_degs = set(k_degs.keys()) - k_marker_genes
m_m_degs = set(m_degs.keys()) & m_marker_genes
m_nm_degs = set(m_degs.keys()) - m_marker_genes

frac_up = [
    sum(1 for g in k_m_degs if k_degs[g] > 0) / len(k_m_degs),
    sum(1 for g in k_nm_degs if k_degs[g] > 0) / len(k_nm_degs),
    sum(1 for g in m_m_degs if m_degs[g] > 0) / len(m_m_degs),
    sum(1 for g in m_nm_degs if m_degs[g] > 0) / len(m_nm_degs),
]
frac_down = [1 - f for f in frac_up]
n_genes = [len(k_m_degs), len(k_nm_degs), len(m_m_degs), len(m_nm_degs)]

x = np.arange(len(categories))
ax_sum.bar(x, frac_up, color=C_RED_D, alpha=0.6, edgecolor=C_RED_D, linewidth=0.5, label='Upregulated')
ax_sum.bar(x, frac_down, bottom=frac_up, color=C_BLUE_D, alpha=0.6, edgecolor=C_BLUE_D, linewidth=0.5, label='Downregulated')
ax_sum.axhline(0.5, color=C_GRAY_D, lw=0.8, ls='--')

for i in range(len(categories)):
    ax_sum.text(i, frac_up[i] / 2, f'{frac_up[i]:.0%}\nup', ha='center', va='center',
                fontsize=7, color='white', fontweight='bold')
    ax_sum.text(i, frac_up[i] + frac_down[i] / 2, f'{frac_down[i]:.0%}\ndown', ha='center', va='center',
                fontsize=7, color='white', fontweight='bold')
    ax_sum.text(i, 1.02, f'n={n_genes[i]:,}', ha='center', va='bottom', fontsize=6, color=C_TEXT2)

ax_sum.set_xticks(x)
ax_sum.set_xticklabels(categories, fontsize=8)
ax_sum.set_ylabel('Fraction of sig DEGs', fontsize=9, fontweight='bold')
ax_sum.set_title('Cell-type markers vs non-markers:\ndirectional bias', fontsize=10, fontweight='bold', color=C_TEXT, pad=10)
ax_sum.set_ylim(0, 1.1)
ax_sum.spines['top'].set_visible(False)
ax_sum.spines['right'].set_visible(False)
ax_sum.legend(loc='upper right', fontsize=7, frameon=True, framealpha=0.9, edgecolor='#E0E0E0')

# ── Top right: Interpretation schematic ──────────────────────────
ax_txt = fig.add_subplot(gs[0, 1])
ax_txt.axis('off')

ax_txt.set_title('Explanations', fontsize=10, fontweight='bold', color=C_TEXT, pad=10)

txt_y = 0.88
notes = [
    (C_PURP_D, 'Cell-type markers are heavily biased UPWARD (63-79%)'),
    (C_PURP_D, '→ Reflects expanding cell populations with aging'),
    ('', ''),
    (C_TEAL_D, 'In muscle: long genes go DOWN (69% in >500 kb bin)'),
    (C_TEAL_D, '→ Opposite to marker bias → cannot be composition'),
    (C_TEAL_D, '→ Genuine transcription stress signal'),
    ('', ''),
    (C_AMBER_D, 'In kidney: long genes are 50/50 (no directional bias)'),
    (C_AMBER_D, '→ Composition (markers UP) may mask transcription stress'),
    (C_AMBER_D, '→ Two opposing forces might cancel each other out'),
]

for i, (color, txt) in enumerate(notes):
    if txt == '':
        txt_y -= 0.03
        continue
    weight = 'bold' if txt.startswith('→') else 'normal'
    ax_txt.text(0.05, txt_y, txt, fontsize=8, color=color if color else C_TEXT,
                fontweight=weight, transform=ax_txt.transAxes, va='top')
    txt_y -= 0.08

# ── Bottom left: Kidney cell types ───────────────────────────────
ax_k = fig.add_subplot(gs[1, 0])

k_cts_top = k_cts[:15]  # top 15 by total
y_k = np.arange(len(k_cts_top))[::-1]
k_up_vals = [k_ct_up[ct] for ct in k_cts_top]
k_down_vals = [-k_ct_down[ct] for ct in k_cts_top]

ax_k.barh(y_k, k_up_vals, height=0.6, color=C_RED_D, alpha=0.6, edgecolor=C_RED_D, linewidth=0.5, label='Up')
ax_k.barh(y_k, k_down_vals, height=0.6, color=C_BLUE_D, alpha=0.6, edgecolor=C_BLUE_D, linewidth=0.5, label='Down')

for i, ct in enumerate(k_cts_top):
    total = k_ct_up[ct] + k_ct_down[ct]
    pct_up = k_ct_up[ct] / total * 100 if total > 0 else 0
    ax_k.text(max(k_up_vals) + 5, y_k[i], f'{pct_up:.0f}% up', va='center', ha='left',
              fontsize=6, color=C_RED_D)

ax_k.axvline(0, color=C_GRAY_D, lw=0.5)
ax_k.set_yticks(y_k)
ax_k.set_yticklabels([clean_name(ct) for ct in k_cts_top], fontsize=7.5)
ax_k.set_xlabel('Number of marker DEGs', fontsize=9, fontweight='bold')
ax_k.set_title('Kidney — aging marker DEGs per cell type', fontsize=10, fontweight='bold', color=C_TEXT, pad=8)
ax_k.spines['top'].set_visible(False)
ax_k.spines['right'].set_visible(False)
ax_k.tick_params(axis='y', length=0)
ax_k.set_xticklabels([str(abs(int(t))) for t in ax_k.get_xticks()])
ax_k.legend(loc='lower right', fontsize=7, frameon=True, framealpha=0.9, edgecolor='#E0E0E0')

# ── Bottom right: Muscle cell types ──────────────────────────────
ax_m = fig.add_subplot(gs[1, 1])

y_m = np.arange(len(m_cts))[::-1]
m_up_vals = [m_ct_up[ct] for ct in m_cts]
m_down_vals = [-m_ct_down[ct] for ct in m_cts]

ax_m.barh(y_m, m_up_vals, height=0.6, color=C_RED_D, alpha=0.6, edgecolor=C_RED_D, linewidth=0.5, label='Up')
ax_m.barh(y_m, m_down_vals, height=0.6, color=C_BLUE_D, alpha=0.6, edgecolor=C_BLUE_D, linewidth=0.5, label='Down')

for i, ct in enumerate(m_cts):
    total = m_ct_up[ct] + m_ct_down[ct]
    pct_up = m_ct_up[ct] / total * 100 if total > 0 else 0
    ax_m.text(max(m_up_vals) + 5, y_m[i], f'{pct_up:.0f}% up', va='center', ha='left',
              fontsize=6, color=C_RED_D)

ax_m.axvline(0, color=C_GRAY_D, lw=0.5)
ax_m.set_yticks(y_m)
ax_m.set_yticklabels([clean_name(ct) for ct in m_cts], fontsize=7)
ax_m.set_xlabel('Number of marker DEGs', fontsize=9, fontweight='bold')
ax_m.set_title('Muscle — aging marker DEGs per cell type', fontsize=10, fontweight='bold', color=C_TEXT, pad=8)
ax_m.spines['top'].set_visible(False)
ax_m.spines['right'].set_visible(False)
ax_m.tick_params(axis='y', length=0)
ax_m.set_xticklabels([str(abs(int(t))) for t in ax_m.get_xticks()])
ax_m.legend(loc='lower right', fontsize=7, frameon=True, framealpha=0.9, edgecolor='#E0E0E0')

# ── Suptitle ─────────────────────────────────────────────────────
fig.suptitle('Fig. 3.1I — Cell-type composition shifts vs transcription stress in aging DEGs',
             fontsize=11, fontweight='bold', color=C_TEXT, y=1.01)

fig.text(0.5, -0.01,
         'Top left: fraction of sig DEGs that are up/downregulated, split by cell-type marker status.\n'
         'Bottom: per-cell-type breakdown of aging marker DEG direction. Markers are biased upward (expanding populations),\n'
         'while the long-gene downward bias in muscle (Fig 3.1H) runs opposite — confirming transcription stress.',
         ha='center', fontsize=6.5, color=C_TEXT2, linespacing=1.4)

plt.tight_layout(rect=[0, 0.04, 1, 0.98])
save_fig(fig, FILENAME)
print('Done — Fig 3.1I')

## Fig 3.1J ─ Cell-type composition: ctrl vs age for both tissues

In [ ]:
# ── Fig 3.1J ─────────────────────────────────────────────────────
# Cell-type composition: ctrl vs age for both tissues
FILENAME = 'Fig_3_1J_celltype_composition_ctrl_vs_age'

from collections import Counter
import matplotlib.patches as mpatches

KIDNEY_COMP_CSV = f"{DATA_BASE}/Kidney/kidney_celltype_condition.csv"
MUSCLE_COMP_CSV = f"{DATA_BASE}/Muscle_old/muscle_celltype_condition.csv"

def load_composition(filepath):
    data = {}
    with open(filepath) as f:
        for r in csv.DictReader(f):
            cond = r['condition']
            ct = r['celltype']
            if cond not in data:
                data[cond] = Counter()
            data[cond][ct] += 1
    return data

k_comp = load_composition(KIDNEY_COMP_CSV)
m_comp = load_composition(MUSCLE_COMP_CSV)

def get_proportions(comp, conditions=['ctrl', 'age']):
    all_cts = set()
    for cond in conditions:
        if cond in comp:
            all_cts.update(comp[cond].keys())
    ct_avg = {ct: sum(comp.get(c, Counter()).get(ct, 0) for c in conditions) for ct in all_cts}
    cts_sorted = sorted(ct_avg, key=ct_avg.get, reverse=True)
    props = {}
    for cond in conditions:
        total = sum(comp[cond].values()) if cond in comp else 1
        props[cond] = {ct: comp.get(cond, Counter()).get(ct, 0) / total * 100 for ct in cts_sorted}
    return cts_sorted, props

k_cts, k_props = get_proportions(k_comp)
m_cts, m_props = get_proportions(m_comp)

k_deltas = {ct: k_props['age'].get(ct, 0) - k_props['ctrl'].get(ct, 0) for ct in k_cts}
m_deltas = {ct: m_props['age'].get(ct, 0) - m_props['ctrl'].get(ct, 0) for ct in m_cts}

k_total_shift = sum(abs(d) for d in k_deltas.values()) / 2
m_total_shift = sum(abs(d) for d in m_deltas.values()) / 2
print(f"Kidney total composition shift: {k_total_shift:.1f} pp")
print(f"Muscle total composition shift: {m_total_shift:.1f} pp")

# palette = ['#E8918D', '#E8A86B', '#D4B56A', '#A3BE8C', '#7DB89E',
#            '#6BBFB0', '#5EAAA8', '#6BA3C7', '#7E94C0', '#9B8EC2',
#            '#B68CB5', '#C98BA3', '#D4A19A', '#BDB5A0', '#A0C4A8',
#            '#D4C5A0', '#8FB8A0', '#A89BC2', '#C4A882', '#9CB8D4',
#            '#B0A8C0', '#C8B090', '#88C0B0']

palette = [
    '#E8918D',  # reddish
    '#E8A86B',  # orange
    '#D4B56A',  # yellowish
    '#A3BE8C',  # green
    '#7DB89E',  # teal
    '#6BBFB0',  # cyan-teal
    '#5EAAA8',  # muted teal
    '#6BA3C7',  # sky blue
    '#7E94C0',  # blue
    '#9B8EC2',  # purple
    '#B68CB5',  # magenta
    '#C98BA3',  # pink
    '#D4A19A',  # peach
    '#BDB5A0',  # beige
    '#A0C4A8',  # mint
    '#D4C5A0',  # light mustard
    '#8FB8A0',  # soft green
    '#A89BC2',  # lavender
    '#C4A882',  # light brown
    '#9CB8D4',  # light blue
    '#B0A8C0',  # dusty purple
    '#C8B090',  # tan
    '#88C0B0',  # muted teal
]

fig, axes = plt.subplots(
    2, 3,
    figsize=(16, 10),
    gridspec_kw={'width_ratios': [1, 1, 1.2], 'hspace': 0.45, 'wspace': 0.6}
)

def plot_stacked_bars(ax, cts, props, title, comp, colors):
    n_ctrl = sum(comp.get('ctrl', Counter()).values())
    n_age = sum(comp.get('age', Counter()).values())
    bottom_ctrl, bottom_age = 0, 0
    for i, ct in enumerate(cts):
        c = colors[i % len(colors)]
        ax.bar(0, props['ctrl'][ct], bottom=bottom_ctrl, width=0.35,
               color=c, edgecolor='white', linewidth=0.3)
        ax.bar(1, props['age'][ct], bottom=bottom_age, width=0.35,
               color=c, edgecolor='white', linewidth=0.3)
        if props['ctrl'][ct] > 4:
            ax.text(0, bottom_ctrl + props['ctrl'][ct]/2, f'{props["ctrl"][ct]:.1f}%',
                    ha='center', va='center', fontsize=5, color='white', fontweight='bold')
        if props['age'][ct] > 4:
            ax.text(1, bottom_age + props['age'][ct]/2, f'{props["age"][ct]:.1f}%',
                    ha='center', va='center', fontsize=5, color='white', fontweight='bold')
        bottom_ctrl += props['ctrl'][ct]
        bottom_age += props['age'][ct]

    ax.set_xticks([0, 1])
    ax.set_xticklabels(['Ctrl', 'Age'], fontsize=10, fontweight='bold')
    ax.set_ylabel('% of nuclei', fontsize=9, fontweight='bold')
    ax.set_title(title, fontsize=11, fontweight='bold', color=C_TEXT, pad=10)
    ax.set_ylim(0, 105)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    # Annotate n= below axis
    ax.annotate(f'n={n_ctrl:,}', (0, 0), xytext=(0, -25),
                textcoords='offset points', ha='center', fontsize=7, color=C_TEXT2)
    ax.annotate(f'n={n_age:,}', (1, 0), xytext=(0, -25),
                textcoords='offset points', ha='center', fontsize=7, color=C_TEXT2)

def plot_delta(ax, cts, deltas, title, colors, total_shift):
    cts_by_delta = sorted(cts, key=lambda ct: abs(deltas[ct]), reverse=True)[:15]
    y = np.arange(len(cts_by_delta))[::-1]
    vals = [deltas[ct] for ct in cts_by_delta]
    ct_colors = [colors[cts.index(ct) % len(colors)] for ct in cts_by_delta]
    for i, (v, c) in enumerate(zip(vals, ct_colors)):
        ax.barh(y[i], v, height=0.55, color=c, edgecolor=c, linewidth=0.5, alpha=0.8)
    ax.axvline(0, color=C_GRAY_D, lw=0.5)
    ax.set_yticks(y)
    ax.set_yticklabels([clean_name(ct) for ct in cts_by_delta], fontsize=8)
    ax.set_xlabel('\u0394 percentage points\n(age \u2212 ctrl)', fontsize=8, fontweight='bold')
    ax.set_title(title, fontsize=10, fontweight='bold', color=C_TEXT, pad=12)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.tick_params(axis='y', length=0)
    ax.text(0.97, 0.03, f'Total shift:\n{total_shift:.1f} pp',
            transform=ax.transAxes, ha='right', va='bottom',
            fontsize=8, fontweight='bold', color=C_TEXT2,
            bbox=dict(boxstyle='round,pad=0.3', fc=C_WHITE, ec='#E0E0E0', lw=0.5))

def make_legend(ax, cts, colors, title):
    ax.axis('off')
    ax.set_title(title, fontsize=9, fontweight='bold', color=C_TEXT, pad=5)
    step = 1 / (len(cts) + 2)  # more vertical spacing
    for i, ct in enumerate(cts):
        y = 1 - (i + 1) * step
        c = colors[i % len(colors)]
        ax.add_patch(mpatches.Rectangle((0, y - 0.015), 0.08, 0.03,
                                        facecolor=c, edgecolor='none', transform=ax.transAxes))
        ax.text(0.12, y, clean_name(ct), fontsize=6, va='center', transform=ax.transAxes, color=C_TEXT)

# ── Plot panels ───────────────────────────────────────────────────
plot_stacked_bars(axes[0, 0], k_cts, k_props, 'Kidney \u2014 composition', k_comp, palette)
plot_delta(axes[0, 1], k_cts, k_deltas, 'Kidney \u2014 composition change', palette, k_total_shift)
make_legend(axes[0, 2], k_cts, palette, 'Kidney cell types')

plot_stacked_bars(axes[1, 0], m_cts, m_props, 'Muscle \u2014 composition', m_comp, palette)
plot_delta(axes[1, 1], m_cts, m_deltas, 'Muscle \u2014 composition change', palette, m_total_shift)
make_legend(axes[1, 2], m_cts, palette, 'Muscle cell types')

fig.suptitle('Fig. 3.1J \u2014 Cell-type composition shifts with aging: kidney vs muscle',
             fontsize=11, fontweight='bold', color=C_TEXT, y=1.01)

fig.text(0.5, -0.01,
         'Left: proportion of nuclei per cell type in ctrl vs age. Center: change in percentage points (age \u2212 ctrl).\n'
         'Total shift = sum of absolute gains across expanding cell types. Higher = more composition change.\n'
         'snRNA-seq: n=1 per condition (exploratory). Composition changes can confound bulk DEG directionality.',
         ha='center', fontsize=6.5, color=C_TEXT2, linespacing=1.4)

plt.tight_layout(rect=[0, 0.04, 1, 0.98])
save_fig(fig, FILENAME)
print('Done \u2014 Fig 3.1J')

## Supplementary Fig S3.1 — Per-cell-type GOBP aging enrichment

---



---



In [ ]:
# ── Supplementary Fig S3.1 — Per-cell-type GOBP aging enrichment ─
# All cell types, both tissues
import textwrap
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
import matplotlib.colors as mcolors

K_AGED = f"{DATA_BASE}/Kidney/2_Enrichment_bulk&sn/GOBP_per_celltype/aged"
M_AGED = f"{DATA_BASE}/Muscle_old/2_Enrichment_tables/GOBP_per_celltype/aged"

def load_enrich_aged(filepath, n=8):
    terms = []
    seen_words = []
    with open(filepath) as f:
        for r in csv.DictReader(f):
            desc = r['Description']
            words = set(desc.lower().split())
            overlap = any(len(words & s) / max(len(words), 1) > 0.6 for s in seen_words)
            if overlap and len(terms) > 2:
                continue
            num, denom = r['GeneRatio'].split('/')
            terms.append({
                'desc': desc,
                'count': int(num),
                'gene_ratio': int(num) / int(denom),
                'padj': float(r['p.adjust']),
                'nlog10p': -np.log10(max(float(r['p.adjust']), 1e-310)),
            })
            seen_words.append(words)
            if len(terms) >= n:
                break
    return terms

def plot_aged_dotplot(ax, terms, ct_label):
    if not terms:
        ax.text(0.5, 0.5, 'No significant\nterms', ha='center', va='center',
                transform=ax.transAxes, fontsize=8, color=C_TEXT2)
        ax.set_title(clean_name(ct_label), fontsize=9, fontweight='bold', color=C_TEXT)
        ax.axis('off')
        return

    n = len(terms)
    y_pos = np.arange(n)[::-1]
    counts = [t['count'] for t in terms]
    nlog10p = [t['nlog10p'] for t in terms]
    gene_ratio = [t['gene_ratio'] for t in terms]
    descs = [textwrap.fill(t['desc'], width=28) if len(t['desc']) > 30 else t['desc'] for t in terms]

    cmap = mcolors.LinearSegmentedColormap.from_list('w2red',
           ['#FFFFFF', '#F5E0D0', '#E8A86B', '#A32D2D', '#6B1A1A'])

    min_c, max_c = min(counts), max(counts)
    sizes = [30 + (c - min_c) / max(max_c - min_c, 1) * 140 for c in counts]

    scatter = ax.scatter(gene_ratio, y_pos, s=sizes, c=nlog10p,
                          cmap=cmap, vmin=min(nlog10p)*0.8, vmax=max(nlog10p)*1.05,
                          edgecolors=C_RED_D, linewidths=0.3, zorder=3, alpha=0.85)

    ax.set_yticks(y_pos)
    ax.set_yticklabels(descs, fontsize=9)
    ax.set_xlabel('Gene ratio', fontsize=7, fontweight='bold')
    ax.set_title(clean_name(ct_label),fontsize=9, fontweight='bold', color=C_TEXT, pad=8)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_linewidth(0.5)
    ax.spines['bottom'].set_linewidth(0.5)
    ax.tick_params(axis='y', length=0)
    ax.grid(axis='x', alpha=0.15, lw=0.5)

    ax.text(0.98, 0.02, f'{len(terms)} terms',
            transform=ax.transAxes, ha='right', va='bottom',
            fontsize=5, color=C_TEXT2, fontstyle='italic')

def build_supp_aged_figure(celltypes_files, tissue, filename, cols=4):
    n = len(celltypes_files)
    rows = int(np.ceil(n / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 4.5, rows * 3.5))
    axes = axes.flatten() if n > 1 else [axes]

    for idx, (ct_label, filepath) in enumerate(celltypes_files):
        terms = load_enrich_aged(filepath, n=8)
        plot_aged_dotplot(axes[idx], terms, clean_name(ct_label))

    # Hide empty axes
    for idx in range(n, len(axes)):
        axes[idx].axis('off')

    fig.suptitle(f'Supplementary — {tissue}: GOBP enrichment of aging DEGs per cell type',
                 fontsize=12, fontweight='bold', color=C_TEXT, y=1.01)
    fig.text(0.5, -0.01,
             'Top 8 GOBP terms per cell type (enrichGO on aging DEGs overlapping cell-type markers).\n'
             'Near-duplicate terms removed. Dot size = gene count, colour = −log10(padj).\n'
             f'YlOrRd colourmap (aging). snRNA-seq marker overlap: exploratory (n=1/condition).',
             ha='center', fontsize=7, color=C_TEXT2, linespacing=1.4)

    plt.tight_layout(rect=[0, 0.03, 1, 0.98])
    save_fig(fig, filename)
    print(f'Done — {filename}')

# ── Kidney aged: all 22 cell types ───────────────────────────────
import os
k_aged_files = []
for subdir in ['ECs', 'IMM', 'Podo', 'Rest']:
    dirpath = f'{K_AGED}/{subdir}'
    if os.path.isdir(dirpath):
        for fname in sorted(os.listdir(dirpath)):
            if fname.endswith('.csv'):
                ct = fname.replace('gobp_aged_', '').replace('.csv', '')
                k_aged_files.append((ct, f'{dirpath}/{fname}'))

print(f"Kidney: {len(k_aged_files)} cell types with aged GOBP")
build_supp_aged_figure(k_aged_files, 'Kidney',
                        'Supp_Fig_S3_1a_kidney_aged_GOBP_per_celltype', cols=4)

# ── Muscle aged: all 13 cell types ───────────────────────────────
m_aged_files = []
for subdir in ['ECs', 'IMM', 'NMJs', 'Rest']:
    dirpath = f'{M_AGED}/{subdir}'
    if os.path.isdir(dirpath):
        for fname in sorted(os.listdir(dirpath)):
            if fname.endswith('.csv'):
                ct = fname.replace('gobp_aged_', '').replace('.csv', '')
                # Shorten long muscle names
                ct_short = ct.replace('Myonuclei (neuromuscular junction nuclei)', 'NMJ nuclei') \
                             .replace('Myonuclei (myotendinous junction nuclei)', 'MTJ nuclei') \
                             .replace('Myonuclei (differentiating myocytes_)', 'Diff. myocytes') \
                             .replace('Myonuclei (Myh4+_)', 'Myh4+ myonuclei') \
                             .replace('Myonuclei (Myh1+)', 'Myh1+ myonuclei') \
                             .replace('Muscle stem cell', 'Muscle stem cells') \
                             .replace('Immune cells_', 'Immune cells')
                m_aged_files.append((ct_short, f'{dirpath}/{fname}'))

print(f"Muscle: {len(m_aged_files)} cell types with aged GOBP")
build_supp_aged_figure(m_aged_files, 'Muscle',
                        'Supp_Fig_S3_1b_muscle_aged_GOBP_per_celltype', cols=4)

#3.2 Transcriptional Rescue by the Combined Intervention


## Fig 3.2A — Rescue framework schematic

In [ ]:
FILENAME = 'Fig_3_2A_rescue_framework'

def rounded_box(ax, x, y, w, h, fc, ec, label1, label2=None,
                fontsize1=9, fontsize2=7.5, bold1=True):
    box = FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.02",
                          facecolor=fc, edgecolor=ec, linewidth=0.8,
                          transform=ax.transAxes, zorder=2)
    ax.add_patch(box)
    if label2:
        ax.text(x+w/2, y+h*0.62, label1, transform=ax.transAxes, ha='center', va='center',
                fontsize=fontsize1, fontweight='bold' if bold1 else 'normal', color=C_TEXT, zorder=3)
        ax.text(x+w/2, y+h*0.28, label2, transform=ax.transAxes, ha='center', va='center',
                fontsize=fontsize2, color=C_TEXT2, zorder=3)
    else:
        ax.text(x+w/2, y+h/2, label1, transform=ax.transAxes, ha='center', va='center',
                fontsize=fontsize1, fontweight='bold' if bold1 else 'normal', color=C_TEXT, zorder=3)

def arrow_down(ax, x, y1, y2, text=None, text_x_offset=0.01):
    ax.annotate('', xy=(x, y2), xytext=(x, y1), xycoords='axes fraction', textcoords='axes fraction',
                arrowprops=dict(arrowstyle='->', color=C_TEXT2, lw=1.0), zorder=1)
    if text:
        ax.text(x + text_x_offset, (y1+y2)/2, text, transform=ax.transAxes,
                ha='left', va='center', fontsize=7, color=C_TEXT2, fontstyle='italic', zorder=3)

def arrow_right(ax, x1, y, x2):
    ax.annotate('', xy=(x2, y), xytext=(x1, y), xycoords='axes fraction', textcoords='axes fraction',
                arrowprops=dict(arrowstyle='->', color=C_TEXT2, lw=0.8), zorder=1)

fig, ax = plt.subplots(1, 1, figsize=(8.5, 11))
ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.axis('off')

# ── Title and subtitle ─────────────────────────────
delta_text = -0.01  # general downward shift for main text
delta_sign_text = -0.02  # extra downward shift for sign(LFC_aging) text

ax.text(0.50, 0.96, 'Fig. 3.2A', transform=ax.transAxes, ha='center', va='center',
        fontsize=11, fontweight='bold', color=C_TEXT)
ax.text(0.50, 0.95 + delta_text, 'Direction-aware rescue framework: three-stage classification of aging DEGs',
        transform=ax.transAxes, ha='center', va='center', fontsize=8.5, color=C_TEXT2)

# ── Level shifts ─────────────────────────────
delta_lvl0 = -0.02   # Aging DEGs
delta_lvl1 = -0.04   # Stage 1 boxes moved down
delta_lvl3 = -0.06   # Stage 2 boxes moved down

# ── Stage 0: Aging DEGs box ─────────────────────────
bw, bh = 0.52, 0.045
rounded_box(ax, 0.24, 0.885 + delta_lvl0, bw, bh, C_RED, C_RED_D,
            'Aging DEGs (padj < 0.05)', 'Kidney: 7,499  \u00b7  Muscle: 1,739')
arrow_down(ax, 0.50, 0.885 + delta_lvl0, 0.835 + delta_lvl0,
           'sign(LFC_aging) \u2260 sign(LFC_combi)?', text_x_offset=0.015)

# ── Stage 1 boxes ─────────────────────────────
rounded_box(ax, 0.12, 0.80 + delta_lvl1, 0.42, 0.045, C_BLUE, C_BLUE_D,
            'Stage 1 \u2014 Rescued (sign reversal by combi)',
            'K: 4,645 (62% of aging DEGs)  \u00b7  M: 926 (53%)')
rounded_box(ax, 0.58, 0.80 + delta_lvl1, 0.30, 0.045, C_GRAY, C_GRAY_D,
            'Not reversed by combi', 'K: 2,854 (38%)  \u00b7  M: 813 (47%)')
arrow_right(ax, 0.54, 0.8225 + delta_lvl1, 0.58)
arrow_down(ax, 0.33, 0.80 + delta_lvl1, 0.755 + delta_lvl1 + delta_text,
           '|LFC combi vs ctrl| < 0.5?')

# ── Stage 2 boxes ─────────────────────────────
rounded_box(ax, 0.08, 0.71 + delta_lvl3, 0.48, 0.045, C_TEAL, C_TEAL_D,
            'Stage 2 \u2014 Rescued-to-normal',
            'K: 1,961 (42% of rescued)  \u00b7  M: 511 (55%)')
rounded_box(ax, 0.60, 0.71 + delta_lvl3, 0.30, 0.045, C_GRAY, C_GRAY_D,
            'Rescued, not normalised', 'K: 2,684 (58%)  \u00b7  M: 415 (45%)')
arrow_right(ax, 0.56, 0.7325 + delta_lvl3, 0.60)
arrow_down(ax, 0.32, 0.71 + delta_lvl3, 0.665 + delta_lvl3 + delta_text,
           'Which intervention drives the reversal?')

# ── Stage 3 boxes ─────────────────────────────
delta_stage3 = -0.065
ax.text(0.50, 0.65 + delta_stage3, 'Stage 3 \u2014 Intervention driver classification',
        transform=ax.transAxes, ha='center', va='center', fontsize=9, fontweight='bold', color=C_TEXT)
ax.text(0.50, 0.636 + delta_stage3, '(within rescued-to-normal set)',
        transform=ax.transAxes, ha='center', va='center', fontsize=7, color=C_TEXT2, fontstyle='italic')
ax.plot([0.13, 0.87], [0.625 + delta_stage3, 0.625 + delta_stage3], color=C_TEXT2, lw=0.6, transform=ax.transAxes, zorder=1)
for xp in [0.13, 0.37, 0.63, 0.87]:
    arrow_down(ax, xp, 0.625 + delta_stage3, 0.60 + delta_stage3)

bw2, bh2, by = 0.21, 0.065, 0.535 + delta_stage3
rounded_box(ax, 0.025, by, bw2, bh2, C_GREEN, C_GREEN_D, 'DR-exclusive',
            'K: 546 (28%)  \u00b7  M: 241 (47%)', fontsize1=8.5)
rounded_box(ax, 0.26,  by, bw2, bh2, C_PURPLE, C_PURP_D, 'sAct-exclusive',
            'K: 108 (6%)  \u00b7  M: 23 (5%)', fontsize1=8.5)
rounded_box(ax, 0.505, by, bw2+0.015, bh2, C_AMBER, C_AMBER_D, 'Combination-only',
            'K: 87 (4%)  \u00b7  M: 18 (4%)', fontsize1=8.5)
rounded_box(ax, 0.755, by, bw2, bh2, C_CORAL, C_CORAL_D, 'Both DR + sAct',
            'K: 1,220 (62%)  \u00b7  M: 229 (45%)', fontsize1=8.5)

ey = 0.505 + delta_stage3
efs = 6.5
for xp, txt in [(0.13, 'Only DR reverses\naging direction'),
                (0.365, 'Only sActRIIB reverses\naging direction'),
                (0.62, 'Neither alone reverses;\nsynergistic rescue'),
                (0.86, 'Both interventions\nindependently reverse')]:
    ax.text(xp, ey, txt, transform=ax.transAxes, ha='center', va='top',
            fontsize=efs, color=C_TEXT2, linespacing=1.3)

ax.plot([0.73, 0.73], [0.80 + delta_stage3, 0.43 + delta_stage3], color=C_GRAY_D, lw=0.6, ls='--', transform=ax.transAxes, zorder=1)
ax.plot([0.73, 0.50], [0.43 + delta_stage3, 0.43 + delta_stage3], color=C_GRAY_D, lw=0.6, ls='--', transform=ax.transAxes, zorder=1)
arrow_down(ax, 0.50, 0.43 + delta_stage3, 0.415 + delta_stage3)

# ── Non-rescued aging DEGs box ─────────────────────────────
nonrescued_y = 0.37 + delta_stage3
rounded_box(ax, 0.22, nonrescued_y, 0.56, 0.045, C_GRAY, C_GRAY_D,
            'Non-rescued aging DEGs',
            'Not reversed by any intervention (combi, DR, or sAct):  K: 1,169 (16%)  \u00b7  M: 300 (17.3%)')

# ── Definitions legend ─────────────────────────────
leg_y = 0.27
ax.text(0.08, leg_y, 'Definitions', transform=ax.transAxes, fontsize=8, fontweight='bold', color=C_TEXT)
for i, (fc, ec, lab) in enumerate([
    (C_RED, C_RED_D, 'Aging DEGs: padj < 0.05 and |log\u2082FC| > 0 in age vs. ctrl (DESeq2)'),
    (C_BLUE, C_BLUE_D, 'Rescued: aging DEG where sign(LFC_aging) \u2260 sign(LFC_combi vs aging)'),
    (C_TEAL, C_TEAL_D, 'Rescued-to-normal: rescued gene with |LFC_combi vs ctrl| < 0.5'),
    (C_GREEN, C_GREEN_D, 'DR-exclusive: Restoration_DR > 0 and Restoration_sAct = 0'),
    (C_PURPLE, C_PURP_D, 'sAct-exclusive: Restoration_sAct > 0 and Restoration_DR = 0'),
    (C_AMBER, C_AMBER_D, 'Combination-only: Restoration_DR = 0 and Restoration_sAct = 0 (synergistic)'),
    (C_CORAL, C_CORAL_D, 'Both DR + sAct: Restoration_DR > 0 and Restoration_sAct > 0 (convergent)'),
]):
    yi = leg_y - 0.020 - i * 0.020
    ax.add_patch(FancyBboxPatch((0.08, yi-0.005), 0.015, 0.013, boxstyle="round,pad=0.001",
                                 facecolor=fc, edgecolor=ec, linewidth=0.5, transform=ax.transAxes, zorder=2))
    ax.text(0.105, yi+0.002, lab, transform=ax.transAxes, ha='left', va='center', fontsize=6.5, color=C_TEXT2)

# ── Footnote ─────────────────────────────
ax.text(0.08, 0.115,
        'K = kidney, M = muscle. Percentages in Stages 1\u20132 are relative to the preceding category;\n'
        'percentages in Stage 3 are relative to the rescued-to-normal set.\n'
        'Pipeline note: kidney processed with STAR + Salmon + DESeq2 (GRCm39); muscle with FeatureCounts (GRCm38).',
        transform=ax.transAxes, ha='left', va='top', fontsize=6, color=C_TEXT2, linespacing=1.5)

plt.tight_layout(pad=0.5)
save_fig(fig, FILENAME)
print('Done \u2014 Fig 3.2A')

## Fig 3.2B — Sankey rescue flow

In [ ]:
# ── Fig 3.2B ─────────────────────────────────────────────────────
FILENAME = 'Fig_3_2B_sankey_rescue'

def draw_band(ax, x_left, x_right, y_top_left, h_left, y_top_right, h_right,
              color, alpha=0.35):
    xm = (x_left + x_right) / 2
    top_verts = [(x_left, y_top_left), (xm, y_top_left), (xm, y_top_right), (x_right, y_top_right)]
    bot_verts = [(x_right, y_top_right+h_right), (xm, y_top_right+h_right),
                 (xm, y_top_left+h_left), (x_left, y_top_left+h_left)]
    verts = top_verts + bot_verts + [(x_left, y_top_left)]
    codes = [Path.MOVETO] + [Path.CURVE4]*3 + [Path.LINETO] + [Path.CURVE4]*3 + [Path.CLOSEPOLY]
    ax.add_patch(mpatches.PathPatch(Path(verts, codes), facecolor=color, alpha=alpha,
                                     edgecolor='none', linewidth=0, zorder=1))

def draw_bar(ax, x, y, w, h, fc, ec):
    ax.add_patch(mpatches.FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.002",
                                          facecolor=fc, edgecolor=ec, linewidth=0.7, zorder=3))

def add_label_inside(ax, x, y, w, h, line1, line2=None, color=C_TEXT, fs1=7.5, fs2=6.5, bold=False):
    cx = x + w / 2
    if line2 and h > 0.05:
        ax.text(cx, y+h/2+0.010, line1, ha='center', va='center', fontsize=fs1, color=color,
                fontweight='bold' if bold else 'normal', zorder=5)
        ax.text(cx, y+h/2-0.012, line2, ha='center', va='center', fontsize=fs2, color=C_TEXT2, zorder=5)
    elif h > 0.025:
        ax.text(cx, y+h/2, line1, ha='center', va='center', fontsize=fs1, color=color,
                fontweight='bold' if bold else 'normal', zorder=5)

def add_label_right(ax, x_right, y, h, line1, count, pct_str, color, fs=6.5):
    cy = y + h / 2
    ax.text(x_right+0.015, cy+0.006, line1, ha='left', va='center', fontsize=fs, color=color,
            fontweight='bold', zorder=5)
    ax.text(x_right+0.015, cy-0.008, f'{count:,} ({pct_str})', ha='left', va='center',
            fontsize=fs-0.5, color=C_TEXT2, zorder=5)

def draw_tissue(ax, title, d):
    bar_w = 0.065
    x = [0.04, 0.26, 0.48, 0.68]
    top_margin, bot_margin = 0.90, 0.05
    usable = top_margin - bot_margin
    gap = 0.018
    total = d['aging']
    scale = usable / total

    h0 = total * scale; y0 = bot_margin
    draw_bar(ax, x[0], y0, bar_w, h0, C_RED, C_RED_D)
    add_label_inside(ax, x[0], y0, bar_w, h0, 'Aging DEGs', f'n = {total:,}', C_RED_D, bold=True)

    h1_r = d['rescued']*scale; h1_n = d['not_reversed']*scale
    y1_r = y0; y1_n = y1_r + h1_r + gap
    draw_bar(ax, x[1], y1_r, bar_w, h1_r, C_BLUE, C_BLUE_D)
    add_label_inside(ax, x[1], y1_r, bar_w, h1_r, 'Rescued', f'n = {d["rescued"]:,}', C_BLUE_D, bold=True)
    draw_bar(ax, x[1], y1_n, bar_w, h1_n, C_GRAY, C_GRAY_D)
    add_label_inside(ax, x[1], y1_n, bar_w, h1_n, 'Not reversed', f'n = {d["not_reversed"]:,}', C_GRAY_D)
    draw_band(ax, x[0]+bar_w, x[1], y0, h1_r, y1_r, h1_r, C_BLUE, 0.25)
    draw_band(ax, x[0]+bar_w, x[1], y0+h1_r, h1_n, y1_n, h1_n, C_GRAY, 0.15)

    h2_rtn = d['rtn']*scale; h2_rnn = d['rescued_not_normal']*scale
    y2_rtn = y1_r; y2_rnn = y2_rtn + h2_rtn + gap
    draw_bar(ax, x[2], y2_rtn, bar_w, h2_rtn, C_TEAL, C_TEAL_D)
    add_label_inside(ax, x[2], y2_rtn, bar_w, h2_rtn, 'Rescued-to-', None, C_TEAL_D, bold=True)
    if h2_rtn > 0.06:
        ax.text(x[2]+bar_w/2, y2_rtn+h2_rtn/2-0.014, 'normal', ha='center', va='center',
                fontsize=7.5, color=C_TEAL_D, fontweight='bold', zorder=5)
        ax.text(x[2]+bar_w/2, y2_rtn+h2_rtn/2-0.032, f'n = {d["rtn"]:,}', ha='center', va='center',
                fontsize=6.5, color=C_TEXT2, zorder=5)
    draw_bar(ax, x[2], y2_rnn, bar_w, h2_rnn, C_GRAY, C_GRAY_D)
    add_label_inside(ax, x[2], y2_rnn, bar_w, h2_rnn, 'Not normalised', f'n = {d["rescued_not_normal"]:,}', C_GRAY_D)
    draw_band(ax, x[1]+bar_w, x[2], y1_r, h2_rtn, y2_rtn, h2_rtn, C_TEAL, 0.25)
    draw_band(ax, x[1]+bar_w, x[2], y1_r+h2_rtn, h2_rnn, y2_rnn, h2_rnn, C_GRAY, 0.15)

    drivers = [('DR-exclusive', d['dr_excl'], C_GREEN, C_GREEN_D),
               ('Both DR+sAct', d['both'], C_CORAL, C_CORAL_D),
               ('sAct-exclusive', d['sact_excl'], C_PURPLE, C_PURP_D),
               ('Combi-only', d['combi_only'], C_AMBER, C_AMBER_D)]
    drivers.sort(key=lambda x: x[1], reverse=True)
    min_bar_h, driver_gap = 0.018, 0.010
    driver_heights = [max(cnt*scale, min_bar_h) for _, cnt, _, _ in drivers]
    y_cursor = y2_rtn; y_src_cursor = y2_rtn; rtn_total = d['rtn']
    for i, (label, count, fc, ec) in enumerate(drivers):
        h_bar = driver_heights[i]
        draw_bar(ax, x[3], y_cursor, bar_w, h_bar, fc, ec)
        pct = f'{count/rtn_total*100:.1f}%'
        add_label_right(ax, x[3]+bar_w, y_cursor, h_bar, label, count, pct, ec, fs=6.5)
        h_src = count * scale
        draw_band(ax, x[2]+bar_w, x[3], y_src_cursor, max(h_src, 0.004), y_cursor, h_bar, fc, 0.28)
        y_src_cursor += h_src; y_cursor += h_bar + driver_gap

    header_y = 0.94
    for i, h in enumerate(['Aging\nsignature', 'Stage 1\nRescue', 'Stage 2\nNormalisation', 'Stage 3\nDrivers']):
        ax.text(x[i]+bar_w/2, header_y, h, ha='center', va='bottom', fontsize=7.5,
                fontweight='bold', color=C_TEXT, linespacing=1.2)
    ax.text(0.50, 0.995, title, ha='center', va='top', fontsize=12, fontweight='bold', color=C_TEXT)

    pct_r = f'{d["rescued"]/total*100:.0f}%'
    ax.text((x[0]+bar_w+x[1])/2, y1_r+h1_r*0.5, pct_r, ha='center', va='center', fontsize=8,
            color=C_BLUE_D, fontweight='bold', zorder=6,
            bbox=dict(boxstyle='round,pad=0.15', fc='white', ec='none', alpha=0.8))
    pct_rtn = f'{d["rtn"]/d["rescued"]*100:.0f}%'
    ax.text((x[1]+bar_w+x[2])/2, y2_rtn+h2_rtn*0.5, pct_rtn, ha='center', va='center', fontsize=8,
            color=C_TEAL_D, fontweight='bold', zorder=6,
            bbox=dict(boxstyle='round,pad=0.15', fc='white', ec='none', alpha=0.8))

kidney_data = dict(aging=7499, rescued=4645, not_reversed=2854, rtn=1961,
                   rescued_not_normal=2684, dr_excl=546, sact_excl=108, combi_only=87, both=1220)
muscle_data = dict(aging=1739, rescued=926, not_reversed=813, rtn=511,
                   rescued_not_normal=415, dr_excl=241, sact_excl=23, combi_only=18, both=229)

fig, (ax_k, ax_m) = plt.subplots(1, 2, figsize=(14, 7.5))
for a in [ax_k, ax_m]:
    a.set_xlim(-0.02, 1.08); a.set_ylim(-0.02, 1.02); a.axis('off')
draw_tissue(ax_k, 'Kidney', kidney_data)
draw_tissue(ax_m, 'Muscle', muscle_data)

fig.suptitle('Fig. 3.2B \u2014 Rescue framework gene flow: aging DEGs to intervention drivers',
             fontsize=11, fontweight='bold', color=C_TEXT, y=0.99)

legend_items = [(C_RED, C_RED_D, 'Aging DEGs'), (C_BLUE, C_BLUE_D, 'Rescued (Stage 1)'),
                (C_TEAL, C_TEAL_D, 'Rescued-to-normal (Stage 2)'), (C_GREEN, C_GREEN_D, 'DR-exclusive'),
                (C_PURPLE, C_PURP_D, 'sAct-exclusive'), (C_AMBER, C_AMBER_D, 'Combi-only'),
                (C_CORAL, C_CORAL_D, 'Both DR+sAct'), (C_GRAY, C_GRAY_D, 'Not rescued / not normalised')]
handles = [mpatches.Patch(facecolor=fc, edgecolor=ec, linewidth=0.5, label=lab) for fc, ec, lab in legend_items]
fig.legend(handles=handles, loc='lower center', ncol=4, fontsize=7, frameon=False,
           bbox_to_anchor=(0.5, -0.01), handlelength=1.2, handleheight=0.8, columnspacing=1.2)

plt.tight_layout(rect=[0, 0.04, 1, 0.97])
save_fig(fig, FILENAME)
print('Done \u2014 Fig 3.2B')

## Fig 3.2C — GOBP rescued-to-normal enrichment,

In [ ]:
FILENAME = 'Fig_3_2C_GOBP_rescued_to_normal_kidney_muscle'

def load_enrich(filepath, n=15):
    with open(filepath) as f:
        rows = list(csv.DictReader(f))
    terms = []
    seen_words = []
    for r in rows:
        desc = r['Description']
        words = set(desc.lower().split())
        overlap = any(len(words & s) / max(len(words), 1) > 0.6 for s in seen_words)
        if overlap and len(terms) > 3:
            continue
        num, denom = r['GeneRatio'].split('/')
        terms.append({
            'desc': desc,
            'count': int(num),
            'gene_ratio': int(num) / int(denom),
            'padj': float(r['p.adjust']),
            'nlog10p': -np.log10(max(float(r['p.adjust']), 1e-310)),
        })
        seen_words.append(words)
        if len(terms) >= n:
            break
    return terms

k_terms = load_enrich(f"{DATA_BASE}/Kidney/2_Enrichment_bulk&sn/GOBP bulk/kidney_enrichment_results_rescued-to-normal.csv", 15)
m_terms = load_enrich(f"{DATA_BASE}/Muscle_old/2_Enrichment_tables/GOBP bulk/muscle_enrichment_results_rescued_to_normal.csv", 15)

print(f"Kidney: {len(k_terms)} terms selected | Muscle: {len(m_terms)} terms selected")

# ── Plot ─────────────────────────────────────────────────────────
fig, (ax_k, ax_m) = plt.subplots(1, 2, figsize=(14, 8),
                                  gridspec_kw={'width_ratios': [1.1, 1]})

def plot_dotplot(ax, terms, title, color_accent, n_rtn, total_terms):
    n = len(terms)
    y_pos = np.arange(n)[::-1]

    counts = [t['count'] for t in terms]
    nlog10p = [t['nlog10p'] for t in terms]
    gene_ratio = [t['gene_ratio'] for t in terms]
    descs = [textwrap.fill(d['desc'], width=42) if len(d['desc']) > 45 else d['desc'] for d in terms]

    min_c, max_c = min(counts), max(counts)
    sizes = [30 + (c - min_c) / max(max_c - min_c, 1) * 200 for c in counts]

    scatter = ax.scatter(
        gene_ratio, y_pos, s=sizes, c=nlog10p,
        cmap='YlGn', vmin=min(nlog10p)*0.8, vmax=max(nlog10p)*1.05,
        edgecolors=color_accent, linewidths=0.4, zorder=3, alpha=0.85
    )

    ax.set_yticks(y_pos)
    ax.set_yticklabels(descs, fontsize=9)
    ax.set_xlabel('Gene ratio', fontsize=9.5, fontweight='bold')
    ax.set_title(title, fontsize=11, fontweight='bold', color=C_TEXT, pad=10)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_linewidth(0.5)
    ax.spines['bottom'].set_linewidth(0.5)
    ax.tick_params(axis='y', length=0)
    ax.grid(axis='x', alpha=0.15, lw=0.5)

    ax.text(0.98, 0.02, f'{n_rtn:,} rescued-to-normal genes\n{len(terms)} of {total_terms} terms shown',
            transform=ax.transAxes, ha='right', va='bottom',
            fontsize=6.5, color=C_TEXT2, fontstyle='italic')

    # Inset colorbar
    cbax = inset_axes(ax, width="30%", height="3%", loc='lower left',
                       bbox_to_anchor=(0.02, 0.02, 1, 1), bbox_transform=ax.transAxes)
    cbar = fig.colorbar(scatter, cax=cbax, orientation='horizontal')
    cbar.set_label('$-$log$_{10}$(padj)', fontsize=6, labelpad=2)
    cbar.ax.tick_params(labelsize=5)

    return scatter

sc_k = plot_dotplot(ax_k, k_terms, 'Kidney — rescued-to-normal GOBP', C_TEAL_D, 1961, 703)
sc_m = plot_dotplot(ax_m, m_terms, 'Muscle — rescued-to-normal GOBP', C_TEAL_D, 511, 42)

# Size legend on muscle panel
for n_leg, lab in [(15, '15'), (40, '40'), (70, '70')]:
    min_c = min(t['count'] for t in m_terms)
    max_c = max(t['count'] for t in m_terms)
    sz = 30 + (n_leg - min_c) / max(max_c - min_c, 1) * 200
    ax_m.scatter([], [], s=max(sz, 10), c='gray', alpha=0.5, edgecolors=C_GRAY_D,
                  linewidths=0.4, label=f'{n_leg} genes')
ax_m.legend(loc='lower right', fontsize=6.5, frameon=True, framealpha=0.9,
            edgecolor='#E0E0E0', title='Gene count', title_fontsize=6.5,
            handletextpad=0.3, borderpad=0.5, labelspacing=0.8)

fig.suptitle('Fig. 3.2C \u2014 GO Biological Process enrichment of rescued-to-normal genes',
             fontsize=11, fontweight='bold', color=C_TEXT, y=1.01)

fig.text(0.5, 0.005,
         'enrichGO (clusterProfiler), padj < 0.05. Near-duplicate terms removed.\n'
         'Kidney: 703 GOBP terms from 1,961 rescued-to-normal genes. Muscle: 42 terms from 511 genes. 8 terms shared.',
         ha='center', fontsize=6.5, color=C_TEXT2)

plt.tight_layout(rect=[0, 0.03, 1, 0.97])
save_fig(fig, FILENAME)
print('Done \u2014 Fig 3.2C')

## Fig 3.2D — GOBP non-rescued aging genes

In [ ]:
FILENAME = 'Fig_3_2D_GOBP_non_rescued_kidney_muscle'

def load_enrich(filepath, n=15):
    with open(filepath) as f:
        rows = list(csv.DictReader(f))
    terms = []
    seen_words = []
    for r in rows:
        desc = r['Description']
        words = set(desc.lower().split())
        overlap = any(len(words & s) / max(len(words), 1) > 0.6 for s in seen_words)
        if overlap and len(terms) > 3:
            continue
        num, denom = r['GeneRatio'].split('/')
        terms.append({
            'desc': desc,
            'count': int(num),
            'gene_ratio': int(num) / int(denom),
            'padj': float(r['p.adjust']),
            'nlog10p': -np.log10(max(float(r['p.adjust']), 1e-310)),
        })
        seen_words.append(words)
        if len(terms) >= n:
            break
    return terms

k_terms = load_enrich(f"{DATA_BASE}/Kidney/2_Enrichment_bulk&sn/GOBP bulk/kidney_enrichment_results_non_rescued_genes.csv", 15)
m_terms = load_enrich(f"{DATA_BASE}/Muscle_old/2_Enrichment_tables/GOBP bulk/muscle_enrichment_results_non_rescued_genes.csv", 15)

print(f"Kidney: {len(k_terms)} terms | Muscle: {len(m_terms)} terms")

fig, (ax_k, ax_m) = plt.subplots(1, 2, figsize=(14, 8),
                                  gridspec_kw={'width_ratios': [1.1, 1]})

def plot_dotplot(ax, terms, title, color_accent, n_genes, total_terms):
    n = len(terms)
    y_pos = np.arange(n)[::-1]
    counts = [t['count'] for t in terms]
    nlog10p = [t['nlog10p'] for t in terms]
    gene_ratio = [t['gene_ratio'] for t in terms]
    descs = [textwrap.fill(d['desc'], width=42) if len(d['desc']) > 45 else d['desc'] for d in terms]

    min_c, max_c = min(counts), max(counts)
    sizes = [30 + (c - min_c) / max(max_c - min_c, 1) * 200 for c in counts]

    scatter = ax.scatter(
        gene_ratio, y_pos, s=sizes, c=nlog10p,
        cmap='OrRd', vmin=min(nlog10p)*0.8, vmax=max(nlog10p)*1.05,
        edgecolors=color_accent, linewidths=0.4, zorder=3, alpha=0.85
    )

    ax.set_yticks(y_pos)
    ax.set_yticklabels(descs, fontsize=9)
    ax.set_xlabel('Gene ratio', fontsize=9.5, fontweight='bold')
    ax.set_title(title, fontsize=11, fontweight='bold', color=C_TEXT, pad=10)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_linewidth(0.5)
    ax.spines['bottom'].set_linewidth(0.5)
    ax.tick_params(axis='y', length=0)
    ax.grid(axis='x', alpha=0.15, lw=0.5)

    ax.text(0.98, 0.02, f'{n_genes:,} non-rescued genes\n{len(terms)} of {total_terms} terms shown',
            transform=ax.transAxes, ha='right', va='bottom',
            fontsize=6.5, color=C_TEXT2, fontstyle='italic')

    cbax = inset_axes(ax, width="30%", height="3%", loc='lower left',
                       bbox_to_anchor=(0.02, 0.02, 1, 1), bbox_transform=ax.transAxes)
    cbar = fig.colorbar(scatter, cax=cbax, orientation='horizontal')
    cbar.set_label('$-$log$_{10}$(padj)', fontsize=6, labelpad=2)
    cbar.ax.tick_params(labelsize=5)

    return scatter

sc_k = plot_dotplot(ax_k, k_terms, 'Kidney — non-rescued GOBP', C_RED_D, 1169, 194)
sc_m = plot_dotplot(ax_m, m_terms, 'Muscle — non-rescued GOBP', C_RED_D, 1739, 1100)

# Size legend
for n_leg in [20, 50, 90]:
    min_c = min(t['count'] for t in m_terms)
    max_c = max(t['count'] for t in m_terms)
    sz = 30 + (n_leg - min_c) / max(max_c - min_c, 1) * 200
    ax_m.scatter([], [], s=max(sz, 10), c='gray', alpha=0.5, edgecolors=C_GRAY_D,
                  linewidths=0.4, label=f'{n_leg} genes')
ax_m.legend(loc='lower right', fontsize=6.5, frameon=True, framealpha=0.9,
            edgecolor='#E0E0E0', title='Gene count', title_fontsize=6.5,
            handletextpad=0.3, borderpad=0.5, labelspacing=0.8)

fig.suptitle('Fig. 3.2D \u2014 GO Biological Process enrichment of non-rescued aging genes',
             fontsize=11, fontweight='bold', color=C_TEXT, y=1.01)

fig.text(0.5, 0.005,
         'enrichGO (clusterProfiler), padj < 0.05. Near-duplicate terms removed.\n'
         'Kidney: 194 GOBP terms from 1,169 non-rescued genes. Muscle: 1,100 terms from 1,739 genes.',
         ha='center', fontsize=6.5, color=C_TEXT2)

plt.tight_layout(rect=[0, 0.03, 1, 0.97])
save_fig(fig, FILENAME)
print('Done \u2014 Fig 3.2D')

## Fig 3.2E: Summary table or heatmap of functional modules, kidney vs muscle

# 3.3 Dissection of Individual Intervention Contributions

## Fig 3.3A — Driver proportions

In [ ]:
FILENAME = 'Fig_3_3A_driver_proportions_kidney_muscle'

# ── Data ─────────────────────────────────────────────────────────
categories = ['Exclusive_DR', 'Exclusive_sAct', 'Only_Active_in_Combi', 'Other']
cat_colors = [C_GREEN_D, C_PURP_D, C_AMBER_D, C_GRAY_D]
cat_labels = ['DR-exclusive', 'sAct-exclusive', 'Combi-only', 'Both DR & sAct']

def load_proportions(filepath):
    data = {}
    with open(filepath) as f:
        for r in csv.DictReader(f):
            data[r['Category']] = {'count': int(r['Count']), 'prop': float(r['Proportion'])}
    return data

k_data = load_proportions(f"{DATA_BASE}/Kidney/3_Effect_comparison_tables/gene_category_proportions.csv")
m_data = load_proportions(f"{DATA_BASE}/Muscle_old/3_Effect_comparison_tables/muscle_gene_category_proportions.csv")

k_total = sum(k_data[c]['count'] for c in categories)
m_total = sum(m_data[c]['count'] for c in categories)

# ── Figure: donut charts + horizontal stacked bar ────────────────
fig = plt.figure(figsize=(13, 6))
gs = fig.add_gridspec(1, 3, width_ratios=[1, 1, 1.3], wspace=0.3)

def plot_donut(ax, data, total, title):
    vals = [data[c]['count'] for c in categories]
    pcts = [data[c]['prop'] * 100 for c in categories]

    wedges, _ = ax.pie(vals, colors=cat_colors, startangle=90,
                        wedgeprops=dict(width=0.45, edgecolor='white', linewidth=1.5))

    # Percentage labels outside
    for i, (wedge, pct, count) in enumerate(zip(wedges, pcts, vals)):
        angle = (wedge.theta2 + wedge.theta1) / 2
        x = np.cos(np.radians(angle)) * 0.78
        y = np.sin(np.radians(angle)) * 0.78

        # Push labels slightly further out for small slices
        radial_multiplier = 1.45 if pct < 10 else 1.35
        ha = 'left' if x > 0 else 'right'
        ax.text(x * radial_multiplier, y * radial_multiplier,
                f'{pct:.0f}%\n({count})', ha=ha, va='center',
                fontsize=8, fontweight='bold', color=cat_colors[i])

    ax.text(0, 0, f'{total}\ngenes', ha='center', va='center',
            fontsize=14, fontweight='bold', color=C_TEXT)
    ax.set_title(title, fontsize=12, fontweight='bold', color=C_TEXT, pad=15)

# ── Kidney donut ─────────────────────────────────────────────────
ax_k = fig.add_subplot(gs[0])
plot_donut(ax_k, k_data, k_total, 'Kidney')

# ── Muscle donut ─────────────────────────────────────────────────
ax_m = fig.add_subplot(gs[1])
plot_donut(ax_m, m_data, m_total, 'Muscle')

# ── Right panel: side-by-side horizontal stacked bar ─────────────
ax_bar = fig.add_subplot(gs[2])

y_pos = [1.2, 0.4]
bar_h = 0.5

for idx, (tissue, data, total) in enumerate([('Kidney', k_data, k_total), ('Muscle', m_data, m_total)]):
    left = 0
    for i, cat in enumerate(categories):
        pct = data[cat]['prop'] * 100
        ax_bar.barh(y_pos[idx], pct, left=left, height=bar_h,
                     color=cat_colors[i], edgecolor='white', linewidth=0.8)
        if pct > 8:
            ax_bar.text(left + pct/2, y_pos[idx], f'{pct:.0f}%',
                        ha='center', va='center', fontsize=8,
                        fontweight='bold', color='white')
        left += pct

ax_bar.set_yticks(y_pos)
ax_bar.set_yticklabels(['Kidney', 'Muscle'], fontsize=10, fontweight='bold')
ax_bar.set_xlabel('% of rescued-to-normal genes', fontsize=9, fontweight='bold')
ax_bar.set_xlim(0, 100)
ax_bar.spines['top'].set_visible(False)
ax_bar.spines['right'].set_visible(False)
ax_bar.spines['left'].set_visible(False)
ax_bar.tick_params(axis='y', length=0)
ax_bar.set_title('Direct comparison', fontsize=11, fontweight='bold', color=C_TEXT, pad=15)

# Key finding annotation
ax_bar.text(50, -0.35, 'DR dominates both tissues:\n28% kidney, 47% muscle',
            ha='center', va='top', fontsize=8, color=C_GREEN_D, fontweight='bold',
            fontstyle='italic')

# ── Shared legend ────────────────────────────────────────────────
legend_items = [mpatches.Patch(facecolor=cat_colors[i], label=cat_labels[i])
                for i in range(len(categories))]
fig.legend(handles=legend_items, loc='lower center', ncol=4, fontsize=8,
           frameon=True, framealpha=0.9, edgecolor='#E0E0E0',
           bbox_to_anchor=(0.5, -0.02))

fig.suptitle('Fig. 3.3A \u2014 Intervention driver proportions among rescued-to-normal genes',
             fontsize=11, fontweight='bold', color=C_TEXT, y=1.01)

fig.text(0.5, -0.07,
         'DR-exclusive = reversed only by DR. sAct-exclusive = reversed only by sAct.\n'
         'Combi-only = reversed only when both interventions combined. Both = reversed by DR and sAct individually.\n'
         'Kidney: 1,961 rescued-to-normal genes. Muscle: 511 genes.',
         ha='center', fontsize=6.5, color=C_TEXT2, linespacing=1.4)

plt.tight_layout(rect=[0, 0.05, 1, 0.97])
save_fig(fig, FILENAME)
print('Done \u2014 Fig 3.3A')

## Fig 3.3B — Restoration scores

In [ ]:
# ── Fig 3.3B ─────────────────────────────────────────────────────
# Restoration scores: DR vs sAct vs Combi per tissue
FILENAME = 'Fig_3_3B_restoration_scores_kidney_muscle'

KIDNEY_IMPACT_CSV = f"{DATA_BASE}/Kidney/3_Effect_comparison_tables/kidney_intervention_impact_comparison.csv"
MUSCLE_IMPACT_CSV = f"{DATA_BASE}/Muscle_old/3_Effect_comparison_tables/muscle_intervention_impact_comparison.csv"

def load_restoration(filepath):
    dr, sact, combi = [], [], []
    with open(filepath) as f:
        for r in csv.DictReader(f):
            dr.append(float(r['Restoration_DR']))
            sact.append(float(r['Restoration_sAct']))
            combi.append(float(r['Restoration_Combined']))
    return {'DR': np.array(dr), 'sAct': np.array(sact), 'Combi': np.array(combi)}

k_rest = load_restoration(KIDNEY_IMPACT_CSV)
m_rest = load_restoration(MUSCLE_IMPACT_CSV)

# Print summaries
for label, data in [('Kidney', k_rest), ('Muscle', m_rest)]:
    print(f"\n{label}:")
    for intv in ['DR', 'sAct', 'Combi']:
        vals = data[intv]
        print(f"  {intv:6s}: median={np.median(vals):.3f}, mean={np.mean(vals):.3f}, "
              f">1 (full reversal): {(vals > 1).sum()} ({(vals > 1).mean()*100:.0f}%)")

# ── Plot ─────────────────────────────────────────────────────────
fig, (ax_k, ax_m) = plt.subplots(1, 2, figsize=(11, 6), sharey=True)

interventions = ['DR', 'sAct', 'Combi']
intv_colors = [C_GREEN_D, C_PURP_D, C_TEAL_D]
intv_colors_light = [C_GREEN, C_PURPLE, C_TEAL]

def plot_restoration(ax, data, title, n_genes):
    positions = [0, 1, 2]
    datasets = [data[intv] for intv in interventions]

    parts = ax.violinplot(datasets, positions=positions, showmeans=False,
                           showmedians=False, showextrema=False)
    for i, body in enumerate(parts['bodies']):
        body.set_facecolor(intv_colors_light[i])
        body.set_edgecolor(intv_colors[i])
        body.set_alpha(0.6)
        body.set_linewidth(0.8)

    # Box plot overlay
    bp = ax.boxplot(datasets, positions=positions, widths=0.15, patch_artist=True,
                     showfliers=False, zorder=3)
    for i, (box, median) in enumerate(zip(bp['boxes'], bp['medians'])):
        box.set_facecolor(intv_colors[i])
        box.set_edgecolor(intv_colors[i])
        box.set_alpha(0.8)
        median.set_color('white')
        median.set_linewidth(1.5)
    for element in ['whiskers', 'caps']:
        for i, line in enumerate(bp[element]):
            line.set_color(intv_colors[i // 2])
            line.set_linewidth(0.8)

    # Median labels
    for i, intv in enumerate(interventions):
        med = np.median(data[intv])
        ax.text(i, med + 0.08, f'{med:.2f}', ha='center', va='bottom',
                fontsize=8, fontweight='bold', color=intv_colors[i])

    # Reference lines
    ax.axhline(1, color=C_AMBER_D, lw=0.8, ls='--', zorder=0, alpha=0.6)
    ax.text(2.45, 1.02, 'full reversal', fontsize=6, color=C_AMBER_D, ha='right', fontstyle='italic')
    ax.axhline(0, color=C_GRAY_D, lw=0.5, ls='-', zorder=0)
    ax.text(2.45, 0.02, 'no effect', fontsize=6, color=C_GRAY_D, ha='right', fontstyle='italic')

    ax.set_xticks(positions)
    ax.set_xticklabels(interventions, fontsize=10, fontweight='bold')
    ax.set_title(title, fontsize=11, fontweight='bold', color=C_TEXT, pad=10)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_linewidth(0.5)
    ax.spines['bottom'].set_linewidth(0.5)
    ax.grid(axis='y', alpha=0.1, lw=0.5)

    ax.text(0.02, 0.97, f'{n_genes:,} genes', transform=ax.transAxes,
            ha='left', va='top', fontsize=7, color=C_TEXT2)

plot_restoration(ax_k, k_rest, 'Kidney', 1961)
plot_restoration(ax_m, m_rest, 'Muscle', 511)

ax_k.set_ylabel('Restoration score', fontsize=10, fontweight='bold')
ax_k.set_ylim(-0.5, 2.5)

fig.suptitle('Fig. 3.3B \u2014 Restoration scores by intervention: kidney vs muscle',
             fontsize=11, fontweight='bold', color=C_TEXT, y=1.01)

fig.text(0.5, -0.02,
         'Restoration score = intervention LFC / |aging LFC|. Score of 1 = full reversal of aging change.\n'
         'Score > 1 = overcorrection. Score < 0 = worsening. Dashed line = full reversal threshold.\n'
         'Computed for rescued-to-normal genes only.',
         ha='center', fontsize=6.5, color=C_TEXT2, linespacing=1.4)

plt.tight_layout(rect=[0, 0.04, 1, 0.97])
save_fig(fig, FILENAME)
print('Done \u2014 Fig 3.3B')

## Fig 3.3C/D — GOBP intervention drivers

In [ ]:
# ── Auto-load v2 driver gene counts and term counts ───────────────
import os

def count_sig_terms(filepath, padj_col='p.adjust', threshold=0.05):
    try:
        with open(filepath) as f:
            rows = list(csv.DictReader(f))
        return sum(1 for r in rows if float(r[padj_col]) < threshold)
    except FileNotFoundError:
        print(f"Not found: {filepath}")
        return 0

def count_genes(filepath):
    try:
        with open(filepath) as f:
            return sum(1 for _ in csv.DictReader(f))
    except FileNotFoundError:
        return 0

#Kidney — auto-loaded
k_dr_genes   = count_genes(f"{DATA_BASE}/Kidney/3_Effect_comparison_tables/kidney_intervention_impact_comparison_DR.csv")
k_sact_genes = count_genes(f"{DATA_BASE}/Kidney/3_Effect_comparison_tables/kidney_intervention_impact_comparison_sAct.csv")
k_combi_genes= count_genes(f"{DATA_BASE}/Kidney/3_Effect_comparison_tables/kidney_intervention_impact_comparison_only_rescued_in_combi.csv")

k_dr_terms   = count_sig_terms(f"{DATA_BASE}/Kidney/2_Enrichment_bulk&sn/GOBP bulk/kidney_enrichment_results_rescued-to-normal__DR.csv")
k_sact_terms = count_sig_terms(f"{DATA_BASE}/Kidney/2_Enrichment_bulk&sn/GOBP bulk/kidney_enrichment_results_rescued-to-normal_sAct.csv")
k_combi_terms= count_sig_terms(f"{DATA_BASE}/Kidney/2_Enrichment_bulk&sn/GOBP bulk/kidney_enrichment_results_rescued-to-normal_only_in_combi.csv")

print(f"Kidney DR:    {k_dr_genes} genes, {k_dr_terms} sig terms")
print(f"Kidney sAct:  {k_sact_genes} genes, {k_sact_terms} sig terms")
print(f"Kidney combi: {k_combi_genes} genes, {k_combi_terms} sig terms")

# Muscle v2 — auto-loaded
m_dr_genes   = count_genes(f"{MUSCLE_V2}/muscle_v2_intervention_impact_comparison_DR.csv")
m_sact_genes = count_genes(f"{MUSCLE_V2}/muscle_v2_intervention_impact_comparison_sAct.csv")
m_combi_genes= count_genes(f"{MUSCLE_V2}/muscle_v2_intervention_impact_comparison_only_rescued_in_combi.csv")

m_dr_terms   = count_sig_terms(f"{MUSCLE_V2}/muscle_v2_enrichment_results_rescued-to-normal__DR.csv")
m_sact_terms = count_sig_terms(f"{MUSCLE_V2}/muscle_v2_enrichment_results_rescued-to-normal_sAct.csv")
m_combi_terms= count_sig_terms(f"{MUSCLE_V2}/muscle_v2_enrichment_results_rescued-to-normal_only_in_combi.csv")

print(f"Muscle v2 DR:    {m_dr_genes} genes, {m_dr_terms} sig terms")
print(f"Muscle v2 sAct:  {m_sact_genes} genes, {m_sact_terms} sig terms")
print(f"Muscle v2 combi: {m_combi_genes} genes, {m_combi_terms} sig terms")

categories = [
    ('DR-exclusive',  C_GREEN,  C_GREEN_D, k_dr_genes,    k_dr_terms,    m_dr_genes,    m_dr_terms),
    ('sAct-exclusive',C_PURPLE, C_PURP_D,  k_sact_genes,  k_sact_terms,  m_sact_genes,  m_sact_terms),
    ('Combi-only',    C_AMBER,  C_AMBER_D, k_combi_genes, k_combi_terms, m_combi_genes, m_combi_terms),
]

In [ ]:
FILENAME = 'Fig_3_3CD_GOBP_intervention_drivers'

def load_enrich(filepath, n=12):
    with open(filepath) as f:
        rows = list(csv.DictReader(f))
    terms = []
    seen_words = []
    for r in rows:
        desc = r['Description']
        words = set(desc.lower().split())
        overlap = any(len(words & s) / max(len(words), 1) > 0.6 for s in seen_words)
        if overlap and len(terms) > 3:
            continue
        num, denom = r['GeneRatio'].split('/')
        padj = float(r['p.adjust'])
        terms.append({
            'desc': desc,
            'count': int(num),
            'gene_ratio': int(num) / int(denom),
            'padj': padj,
            'nlog10p': -np.log10(max(padj, 1e-310)),
            'sig': padj < 0.05,
        })
        seen_words.append(words)
        if len(terms) >= n:
            break
    return terms

# Load all
k_dr = load_enrich(f"{DATA_BASE}/Kidney/2_Enrichment_bulk&sn/GOBP bulk/kidney_enrichment_results_rescued-to-normal__DR.csv", 12)
m_dr = load_enrich(f"{DATA_BASE}/Muscle_old/2_Enrichment_tables/GOBP bulk/muscle_enrichment_results_rescued-to-normal__DR.csv", 12)

# ── Figure: 2 dot plots + 1 summary panel ────────────────────────
fig = plt.figure(figsize=(15, 8))
# was: wspace=0.35
gs = fig.add_gridspec(1, 3, width_ratios=[1.1, 1.1, 0.6], wspace=0.55)
ax_k = fig.add_subplot(gs[0])
ax_m = fig.add_subplot(gs[1])
ax_sum = fig.add_subplot(gs[2])

def plot_dotplot(ax, terms, title, color_accent, n_genes, total_terms):
    n = len(terms)
    y_pos = np.arange(n)[::-1]
    counts = [t['count'] for t in terms]
    nlog10p = [t['nlog10p'] for t in terms]
    gene_ratio = [t['gene_ratio'] for t in terms]
    descs = [textwrap.fill(d['desc'], width=38) if len(d['desc']) > 40 else d['desc'] for d in terms]

    min_c, max_c = min(counts), max(counts)
    sizes = [30 + (c - min_c) / max(max_c - min_c, 1) * 180 for c in counts]

    scatter = ax.scatter(gene_ratio, y_pos, s=sizes, c=nlog10p,
                          cmap='YlGn', vmin=min(nlog10p)*0.8, vmax=max(nlog10p)*1.05,
                          edgecolors=color_accent, linewidths=0.4, zorder=3, alpha=0.85)

    ax.set_yticks(y_pos)
    ax.set_yticklabels(descs, fontsize=9)
    ax.set_xlabel('Gene ratio', fontsize=9, fontweight='bold')
    ax.set_title(title, fontsize=10, fontweight='bold', color=C_TEXT, pad=10)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_linewidth(0.5)
    ax.spines['bottom'].set_linewidth(0.5)
    ax.tick_params(axis='y', length=0)
    ax.grid(axis='x', alpha=0.15, lw=0.5)

    ax.text(0.98, 0.02, f'{n_genes} DR-excl. genes\n{len(terms)} of {total_terms} terms',
            transform=ax.transAxes, ha='right', va='bottom',
            fontsize=6, color=C_TEXT2, fontstyle='italic')

    cbax = inset_axes(ax, width="30%", height="3%", loc='lower left',
                       bbox_to_anchor=(0.02, 0.02, 1, 1), bbox_transform=ax.transAxes)
    cbar = fig.colorbar(scatter, cax=cbax, orientation='horizontal')
    cbar.set_label('$-$log$_{10}$(padj)', fontsize=5.5, labelpad=2)
    cbar.ax.tick_params(labelsize=5)

plot_dotplot(ax_k, k_dr, 'Kidney — DR-exclusive GOBP', C_GREEN_D, k_dr_genes, k_dr_terms)
plot_dotplot(ax_m, m_dr, 'Muscle — DR-exclusive GOBP', C_GREEN_D, m_dr_genes, m_dr_terms)

# ── Summary panel: all 3 drivers compared ────────────────────────
ax_sum.set_xlim(0, 1)
ax_sum.set_ylim(0, 1)
ax_sum.axis('off')
ax_sum.set_title('Other drivers:\nenriched GOBP terms', fontsize=9, fontweight='bold',
                  color=C_TEXT, pad=10)

# Summary table
categories = [
    ('DR-exclusive',  C_GREEN,  C_GREEN_D, k_dr_genes,    k_dr_terms,    m_dr_genes,    m_dr_terms),
    ('sAct-exclusive',C_PURPLE, C_PURP_D,  k_sact_genes,  k_sact_terms,  m_sact_genes,  m_sact_terms),
    ('Combi-only',    C_AMBER,  C_AMBER_D, k_combi_genes, k_combi_terms, m_combi_genes, m_combi_terms),
]

y_start = 0.85
row_h = 0.07

# Header
ax_sum.text(0.50, 0.95, 'Kidney', ha='center', fontsize=8, fontweight='bold', color=C_RED_D)
ax_sum.text(0.85, 0.95, 'Muscle', ha='center', fontsize=8, fontweight='bold', color=C_BLUE_D)

for i, (label, fc, ec, k_genes, k_terms, m_genes, m_terms) in enumerate(categories):
    y = y_start - i * (row_h * 3 + 0.02)

    ax_sum.add_patch(FancyBboxPatch((0.02, y - row_h*0.6), 0.96, row_h*2.8,
                     boxstyle="round,pad=0.02", facecolor=fc, edgecolor=ec,
                     linewidth=0.6, alpha=0.3, transform=ax_sum.transAxes, zorder=0))

    ax_sum.text(0.10, y + row_h, label, ha='left', va='center',
                fontsize=8, fontweight='bold', color=ec, transform=ax_sum.transAxes)

    # Kidney stats
    ax_sum.text(0.50, y + row_h*0.3, f'{k_genes} genes', ha='center', va='center',
                fontsize=7, color=C_TEXT, transform=ax_sum.transAxes)
    ax_sum.text(0.50, y - row_h*0.3,
                f'{k_terms} sig. terms' if k_terms > 0 else '0 sig. terms',
                ha='center', va='center',
                fontsize=6.5, color=C_TEXT2, transform=ax_sum.transAxes)

    # Muscle stats
    ax_sum.text(0.85, y + row_h*0.3, f'{m_genes} genes', ha='center', va='center',
                fontsize=7, color=C_TEXT, transform=ax_sum.transAxes)
    ax_sum.text(0.85, y - row_h*0.3,
                f'{m_terms} sig. terms' if m_terms > 0 else '0 sig. terms',
                ha='center', va='center',
                fontsize=6.5, color=C_TEXT2, transform=ax_sum.transAxes)

# Key insight box
ax_sum.add_patch(FancyBboxPatch((0.02, 0.02), 0.96, 0.15,
                 boxstyle="round,pad=0.015", facecolor='#F8F7F5',
                 edgecolor=C_GRAY_D, linewidth=0.6, transform=ax_sum.transAxes, zorder=2))
ax_sum.text(0.50, 0.12, 'Key finding', ha='center', fontsize=7, fontweight='bold',
            color=C_TEXT, transform=ax_sum.transAxes, zorder=3)
ax_sum.text(0.50, 0.06,
            'DR-exclusive: strong, specific\npathway enrichment in both tissues.\nsAct/combi-only: too few genes\nfor robust pathway-level signal.',
            ha='center', va='center', fontsize=6, color=C_TEXT2,
            transform=ax_sum.transAxes, zorder=3, linespacing=1.4)

fig.suptitle('Fig. 3.3C/D — Pathway specificity of intervention drivers',
             fontsize=11, fontweight='bold', color=C_TEXT, y=1.01)

fig.text(0.37, -0.01,
         'Left/center: Top DR-exclusive GOBP terms (enrichGO, padj < 0.05). Right: summary of all three driver categories.\n'
         'sAct-exclusive and combi-only gene sets are too small (23-108 genes) for robust over-representation analysis.',
         ha='center', fontsize=6.5, color=C_TEXT2, linespacing=1.4)

plt.tight_layout(rect=[0, 0.03, 1, 0.97])
save_fig(fig, FILENAME)
print('Done — Fig 3.3C/D')

## Table 3.3A — Top genes per driver category

In [ ]:
# ── Table 3.3A ───────────────────────────────────────────────────
# Top genes per driver category with LFC values
FILENAME = 'Table_3_3A_top_genes_per_driver'

KIDNEY_BASE = f"{DATA_BASE}/Kidney/3_Effect_comparison_tables"
MUSCLE_BASE = f"{DATA_BASE}/Muscle_old/3_Effect_comparison_tables"

def load_driver_genes(base, tissue):
    cats = {}
    for cat, suffix, rest_col, intv_col in [
        ('DR-exclusive', f'{tissue}_intervention_impact_comparison_DR.csv', 'Restoration_DR', 'LFC_DR'),
        ('sAct-exclusive', f'{tissue}_intervention_impact_comparison_sAct.csv', 'Restoration_sAct', 'LFC_sAct'),
        ('Combi-only', f'{tissue}_intervention_impact_comparison_only_rescued_in_combi.csv', 'Restoration_Combined', 'LFC_Combined'),
    ]:
        with open(f'{base}/{suffix}') as f:
            rows = list(csv.DictReader(f))
        # Filter out genes without symbol
        rows = [r for r in rows if r.get('Symbol', '') and not r['Symbol'].startswith('ENSMUSG') and r['Symbol'] != 'NA']
        # Sort by restoration score (best rescued first)
        rows.sort(key=lambda r: abs(float(r['LFC_Age'])), reverse=True)
        cats[cat] = rows[:5]
    return cats

k_cats = load_driver_genes(KIDNEY_BASE, 'kidney')
m_cats = load_driver_genes(MUSCLE_BASE, 'muscle')

# ── Build figure table ───────────────────────────────────────────
fig, (ax_k, ax_m) = plt.subplots(2, 1, figsize=(13, 10))

cat_colors_map = {'DR-exclusive': C_GREEN_D, 'sAct-exclusive': C_PURP_D, 'Combi-only': C_AMBER_D}
cat_bg_map = {'DR-exclusive': C_GREEN, 'sAct-exclusive': C_PURPLE, 'Combi-only': C_AMBER}

def draw_table(ax, cats, title, tissue):
    ax.axis('off')
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_title(title, fontsize=12, fontweight='bold', color=C_TEXT, pad=15)

    # Column positions
    col_x = [0.02, 0.14, 0.32, 0.46, 0.60, 0.74, 0.88]
    headers = ['Category', 'Gene', 'Aging LFC', 'DR LFC', 'sAct LFC', 'Combi LFC', 'Restoration']

    y = 0.95
    row_h = 0.038

    # Header
    for j, (x, h) in enumerate(zip(col_x, headers)):
        ax.text(x, y, h, fontsize=7.5, fontweight='bold', color=C_TEXT,
                va='top', ha='left')
    y -= row_h * 0.7
    ax.axhline(y, xmin=0.01, xmax=0.99, color=C_GRAY_D, lw=0.5)
    y -= row_h * 0.5

    for cat_name in ['DR-exclusive', 'sAct-exclusive', 'Combi-only']:
        genes = cats[cat_name]
        color = cat_colors_map[cat_name]
        bg = cat_bg_map[cat_name]

        # Category label (spanning rows)
        cat_y_start = y
        for i, g in enumerate(genes):
            # Background stripe
            if i % 2 == 0:
                ax.add_patch(plt.Rectangle((0.01, y - row_h * 0.3), 0.98, row_h * 0.9,
                             facecolor='#F8F7F5', edgecolor='none', transform=ax.transAxes, zorder=0))

            sym = g.get('Symbol', '?')
            age_lfc = float(g['LFC_Age'])
            dr_lfc = float(g.get('LFC_DR', 0))
            sact_lfc = float(g.get('LFC_sAct', 0))
            combi_lfc = float(g['LFC_Combined'])

            # Pick restoration score based on category
            if cat_name == 'DR-exclusive':
                rest = float(g['Restoration_DR'])
            elif cat_name == 'sAct-exclusive':
                rest = float(g['Restoration_sAct'])
            else:
                rest = float(g['Restoration_Combined'])

            values = [
                '',
                sym,
                f'{age_lfc:+.2f}',
                f'{dr_lfc:+.2f}',
                f'{sact_lfc:+.2f}',
                f'{combi_lfc:+.2f}',
                f'{rest:.2f}',
            ]
            lfc_colors = [
                C_TEXT,
                C_TEXT,
                C_RED_D if age_lfc > 0 else C_BLUE_D,
                C_GREEN_D,
                C_PURP_D,
                C_TEAL_D,
                C_AMBER_D if rest >= 0.8 else C_TEXT,
            ]

            for j, (x, v, c) in enumerate(zip(col_x, values, lfc_colors)):
                weight = 'bold' if j == 1 else 'normal'
                style = 'italic' if j == 1 else 'normal'
                ax.text(x, y, v, fontsize=7, color=c, va='top', ha='left',
                        fontweight=weight, fontstyle=style)
            y -= row_h

        # Category label centered on its rows
        cat_y_mid = (cat_y_start + y) / 2 + row_h * 0.3
        ax.add_patch(plt.Rectangle((0.01, y + row_h * 0.2), 0.12, cat_y_start - y,
                     facecolor=bg, edgecolor=color, linewidth=0.5,
                     transform=ax.transAxes, zorder=1, alpha=0.5))
        # Split category name
        parts = cat_name.split('-')
        ax.text(0.07, cat_y_mid + 0.01, parts[0] + '-', fontsize=6.5, fontweight='bold',
                color=color, va='center', ha='center')
        ax.text(0.07, cat_y_mid - 0.015, parts[1] if len(parts) > 1 else '', fontsize=6.5,
                fontweight='bold', color=color, va='center', ha='center')

        # Separator line
        ax.axhline(y + row_h * 0.2, xmin=0.01, xmax=0.99, color=C_GRAY, lw=0.3)
        y -= row_h * 0.3

draw_table(ax_k, k_cats, 'Kidney — top genes per driver category', 'kidney')
draw_table(ax_m, m_cats, 'Muscle — top genes per driver category', 'muscle')

fig.suptitle('Table 3.3A \u2014 Top rescued-to-normal genes per intervention driver category',
             fontsize=11, fontweight='bold', color=C_TEXT, y=1.01)

fig.text(0.5, -0.01,
         'Top 5 genes per category ranked by |aging LFC|. Restoration = intervention LFC / |aging LFC| (1 = full reversal).\n'
         'LFC values colour-coded: red/blue = aging direction, green = DR, purple = sAct, teal = combi.',
         ha='center', fontsize=6.5, color=C_TEXT2, linespacing=1.4)

plt.tight_layout(rect=[0, 0.03, 1, 0.98])
save_fig(fig, FILENAME)
print('Done \u2014 Table 3.3A')

# 3.4 Cell-Type-Specific Resolution of Aging and Rescue

## Fig 3.4-0A/B — UMAP atlases

In [ ]:
# ── Fig 3.4-0A/B ────────────────────────────────────────────────
# UMAP of snRNA-seq atlases colored by cell type
FILENAME_K = 'Fig_3_4_0A_UMAP_kidney'
FILENAME_M = 'Fig_3_4_0B_UMAP_muscle'

from collections import defaultdict

KIDNEY_UMAP_CSV = f"{DATA_BASE}/Kidney/kidney_umap_coordinates.csv"
MUSCLE_UMAP_CSV = f"{DATA_BASE}/Muscle_old/muscle_umap_coordinates.csv"

def load_umap(filepath):
    data = defaultdict(lambda: {'x': [], 'y': []})
    with open(filepath) as f:
        for r in csv.DictReader(f):
            ct = r['celltype']
            data[ct]['x'].append(float(r['UMAP_1']))
            data[ct]['y'].append(float(r['UMAP_2']))
    # Convert to arrays
    for ct in data:
        data[ct]['x'] = np.array(data[ct]['x'])
        data[ct]['y'] = np.array(data[ct]['y'])
    return data

palette = ['#E8918D', '#E8A86B', '#D4B56A', '#A3BE8C', '#7DB89E',
           '#6BBFB0', '#5EAAA8', '#6BA3C7', '#7E94C0', '#9B8EC2',
           '#B68CB5', '#C98BA3', '#D4A19A', '#BDB5A0', '#A0C4A8',
           '#D4C5A0', '#8FB8A0', '#A89BC2', '#C4A882', '#9CB8D4',
           '#B0A8C0', '#C8B090', '#88C0B0', '#C0A8A8', '#A8C0B8']

def plot_umap(umap_data, title, filename, figsize=(10, 8)):
    # Sort cell types by size (largest first = plotted first = behind)
    cts_sorted = sorted(umap_data.keys(), key=lambda ct: len(umap_data[ct]['x']), reverse=True)
    n = len(cts_sorted)

    fig, ax = plt.subplots(figsize=figsize)

    for i, ct in enumerate(cts_sorted):
        c = palette[i % len(palette)]
        ax.scatter(umap_data[ct]['x'], umap_data[ct]['y'],
                   s=1.5, c=c, alpha=0.4, edgecolors='none',
                   rasterized=True, zorder=2, label=ct)

    # Centroid labels
    for i, ct in enumerate(cts_sorted):
        cx = np.median(umap_data[ct]['x'])
        cy = np.median(umap_data[ct]['y'])
        c = palette[i % len(palette)]
        ax.annotate(ct, (cx, cy), fontsize=6, fontweight='bold', color='#1a1a1a',
                    ha='center', va='center', zorder=5,
                    bbox=dict(boxstyle='round,pad=0.15', fc='white', ec=c,
                              lw=0.5, alpha=0.85))

    ax.set_xlabel('UMAP 1', fontsize=10, fontweight='bold')
    ax.set_ylabel('UMAP 2', fontsize=10, fontweight='bold')
    ax.set_title(title, fontsize=12, fontweight='bold', color=C_TEXT, pad=12)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_linewidth(0.5)
    ax.spines['bottom'].set_linewidth(0.5)
    ax.tick_params(labelsize=8)

    # Cell count
    total = sum(len(umap_data[ct]['x']) for ct in cts_sorted)
    ax.text(0.02, 0.02, f'{total:,} nuclei, {n} cell types',
            transform=ax.transAxes, fontsize=7, color=C_TEXT2,
            bbox=dict(boxstyle='round,pad=0.3', fc=C_WHITE, ec='#E0E0E0', lw=0.5))

    plt.tight_layout()
    save_fig(fig, filename)
    print(f'Done — {filename}')

k_umap = load_umap(KIDNEY_UMAP_CSV)
plot_umap(k_umap, 'Kidney snRNA-seq atlas — cell types', FILENAME_K)

m_umap = load_umap(MUSCLE_UMAP_CSV)
plot_umap(m_umap, 'Muscle snRNA-seq atlas — cell types', FILENAME_M)

## Fig 3.4-0C — Composition all conditions

In [ ]:
# ── Fig 3.4-0C ───────────────────────────────────────────────────
# Cell-type composition per condition (all 5 conditions)
FILENAME = 'Fig_3_4_0C_composition_all_conditions'

from collections import Counter

KIDNEY_COMP_CSV = f"{DATA_BASE}/Kidney/kidney_celltype_condition.csv"
MUSCLE_COMP_CSV = f"{DATA_BASE}/Muscle_old/muscle_celltype_condition.csv"

conditions_all = ['ctrl', 'age', 'DR', 'sAct', 'combi']
cond_labels_all = ['Ctrl', 'Age', 'DR', 'sAct', 'Combi']

def load_composition(filepath):
    data = {}
    with open(filepath) as f:
        for r in csv.DictReader(f):
            cond = r['condition']
            ct = r['celltype']
            if cond not in data:
                data[cond] = Counter()
            data[cond][ct] += 1
    return data

def get_proportions_all(comp, conditions):
    all_cts = set()
    for cond in conditions:
        if cond in comp:
            all_cts.update(comp[cond].keys())
    # Sort by average proportion
    ct_avg = {ct: sum(comp.get(c, Counter()).get(ct, 0) for c in conditions) for ct in all_cts}
    cts_sorted = sorted(ct_avg, key=ct_avg.get, reverse=True)
    props = {}
    for cond in conditions:
        total = sum(comp[cond].values()) if cond in comp else 1
        props[cond] = {ct: comp.get(cond, Counter()).get(ct, 0) / total * 100 for ct in cts_sorted}
    return cts_sorted, props

k_comp = load_composition(KIDNEY_COMP_CSV)
m_comp = load_composition(MUSCLE_COMP_CSV)

k_cts, k_props = get_proportions_all(k_comp, conditions_all)
m_cts, m_props = get_proportions_all(m_comp, conditions_all)

palette = ['#E8918D', '#E8A86B', '#D4B56A', '#A3BE8C', '#7DB89E',
           '#6BBFB0', '#5EAAA8', '#6BA3C7', '#7E94C0', '#9B8EC2',
           '#B68CB5', '#C98BA3', '#D4A19A', '#BDB5A0', '#A0C4A8',
           '#D4C5A0', '#8FB8A0', '#A89BC2', '#C4A882', '#9CB8D4',
           '#B0A8C0', '#C8B090', '#88C0B0', '#C0A8A8', '#A8C0B8']

fig, (ax_k, ax_m) = plt.subplots(1, 2, figsize=(14, 7), gridspec_kw={'wspace': 0.35})

def plot_composition(ax, cts, props, comp, conditions, title, colors):
    n_cond = len(conditions)
    x = np.arange(n_cond) * 2  # space out bars

    for ci, cond in enumerate(conditions):
        bottom = 0
        for i, ct in enumerate(cts):
            val = props[cond].get(ct, 0)
            c = colors[i % len(colors)]
            ax.bar(x[ci], val, bottom=bottom, width=0.8,
                   color=c, edgecolor='white', linewidth=0.3)
            if val > 5:
                ax.text(x[ci], bottom + val / 2, f'{val:.0f}%',
                        ha='center', va='center', fontsize=5, color='white', fontweight='bold')
            bottom += val

    # N cells below each bar
    for ci, cond in enumerate(conditions):
        n = sum(comp.get(cond, Counter()).values())
        ax.text(x[ci], -4, f'n={n:,}', ha='center', fontsize=6, color=C_TEXT2)

    ax.set_xticks(x)
    ax.set_xticklabels(cond_labels_all, fontsize=10, fontweight='bold')
    ax.set_ylabel('% of nuclei', fontsize=10, fontweight='bold')
    ax.set_title(title, fontsize=12, fontweight='bold', color=C_TEXT, pad=12)
    ax.set_ylim(0, 102)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plot_composition(ax_k, k_cts, k_props, k_comp, conditions_all,
                 'Kidney — cell-type composition', palette)
plot_composition(ax_m, m_cts, m_props, m_comp, conditions_all,
                 'Muscle — cell-type composition', palette)

# Legends
def add_legend(ax, cts, colors):
    handles = [mpatches.Patch(facecolor=colors[i % len(colors)], label=ct)
               for i, ct in enumerate(cts)]
    ax.legend(handles=handles, loc='center left', bbox_to_anchor=(1.02, 0.5),
              fontsize=5.5, frameon=True, framealpha=0.9, edgecolor='#E0E0E0',
              title='Cell types', title_fontsize=6.5, handlelength=1, handleheight=1,
              labelspacing=0.3, borderpad=0.4)

add_legend(ax_k, k_cts, palette)
add_legend(ax_m, m_cts, palette)

fig.suptitle('Cell-type composition across all conditions',
             fontsize=11, fontweight='bold', color=C_TEXT, y=1.01)

fig.text(0.5, -0.02,
         'Stacked bars show proportion of nuclei per cell type in each condition.\n'
         'snRNA-seq: n=1 per condition (exploratory). Changes may reflect true composition shifts or sampling variation.',
         ha='center', fontsize=6.5, color=C_TEXT2, linespacing=1.4)

plt.tight_layout(rect=[0, 0.03, 1, 0.98])
save_fig(fig, FILENAME)
print('Done — Fig 3.4-0C')

## Fig 3.4A/B/E ─ Violin plots of rescued module scores by cell type

In [ ]:
# ── Fig 3.4A/B/E ─────────────────────────────────────────────────
# Violin plots of rescued module scores by cell type
# Requires: kidney_module_scores_all.csv, muscle_module_scores_all.csv

import textwrap
from collections import defaultdict

KIDNEY_ALL_CSV = f"{DATA_BASE}/Kidney/kidney_module_scores_all.csv"
MUSCLE_ALL_CSV = f"{DATA_BASE}/Muscle_old/muscle_module_scores_all.csv"

def load_all_scores(filepath):
    """Load per-cell module scores for all score types."""
    scores = defaultdict(lambda: defaultdict(list))
    with open(filepath) as f:
        for r in csv.DictReader(f):
            ct = r['celltype']
            for col in ['Reversal_norm', 'Reversal_DR', 'Reversal_sAct', 'Aged']:
                if col in r and r[col] not in ('NA', '', 'NaN'):
                    scores[col][ct].append(float(r[col]))
    return scores

def plot_violin(ct_scores, title, filename, figsize=(12, 6)):
    cts_sorted = sorted(
        ct_scores.keys(),
        key=lambda ct: np.median(ct_scores[ct]),
        reverse=True
    )
    data = [ct_scores[ct] for ct in cts_sorted]
    n = len(cts_sorted)

    fig, ax = plt.subplots(figsize=figsize)

    palette = ['#E8918D', '#E8A86B', '#D4B56A', '#A3BE8C', '#7DB89E',
               '#6BBFB0', '#5EAAA8', '#6BA3C7', '#7E94C0', '#9B8EC2',
               '#B68CB5', '#C98BA3', '#D4A19A', '#BDB5A0', '#A0C4A8']
    colors = [palette[i % len(palette)] for i in range(n)]

    parts = ax.violinplot(
        data,
        positions=range(n),
        showmeans=False,
        showmedians=False,
        showextrema=False
    )

    for i, body in enumerate(parts['bodies']):
        body.set_facecolor(colors[i])
        body.set_edgecolor(colors[i])
        body.set_alpha(0.75)
        body.set_linewidth(0.5)

    for i, d in enumerate(data):
        med = np.median(d)
        q1, q3 = np.percentile(d, [25, 75])
        ax.vlines(i, q1, q3, color='#333333', linewidth=0.8, zorder=3)
        ax.scatter(i, med, color='white', s=8, edgecolors='#333333',
                   linewidths=0.5, zorder=4)

    labels = [textwrap.fill(ct, width=15) if len(ct) > 12 else ct for ct in cts_sorted]
    ax.set_xticks(range(n))
    ax.set_xticklabels(labels, fontsize=7, rotation=45, ha='right')
    ax.set_ylabel('Module score', fontsize=10, fontweight='bold')
    ax.set_title(title, fontsize=12, fontweight='bold', color=C_TEXT, pad=12)

    ax.axhline(0, color=C_GRAY_D, lw=0.5, ls='--', zorder=0)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_linewidth(0.5)
    ax.spines['bottom'].set_linewidth(0.5)
    ax.tick_params(axis='y', labelsize=8)
    ax.grid(axis='y', alpha=0.1, lw=0.5)

    top3 = [clean_name(ct) for ct in cts_sorted[:3]]
    ax.text(
        0.98, 0.97,
        f'Top 3: {", ".join(top3)}',
        transform=ax.transAxes,
        ha='right',
        va='top',
        fontsize=7,
        color=C_TEXT2,
        fontstyle='italic',
        bbox=dict(boxstyle='round,pad=0.3', fc=C_WHITE, ec='#E0E0E0', lw=0.5)
    )

    total_cells = sum(len(d) for d in data)
    ax.text(
        0.02, 0.97,
        f'{total_cells:,} nuclei across {n} cell types',
        transform=ax.transAxes,
        ha='left',
        va='top',
        fontsize=7,
        color=C_TEXT2
    )

    plt.tight_layout()
    save_fig(fig, filename)
    print(f'Done — {filename}')


# ── Tissue-specific figure definitions ────────────────────────────
figures = {
    'Kidney': [
        ('Reversal_norm', '3.4A', 'rescued-to-normal',     'Fig_3_4A_violin_rescued_module'),
        ('Reversal_DR',   '3.4B', 'DR-exclusive rescued',  'Fig_3_4B1_violin_DR_module'),
        ('Reversal_sAct', '3.4B', 'sAct-exclusive rescued','Fig_3_4B2_violin_sAct_module'),
    ],
    'Muscle': [
        ('Reversal_norm', '3.4E', 'rescued-to-normal',     'Fig_3_4E_violin_rescued_module'),
        ('Reversal_DR',   '3.4F', 'DR-exclusive rescued',  'Fig_3_4F1_violin_DR_module'),
        ('Reversal_sAct', '3.4F', 'sAct-exclusive rescued','Fig_3_4F2_violin_sAct_module'),
    ]
}


# ── Load and plot ────────────────────────────────────────────────
for tissue, csv_path, tissue_label, figw in [
    ('Kidney', KIDNEY_ALL_CSV, 'kidney', 13),
    ('Muscle', MUSCLE_ALL_CSV, 'muscle', 11),
]:
    try:
        all_scores = load_all_scores(csv_path)
        print(f"\n{tissue}: loaded {list(all_scores.keys())}")
    except FileNotFoundError:
        print(f"{tissue} CSV not found at {csv_path} — skip")
        continue

    for score_col, fig_id, title_suffix, fname_base in figures[tissue]:
        if score_col not in all_scores or not all_scores[score_col]:
            print(f"  Skipping {score_col} — no data")
            continue

        fname = f'{fname_base}_{tissue_label}'
        title = f'Fig. {fig_id} — {tissue} {title_suffix} module scores by cell type'

        plot_violin(all_scores[score_col], title, fname, figsize=(figw, 6))

## Fig 3.4C — Reversal_marker_celltype_heatmap

In [ ]:
from collections import defaultdict
import matplotlib.colors as mcolors

In [ ]:
FILENAME = 'Fig_3_4C_reversal_marker_celltype_heatmap'

def load_driver_genes(base_path):
    dr, sact, combi, both = set(), set(), set(), set()
    with open(f"{base_path}_DR.csv") as f:
        for r in csv.DictReader(f): dr.add(r['Symbol'])
    with open(f"{base_path}_sAct.csv") as f:
        for r in csv.DictReader(f): sact.add(r['Symbol'])
    with open(f"{base_path}_only_rescued_in_combi.csv") as f:
        for r in csv.DictReader(f): combi.add(r['Symbol'])
    with open(f"{base_path}.csv") as f:
        for r in csv.DictReader(f):
            sym = r['Symbol']
            if sym not in dr and sym not in sact and sym not in combi:
                both.add(sym)
    return {'DR-exclusive': dr, 'sAct-exclusive': sact, 'Combi-only': combi, 'Both DR+sAct': both}

kidney_drivers = load_driver_genes(f"{DATA_BASE}/Kidney/3_Effect_comparison_tables/kidney_intervention_impact_comparison")
muscle_drivers = load_driver_genes(f"{DATA_BASE}/Muscle_old/3_Effect_comparison_tables/muscle_intervention_impact_comparison")

def build_matrix(marker_csv, driver_sets):
    categories = ['DR-exclusive', 'Both DR+sAct', 'sAct-exclusive', 'Combi-only']
    ct_counts = defaultdict(lambda: {c: 0 for c in categories})
    with open(marker_csv) as f:
        for row in csv.DictReader(f):
            gene, ct = row['gene'], row['cluster']
            for cat in categories:
                if gene in driver_sets[cat]:
                    ct_counts[ct][cat] += 1
                    break
    totals = {ct: sum(d.values()) for ct, d in ct_counts.items()}
    sorted_cts = sorted(totals, key=totals.get, reverse=True)
    matrix = [[ct_counts[ct][c] for c in categories] for ct in sorted_cts]
    return sorted_cts, categories, matrix, totals

k_cts, cats, k_mat, k_totals = build_matrix(
    f"{DATA_BASE}/Kidney/4_Aged-Rescued_genes_celltype_breakdown/kidney_markers_reversal_genes_celltypes.csv",
    kidney_drivers)
m_cts, _, m_mat, m_totals = build_matrix(
    f"{DATA_BASE}/Muscle_old/4_Aged-Rescued_genes_celltype_breakdown/muscle_markers_reversal_genes_celltypes.csv",
    muscle_drivers)

cat_colors = [C_GREEN_D, C_CORAL_D, C_PURP_D, C_AMBER_D]
cat_fills  = [C_GREEN,   C_CORAL,   C_PURPLE,  C_AMBER]

fig, (ax_k, ax_m) = plt.subplots(1, 2, figsize=(14, 9),
                                  gridspec_kw={'width_ratios': [1.3, 1]})

def draw_heatmap(ax, cell_types, matrix, totals, title, max_val=None):
    n_ct = len(cell_types)
    n_cat = len(cats)
    if max_val is None:
        max_val = max(max(row) for row in matrix) if matrix else 1
    for i, ct in enumerate(cell_types):
        y = n_ct - 1 - i
        for j, cat in enumerate(cats):
            val = matrix[i][j]
            intensity = val / max_val if max_val > 0 else 0
            r1, g1, b1 = mcolors.to_rgb(C_WHITE)
            r2, g2, b2 = mcolors.to_rgb(cat_fills[j])
            fc = (r1+(r2-r1)*intensity, g1+(g2-g1)*intensity, b1+(b2-b1)*intensity)
            rect = mpatches.FancyBboxPatch((j, y), 0.92, 0.85, boxstyle="round,pad=0.02",
                                            facecolor=fc, edgecolor='#E0E0E0', linewidth=0.3, zorder=2)
            ax.add_patch(rect)
            if val > 0:
                txt_color = cat_colors[j] if intensity > 0.3 else C_TEXT2
                ax.text(j+0.46, y+0.42, str(val), ha='center', va='center', fontsize=7.5,
                        fontweight='bold' if val >= 10 else 'normal', color=txt_color, zorder=3)
        ax.text(-0.15, y+0.42, clean_name(ct), ha='right', va='center', fontsize=7.5, color=C_TEXT)
        ax.text(n_cat+0.15, y+0.42, str(totals[ct]), ha='left', va='center', fontsize=7.5,
                fontweight='bold', color=C_TEXT2)

    # Column headers — pushed higher with more room above grid
    header_y = n_ct + 0.5
    for j, cat in enumerate(cats):
        ax.text(j+0.46, header_y, cat, ha='center', va='bottom', fontsize=7.5,
                fontweight='bold', color=cat_colors[j], rotation=35)
    ax.text(n_cat+0.15, header_y, 'Total', ha='left', va='bottom', fontsize=7.5,
            fontweight='bold', color=C_TEXT2, rotation=35)

    # Title — pushed higher above column headers
    ax.text(n_cat/2, n_ct + 3.5, title, ha='center', va='center', fontsize=12,
            fontweight='bold', color=C_TEXT)

    ax.set_xlim(-0.3, n_cat+0.8)
    ax.set_ylim(-0.5, n_ct + 4.5)
    ax.set_aspect('equal')
    ax.axis('off')

global_max = max(max(max(row) for row in k_mat), max(max(row) for row in m_mat))
draw_heatmap(ax_k, k_cts, k_mat, k_totals, 'Kidney', max_val=global_max)
draw_heatmap(ax_m, m_cts, m_mat, m_totals, 'Muscle', max_val=global_max)

fig.suptitle('Fig. 3.4C \u2014 Reversal gene markers by cell type and intervention driver',
             fontsize=11, fontweight='bold', color=C_TEXT, y=0.99)

# Legend — positioned well below the figure content
legend_items = [(C_GREEN, C_GREEN_D, 'DR-exclusive'), (C_CORAL, C_CORAL_D, 'Both DR+sAct'),
                (C_PURPLE, C_PURP_D, 'sAct-exclusive'), (C_AMBER, C_AMBER_D, 'Combi-only')]
handles = [mpatches.Patch(facecolor=fc, edgecolor=ec, linewidth=0.5, label=lab) for fc, ec, lab in legend_items]
fig.legend(handles=handles, loc='lower center', ncol=4, fontsize=8, frameon=False,
           bbox_to_anchor=(0.5, 0.005), handlelength=1.5, handleheight=0.9, columnspacing=2.0)

# Footnote — below the legend
fig.text(0.5, 0.035,
         'Values = number of rescued-to-normal genes that are also cell-type markers (FindAllMarkers, padj < 0.05).\n'
         'Color intensity scales with count. Cell types ordered by total reversal marker count (descending).',
         ha='center', va='bottom', fontsize=7, color=C_TEXT2)

plt.tight_layout(rect=[0, 0.065, 1, 0.97])
save_fig(fig, FILENAME)
print('Done \u2014 Fig 3.4C')

## Fig 3.4D/H — Per-cell-type GOBP enrichment (rescued)[link text](https://)

In [ ]:
# ── Fig 3.4D/H ───────────────────────────────────────────────────
# Per-cell-type GOBP enrichment for rescued genes in key cell types
FILENAME_K = 'Fig_3_4D_GOBP_per_celltype_kidney'
FILENAME_M = 'Fig_3_4H_GOBP_per_celltype_muscle'

import textwrap
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

def load_enrich_ct(filepath, n=8):
    terms = []
    seen_words = []
    with open(filepath) as f:
        for r in csv.DictReader(f):
            desc = r['Description']
            words = set(desc.lower().split())
            overlap = any(len(words & s) / max(len(words), 1) > 0.6 for s in seen_words)
            if overlap and len(terms) > 2:
                continue
            num, denom = r['GeneRatio'].split('/')
            terms.append({
                'desc': desc,
                'count': int(num),
                'gene_ratio': int(num) / int(denom),
                'padj': float(r['p.adjust']),
                'nlog10p': -np.log10(max(float(r['p.adjust']), 1e-310)),
            })
            seen_words.append(words)
            if len(terms) >= n:
                break
    return terms

def plot_ct_enrichment(celltypes_files, title, filename, figsize=(16, 9)):
    n_ct = len(celltypes_files)

    # More space between panels + taller figure
    fig, axes = plt.subplots(1, n_ct, figsize=figsize,
                              gridspec_kw={'wspace': 0.6})  # ← more horizontal space
    if n_ct == 1:
        axes = [axes]

    for idx, (ct_label, filepath) in enumerate(celltypes_files):
        ax = axes[idx]
        terms = load_enrich_ct(filepath, n=8)

        if not terms:
            ax.text(0.5, 0.5, 'No significant\nterms', ha='center', va='center',
                    transform=ax.transAxes, fontsize=9, color=C_TEXT2)
            ax.set_title(clean_name(ct_label), fontsize=10, fontweight='bold', color=C_TEXT)
            ax.axis('off')
            continue

        n = len(terms)
        y_pos = np.arange(n)[::-1]
        counts    = [t['count']      for t in terms]
        nlog10p   = [t['nlog10p']    for t in terms]
        gene_ratio= [t['gene_ratio'] for t in terms]

        # Wrap labels more aggressively
        descs = [textwrap.fill(t['desc'], width=22) for t in terms]  # ← shorter wrap

        min_c, max_c = min(counts), max(counts)
        sizes = [30 + (c - min_c) / max(max_c - min_c, 1) * 150 for c in counts]

        scatter = ax.scatter(gene_ratio, y_pos, s=sizes, c=nlog10p,
                              cmap='YlGn', vmin=min(nlog10p)*0.8, vmax=max(nlog10p)*1.05,
                              edgecolors=C_TEAL_D, linewidths=0.3, zorder=3, alpha=0.85)

        ax.set_yticks(y_pos)
        ax.set_yticklabels(descs, fontsize=9)  # ← smaller font
        ax.set_xlabel('Gene ratio', fontsize=8, fontweight='bold')
        ax.set_title(clean_name(ct_label), fontsize=9, fontweight='bold', color=C_TEXT,
                     pad=10, wrap=True)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.spines['left'].set_linewidth(0.5)
        ax.spines['bottom'].set_linewidth(0.5)
        ax.tick_params(axis='y', length=0)
        ax.grid(axis='x', alpha=0.15, lw=0.5)

        ax.text(0.98, 0.02, f'{len(terms)} terms shown',
                transform=ax.transAxes, ha='right', va='bottom',
                fontsize=5.5, color=C_TEXT2, fontstyle='italic')

        cbax = inset_axes(ax, width="30%", height="3%", loc='lower left',
                           bbox_to_anchor=(0.02, 0.02, 1, 1), bbox_transform=ax.transAxes)
        cbar = fig.colorbar(scatter, cax=cbax, orientation='horizontal')
        cbar.set_label('$-$log$_{10}$(padj)', fontsize=5, labelpad=2)
        cbar.ax.tick_params(labelsize=4)

    fig.suptitle(title, fontsize=11, fontweight='bold', color=C_TEXT, y=1.01)
    fig.text(0.5, -0.01,
             'Top GOBP terms per cell type (enrichGO on rescued-to-normal genes overlapping cell-type markers).\n'
             'Near-duplicate terms removed. snRNA-seq marker overlap: exploratory.',
             ha='center', fontsize=6.5, color=C_TEXT2, linespacing=1.4)

    plt.tight_layout(rect=[0, 0.03, 1, 0.98])
    save_fig(fig, filename)
    print(f'Done — {filename}')

# ── Kidney: key cell types ───────────────────────────────────────
k_base = f"{DATA_BASE}/Kidney/2_Enrichment_bulk&sn/GOBP_per_celltype/rescued"
k_celltypes = [
    ('EC-1(gEC)', f'{k_base}/ECs/gobp_rescued_EC-1(gEC).csv'),
    ('EC-2', f'{k_base}/ECs/gobp_rescued_EC-2.csv'),
    #('EC-3', f'{k_base}/ECs/gobp_rescued_EC-3.csv'),
    #('EC-4', f'{k_base}/ECs/gobp_rescued_EC-4.csv'),
    #('EC-5', f'{k_base}/ECs/gobp_rescued_EC-5.csv'),
    #('EC-6', f'{k_base}/ECs/gobp_rescued_EC-6.csv'),
    ('IMM',  f'{k_base}/IMM/gobp_rescued_IMM.csv'),
    ('PODO', f'{k_base}/Podo/gobp_rescued_PODO.csv'),
    ('FIB',  f'{k_base}/Rest/gobp_rescued_FIB.csv'),
]

plot_ct_enrichment(k_celltypes,
    'Fig. 3.4D — Kidney: GOBP enrichment of rescued genes per cell type',
    FILENAME_K, figsize=(22, 8))

# ── Muscle: key cell types ───────────────────────────────────────
m_base = f"{DATA_BASE}/Muscle_old/2_Enrichment_tables/GOBP_per_celltype/rescued"
m_celltypes = [
    ('Macrophages', f'{m_base}/IMM/gobp_rescued_Macrophages.csv'),
    ('Immune cells?', f'{m_base}/IMM/gobp_rescued_Immune cells_.csv'),
    ('NMJ nuclei',  f'{m_base}/NMJs/gobp_rescued_Myonuclei (neuromuscular junction nuclei).csv'),
    ('Myonuclei\n(Myh4+)',  f'{m_base}/Rest/gobp_rescued_Myonuclei (Myh4+_).csv'),
    ('Endothelial',  f'{m_base}/ECs/gobp_rescued_Endothelial cells.csv'),
]

plot_ct_enrichment(m_celltypes,
    'Fig. 3.4H — Muscle: GOBP enrichment of rescued genes per cell type',
    FILENAME_M, figsize=(20, 8))

## Fig 3.4I — Cross-tissue rescue cell-type comparison

In [ ]:
# ── Fig 3.4I — Cross-tissue rescue cell types ────────────────────
# Consistent font sizes + black text throughout
FILENAME = 'Fig_3_4I_cross_tissue_rescue_celltypes'

fig, ax = plt.subplots(figsize=(13, 10))
ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.axis('off')

def rbox(cx, cy, w, h, fc, ec, title, sub=None, fs=10, fs_sub=7.5, lw=0.8):
    ax.add_patch(FancyBboxPatch((cx-w/2, cy-h/2), w, h, boxstyle="round,pad=0.012",
                                 facecolor=fc, edgecolor=ec, linewidth=lw,
                                 transform=ax.transAxes, zorder=2))
    if sub:
        ax.text(cx, cy+0.009, title, transform=ax.transAxes, ha='center', va='center',
                fontsize=fs, fontweight='bold', color=ec, zorder=3)
        ax.text(cx, cy-0.009, sub, transform=ax.transAxes, ha='center', va='center',
                fontsize=fs_sub, color=C_TEXT, zorder=3)          # ← black
    else:
        ax.text(cx, cy, title, transform=ax.transAxes, ha='center', va='center',
                fontsize=fs, fontweight='bold', color=ec, zorder=3)

def hline(x1, y1, x2, y2, color, lw=0.7, alpha=0.5):
    ax.plot([x1, x2], [y1, y2], color=color, lw=lw, alpha=alpha,
            transform=ax.transAxes, zorder=1)

kx = 0.15;  mx = 0.85;  cx = 0.50
bw = 0.22;  bh = 0.042; step = 0.058

# ── Title ─────────────────────────────────────────────────────────
ax.text(0.50, 0.97, 'Fig. 3.4I \u2014 Cross-tissue comparison of rescue-driving cell types',
        ha='center', fontsize=13, fontweight='bold', color=C_TEXT, transform=ax.transAxes)

# ── Tissue headers ─────────────────────────────────────────────────
rbox(kx, 0.91, 0.26, 0.050, C_RED,  C_RED_D,  'KIDNEY',
     '7,499 aging DEGs \u2192 1,961 rescued-to-normal', fs=13, fs_sub=8)
rbox(mx, 0.91, 0.26, 0.050, C_BLUE, C_BLUE_D, 'MUSCLE',
     '1,739 aging DEGs \u2192 511 rescued-to-normal',   fs=13, fs_sub=8)

# ── Kidney cell types ──────────────────────────────────────────────
k_cells = [
    ('PT-2 (119)',     'Gstm1, Acox2, Rdh16',       C_TEAL,   C_TEAL_D),
    ('FIB (101)',      'Flt1, Prkd1, ECM genes',     C_TEAL,   C_TEAL_D),
    ('vSMC/MC (85)',   'Prkd1, vascular tone',       C_TEAL,   C_TEAL_D),
    ('EC-2/3/4 (122)', 'Tgfb1, Flt1, B2m',          C_GREEN,  C_GREEN_D),
    ('IMM (51)',        'Tgfb1, Hcls1, Pld4, Nrros', C_PURPLE, C_PURP_D),
    ('PODO (43)',       'Prkd1, Plekho1',             C_TEAL,   C_TEAL_D),
]
ky_start = 0.84
k_ys = []
for i, (name, genes, fc, ec) in enumerate(k_cells):
    y = ky_start - i * step
    k_ys.append(y)
    rbox(kx, y, bw, bh, fc, ec, name, genes)

# ── Muscle cell types ──────────────────────────────────────────────
m_cells = [
    ('Myonuclei Myh4+ (64)', 'Kcnq5, Wwox',               C_TEAL,   C_TEAL_D),
    ('Macrophages (34)',      'Tgfb1, Hcls1, Pld4',        C_PURPLE, C_PURP_D),
    ('Myonuclei diff. (29)', 'Kcnq5, regeneration',        C_TEAL,   C_TEAL_D),
    ('NMJ nuclei (25)',      'Ryr3, Kcnq5, Ptprm, Wwox',  C_AMBER,  C_AMBER_D),
    ('Muscle stem (23)',     'Ryr3, Pxdn',                  C_TEAL,   C_TEAL_D),
    ('Endothelial (20)',     'Minimal response',            C_GRAY,   C_GRAY_D),
]
m_ys = []
for i, (name, genes, fc, ec) in enumerate(m_cells):
    y = ky_start - i * step
    m_ys.append(y)
    rbox(mx, y, bw, bh, fc, ec, name, genes)

# ── Center: shared themes ──────────────────────────────────────────
shared = [
    ('Tgfb1 signalling',     'Top upstream regulator in both tissues', C_CORAL,  C_CORAL_D, 0.80),
    ('Immune rebalancing',   'Hcls1, Pld4, Nrros, Ly6e shared',       C_PURPLE, C_PURP_D,  0.73),
    ('12 shared marker genes','Reversal markers in both tissues',      C_AMBER,  C_AMBER_D, 0.66),
    ('DR as dominant driver', 'K: 28% | M: 47% DR-exclusive',         C_GREEN,  C_GREEN_D, 0.59),
]
for label, sub, fc, ec, y in shared:
    rbox(cx, y, 0.22, 0.045, fc, ec, label, sub, fs=9, fs_sub=7)

# ── Connectors ─────────────────────────────────────────────────────
hline(kx+bw/2, k_ys[3], cx-0.11, 0.80, C_CORAL_D)
hline(kx+bw/2, k_ys[4], cx-0.11, 0.80, C_CORAL_D)
hline(mx-bw/2, m_ys[1], cx+0.11, 0.80, C_CORAL_D)
hline(kx+bw/2, k_ys[4], cx-0.11, 0.73, C_PURP_D)
hline(mx-bw/2, m_ys[1], cx+0.11, 0.73, C_PURP_D)

# ── Divider ────────────────────────────────────────────────────────
ax.plot([0.05, 0.95], [0.50, 0.50], color='#E0E0E0', lw=0.5,
        transform=ax.transAxes, zorder=0)

# ── Tissue-specific highlights ─────────────────────────────────────
hl_y = 0.47
ax.text(kx, hl_y, 'Kidney-specific rescue features', ha='center',
        fontsize=10, fontweight='bold', color=C_RED_D, transform=ax.transAxes)
k_hl = [
    'EC-driven vascular rescue: Flt1 in 7 EC subtypes + FIB',
    'Broadly distributed: 23 cell types contribute markers',
    'Podocyte-specific: Prkd1 (Golgi), Plekho1',
    'B2m (senescence marker) in IMM + EC-3',
    'ECM remodelling centred on FIB + vSMC/MC',
]
for i, t in enumerate(k_hl):
    ax.text(kx, hl_y-0.03-i*0.028, '\u2022  '+t, ha='center',
            fontsize=8, color=C_TEXT, transform=ax.transAxes)    # ← black

ax.text(mx, hl_y, 'Muscle-specific rescue features', ha='center',
        fontsize=10, fontweight='bold', color=C_BLUE_D, transform=ax.transAxes)
m_hl = [
    'NMJ long-gene vulnerability: Ryr3, Kcnq5, Wwox (>500 kb)',
    'Concentrated: top 3 cell types carry 73% of markers',
    'Myonuclei fibre-type maintenance (Myh4+)',
    'ECs transcriptionally quiescent (contrast to kidney)',
    'DR disproportionately dominant (47% vs 28%)',
]
for i, t in enumerate(m_hl):
    ax.text(mx, hl_y-0.03-i*0.028, '\u2022  '+t, ha='center',
            fontsize=8, color=C_TEXT, transform=ax.transAxes)    # ← black

# ── Key finding box ────────────────────────────────────────────────
fy = 0.155
ax.add_patch(FancyBboxPatch((0.12, fy-0.04), 0.76, 0.08,
             boxstyle="round,pad=0.015", facecolor='#F8F7F5',
             edgecolor=C_GRAY_D, linewidth=0.8,
             transform=ax.transAxes, zorder=2))
ax.text(0.50, fy+0.02, 'Key finding', ha='center', fontsize=10,
        fontweight='bold', color=C_TEXT, transform=ax.transAxes, zorder=3)
ax.text(0.50, fy-0.01,
        'Immune cells (IMM/macrophages) and Tgfb1 signalling are rescue hubs in both tissues.\n'
        'Kidney rescue is vascular/stromal (ECs, FIB); muscle rescue is neuromuscular (NMJ, myonuclei).\n'
        'ECs are major rescue contributors in kidney but transcriptionally silent in muscle.',
        ha='center', va='center', fontsize=8, color=C_TEXT,        # ← black
        transform=ax.transAxes, zorder=3, linespacing=1.5)

# ── Footnote ───────────────────────────────────────────────────────
ax.text(0.50, 0.05,
        'Marker counts from FindAllMarkers (Seurat) intersected with bulk rescued-to-normal gene sets.\n'
        'Numbers in parentheses = reversal gene markers per cell type. EC-2/3/4 counts combined for kidney.',
        ha='center', fontsize=7.5, color=C_TEXT, transform=ax.transAxes)  # ← black

plt.tight_layout(pad=0.5)
save_fig(fig, FILENAME)
print('Done \u2014 Fig 3.4I')

## Fig. 3.4J - NMJ long genes directionality check

In [ ]:
FILENAME = 'Fig_3_4J_NMJ_long_gene_directionality'

# ── Data ─────────────────────────────────────────────────────────
genes = [
    ('Ptprm',  1148, -0.605, +0.388, -0.217, 'Receptor-type PTPase'),
    ('Wwox',    836, -0.531, +0.054, -0.477, 'WW domain oxidoreductase'),
    ('Ryr3',    791, -1.040, +1.259, +0.218, 'Ryanodine receptor 3'),
    ('Kcnq5',   588, +0.820, -1.224, -0.404, 'Voltage-gated K+ channel'),
    ('Nav3',    570, +0.711, -0.619, +0.093, 'Neuron navigator 3'),
]
genes.sort(key=lambda x: x[1], reverse=True)

syms     = [g[0] for g in genes]
lengths  = [g[1] for g in genes]
aging    = [g[2] for g in genes]
interv   = [g[3] for g in genes]
combi_c  = [g[4] for g in genes]
funcs    = [g[5] for g in genes]

n = len(genes)
y = np.arange(n)

# ── Figure ───────────────────────────────────────────────────────
fig, (ax_main, ax_len) = plt.subplots(1, 2, figsize=(12, 5),
                                       gridspec_kw={'width_ratios': [2.8, 1], 'wspace': 0.35})

bar_h = 0.28

# Aging bars (top of each row)
for i in range(n):
    color = C_RED_D if aging[i] < 0 else C_CORAL_D
    ax_main.barh(y[i] + bar_h/2, aging[i], height=bar_h, color=color, alpha=0.7,
                  edgecolor=color, linewidth=0.5, zorder=2)

# Intervention bars (bottom of each row)
for i in range(n):
    color = C_TEAL_D if (aging[i] < 0 and interv[i] > 0) or (aging[i] > 0 and interv[i] < 0) else C_GRAY_D
    ax_main.barh(y[i] - bar_h/2, interv[i], height=bar_h, color=color, alpha=0.7,
                  edgecolor=color, linewidth=0.5, zorder=2)

# Combi vs ctrl markers
for i in range(n):
    ax_main.plot(combi_c[i], y[i], 'D', color=C_PURP_D, markersize=7,
                  markeredgecolor='white', markeredgewidth=0.8, zorder=4)

# Zero line and rescue zone
ax_main.axvline(0, color=C_GRAY_D, lw=0.6, zorder=1)
ax_main.axvspan(-0.5, 0.5, color=C_GREEN, alpha=0.08, zorder=0)
ax_main.text(0, n + 0.1, '|LFC| < 0.5 (rescue zone)', ha='center', va='bottom',
             fontsize=6, color=C_GREEN_D, fontstyle='italic')

# Y labels with direction arrows
labels = []
for i in range(n):
    arrow = '\u25BC' if aging[i] < 0 else '\u25B2'
    labels.append(f'{syms[i]}  {arrow}')
ax_main.set_yticks(y)
ax_main.set_yticklabels(labels, fontsize=9.5, fontweight='bold')

ax_main.set_xlabel('log2 fold change', fontsize=10, fontweight='bold')
ax_main.set_title('NMJ long-gene expression changes', fontsize=11,
                    fontweight='bold', color=C_TEXT, pad=10)
ax_main.spines['top'].set_visible(False)
ax_main.spines['right'].set_visible(False)
ax_main.spines['left'].set_linewidth(0.5)
ax_main.spines['bottom'].set_linewidth(0.5)
ax_main.tick_params(axis='y', length=0)
ax_main.set_xlim(-1.55, 1.55)

# Legend
legend_items = [
    mpatches.Patch(facecolor=C_RED_D, alpha=0.7, label='Aging LFC (down with aging)'),
    mpatches.Patch(facecolor=C_CORAL_D, alpha=0.7, label='Aging LFC (up with aging)'),
    mpatches.Patch(facecolor=C_TEAL_D, alpha=0.7, label='Intervention LFC (combi vs age)'),
    plt.Line2D([0], [0], marker='D', color='w', markerfacecolor=C_PURP_D,
               markersize=7, markeredgecolor='white', label='Net effect (combi vs ctrl)'),
]
ax_main.legend(handles=legend_items, loc='lower left', fontsize=6,
               frameon=True, framealpha=0.9, edgecolor='#E0E0E0',
               handlelength=1.5, borderpad=0.4)

# ── RIGHT: Gene length + function ────────────────────────────────
ax_len.barh(y, lengths, height=0.55, color=C_AMBER, edgecolor=C_AMBER_D,
            linewidth=0.5, alpha=0.75, zorder=2)

for i in range(n):
    ax_len.text(lengths[i] + 20, y[i] + 0.08, f'{lengths[i]:,} kb', va='center', ha='left',
                fontsize=7.5, fontweight='bold', color=C_AMBER_D)
    ax_len.text(lengths[i] + 20, y[i] - 0.15, funcs[i], va='center', ha='left',
                fontsize=5.5, color=C_TEXT2, fontstyle='italic')

ax_len.set_xlabel('Gene length (kb)', fontsize=9, fontweight='bold')
ax_len.set_title('Genomic span', fontsize=10, fontweight='bold', color=C_TEXT, pad=10)
ax_len.set_yticks([])
ax_len.set_xlim(0, 1650)
ax_len.axvline(500, color=C_GRAY_D, lw=0.6, ls='--', zorder=1)
ax_len.text(510, n - 0.1, '500 kb', ha='left', va='bottom', fontsize=6, color=C_GRAY_D)
ax_len.spines['top'].set_visible(False)
ax_len.spines['right'].set_visible(False)
ax_len.spines['left'].set_visible(False)
ax_len.spines['bottom'].set_linewidth(0.5)

# ── Suptitle ─────────────────────────────────────────────────────
fig.suptitle('Fig. 3.4J \u2014 NMJ-associated long genes: aging directionality and rescue',
             fontsize=11, fontweight='bold', color=C_TEXT, y=1.02)

fig.text(0.5, -0.06,
         '3/5 NMJ long genes (Ryr3, Ptprm, Wwox) are downregulated with aging \u2014 consistent with transcription stress on long genes.\n'
         '2/5 (Kcnq5, Nav3) are upregulated, suggesting compensatory or alternative mechanisms.\n'
         'All 5 are rescued-to-normal by the combined intervention (purple diamonds within green rescue zone).\n'
         'All genes are NMJ nuclei markers in muscle snRNA-seq and exceed 500 kb genomic span.',
         ha='center', fontsize=6.5, color=C_TEXT2, linespacing=1.5)

plt.tight_layout(rect=[0, 0.05, 1, 0.98])
save_fig(fig, FILENAME)
print('Done \u2014 Fig 3.4J')

#3.5 Cell–Cell Communication Networks and Rescue Signaling

In [ ]:
# ── Restore plot_pathway_heatmap ──────────────────────────────────

# Ensure these are defined in case the session was restarted
conditions  = ['ctrl', 'age', 'DR', 'sAct', 'combi']
cond_labels = ['Ctrl', 'Age', 'DR', 'sAct', 'Combi']
cond_colors = ['#888888', C_RED_D, C_GREEN_D, C_PURP_D, C_TEAL_D]

def load_all_lr(base_path, suffix=''):
    data = {}
    for cond in conditions:
        fpath = f"{base_path}/{cond}_lr_interactions_reversal_genes{suffix}.csv"
        with open(fpath) as f:
            data[cond] = list(csv.DictReader(f))
    return data

def get_pathway_pairs(lr_data, pathway):
    all_pairs = set()
    pair_probs = defaultdict(lambda: defaultdict(float))
    for cond in conditions:
        for r in lr_data[cond]:
            if r['pathway_name'] == pathway:
                pair = (r['source'], r['target'])
                all_pairs.add(pair)
                pair_probs[pair][cond] += float(r['prob'])
    pairs_sorted = sorted(all_pairs,
                           key=lambda p: sum(pair_probs[p].values()), reverse=True)
    return pairs_sorted, pair_probs

def plot_pathway_heatmap(ax, lr_data, pathway, title, max_pairs=12):
    pairs, probs = get_pathway_pairs(lr_data, pathway)
    pairs = pairs[:max_pairs]

    if len(pairs) == 0:
        ax.text(0.5, 0.5, f'No interactions\nfor {pathway}', ha='center', va='center',
                transform=ax.transAxes, fontsize=9, color=C_TEXT2)
        ax.set_title(title, fontsize=9, fontweight='bold', color=C_TEXT)
        return

    n_pairs = len(pairs)
    n_cond  = len(conditions)

    mat = np.zeros((n_pairs, n_cond))
    for i, pair in enumerate(pairs):
        for j, cond in enumerate(conditions):
            mat[i, j] = probs[pair].get(cond, 0)

    cmap    = mcolors.LinearSegmentedColormap.from_list('w2teal',
              ['#FFFFFF','#D0EDE2','#6BBFB0','#0F6E56','#0A4A3A'])
    max_val = mat.max() if mat.max() > 0 else 1

    for i in range(n_pairs):
        for j in range(n_cond):
            val = mat[i, j]
            if val > 0:
                size  = 20 + (val / max_val) * 180
                color = cmap(val / max_val)
                ax.scatter(j, n_pairs - 1 - i, s=size, c=[color],
                           edgecolors=C_TEAL_D, linewidths=0.3, zorder=3)

    # ── CHANGED: apply clean_name to source and target labels ─────
    pair_labels = [f'{clean_name(s)}\u2192{clean_name(t)}' for s, t in pairs]
    ax.set_yticks(range(n_pairs))
    ax.set_yticklabels(pair_labels[::-1], fontsize=9)
    ax.set_xticks(range(n_cond))
    ax.set_xticklabels(cond_labels, fontsize=8, fontweight='bold')
    ax.xaxis.set_ticks_position('top')
    ax.set_xlim(-0.5, n_cond - 0.5)
    ax.set_ylim(-0.5, n_pairs - 0.5)

    for i in range(n_pairs):
        ax.axhline(i - 0.5, color='#F0F0F0', lw=0.5, zorder=0)
    for j in range(n_cond):
        ax.axvline(j - 0.5, color='#F0F0F0', lw=0.5, zorder=0)

    ax.set_title(title, fontsize=9, fontweight='bold', color=C_TEXT, pad=15)
    ax.spines['top'].set_visible(False);  ax.spines['right'].set_visible(False)
    ax.spines['left'].set_linewidth(0.3); ax.spines['bottom'].set_linewidth(0.3)
    ax.tick_params(axis='y', length=0);   ax.tick_params(axis='x', length=0)

# Load kidney LR data if not already loaded
if 'k_lr' not in dir() or not k_lr:
    k_lr = {}
    for cond in conditions:
        with open(f"{DATA_BASE}/Kidney/5_Cellchat_LR_analysis/{cond}_lr_interactions_reversal_genes.csv") as f:
            k_lr[cond] = list(csv.DictReader(f))
    print(f"k_lr loaded: {list(k_lr.keys())}")

# Load muscle LR data — use v2
if 'm_lr' not in dir() or not m_lr:
    m_lr = {}
    for cond in conditions:
        with open(f"{MUSCLE_V2}/{cond}_lr_interactions_reversal_genes_v2.csv") as f:
            m_lr[cond] = list(csv.DictReader(f))
    print(f"m_lr loaded: {list(m_lr.keys())}")

print("plot_pathway_heatmap restored ✓")

In [ ]:
# ── Fig 3.5B-circles ─────────────────────────────────────────────
FILENAME_KB = 'Fig_3_5B_kidney_network_circles'

from matplotlib.patches import FancyArrowPatch

# ── Load LR data ────────────────────────────────────────────────
k_lr = {}
for cond in ['ctrl', 'age', 'DR', 'sAct', 'combi']:
    with open(f"{DATA_BASE}/Kidney/5_Cellchat_LR_analysis/{cond}_lr_interactions_reversal_genes.csv") as f:
        k_lr[cond] = list(csv.DictReader(f))
    print(f"{cond}: {len(k_lr[cond])} LR interactions")

# ── Cell types (CLEANED) ────────────────────────────────────────
k_celltypes = sorted(
    set(r['source'].strip() for cond in k_lr for r in k_lr[cond]) |
    set(r['target'].strip() for cond in k_lr for r in k_lr[cond])
)

print(f"Total kidney cell types: {len(k_celltypes)}")
print("Example cell types:", k_celltypes[:10])

# ── Colors ──────────────────────────────────────────────────────
k_ct_colors = {}
k_palette = [
    '#E8918D', '#E8A86B', '#D4B56A', '#A3BE8C', '#7DB89E',
    '#6BBFB0', '#5EAAA8', '#6BA3C7', '#7E94C0', '#9B8EC2',
    '#B68CB5', '#C98BA3', '#D4A19A', '#BDB5A0', '#A0C4A8',
]

for i, ct in enumerate(k_celltypes):
    k_ct_colors[ct] = k_palette[i % len(k_palette)]

# ── Build adjacency (FIXED CLEANING) ────────────────────────────
def build_adjacency_general(rows, cell_types):
    n = len(cell_types)
    ct_idx = {ct: i for i, ct in enumerate(cell_types)}

    mat = np.zeros((n, n))

    for r in rows:
        s = r['source'].strip()
        t = r['target'].strip()

        if s in ct_idx and t in ct_idx:
            try:
                mat[ct_idx[s], ct_idx[t]] += float(r['prob'])
            except:
                continue

    return mat

# ── Debug wrapper (IMPORTANT) ───────────────────────────────────
def debug_matrix(mat, cond):
    print(f"{cond}: max={mat.max():.6f}, sum={mat.sum():.6f}, nonzero={(mat>0).sum()}")

# ── Plot function (same, but threshold passed explicitly) ───────
def draw_network_single(ax, mat, cell_types, title, colors,
                        max_edge_width=4, prob_threshold=0.005, label_size=5):

    n = len(cell_types)

    angles = np.linspace(np.pi/2, -3*np.pi/2, n, endpoint=False)
    node_x = np.cos(angles)
    node_y = np.sin(angles)

    max_prob = mat.max() if mat.max() > 0 else 1

    # ── edges ─────────────────────────
    for i in range(n):
        for j in range(n):
            if mat[i, j] < prob_threshold:
                continue

            width = 0.2 + (mat[i, j] / max_prob) * max_edge_width
            alpha = 0.1 + (mat[i, j] / max_prob) * 0.5
            color = colors.get(cell_types[i], C_GRAY_D)

            ax.add_patch(FancyArrowPatch(
                (node_x[i], node_y[i]),
                (node_x[j], node_y[j]),
                connectionstyle="arc3,rad=0.25",
                arrowstyle='->',
                mutation_scale=7,
                color=color,
                alpha=alpha,
                linewidth=width,
                zorder=1
            ))

    # ── nodes ─────────────────────────
    for i, ct in enumerate(cell_types):
        color = colors.get(ct, C_GRAY_D)

        ax.scatter(node_x[i], node_y[i], s=350, c=color,
                   edgecolors='white', linewidths=1, zorder=3)

        ax.text(node_x[i]*1.35, node_y[i]*1.35, clean_name(ct),
                ha='center', va='center', fontsize=label_size,
                fontweight='bold', color=color)

    total = mat.sum()
    n_int = (mat > prob_threshold).sum()

    ax.text(0, -1.65, f'{n_int} int.\nΣ={total:.3f}',
            ha='center', fontsize=10, fontweight='bold', color=C_TEXT2)

    ax.set_title(title)
    ax.set_xlim(-1.8, 1.8)
    ax.set_ylim(-1.85, 1.6)
    ax.set_aspect('equal')
    ax.axis('off')

# ── FIGURE ──────────────────────────────────────────────────────
fig_k, axes_k = plt.subplots(1, 5, figsize=(22, 5.5))
cond_titles = ['Ctrl', 'Age', 'DR', 'sAct', 'Combi']

for ci, cond in enumerate(['ctrl','age','DR','sAct','combi']):

    mat = build_adjacency_general(k_lr[cond], k_celltypes)

    # 🔥 DEBUG OUTPUT (THIS IS KEY)
    debug_matrix(mat, cond)

    draw_network_single(
        axes_k[ci],
        mat,
        k_celltypes,
        cond_titles[ci],
        k_ct_colors,
        prob_threshold=0.005,
        label_size=4.5
    )

fig_k.suptitle('Kidney intra-tissue communication networks across conditions')
plt.tight_layout()
save_fig(fig_k, FILENAME_KB)

print("Done — Fig 3.5B kidney circles")

## Fig 3.5A/D -- cellchat_interaction_overview

In [ ]:
FILENAME = 'Fig_3_5AD_cellchat_interaction_overview'

conditions = ['ctrl', 'age', 'DR', 'sAct', 'combi']
cond_labels = ['Ctrl', 'Age', 'DR', 'sAct', 'Combi']
cond_colors = ['#888888', C_RED_D, C_GREEN_D, C_PURP_D, C_TEAL_D]

def load_lr_stats(base_path, suffix=''):
    """Load interaction count, total prob, unique pairs, pathways per condition."""
    stats = {}
    for cond in conditions:
        fpath = f"{base_path}/{cond}_lr_interactions_reversal_genes{suffix}.csv"
        n_interactions = 0
        total_prob = 0
        pairs = set()
        pathways = set()
        with open(fpath) as f:
            for r in csv.DictReader(f):
                n_interactions += 1
                total_prob += float(r['prob'])
                pairs.add((r['source'], r['target']))
                pathways.add(r['pathway_name'])
        stats[cond] = {
            'n_interactions': n_interactions,
            'total_prob': total_prob,
            'n_pairs': len(pairs),
            'n_pathways': len(pathways),
        }
    return stats

k_stats = load_lr_stats(f"{DATA_BASE}/Kidney/5_Cellchat_LR_analysis")
m_stats = load_lr_stats(f"{DATA_BASE}/Muscle_old/5_Cellchat_LR_analysis")

# Print summary
for label, stats in [('Kidney', k_stats), ('Muscle', m_stats)]:
    print(f"\n{label}:")
    for c in conditions:
        s = stats[c]
        print(f"  {c:6s}: {s['n_interactions']:>5d} interactions, prob={s['total_prob']:.1f}, {s['n_pairs']} pairs, {s['n_pathways']} pathways")

# ── Figure: 2×2 grid ─────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(13, 9), gridspec_kw={'hspace': 0.35, 'wspace': 0.3})

def plot_overview(ax_count, ax_prob, stats, tissue):
    x = np.arange(len(conditions))
    w = 0.55

    # Top: interaction count
    counts = [stats[c]['n_interactions'] for c in conditions]
    bars = ax_count.bar(x, counts, width=w, color=cond_colors, alpha=0.7,
                         edgecolor=[c for c in cond_colors], linewidth=0.5)
    for i, (v, c) in enumerate(zip(counts, conditions)):
        ax_count.text(i, v + max(counts)*0.02, str(v), ha='center', va='bottom',
                      fontsize=8, fontweight='bold', color=cond_colors[i])

    ax_count.set_xticks(x)
    ax_count.set_xticklabels(cond_labels, fontsize=9, fontweight='bold')
    ax_count.set_ylabel('Number of LR interactions', fontsize=9, fontweight='bold')
    ax_count.set_title(f'{tissue} — interaction count', fontsize=11,
                        fontweight='bold', color=C_TEXT, pad=10)
    ax_count.spines['top'].set_visible(False)
    ax_count.spines['right'].set_visible(False)

    # Annotate pathways (moved slightly left to avoid Combi bar)
    n_pw = [stats[c]['n_pathways'] for c in conditions]
    ax_count.text(0.94, 0.95, f'Pathways: {min(n_pw)}\u2013{max(n_pw)}',
                  transform=ax_count.transAxes, ha='right', va='top',
                  fontsize=7, color=C_TEXT2, fontstyle='italic')

    # Bottom: total probability (interaction strength)
    probs = [stats[c]['total_prob'] for c in conditions]
    bars2 = ax_prob.bar(x, probs, width=w, color=cond_colors, alpha=0.5,
                         edgecolor=cond_colors, linewidth=0.5)
    for i, v in enumerate(probs):
        ax_prob.text(i, v + max(probs)*0.02, f'{v:.1f}', ha='center', va='bottom',
                     fontsize=8, fontweight='bold', color=cond_colors[i])

    ax_prob.set_xticks(x)
    ax_prob.set_xticklabels(cond_labels, fontsize=9, fontweight='bold')
    ax_prob.set_ylabel('Total interaction probability', fontsize=9, fontweight='bold')
    ax_prob.set_title(f'{tissue} — interaction strength', fontsize=11,
                       fontweight='bold', color=C_TEXT, pad=10)
    ax_prob.spines['top'].set_visible(False)
    ax_prob.spines['right'].set_visible(False)

    # Aging change annotation (moved above like in other figure)
    ctrl_p, age_p, combi_p = probs[0], probs[1], probs[4]
    if age_p != ctrl_p:
        change = ((age_p - ctrl_p) / ctrl_p * 100)
        direction = '\u2191' if change > 0 else '\u2193'
        ax_prob.annotate(f'{direction} {abs(change):.0f}% with aging',
                         xy=(1, age_p), xytext=(1.5, age_p + max(probs)*0.14),
                         fontsize=7, color=C_RED_D, fontweight='bold',
                         arrowprops=dict(arrowstyle='->', color=C_RED_D, lw=1))

plot_overview(axes[0, 0], axes[0, 1], k_stats, 'Kidney')
plot_overview(axes[1, 0], axes[1, 1], m_stats, 'Muscle')

fig.suptitle('Fig. 3.5A/D — CellChat interaction overview: kidney and muscle',
             fontsize=11, fontweight='bold', color=C_TEXT, y=1.01)

fig.text(0.5, -0.01,
         'LR interactions filtered to reversal genes (rescued-to-normal gene set).\n'
         'Left: number of significant LR interactions per condition. Right: summed interaction probability (strength).\n'
         'CellChat analysis: exploratory (n=1 per condition).',
         ha='center', fontsize=6.5, color=C_TEXT2, linespacing=1.4)

plt.tight_layout(rect=[0, 0.04, 1, 0.98])
save_fig(fig, FILENAME)
print('Done — Fig 3.5A/D')

## Fig 3.5B/E pathway communication (kidney, muscle)

In [ ]:
# ── KIDNEY: Fig 3.5B — 2-row layout ──────────────────────────────
k_pathways = ['VEGF', 'SEMA3', 'EGF', 'EPHB', 'PDGF']
n_cols = 3
n_rows = int(np.ceil(len(k_pathways) / n_cols))

fig_k, axes_k = plt.subplots(n_rows, n_cols,
                               figsize=(n_cols * 5.5, n_rows * 7),
                               gridspec_kw={'hspace': 0.5, 'wspace': 0.5})
axes_k_flat = axes_k.flatten()

for i, pw in enumerate(k_pathways):
    plot_pathway_heatmap(axes_k_flat[i], k_lr, pw, pw, max_pairs=15)

# Hide empty axes
for i in range(len(k_pathways), len(axes_k_flat)):
    axes_k_flat[i].axis('off')

fig_k.suptitle('Fig. 3.5B — Kidney: cell-type communication per pathway across conditions',
               fontsize=11, fontweight='bold', color=C_TEXT, y=1.01)
fig_k.text(0.5, -0.01,
           'Dot size and colour = summed LR interaction probability for each source\u2192target pair.\n'
           'Top 12\u201315 cell-type pairs shown per pathway, ranked by total probability across conditions.\n'
           'CellChat on reversal genes, exploratory (n=1/condition).',
           ha='center', fontsize=8, color=C_TEXT, linespacing=1.4)
plt.tight_layout(rect=[0, 0.04, 1, 0.98])
save_fig(fig_k, 'Fig_3_5B_kidney_pathway_communication')
print('Done — Fig 3.5B (kidney)')

# ── MUSCLE: Fig 3.5E — 2-row layout ──────────────────────────────
m_pathways = ['PTPRM', 'KIT', 'CDH5', 'PROS']
n_cols_m = 2
n_rows_m = int(np.ceil(len(m_pathways) / n_cols_m))

fig_m, axes_m = plt.subplots(n_rows_m, n_cols_m,
                               figsize=(n_cols_m * 6, n_rows_m * 7),
                               gridspec_kw={'hspace': 0.5, 'wspace': 0.5})
axes_m_flat = axes_m.flatten()

for i, pw in enumerate(m_pathways):
    plot_pathway_heatmap(axes_m_flat[i], m_lr, pw, pw, max_pairs=12)

for i in range(len(m_pathways), len(axes_m_flat)):
    axes_m_flat[i].axis('off')

fig_m.suptitle('Fig. 3.5E — Muscle: cell-type communication per pathway across conditions',
               fontsize=11, fontweight='bold', color=C_TEXT, y=1.01)
fig_m.text(0.5, -0.01,
           'Dot size and colour = summed LR interaction probability for each source\u2192target pair.\n'
           'Top cell-type pairs shown per pathway. CellChat on reversal genes, exploratory (n=1/condition).',
           ha='center', fontsize=8, color=C_TEXT, linespacing=1.4)
plt.tight_layout(rect=[0, 0.04, 1, 0.98])
save_fig(fig_m, 'Fig_3_5E_muscle_pathway_communication')
print('Done — Fig 3.5E (muscle)')

## Fig 3.5B/E — Kidney/Muscle CellChat network circles (overall + per pathway)

In [ ]:
# ── Fig 3.5B-circles — Kidney intra-tissue circular networks ─────
FILENAME_KB  = 'Fig_3_5B_kidney_network_circles'
FILENAME_KB2 = 'Fig_3_5B_kidney_network_by_pathway'
FILENAME_ME  = 'Fig_3_5E_muscle_network_circles'
FILENAME_ME2 = 'Fig_3_5E_muscle_network_by_pathway'

from matplotlib.patches import FancyArrowPatch

# ── Global font scale ─────────────────────────────────────────────
FS_LABEL    = 16   # node labels in circles
FS_STATS    = 20   # interaction count + sum below each circle
FS_TITLE    = 22   # condition titles
FS_PATHWAY  = 28   # pathway name on left side
FS_SUPTITLE = 23   # figure supertitle
FS_FOOTNOTE = 19   # bottom footnote text

k_lr = {}
for cond in conditions:
    with open(f"{DATA_BASE}/Kidney/5_Cellchat_LR_analysis/{cond}_lr_interactions_reversal_genes.csv") as f:
        k_lr[cond] = list(csv.DictReader(f))

k_celltypes = sorted(set(r['source'] for cond in k_lr for r in k_lr[cond]) |
                      set(r['target'] for cond in k_lr for r in k_lr[cond]))
k_ct_colors = {}
k_palette = ['#E8918D','#E8A86B','#D4B56A','#A3BE8C','#7DB89E',
             '#6BBFB0','#5EAAA8','#6BA3C7','#7E94C0','#9B8EC2',
             '#B68CB5','#C98BA3','#D4A19A','#BDB5A0','#A0C4A8',
             '#D4C5A0','#8FB8A0','#A89BC2','#C4A882','#9CB8D4']
for i, ct in enumerate(k_celltypes):
    k_ct_colors[ct] = k_palette[i % len(k_palette)]

def build_adjacency_general(rows, cell_types):
    n = len(cell_types)
    ct_idx = {ct: i for i, ct in enumerate(cell_types)}
    mat = np.zeros((n, n))
    for r in rows:
        s, t = r['source'], r['target']
        if s in ct_idx and t in ct_idx:
            mat[ct_idx[s], ct_idx[t]] += float(r['prob'])
    return mat

def build_adjacency_pathway(rows, cell_types, pathway_name):
    n = len(cell_types)
    ct_idx = {ct: i for i, ct in enumerate(cell_types)}
    mat = np.zeros((n, n))
    for r in rows:
        s, t = r['source'], r['target']
        p = r.get('pathway_name', r.get('pathway', ''))
        if s in ct_idx and t in ct_idx and p == pathway_name:
            mat[ct_idx[s], ct_idx[t]] += float(r['prob'])
    return mat

def draw_network_single(ax, mat, cell_types, title, colors,
                         max_edge_width=4, prob_threshold=0.01, label_size=FS_LABEL):
    n = len(cell_types)
    angles = np.linspace(np.pi/2, -3*np.pi/2, n, endpoint=False)
    node_x = [np.cos(a) for a in angles]
    node_y = [np.sin(a) for a in angles]
    max_prob = mat.max() if mat.max() > 0 else 1

    for i in range(n):
        for j in range(n):
            if mat[i, j] < prob_threshold: continue
            width = 0.2 + (mat[i, j] / max_prob) * max_edge_width
            alpha = 0.1 + (mat[i, j] / max_prob) * 0.5
            color = colors.get(cell_types[i], C_GRAY_D)
            ax.add_patch(FancyArrowPatch(
                (node_x[i], node_y[i]), (node_x[j], node_y[j]),
                connectionstyle="arc3,rad=0.25", arrowstyle='->',
                mutation_scale=9, color=color, alpha=alpha,
                linewidth=width, zorder=1))

    for i, ct in enumerate(cell_types):
        color = colors.get(ct, C_GRAY_D)
        ax.scatter(node_x[i], node_y[i], s=400, c=color,
                   edgecolors='white', linewidths=1.2, zorder=3)
        lx = node_x[i] * 1.38
        ly = node_y[i] * 1.38
        ha = 'right' if node_x[i] < -0.1 else ('left' if node_x[i] > 0.1 else 'center')

        # ── Shorten long cell type names ──────────────────────────
        short = short_name(ct)

        ax.text(lx, ly, short, ha=ha, va='center', fontsize=label_size,
                fontweight='bold', color=color,
                bbox=dict(boxstyle='round,pad=0.15', fc='white', ec=color,
                          lw=0.4, alpha=0.85))

    total = mat.sum()
    n_int = (mat > prob_threshold).sum()
    ax.text(0, -1.72, f'{n_int} int.\nΣ={total:.1f}',
            ha='center', fontsize=FS_STATS, fontweight='bold', color=C_TEXT)
    ax.set_title(title, fontsize=FS_TITLE, fontweight='bold', color=C_TEXT, pad=10)
    ax.set_xlim(-1.9, 1.9); ax.set_ylim(-1.95, 1.7)
    ax.set_aspect('equal'); ax.axis('off')

cond_titles = ['Ctrl', 'Age', 'DR', 'sAct', 'Combi']

# ── Kidney overall ────────────────────────────────────────────────
fig_k, axes_k = plt.subplots(1, 5, figsize=(34, 8))
for ci, cond in enumerate(conditions):
    mat = build_adjacency_general(k_lr[cond], k_celltypes)
    draw_network_single(axes_k[ci], mat, k_celltypes, cond_titles[ci], k_ct_colors)

fig_k.suptitle('Kidney intra-tissue communication networks across conditions',
               fontsize=FS_SUPTITLE, fontweight='bold', color=C_TEXT, y=0.985)
fig_k.text(0.5, 0.93, 'Reversal genes only',
           ha='center', fontsize=FS_FOOTNOTE+1, fontweight='bold', color=C_TEXT2)
fig_k.text(0.5, -0.01,
           'Arrows show LR interaction probability between kidney cell types (reversal gene set).\n'
           'Arrow thickness and opacity proportional to probability. Arrow colour = source cell type.\n'
           'CellChat: exploratory (n=1/condition).',
           ha='center', fontsize=FS_FOOTNOTE, color=C_TEXT, linespacing=1.4)
plt.tight_layout(rect=[0, 0.04, 1, 0.95])
save_fig(fig_k, FILENAME_KB)
print('Done — Fig 3.5B kidney circles')

# ── Kidney per pathway ────────────────────────────────────────────
k_pathways_top = ['VEGF', 'SEMA3', 'EGF', 'EPHA', 'EPHB', 'PDGF']
fig_kp, axes_kp = plt.subplots(len(k_pathways_top), 5,
                                 figsize=(34, len(k_pathways_top) * 6.5))

for pi, pathway in enumerate(k_pathways_top):
    all_mats = [build_adjacency_pathway(k_lr[c], k_celltypes, pathway)
                for c in conditions]
    global_max = max(m.max() for m in all_mats) if max(m.max() for m in all_mats) > 0 else 1

    for ci, cond in enumerate(conditions):
        ax = axes_kp[pi, ci]
        mat = all_mats[ci]
        n = len(k_celltypes)
        angles = np.linspace(np.pi/2, -3*np.pi/2, n, endpoint=False)
        node_x = [np.cos(a) for a in angles]
        node_y = [np.sin(a) for a in angles]

        prob_threshold = 0.005
        for i in range(n):
            for j in range(n):
                if mat[i, j] < prob_threshold: continue
                width = 0.3 + (mat[i, j] / global_max) * 4
                alpha = 0.15 + (mat[i, j] / global_max) * 0.55
                color = k_ct_colors.get(k_celltypes[i], C_GRAY_D)
                ax.add_patch(FancyArrowPatch(
                    (node_x[i], node_y[i]), (node_x[j], node_y[j]),
                    connectionstyle="arc3,rad=0.25", arrowstyle='->',
                    mutation_scale=9, color=color, alpha=alpha,
                    linewidth=width, zorder=1))

        for i, ct in enumerate(k_celltypes):
            color = k_ct_colors.get(ct, C_GRAY_D)
            activity = mat[i,:].sum() + mat[:,i].sum()
            node_size = 120 + (activity / max(global_max, 0.01)) * 350
            ax.scatter(node_x[i], node_y[i], s=node_size, c=color,
                       edgecolors='white', linewidths=0.8, zorder=3)
            if ci == 0:
                lx, ly = node_x[i]*1.42, node_y[i]*1.42
                ha = 'right' if node_x[i] < -0.1 else ('left' if node_x[i] > 0.1 else 'center')
                ax.text(lx, ly, short_name(ct), ha=ha, va='center', fontsize=FS_LABEL,
                        fontweight='bold', color=color)

        total = mat.sum()
        n_int = (mat > prob_threshold).sum()
        ax.text(0, -1.52, f'{n_int} int.\nΣ={total:.2f}',
                ha='center', fontsize=FS_STATS, fontweight='bold', color=C_TEXT)

        if pi == 0:
            ax.set_title(cond_titles[ci], fontsize=FS_TITLE,
                         fontweight='bold', color=C_TEXT, pad=10)
        if ci == 0:
            ax.text(-2.3, 0, pathway, ha='center', va='center',
                    fontsize=FS_PATHWAY, fontweight='bold', color=C_TEAL_D, rotation=90)

        ax.set_xlim(-1.9, 1.9); ax.set_ylim(-1.65, 1.6)
        ax.set_aspect('equal'); ax.axis('off')

fig_kp.suptitle('Kidney communication networks by pathway and condition',
                fontsize=FS_SUPTITLE, fontweight='bold', color=C_TEXT, y=0.985)
fig_kp.text(0.5, 0.955, 'Reversal genes only',
            ha='center', fontsize=FS_FOOTNOTE+1, fontweight='bold', color=C_TEXT2)
fig_kp.text(0.5, -0.01,
            'Each panel = one pathway × one condition. Node size = pathway activity.\n'
            'Arrow width = interaction probability. Scaling consistent within each pathway row.\n'
            'CellChat: exploratory (n=1/condition).',
            ha='center', fontsize=FS_FOOTNOTE, color=C_TEXT, linespacing=1.4)
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
save_fig(fig_kp, FILENAME_KB2)
print('Done — Fig 3.5B kidney pathway breakdown')

# ── Muscle overall ────────────────────────────────────────────────
m_lr = {}
for cond in conditions:
    with open(f"{MUSCLE_V2}/{cond}_lr_interactions_reversal_genes_v2.csv") as f:
        m_lr[cond] = list(csv.DictReader(f))

m_celltypes = sorted(set(r['source'] for cond in m_lr for r in m_lr[cond]) |
                      set(r['target'] for cond in m_lr for r in m_lr[cond]))
m_ct_colors = {}
m_palette = ['#6BA3C7','#7E94C0','#9B8EC2','#B68CB5','#C98BA3',
             '#D4A19A','#BDB5A0','#A0C4A8','#D4C5A0','#8FB8A0',
             '#A89BC2','#C4A882','#9CB8D4','#B0A8C0']
for i, ct in enumerate(m_celltypes):
    m_ct_colors[ct] = m_palette[i % len(m_palette)]

fig_m, axes_m = plt.subplots(1, 5, figsize=(34, 8))
for ci, cond in enumerate(conditions):
    mat = build_adjacency_general(m_lr[cond], m_celltypes)
    draw_network_single(axes_m[ci], mat, m_celltypes, cond_titles[ci], m_ct_colors)

fig_m.suptitle('Muscle intra-tissue communication networks across conditions',
               fontsize=FS_SUPTITLE, fontweight='bold', color=C_TEXT, y=0.985)
fig_m.text(0.5, 0.93, 'Reversal genes only',
           ha='center', fontsize=FS_FOOTNOTE+1, fontweight='bold', color=C_TEXT2)
fig_m.text(0.5, -0.01,
           'Arrows show LR interaction probability between muscle cell types (reversal gene set).\n'
           'Arrow thickness and opacity proportional to probability. Arrow colour = source cell type.\n'
           'CellChat: exploratory (n=1/condition).',
           ha='center', fontsize=FS_FOOTNOTE, color=C_TEXT, linespacing=1.4)
plt.tight_layout(rect=[0, 0.04, 1, 0.95])
save_fig(fig_m, FILENAME_ME)
print('Done — Fig 3.5E muscle circles')

# ── Muscle per pathway ────────────────────────────────────────────
m_pathways_top = ['PTPRM', 'KIT', 'CDH5', 'PROS']
fig_mp, axes_mp = plt.subplots(len(m_pathways_top), 5,
                                 figsize=(34, len(m_pathways_top) * 6.5))

for pi, pathway in enumerate(m_pathways_top):
    all_mats = [build_adjacency_pathway(m_lr[c], m_celltypes, pathway)
                for c in conditions]
    global_max = max(m.max() for m in all_mats) if max(m.max() for m in all_mats) > 0 else 1

    for ci, cond in enumerate(conditions):
        ax = axes_mp[pi, ci]
        mat = all_mats[ci]
        n = len(m_celltypes)
        angles = np.linspace(np.pi/2, -3*np.pi/2, n, endpoint=False)
        node_x = [np.cos(a) for a in angles]
        node_y = [np.sin(a) for a in angles]

        prob_threshold = 0.005
        for i in range(n):
            for j in range(n):
                if mat[i, j] < prob_threshold: continue
                width = 0.3 + (mat[i, j] / global_max) * 4
                alpha = 0.15 + (mat[i, j] / global_max) * 0.55
                color = m_ct_colors.get(m_celltypes[i], C_GRAY_D)
                ax.add_patch(FancyArrowPatch(
                    (node_x[i], node_y[i]), (node_x[j], node_y[j]),
                    connectionstyle="arc3,rad=0.25", arrowstyle='->',
                    mutation_scale=9, color=color, alpha=alpha,
                    linewidth=width, zorder=1))

        for i, ct in enumerate(m_celltypes):
            color = m_ct_colors.get(ct, C_GRAY_D)
            activity = mat[i,:].sum() + mat[:,i].sum()
            node_size = 170 + (activity / max(global_max, 0.01)) * 450
            ax.scatter(node_x[i], node_y[i], s=node_size, c=color,
                       edgecolors='white', linewidths=0.8, zorder=3)
            if ci == 0:
                short = short_name(ct)
                lx, ly = node_x[i]*1.42, node_y[i]*1.42
                ha = 'right' if node_x[i] < -0.1 else ('left' if node_x[i] > 0.1 else 'center')
                ax.text(lx, ly, short, ha=ha, va='center', fontsize=FS_LABEL,
                        fontweight='bold', color=color)

        total = mat.sum()
        n_int = (mat > prob_threshold).sum()
        ax.text(0, -1.72, f'{n_int} int.\nΣ={total:.2f}',
                ha='center', fontsize=FS_STATS, fontweight='bold', color=C_TEXT)

        if pi == 0:
            ax.set_title(cond_titles[ci], fontsize=FS_TITLE,
                         fontweight='bold', color=C_TEXT, pad=10)
        if ci == 0:
            ax.text(-2.3, 0, pathway, ha='center', va='center',
                    fontsize=FS_PATHWAY, fontweight='bold', color=C_TEAL_D, rotation=90)

        ax.set_xlim(-1.9, 1.9); ax.set_ylim(-1.8, 1.6)
        ax.set_aspect('equal'); ax.axis('off')

fig_mp.suptitle('Muscle communication networks by pathway and condition',
                fontsize=FS_SUPTITLE, fontweight='bold', color=C_TEXT, y=0.985)
fig_mp.text(0.5, 0.955, 'Reversal genes only',
            ha='center', fontsize=FS_FOOTNOTE+1, fontweight='bold', color=C_TEXT2)
fig_mp.text(0.5, -0.01,
            'Each panel = one pathway × one condition. Node size = pathway activity.\n'
            'Arrow width = interaction probability. Scaling consistent within each pathway row.\n'
            'CellChat: exploratory (n=1/condition).',
            ha='center', fontsize=FS_FOOTNOTE, color=C_TEXT, linespacing=1.4)
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
save_fig(fig_mp, FILENAME_ME2)
print('Done — Fig 3.5E muscle pathway breakdown')

## Fig 3.5C — CellChat pathway heatmap

In [ ]:
FILENAME = 'Fig_3_5C_cellchat_pathway_heatmap'

conditions = ['ctrl', 'age', 'DR', 'sAct', 'combi']
cond_labels = ['Ctrl', 'Age', 'DR', 'sAct', 'Combi']

def load_pathway_probs(base_path, prefix):
    pathway_prob = defaultdict(lambda: defaultdict(float))
    pathway_count = defaultdict(lambda: defaultdict(int))
    for cond in conditions:
        fpath = f"{base_path}/{cond}_lr_interactions_reversal_genes.csv"
        with open(fpath) as f:
            for r in csv.DictReader(f):
                pw = r['pathway_name']
                prob = float(r['prob'])
                pathway_prob[pw][cond] += prob
                pathway_count[pw][cond] += 1
    return pathway_prob, pathway_count

k_prob, k_count = load_pathway_probs(
    f"{DATA_BASE}/Kidney/5_Cellchat_LR_analysis", 'kidney')
m_prob, m_count = load_pathway_probs(
    f"{DATA_BASE}/Muscle_old/5_Cellchat_LR_analysis", 'muscle')

# Sort pathways by total probability (most active first)
k_pathways = sorted(k_prob.keys(),
                     key=lambda pw: sum(k_prob[pw].values()), reverse=True)
m_pathways = sorted(m_prob.keys(),
                     key=lambda pw: sum(m_prob[pw].values()), reverse=True)

# Build matrices
def build_matrix(pathways, prob_dict):
    mat = []
    for pw in pathways:
        row = [prob_dict[pw].get(c, 0) for c in conditions]
        mat.append(row)
    return np.array(mat)

k_mat = build_matrix(k_pathways, k_prob)
m_mat = build_matrix(m_pathways, m_prob)

# Normalise per row (pathway) to show relative change across conditions
def row_normalise(mat):
    """Scale each row to 0-1 range for visual comparison."""
    normed = np.zeros_like(mat)
    for i in range(mat.shape[0]):
        row_max = mat[i].max()
        if row_max > 0:
            normed[i] = mat[i] / row_max
    return normed

k_norm = row_normalise(k_mat)
m_norm = row_normalise(m_mat)

# ── Plot ─────────────────────────────────────────────────────────
fig, (ax_k, ax_m) = plt.subplots(1, 2, figsize=(12, 8),
                                  gridspec_kw={'width_ratios': [1.3, 1], 'wspace': 0.4})

def draw_heatmap(ax, pathways, mat_raw, mat_norm, title, n_interactions):
    n_pw = len(pathways)
    n_cond = len(conditions)

    # Custom colourmap: white → teal
    cmap = mcolors.LinearSegmentedColormap.from_list('w2teal',
           ['#FFFFFF', '#D0EDE2', '#6BBFB0', '#0F6E56', '#0A4A3A'])

    im = ax.imshow(mat_norm, cmap=cmap, aspect='auto', vmin=0, vmax=1,
                    interpolation='nearest')

    # Annotate with raw values
    for i in range(n_pw):
        for j in range(n_cond):
            val = mat_raw[i, j]
            if val >= 1:
                txt = f'{val:.1f}'
            elif val >= 0.01:
                txt = f'{val:.2f}'
            else:
                txt = ''
            intensity = mat_norm[i, j]
            color = 'white' if intensity > 0.6 else C_TEXT
            ax.text(j, i, txt, ha='center', va='center', fontsize=6,
                    color=color, fontweight='bold' if intensity > 0.5 else 'normal')

    ax.set_xticks(range(n_cond))
    ax.set_xticklabels(cond_labels, fontsize=8, fontweight='bold')
    ax.xaxis.set_ticks_position('top')
    ax.xaxis.set_label_position('top')
    ax.set_yticks(range(n_pw))
    ax.set_yticklabels(pathways, fontsize=8)
    ax.set_title(title, fontsize=11, fontweight='bold', color=C_TEXT, pad=12)

    ax.tick_params(axis='y', length=0)
    ax.tick_params(axis='x', length=0)

    # Grid lines between cells
    for i in range(n_pw + 1):
        ax.axhline(i - 0.5, color='white', lw=1.5)
    for j in range(n_cond + 1):
        ax.axvline(j - 0.5, color='white', lw=1.5)

    # Mark "aging change" vs "rescue" patterns
    # Arrows showing direction of change from ctrl→age and age→combi
    for i in range(n_pw):
        ctrl_val = mat_raw[i, 0]
        age_val = mat_raw[i, 1]
        combi_val = mat_raw[i, 4]

        # Aging effect arrow (right side)
        if ctrl_val > 0 or age_val > 0:
            if age_val > ctrl_val * 1.3:
                ax.text(n_cond - 0.3, i, '\u2191', ha='left', va='center',
                        fontsize=8, color=C_RED_D, fontweight='bold')
            elif age_val < ctrl_val * 0.7:
                ax.text(n_cond - 0.3, i, '\u2193', ha='left', va='center',
                        fontsize=8, color=C_BLUE_D, fontweight='bold')

    return im

im_k = draw_heatmap(ax_k, k_pathways, k_mat, k_norm,
                      'Kidney — CellChat pathways', None)
im_m = draw_heatmap(ax_m, m_pathways, m_mat, m_norm,
                      'Muscle — CellChat pathways', None)

# Colourbar
cbar = fig.colorbar(im_k, ax=[ax_k, ax_m], shrink=0.3, aspect=12, pad=0.02,
                     location='bottom')
cbar.set_label('Relative interaction probability (row-normalised)', fontsize=8)
cbar.ax.tick_params(labelsize=7)

fig.suptitle('Fig. 3.5C \u2014 CellChat pathway activity across conditions (reversal genes)',
             fontsize=11, fontweight='bold', color=C_TEXT, y=0.99)

fig.text(0.5, 0.02,
         'Values = summed LR interaction probability per pathway across all cell-type pairs.\n'
         'Colour = row-normalised (brightest = condition with highest activity for that pathway).\n'
         'Arrows: \u2191 = increased with aging vs ctrl; \u2193 = decreased with aging. CellChat: exploratory (n=1/condition).',
         ha='center', fontsize=6.5, color=C_TEXT2, linespacing=1.4)

plt.tight_layout(rect=[0, 0.09, 1, 0.97])
save_fig(fig, FILENAME)
print('Done \u2014 Fig 3.5C')

## Fig 3.5F/G/H — Interorgan CellChat

In [ ]:
# ── Fig 3.5F/G/H — Interorgan CellChat ──────────────────────────
# Requires: interorgan_*_lr_interactions.csv in {DATA_BASE}/Cellchat/interorgan/
FILENAME_F = 'Fig_3_5F_interorgan_overview'
FILENAME_G = 'Fig_3_5G_interorgan_pathways'
FILENAME_H = 'Fig_3_5H_interorgan_information_flow'

from collections import Counter, defaultdict
import textwrap
import matplotlib.colors as mcolors

conditions = ['ctrl', 'age', 'DR', 'sAct', 'combi']
cond_labels = ['Ctrl', 'Age', 'DR', 'sAct', 'Combi']
cond_colors = ['#888888', C_RED_D, C_GREEN_D, C_PURP_D, C_TEAL_D]

IO_BASE = f"{DATA_BASE}/Cellchat/interorgan"

def load_interorgan_lr(base_path):
    data = {}
    for cond in conditions:
        fpath = f"{base_path}/interorgan_{cond}_lr_interactions.csv"
        with open(fpath) as f:
            data[cond] = list(csv.DictReader(f))
    return data

io_data = load_interorgan_lr(IO_BASE)

# Print summary
for cond in conditions:
    rows = io_data[cond]
    pathways = set(r['pathway_name'] for r in rows)
    total_prob = sum(float(r['prob']) for r in rows)
    print(f"  {cond:6s}: {len(rows):>5d} interactions, {len(pathways)} pathways, prob={total_prob:.1f}")

# ── Fig 3.5F: Interaction count + strength overview ──────────────
fig_f, axes_f = plt.subplots(1, 2, figsize=(11, 5), gridspec_kw={'wspace': 0.3})

counts = [len(io_data[c]) for c in conditions]
probs = [sum(float(r['prob']) for r in io_data[c]) for c in conditions]
x = np.arange(len(conditions))
w = 0.55

# Count
for i, (v, c) in enumerate(zip(counts, cond_colors)):
    axes_f[0].bar(x[i], counts[i], width=w, color=c, alpha=0.7, edgecolor=c, linewidth=0.5)
    axes_f[0].text(i, counts[i] + max(counts)*0.02, str(counts[i]), ha='center',
                    va='bottom', fontsize=9, fontweight='bold', color=cond_colors[i])
axes_f[0].set_xticks(x)
axes_f[0].set_xticklabels(cond_labels, fontsize=10, fontweight='bold')
axes_f[0].set_ylabel('Number of LR interactions', fontsize=9, fontweight='bold')
axes_f[0].set_title('Interorgan — interaction count', fontsize=11, fontweight='bold', color=C_TEXT, pad=10)
axes_f[0].spines['top'].set_visible(False)
axes_f[0].spines['right'].set_visible(False)

# Strength
for i, (v, c) in enumerate(zip(probs, cond_colors)):
    axes_f[1].bar(x[i], probs[i], width=w, color=c, alpha=0.5, edgecolor=c, linewidth=0.5)
    axes_f[1].text(i, probs[i] + max(probs)*0.02, f'{probs[i]:.1f}', ha='center',
                    va='bottom', fontsize=9, fontweight='bold', color=cond_colors[i])
axes_f[1].set_xticks(x)
axes_f[1].set_xticklabels(cond_labels, fontsize=10, fontweight='bold')
axes_f[1].set_ylabel('Total interaction probability', fontsize=9, fontweight='bold')
axes_f[1].set_title('Interorgan — interaction strength', fontsize=11, fontweight='bold', color=C_TEXT, pad=10)
axes_f[1].spines['top'].set_visible(False)
axes_f[1].spines['right'].set_visible(False)

# Aging change annotation
ctrl_p, age_p, combi_p = probs[0], probs[1], probs[4]
if age_p != ctrl_p:
    change = ((age_p - ctrl_p) / ctrl_p * 100)
    direction = '\u2191' if change > 0 else '\u2193'
    axes_f[1].annotate(f'{direction} {abs(change):.0f}% with aging',
                        xy=(1, age_p), xytext=(1.5, age_p + max(probs)*0.1),
                        fontsize=7, color=C_RED_D, fontweight='bold',
                        arrowprops=dict(arrowstyle='->', color=C_RED_D, lw=1))

fig_f.suptitle('Fig. 3.5F \u2014 Interorgan CellChat: interaction overview',
               fontsize=11, fontweight='bold', color=C_TEXT, y=1.01)
fig_f.text(0.5, -0.02, 'Inter-organ LR interactions across kidney + muscle cell types.\nCellChat: exploratory (n=1/condition).',
           ha='center', fontsize=6.5, color=C_TEXT2, linespacing=1.4)
plt.tight_layout(rect=[0, 0.03, 1, 0.97])
save_fig(fig_f, FILENAME_F)
print('Done \u2014 Fig 3.5F')

# ── Fig 3.5G: Pathway heatmap across conditions ─────────────────
pathway_prob = defaultdict(lambda: defaultdict(float))
pathway_count = defaultdict(lambda: defaultdict(int))
for cond in conditions:
    for r in io_data[cond]:
        pw = r['pathway_name']
        pathway_prob[pw][cond] += float(r['prob'])
        pathway_count[pw][cond] += 1

# Sort by total probability
pathways_sorted = sorted(pathway_prob.keys(),
                          key=lambda pw: sum(pathway_prob[pw].values()), reverse=True)

# Build matrix
mat_raw = np.array([[pathway_prob[pw].get(c, 0) for c in conditions] for pw in pathways_sorted])

# Row-normalise
mat_norm = np.zeros_like(mat_raw)
for i in range(mat_raw.shape[0]):
    row_max = mat_raw[i].max()
    if row_max > 0:
        mat_norm[i] = mat_raw[i] / row_max

n_pw = len(pathways_sorted)
n_cond = len(conditions)

fig_g, ax_g = plt.subplots(figsize=(8, max(6, n_pw * 0.35 + 2)))

cmap = mcolors.LinearSegmentedColormap.from_list('w2teal',
       ['#FFFFFF', '#D0EDE2', '#6BBFB0', '#0F6E56', '#0A4A3A'])

im = ax_g.imshow(mat_norm, cmap=cmap, aspect='auto', vmin=0, vmax=1, interpolation='nearest')

for i in range(n_pw):
    for j in range(n_cond):
        val = mat_raw[i, j]
        if val >= 1:
            txt = f'{val:.1f}'
        elif val >= 0.01:
            txt = f'{val:.2f}'
        else:
            txt = ''
        intensity = mat_norm[i, j]
        color = 'white' if intensity > 0.6 else C_TEXT
        ax_g.text(j, i, txt, ha='center', va='center', fontsize=6,
                  color=color, fontweight='bold' if intensity > 0.5 else 'normal')

ax_g.set_xticks(range(n_cond))
ax_g.set_xticklabels(cond_labels, fontsize=9, fontweight='bold')
ax_g.xaxis.set_ticks_position('top')
ax_g.set_yticks(range(n_pw))
ax_g.set_yticklabels(pathways_sorted, fontsize=8)
ax_g.set_title('Fig. 3.5G \u2014 Interorgan pathway activity across conditions',
                fontsize=11, fontweight='bold', color=C_TEXT, pad=15)

for i in range(n_pw + 1):
    ax_g.axhline(i - 0.5, color='white', lw=1.5)
for j in range(n_cond + 1):
    ax_g.axvline(j - 0.5, color='white', lw=1.5)

ax_g.spines['top'].set_visible(False)
ax_g.spines['right'].set_visible(False)
ax_g.tick_params(axis='y', length=0)
ax_g.tick_params(axis='x', length=0)

# Arrows for aging direction
for i in range(n_pw):
    ctrl_val = mat_raw[i, 0]
    age_val = mat_raw[i, 1]
    if ctrl_val > 0 or age_val > 0:
        if age_val > ctrl_val * 1.3:
            ax_g.text(n_cond - 0.3, i, '\u2191', ha='left', va='center',
                      fontsize=8, color=C_RED_D, fontweight='bold')
        elif age_val < ctrl_val * 0.7:
            ax_g.text(n_cond - 0.3, i, '\u2193', ha='left', va='center',
                      fontsize=8, color=C_BLUE_D, fontweight='bold')

from mpl_toolkits.axes_grid1.inset_locator import inset_axes
cbax = inset_axes(ax_g, width="30%", height="3%", loc='lower right',
                   bbox_to_anchor=(0, 0.02, 1, 1), bbox_transform=ax_g.transAxes)
cbar = fig_g.colorbar(im, cax=cbax, orientation='horizontal')
cbar.set_label('Relative probability (row-normalised)', fontsize=6, labelpad=2)
cbar.ax.tick_params(labelsize=5)

fig_g.text(0.5, -0.01,
           'Values = summed LR probability per pathway. Colour = row-normalised.\n'
           'Arrows: \u2191 increased / \u2193 decreased with aging vs ctrl. CellChat: exploratory (n=1/condition).',
           ha='center', fontsize=6.5, color=C_TEXT2, linespacing=1.4)

plt.tight_layout(rect=[0, 0.04, 1, 0.98])
save_fig(fig_g, FILENAME_G)
print('Done \u2014 Fig 3.5G')

# ── Fig 3.5H: Information flow ranking ───────────────────────────
# Rank pathways by total probability per condition, show as bump chart
fig_h, ax_h = plt.subplots(figsize=(10, max(6, n_pw * 0.35 + 2)))

# Get total probability per pathway per condition (already in pathway_prob)
# Rank within each condition (1 = highest)
ranks = {}
for cond in conditions:
    pw_probs = [(pw, pathway_prob[pw].get(cond, 0)) for pw in pathways_sorted]
    pw_probs.sort(key=lambda x: x[1], reverse=True)
    for rank, (pw, prob) in enumerate(pw_probs):
        if cond not in ranks:
            ranks[cond] = {}
        ranks[cond][pw] = rank

# Horizontal grouped bar: total probability per pathway, grouped by condition
bar_h = 0.15
y_base = np.arange(n_pw)[::-1]

for ci, cond in enumerate(conditions):
    vals = [pathway_prob[pw].get(cond, 0) for pw in pathways_sorted]
    y_offset = y_base + (ci - 2) * bar_h
    ax_h.barh(y_offset, vals, height=bar_h, color=cond_colors[ci], alpha=0.7,
              edgecolor=cond_colors[ci], linewidth=0.3, label=cond_labels[ci])

ax_h.set_yticks(y_base)
ax_h.set_yticklabels(pathways_sorted, fontsize=7.5)
ax_h.set_xlabel('Total interaction probability', fontsize=9, fontweight='bold')
ax_h.set_title('Fig. 3.5H \u2014 Interorgan information flow: pathway ranking by condition',
                fontsize=11, fontweight='bold', color=C_TEXT, pad=12)
ax_h.spines['top'].set_visible(False)
ax_h.spines['right'].set_visible(False)
ax_h.tick_params(axis='y', length=0)
ax_h.legend(loc='lower right', fontsize=7, frameon=True, framealpha=0.9,
            edgecolor='#E0E0E0', ncol=1)

fig_h.text(0.5, -0.01,
           'Grouped bars show total LR interaction probability per pathway per condition.\n'
           'Allows comparison of which pathways gain or lose signaling strength across conditions.\n'
           'CellChat interorgan analysis: exploratory (n=1/condition).',
           ha='center', fontsize=6.5, color=C_TEXT2, linespacing=1.4)

plt.tight_layout(rect=[0, 0.04, 1, 0.98])
save_fig(fig_h, FILENAME_H)
print('Done \u2014 Fig 3.5H')

## Fig 3.5G-detail ─ Interorgan pathway dot matrix: cell-type pairs × conditions

In [ ]:
# ── Fig 3.5G-detail ──────────────────────────────────────────────
# Interorgan pathway dot matrix: cell-type pairs × conditions
# Filtered to cross-tissue interactions only
FILENAME_G2 = 'Fig_3_5G_interorgan_pathway_communication'

import matplotlib.colors as mcolors

kidney_types = {'EC-1(gEC)', 'PECs?', 'PODO', 'PT-1', 'PT-2', 'vSMC/MC', 'IMM'}
muscle_types = {'Endothelial cells', 'Myonuclei'}

def get_cross_tissue_pairs(io_data, pathway, max_pairs=12):
    all_pairs = set()
    pair_probs = defaultdict(lambda: defaultdict(float))
    for cond in conditions:
        for r in io_data[cond]:
            if r['pathway_name'] != pathway:
                continue
            s, t = r['source'], r['target']
            # Only cross-tissue pairs
            if (s in kidney_types and t in muscle_types) or (s in muscle_types and t in kidney_types):
                pair = (s, t)
                all_pairs.add(pair)
                pair_probs[pair][cond] += float(r['prob'])
    pairs_sorted = sorted(all_pairs, key=lambda p: sum(pair_probs[p].values()), reverse=True)
    return pairs_sorted[:max_pairs], pair_probs

cmap = mcolors.LinearSegmentedColormap.from_list('w2teal',
       ['#FFFFFF', '#D0EDE2', '#6BBFB0', '#0F6E56', '#0A4A3A'])

def plot_io_pathway(ax, io_data, pathway, max_pairs=12):
    pairs, probs = get_cross_tissue_pairs(io_data, pathway, max_pairs)

    if not pairs:
        ax.text(0.5, 0.5, 'No cross-tissue\ninteractions', ha='center', va='center',
                transform=ax.transAxes, fontsize=9, color=C_TEXT2)
        ax.set_title(pathway, fontsize=10, fontweight='bold', color=C_TEXT)
        ax.axis('off')
        return

    n_pairs = len(pairs)
    n_cond = len(conditions)

    mat = np.zeros((n_pairs, n_cond))
    for i, pair in enumerate(pairs):
        for j, cond in enumerate(conditions):
            mat[i, j] = probs[pair].get(cond, 0)

    max_val = mat.max() if mat.max() > 0 else 1

    for i in range(n_pairs):
        for j in range(n_cond):
            val = mat[i, j]
            if val > 0:
                size = 20 + (val / max_val) * 180
                color = cmap(val / max_val)
                ax.scatter(j, n_pairs - 1 - i, s=size, c=[color],
                          edgecolors=C_TEAL_D, linewidths=0.3, zorder=3)

    # Label pairs with tissue origin indicators
    pair_labels = []
    for s, t in pairs:
        s_tissue = 'K' if s in kidney_types else 'M'
        t_tissue = 'K' if t in kidney_types else 'M'
        pair_labels.append(f'{s} [{s_tissue}]\u2192{t} [{t_tissue}]')

    ax.set_yticks(range(n_pairs))
    ax.set_yticklabels(pair_labels[::-1], fontsize=7)
    ax.set_xticks(range(n_cond))
    ax.set_xticklabels(cond_labels, fontsize=8, fontweight='bold')
    ax.xaxis.set_ticks_position('top')
    ax.set_xlim(-0.5, n_cond - 0.5)
    ax.set_ylim(-0.5, n_pairs - 0.5)

    for i in range(n_pairs):
        ax.axhline(i - 0.5, color='#F0F0F0', lw=0.5, zorder=0)
    for j in range(n_cond):
        ax.axvline(j - 0.5, color='#F0F0F0', lw=0.5, zorder=0)

    ax.set_title(pathway, fontsize=10, fontweight='bold', color=C_TEXT, pad=15)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_linewidth(0.3)
    ax.spines['bottom'].set_linewidth(0.3)
    ax.tick_params(axis='y', length=0)
    ax.tick_params(axis='x', length=0)

# Select top cross-tissue pathways
io_pathways = ['SEMA3', 'PTPRM', 'SEMA6', 'EPHA', 'PDGF']

fig, axes = plt.subplots(1, len(io_pathways), figsize=(18, 7))

for i, pw in enumerate(io_pathways):
    plot_io_pathway(axes[i], io_data, pw, max_pairs=12)

fig.suptitle('Interorgan cross-tissue cell-type communication per pathway',
             fontsize=11, fontweight='bold', color=C_TEXT, y=1.02)

fig.text(0.5, -0.01,
         'Dot size and colour = LR interaction probability. Only cross-tissue pairs shown (kidney [K] \u2194 muscle [M]).\n'
         'Top 12 cell-type pairs per pathway ranked by total probability. CellChat: exploratory (n=1/condition).',
         ha='center', fontsize=6.5, color=C_TEXT2, linespacing=1.4)

plt.tight_layout(rect=[0, 0.03, 1, 0.98])
save_fig(fig, FILENAME_G2)
print('Done \u2014 Fig 3.5G-detail')

## Fig 3.5G-detail ─ Interorgan circular network plots per condition

In [ ]:
# ── Fig 3.5G-detail ──────────────────────────────────────────────
# Interorgan circular network plots per condition
FILENAME_G2 = 'Fig_3_5G_interorgan_network_circles'

kidney_types = ['EC-1(gEC)', 'PODO', 'PT-1', 'PT-2', 'vSMC/MC', 'IMM', 'PECs?']
muscle_types = ['Endothelial cells', 'Myonuclei']
all_types = kidney_types + muscle_types

# Colours: kidney warm, muscle cool
ct_colors = {
    'EC-1(gEC)': '#E8918D', 'PODO': '#E8A86B', 'PT-1': '#D4B56A',
    'PT-2': '#A3BE8C', 'vSMC/MC': '#7DB89E', 'IMM': '#6BBFB0', 'PECs?': '#5EAAA8',
    'Endothelial cells': '#6BA3C7', 'Myonuclei': '#7E94C0',
}

IO_BASE = f"{DATA_BASE}/Cellchat/interorgan"

def load_io(base_path):
    data = {}
    for cond in ['ctrl', 'age', 'DR', 'sAct', 'combi']:
        with open(f"{base_path}/interorgan_{cond}_lr_interactions.csv") as f:
            data[cond] = list(csv.DictReader(f))
    return data

io_data = load_io(IO_BASE)

def build_adjacency(rows, cell_types):
    n = len(cell_types)
    ct_idx = {ct: i for i, ct in enumerate(cell_types)}
    mat = np.zeros((n, n))
    for r in rows:
        s, t = r['source'], r['target']
        if s in ct_idx and t in ct_idx:
            mat[ct_idx[s], ct_idx[t]] += float(r['prob'])
    return mat


def draw_network(ax, mat, cell_types, title, ct_colors,
                 max_edge_width=4, prob_threshold=0.01):

    n = len(cell_types)

    n_kidney = len(kidney_types)
    k_angles = np.linspace(np.pi * 0.8, np.pi * 0.2, n_kidney, endpoint=True)
    m_angles = np.linspace(-np.pi * 0.2, -np.pi * 0.8, len(muscle_types), endpoint=True)
    node_angles = list(k_angles) + list(m_angles)

    radius = 1.0
    node_x = [radius * np.cos(a) for a in node_angles]
    node_y = [radius * np.sin(a) for a in node_angles]

    max_prob = mat.max() if mat.max() > 0 else 1

    # ── Edges ─────────────────────────────────────────
    from matplotlib.patches import FancyArrowPatch
    for i in range(n):
        for j in range(n):
            if mat[i, j] < prob_threshold:
                continue
            width = 0.3 + (mat[i, j] / max_prob) * max_edge_width
            alpha = 0.15 + (mat[i, j] / max_prob) * 0.5
            color = ct_colors.get(cell_types[i], C_GRAY_D)

            arrow = FancyArrowPatch(
                (node_x[i], node_y[i]), (node_x[j], node_y[j]),
                connectionstyle="arc3,rad=0.2",
                arrowstyle='->', mutation_scale=8,
                color=color, alpha=alpha, linewidth=width, zorder=1
            )
            ax.add_patch(arrow)

    # ── Nodes ─────────────────────────────────────────
    for i, ct in enumerate(cell_types):
        color = ct_colors.get(ct, C_GRAY_D)
        node_size = 600
        ax.scatter(node_x[i], node_y[i], s=node_size, c=color,
                   edgecolors='white', linewidths=1.5, zorder=3)

        lx = node_x[i] * 1.35
        ly = node_y[i] * 1.35
        ha = 'right' if node_x[i] < 0 else 'left'
        if abs(node_x[i]) < 0.15:
            ha = 'center'

        short_name = ct.replace('Endothelial cells', 'ECs (M)') \
                       .replace('Myonuclei', 'Myo (M)')

        ax.text(lx, ly, short_name, ha=ha, va='center', fontsize=6,
                fontweight='bold', color=color,
                bbox=dict(boxstyle='round,pad=0.1', fc='white', ec=color,
                          lw=0.4, alpha=0.8))

    # ── Tissue labels ─────────────────────────
    ax.text(-1.0, 1.35, 'Kidney', ha='center', fontsize=8,
            fontweight='bold', color=C_RED_D, fontstyle='italic')
    ax.text(1.0, 1.35, 'Muscle', ha='center', fontsize=8,
            fontweight='bold', color=C_BLUE_D, fontstyle='italic')

    # Divider
    ax.axvline(0, color=C_GRAY, lw=0.5, ls=':', zorder=0, ymin=0.1, ymax=0.9)

    # ── UPDATED: Bigger + bold annotation ─────────────
    total = mat.sum()
    n_interactions = (mat > prob_threshold).sum()
    ax.text(0, -1.55,
            f'{n_interactions} int.\nΣ={total:.2f}',
            ha='center',
            fontsize=10,
            fontweight='bold',
            color=C_TEXT2)

    ax.set_title(title, fontsize=10, fontweight='bold', color=C_TEXT, pad=8)
    ax.set_xlim(-1.8, 1.8)
    ax.set_ylim(-1.7, 1.6)
    ax.set_aspect('equal')
    ax.axis('off')


# ── Plot ───────────────────────────────────────────
fig, axes = plt.subplots(1, 5, figsize=(22, 5.5))

cond_titles = ['Ctrl', 'Age', 'DR', 'sAct', 'Combi']

for ci, cond in enumerate(['ctrl', 'age', 'DR', 'sAct', 'combi']):
    mat = build_adjacency(io_data[cond], all_types)
    draw_network(axes[ci], mat, all_types, cond_titles[ci], ct_colors)

# ── Title + centered subtitle (MATCHED STYLE) ──────
fig.suptitle('Interorgan cell-type communication networks across conditions',
             fontsize=12, fontweight='bold', color=C_TEXT, y=0.985)

fig.text(0.5, 0.94, 'Kidney ↔ Muscle',
         ha='center', fontsize=10,
         fontweight='bold', color=C_TEXT2)

# ── Bigger caption ─────────────────────────────────
fig.text(0.5, -0.01,
         'Arrows show LR interaction probability between kidney (left) and muscle (right) cell types.\n'
         'Arrow thickness and opacity proportional to summed probability. Arrow colour = source cell type.\n'
         'CellChat interorgan analysis: exploratory (n=1/condition).',
         ha='center', fontsize=10, color=C_TEXT2, linespacing=1.4)

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
save_fig(fig, FILENAME_G2)
print('Done — Fig 3.5G interorgan network circles')

## Fig 3.5G —  Interorgan network circles pathway breakdowns

In [ ]:
# ── Fig 3.5G-pathways ────────────────────────────────────────────
FILENAME_G3 = 'Fig_3_5G_interorgan_network_by_pathway'

io_pathways_top = ['SEMA3', 'PTPRM', 'SEMA6', 'EPHA', 'PDGF']

def build_adjacency_pathway(rows, cell_types, pathway):
    n = len(cell_types)
    ct_idx = {ct: i for i, ct in enumerate(cell_types)}
    mat = np.zeros((n, n))
    for r in rows:
        if r['pathway_name'] != pathway:
            continue
        s, t = r['source'], r['target']
        if s in ct_idx and t in ct_idx:
            mat[ct_idx[s], ct_idx[t]] += float(r['prob'])
    return mat

n_pw = len(io_pathways_top)
n_cond = 5

fig, axes = plt.subplots(n_pw, n_cond, figsize=(22, n_pw * 4.5))

cond_list = ['ctrl', 'age', 'DR', 'sAct', 'combi']
cond_titles = ['Ctrl', 'Age', 'DR', 'sAct', 'Combi']

for pi, pathway in enumerate(io_pathways_top):
    all_mats = []
    for cond in cond_list:
        mat = build_adjacency_pathway(io_data[cond], all_types, pathway)
        all_mats.append(mat)
    global_max = max(m.max() for m in all_mats) if max(m.max() for m in all_mats) > 0 else 1

    for ci, cond in enumerate(cond_list):
        ax = axes[pi, ci]
        mat = all_mats[ci]

        n = len(all_types)
        n_kidney = len(kidney_types)
        k_angles = np.linspace(np.pi * 0.8, np.pi * 0.2, n_kidney, endpoint=True)
        m_angles = np.linspace(-np.pi * 0.2, -np.pi * 0.8, len(muscle_types), endpoint=True)
        node_angles = list(k_angles) + list(m_angles)

        radius = 1.0
        node_x = [radius * np.cos(a) for a in node_angles]
        node_y = [radius * np.sin(a) for a in node_angles]

        from matplotlib.patches import FancyArrowPatch
        prob_threshold = 0.005

        for i in range(n):
            for j in range(n):
                if mat[i, j] < prob_threshold:
                    continue
                width = 0.3 + (mat[i, j] / global_max) * 4
                alpha = 0.15 + (mat[i, j] / global_max) * 0.55
                color = ct_colors.get(all_types[i], C_GRAY_D)
                arrow = FancyArrowPatch(
                    (node_x[i], node_y[i]), (node_x[j], node_y[j]),
                    connectionstyle="arc3,rad=0.2",
                    arrowstyle='->', mutation_scale=8,
                    color=color, alpha=alpha, linewidth=width, zorder=1
                )
                ax.add_patch(arrow)

        for i, ct in enumerate(all_types):
            color = ct_colors.get(ct, C_GRAY_D)
            activity = mat[i, :].sum() + mat[:, i].sum()
            node_size = 150 + (activity / max(global_max, 0.01)) * 400
            ax.scatter(node_x[i], node_y[i], s=node_size, c=color,
                       edgecolors='white', linewidths=1, zorder=3)

        if ci == 0:
            for i, ct in enumerate(all_types):
                color = ct_colors.get(ct, C_GRAY_D)
                lx = node_x[i] * 1.4
                ly = node_y[i] * 1.4
                ha = 'right' if node_x[i] < 0 else 'left'
                short = ct.replace('Endothelial cells', 'ECs (M)').replace('Myonuclei', 'Myo (M)')
                ax.text(lx, ly, short, ha=ha, va='center', fontsize=5,
                        fontweight='bold', color=color)

        # ✅ UPDATED: bigger + bold annotation
        total = mat.sum()
        n_int = (mat > prob_threshold).sum()
        ax.text(0, -1.45,
                f'{n_int} int.\nΣ={total:.2f}',
                ha='center',
                fontsize= 10,              # was 5
                fontweight='bold',         # NEW
                color=C_TEXT2)

        if pi == 0:
            ax.set_title(cond_titles[ci], fontsize=10, fontweight='bold', color=C_TEXT, pad=8)

        if ci == 0:
            ax.text(-2.2, 0, pathway, ha='center', va='center', fontsize=14,
                    fontweight='bold', color=C_TEAL_D, rotation=90)

        ax.set_xlim(-1.8, 1.8)
        ax.set_ylim(-1.6, 1.5)
        ax.set_aspect('equal')
        ax.axis('off')

        ax.axvline(0, color=C_GRAY, lw=0.3, ls=':', zorder=0, ymin=0.15, ymax=0.85)

# ── Title + centered subtitle (UPDATED) ──────────────────────────
fig.suptitle('Interorgan communication networks by pathway and condition',
             fontsize=12, fontweight='bold', color=C_TEXT, y=0.985)

# ✅ moved + centered properly under title
fig.text(0.5, 0.97, 'Kidney ↔ Muscle',
         ha='center', fontsize=10,
         fontweight='bold', color=C_TEXT2)

fig.text(0.5, -0.01,
         'Each panel = one pathway × one condition. Node size = pathway activity. Arrow width = interaction probability.\n'
         'Cell type labels on left column. Arrow colour = source cell type. Scaling consistent within each pathway row.\n'
         'CellChat interorgan: exploratory (n=1/condition).',
         ha='center', fontsize=10, color=C_TEXT2, linespacing=1.4)

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
save_fig(fig, FILENAME_G3)
print('Done — Fig 3.5G pathway breakdown')

##Fig 3.5H — Interorgan rescued pathways per condition ──────────

In [ ]:
# ── Fig 3.5H — Interorgan rescued pathways per condition ──────────
FILENAME_H = 'Fig_3_5H_interorgan_rescued_pathways'

import matplotlib.gridspec as gridspec
from matplotlib.patches import FancyArrowPatch

rescued_pathways = ['SEMA3', 'SEMA6', 'PECAM1', 'CADM', 'CDH5',
                    'THBS', 'COLLAGEN', 'VEGF', 'LAMININ', 'KIT', 'ESAM']

kidney_types = ['EC-1(gEC)', 'PODO', 'PT-1', 'PT-2', 'vSMC/MC', 'IMM', 'PECs?']
muscle_types = ['Endothelial cells', 'Myonuclei']
all_types = kidney_types + muscle_types

ct_colors = {
    'EC-1(gEC)': '#E8918D', 'PODO': '#E8A86B', 'PT-1': '#D4B56A',
    'PT-2': '#A3BE8C', 'vSMC/MC': '#7DB89E', 'IMM': '#6BBFB0', 'PECs?': '#5EAAA8',
    'Endothelial cells': '#6BA3C7', 'Myonuclei': '#7E94C0',
}

conditions = ['ctrl', 'age', 'DR', 'sAct', 'combi']
cond_titles = ['Ctrl', 'Age', 'DR', 'sAct', 'Combi']

def build_adjacency_pathway_io(rows, cell_types, pathway_name):
    n = len(cell_types)
    ct_idx = {ct: i for i, ct in enumerate(cell_types)}
    mat = np.zeros((n, n))
    for r in rows:
        s, t = r['source'], r['target']
        p = r.get('pathway_name', '')
        if s in ct_idx and t in ct_idx and p == pathway_name:
            mat[ct_idx[s], ct_idx[t]] += float(r['prob'])
    return mat

def draw_pathway_network(ax, mat, cell_types, title, ct_colors, global_max,
                          max_edge_width=4, prob_threshold=0.005):
    n = len(cell_types)
    n_kidney = len(kidney_types)
    k_angles = np.linspace(np.pi * 0.8, np.pi * 0.2, n_kidney, endpoint=True)
    m_angles = np.linspace(-np.pi * 0.2, -np.pi * 0.8, len(muscle_types), endpoint=True)
    node_angles = list(k_angles) + list(m_angles)

    radius = 1.0
    node_x = [radius * np.cos(a) for a in node_angles]
    node_y = [radius * np.sin(a) for a in node_angles]

    for i in range(n):
        for j in range(n):
            if mat[i, j] < prob_threshold:
                continue
            width = 0.3 + (mat[i, j] / global_max) * max_edge_width
            alpha = 0.15 + (mat[i, j] / global_max) * 0.55
            color = ct_colors.get(cell_types[i], C_GRAY_D)
            arrow = FancyArrowPatch(
                (node_x[i], node_y[i]), (node_x[j], node_y[j]),
                connectionstyle="arc3,rad=0.2",
                arrowstyle='->', mutation_scale=8,
                color=color, alpha=alpha, linewidth=width, zorder=1)
            ax.add_patch(arrow)

    for i, ct in enumerate(cell_types):
        color = ct_colors.get(ct, C_GRAY_D)
        activity = mat[i,:].sum() + mat[:,i].sum()
        node_size = 150 + (activity / max(global_max, 0.01)) * 400
        ax.scatter(node_x[i], node_y[i], s=node_size, c=color,
                   edgecolors='white', linewidths=0.8, zorder=3)

    total = mat.sum()
    n_int = (mat > prob_threshold).sum()
    ax.text(0, -1.52, f'{n_int} int.\nΣ={total:.3f}',
            ha='center', fontsize=FS_STATS - 4, fontweight='bold', color=C_TEXT)
    ax.set_title(title, fontsize=FS_TITLE - 6, fontweight='bold', color=C_TEXT, pad=6)
    ax.set_xlim(-1.7, 1.7)
    ax.set_ylim(-1.65, 1.5)
    ax.set_aspect('equal')
    ax.axis('off')

# ── Build figure ──────────────────────────────────────────────────
n_pw = len(rescued_pathways)
fig, axes = plt.subplots(n_pw, 5, figsize=(34, n_pw * 6))

for pi, pathway in enumerate(rescued_pathways):
    # Build all matrices for this pathway across conditions
    all_mats = [build_adjacency_pathway_io(io_data[c], all_types, pathway)
                for c in conditions]
    global_max = max(m.max() for m in all_mats) if max(m.max() for m in all_mats) > 0 else 1

    for ci, cond in enumerate(conditions):
        ax = axes[pi, ci]
        mat = all_mats[ci]
        draw_pathway_network(ax, mat, all_types, '', ct_colors, global_max)

        # Column titles on top row only
        if pi == 0:
            ax.set_title(cond_titles[ci], fontsize=FS_TITLE,
                         fontweight='bold', color=C_TEXT, pad=10)

        # Pathway label on left column only
        if ci == 0:
            ax.text(-2.2, 0, pathway, ha='center', va='center',
                    fontsize=FS_PATHWAY, fontweight='bold', color=C_TEAL_D, rotation=90)

        # Add tissue labels on first panel only
        if pi == 0 and ci == 0:
            ax.text(-0.8, 1.35, 'Kidney', ha='center', fontsize=FS_LABEL - 2,
                    fontweight='bold', color=C_RED_D, fontstyle='italic')
            ax.text(0.8, 1.35, 'Muscle', ha='center', fontsize=FS_LABEL - 2,
                    fontweight='bold', color=C_BLUE_D, fontstyle='italic')

# ── Node label legend on first panel ─────────────────────────────
# Add cell type labels only to first pathway row, first condition
ax_legend = axes[0, 0]
n_k = len(kidney_types)
k_angles = np.linspace(np.pi * 0.8, np.pi * 0.2, n_k, endpoint=True)
m_angles = np.linspace(-np.pi * 0.2, -np.pi * 0.8, len(muscle_types), endpoint=True)
all_angles = list(k_angles) + list(m_angles)
for i, ct in enumerate(all_types):
    lx = 1.0 * np.cos(all_angles[i]) * 1.45
    ly = 1.0 * np.sin(all_angles[i]) * 1.45
    ha = 'right' if lx < 0 else 'left'
    short = ct.replace('Endothelial cells', 'ECs (M)').replace('Myonuclei', 'Myo (M)')
    ax_legend.text(lx, ly, short, ha=ha, va='center', fontsize=FS_LABEL - 2,
                   fontweight='bold', color=ct_colors.get(ct, C_GRAY_D))

fig.suptitle('Interorgan communication: combi-rescued pathways across conditions',
             fontsize=FS_SUPTITLE, fontweight='bold', color=C_TEXT, y=0.995)
fig.text(0.5, 0.005,
         'Each row = one pathway; columns = conditions. Node size = pathway activity.\n'
         'Arrow width = interaction probability, scaled within each pathway row.\n'
         'Kidney cell types (left): EC-1/gEC, PODO, PT-1, PT-2, vSMC/MC, IMM, PECs?. '
         'Muscle cell types (right): ECs, Myonuclei.\n'
         'CellChat interorgan: exploratory (n=1/condition).',
         ha='center', fontsize=FS_FOOTNOTE, color=C_TEXT2, linespacing=1.4)

plt.tight_layout(rect=[0, 0.04, 1, 0.98])
save_fig(fig, FILENAME_H)
print('Done — Fig 3.5H interorgan rescued pathways')

#3.6 Cross-Tissue Integration: Conserved and Tissue-Specific Rescue Programs

## Fig 3.6A — Venn diagram rescued-to-normal

In [ ]:
FILENAME = 'Fig_3_6A_venn_rescued_to_normal'

# ── Auto-load counts ──────────────────────────────────────────────
with open(f"{DATA_BASE}/Kidney/1_DEGs_tables/kidney_rescued_to_normal.csv") as f:
    k_rtn_set = {r['Symbol'] for r in csv.DictReader(f) if r.get('Symbol','')}
with open(f"{DATA_BASE}/Muscle_old/1_DEGs_tables/muscle_rescued_to_normal.csv") as f:
    m_rtn_set = {r['Symbol'] for r in csv.DictReader(f) if r.get('Symbol','')}

shared_set = k_rtn_set & m_rtn_set
k_total = len(k_rtn_set)
m_total = len(m_rtn_set)
shared  = len(shared_set)
k_only  = k_total - shared
m_only  = m_total - shared

with open(f"{DATA_BASE}/Kidney/1_DEGs_tables/kidney_rescued_to_normal.csv") as f:
    k_lfc_d = {r['Symbol']: float(r['log2FoldChange'])
               for r in csv.DictReader(f) if r.get('Symbol','')}
with open(f"{DATA_BASE}/Muscle_old/1_DEGs_tables/muscle_rescued_to_normal.csv") as f:
    m_lfc_d = {r['Symbol']: float(r['log2FoldChange'])
               for r in csv.DictReader(f) if r.get('Symbol','')}
same_dir_n = sum(1 for g in shared_set
                 if g in k_lfc_d and g in m_lfc_d
                 and np.sign(k_lfc_d[g]) == np.sign(m_lfc_d[g]))

# Notable shared genes — filter to those actually in shared set
original_notable = ['Tgfb1','Bax','Lmna','Ctnna1','Maf','Pcbp2',
                    'Ssr3','Tnks2','Ly6e','Pxdn','Cert1','Mxi1']
notable_shared = [g for g in original_notable if g in shared_set]
if len(notable_shared) < 12:
    extras = [g for g in sorted(shared_set) if g not in notable_shared]
    notable_shared += extras[:12 - len(notable_shared)]
notable_shared = notable_shared[:12]

fig, ax = plt.subplots(figsize=(10, 8))
ax.set_xlim(-5, 5)
ax.set_ylim(-4, 5)
ax.set_aspect('equal')
ax.axis('off')

# ── Title ────────────────────────────────────────────────────────
ax.text(0, 4.7, 'Fig. 3.6A', ha='center', va='center',
        fontsize=12, fontweight='bold', color=C_TEXT)
ax.text(0, 4.3, 'Rescued-to-normal genes shared between kidney and muscle',
        ha='center', va='center', fontsize=9.5, color=C_TEXT2)

# ── Ellipses ─────────────────────────────────────────────────────
# Same aspect ratio (1:1 = circles), different sizes
# Kidney circle (larger, left-shifted)
kidney_e = Ellipse((-0.8, 0.4), width=5.0, height=5.0, angle=0,
                    facecolor=C_RED, edgecolor=C_RED_D, linewidth=1.2,
                    alpha=0.45, zorder=1)
ax.add_patch(kidney_e)

# Muscle circle (smaller, right-shifted)
muscle_e = Ellipse((1.5, 0.4), width=3.4, height=3.4, angle=0,
                    facecolor=C_BLUE, edgecolor=C_BLUE_D, linewidth=1.2,
                    alpha=0.45, zorder=1)
ax.add_patch(muscle_e)

# ── Labels: circle titles ────────────────────────────────────────
ax.text(-2.8, 3.2, 'Kidney', ha='center', va='center',
        fontsize=14, fontweight='bold', color=C_RED_D)
ax.text(-2.8, 2.8, 'rescued-to-normal', ha='center', va='center',
        fontsize=9, color=C_RED_D)

ax.text(3.2, 3.2, 'Muscle', ha='center', va='center',
        fontsize=14, fontweight='bold', color=C_BLUE_D)
ax.text(3.2, 2.8, 'rescued-to-normal', ha='center', va='center',
        fontsize=9, color=C_BLUE_D)

# ── Numbers in each region ───────────────────────────────────────
# Kidney-only
ax.text(-2.5, 0.6, f'{k_only:,}', ha='center', va='center',
        fontsize=28, fontweight='bold', color=C_RED_D, zorder=5)
ax.text(-2.5, 0.05, 'kidney-only', ha='center', va='center',
        fontsize=9, fontweight='bold', color=C_RED_D, zorder=5)

# Muscle-only
ax.text(2.4, 0.6, f'{m_only}', ha='center', va='center',
        fontsize=28, fontweight='bold', color=C_BLUE_D, zorder=5)
ax.text(2.4, 0.05, 'muscle-only', ha='center', va='center',
        fontsize=9, fontweight='bold', color=C_BLUE_D, zorder=5)

# Shared (overlap region)
ax.text(0.7, 0.6, f'{shared}', ha='center', va='center',
        fontsize=28, fontweight='bold', color=C_PURP_D, zorder=5)
ax.text(0.7, 0.05, 'shared', ha='center', va='center',
        fontsize=9, fontweight='bold', color=C_PURP_D, zorder=5)
ax.text(0.7, -0.3, f'({same_dir_n/shared*100:.0f}% same direction)', ha='center', va='center',
        fontsize=7.5, color=C_TEXT2, zorder=5)

# ── Notable shared genes box ─────────────────────────────────────
box_y = -2.2
box = FancyBboxPatch((-3.2, box_y - 0.7), 6.4, 1.4,
                      boxstyle="round,pad=0.15", facecolor='#F8F7F5',
                      edgecolor=C_GRAY_D, linewidth=0.6, zorder=2)
ax.add_patch(box)

ax.text(0,  box_y + 0.45, f'Notable shared rescued genes ({len(notable_shared)} of {shared})',
        ha='center', va='center', fontsize=8.5, fontweight='bold', color=C_TEXT, zorder=3)

# Arrange genes in 3 rows of 4
genes_per_row = 4
for i, gene in enumerate(notable_shared):
    row = i // genes_per_row
    col = i % genes_per_row
    gx = -2.2 + col * 1.5
    gy = box_y + 0.05 - row * 0.35
    # Gene name pill
    pill = FancyBboxPatch((gx - 0.55, gy - 0.12), 1.1, 0.28,
                           boxstyle="round,pad=0.04",
                           facecolor=C_PURPLE, edgecolor=C_PURP_D,
                           linewidth=0.4, zorder=3)
    ax.add_patch(pill)
    ax.text(gx, gy, gene, ha='center', va='center',
            fontsize=7, fontweight='bold', color=C_PURP_D,
            fontstyle='italic', zorder=4)

# ── Percentage annotations ───────────────────────────────────────
# Fraction of each tissue's genes that are shared
k_pct = shared / k_total * 100
m_pct = shared / m_total * 100

ax.text(-2.5, -0.3, f'({k_total:,} total)', ha='center', va='center',
        fontsize=7.5, color=C_TEXT2)
ax.text(2.4,  -0.3, f'({m_total} total)', ha='center', va='center',
        fontsize=7.5, color=C_TEXT2)

# ── Footnote ─────────────────────────────────────────────────────
ax.text(0, -3.6,
        f'Overlap: {shared} genes = {k_pct:.1f}% of kidney rescued-to-normal, '
        f'{m_pct:.1f}% of muscle rescued-to-normal.\n'
        f'{same_dir_n/shared*100:.0f}% ({same_dir_n}/{shared}) show the same direction of aging change in both tissues.\n'
        'Gene symbols mapped via org.Mm.eg.db; 25 kidney genes without valid symbols excluded from Venn.',
        ha='center', va='center', fontsize=6.5, color=C_TEXT2, linespacing=1.5)

plt.tight_layout(pad=0.5)
save_fig(fig, FILENAME)
print('Done \u2014 Fig 3.6A')

## Fig 3.6B — Aging LFC concordance scatter

In [ ]:
FILENAME = 'Fig_3_6B_scatter_aging_LFC_kidney_vs_muscle'

COMMON_CSV = f"{DATA_BASE}/Common Genes and Pathways Kidney & Muscle/common_rescued_genes_kidney_muscle.csv"

with open(COMMON_CSV) as f:
    rows = list(csv.DictReader(f))

symbols = [r['Symbol_kidney'] for r in rows]
k_lfc = np.array([float(r['log2FoldChange_kidney']) for r in rows])
m_lfc = np.array([float(r['log2FoldChange_muscle']) for r in rows])

same_dir = np.array([(k > 0) == (m > 0) for k, m in zip(k_lfc, m_lfc)])
n_same = int(same_dir.sum())
n_opp = len(same_dir) - n_same
r_val = float(np.corrcoef(k_lfc, m_lfc)[0, 1])

genes_to_label = {
    'Tgfb1', 'Bax', 'Lmna', 'Ctnna1', 'Maf', 'Pld4', 'Tmem158',
    'Ly6e', 'Nrros', 'Hcls1', 'Tppp3', 'Pxdn', 'Grap',
    'Tbc1d1', 'Prcp', 'Man2b1', 'Pcbp2', 'Ssr3',
}

label_offsets = {
    'Tgfb1': (8, 12), 'Bax': (10, -14), 'Lmna': (8, 12),
    'Ctnna1': (10, -12), 'Maf': (-10, -10), 'Pld4': (8, -10),
    'Tmem158': (8, 8), 'Ly6e': (-14, -8), 'Nrros': (10, -8),
    'Hcls1': (-14, 10), 'Tppp3': (-12, 8), 'Pxdn': (-12, 12),
    'Grap': (8, 12), 'Tbc1d1': (8, -10), 'Prcp': (-10, 8),
    'Man2b1': (-10, -10), 'Pcbp2': (10, -10), 'Ssr3': (-12, -12),
}

fig, ax = plt.subplots(figsize=(8, 8))
lim = max(abs(k_lfc).max(), abs(m_lfc).max()) * 1.15
ax.set_xlim(-lim, lim)
ax.set_ylim(-lim, lim)

# Quadrant shading
ax.fill_between([0, lim], [0, 0], [lim, lim], color=C_TEAL, alpha=0.08, zorder=0)
ax.fill_between([-lim, 0], [-lim, -lim], [0, 0], color=C_TEAL, alpha=0.08, zorder=0)
ax.fill_between([-lim, 0], [0, 0], [lim, lim], color=C_CORAL, alpha=0.06, zorder=0)
ax.fill_between([0, lim], [-lim, -lim], [0, 0], color=C_CORAL, alpha=0.06, zorder=0)

ax.axhline(0, color=C_GRAY_D, lw=0.5, ls='--', zorder=1)
ax.axvline(0, color=C_GRAY_D, lw=0.5, ls='--', zorder=1)
ax.plot([-lim, lim], [-lim, lim], color=C_GRAY_D, lw=0.8, ls=':', alpha=0.5, zorder=1)

ax.scatter(k_lfc[same_dir], m_lfc[same_dir],
           c=C_TEAL_D, s=40, alpha=0.7, edgecolors='white', linewidths=0.3,
           zorder=3, label=f'Same direction (n={n_same})')
ax.scatter(k_lfc[~same_dir], m_lfc[~same_dir],
           c=C_CORAL_D, s=40, alpha=0.7, edgecolors='white', linewidths=0.3,
           zorder=3, label=f'Opposite direction (n={n_opp})')

for i, sym in enumerate(symbols):
    if sym in genes_to_label:
        kl, ml = k_lfc[i], m_lfc[i]
        is_same = same_dir[i]
        color = C_TEAL_D if is_same else C_CORAL_D
        offset = label_offsets.get(sym, (8, 8))
        ax.annotate(sym, xy=(kl, ml), xytext=offset, textcoords='offset points',
                    fontsize=7, fontstyle='italic', fontweight='bold', color=color,
                    arrowprops=dict(arrowstyle='-', color=color, lw=0.5, alpha=0.6), zorder=5)

ax.set_xlabel('log$_2$FC aging (kidney)', fontsize=11, fontweight='bold', color=C_TEXT)
ax.set_ylabel('log$_2$FC aging (muscle)', fontsize=11, fontweight='bold', color=C_TEXT)
ax.tick_params(labelsize=9)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_linewidth(0.5)
ax.spines['bottom'].set_linewidth(0.5)

qa = 0.55
ax.text(lim*0.65, lim*0.85, 'Both upregulated\nwith aging', ha='center', va='center',
        fontsize=7, color=C_TEAL_D, alpha=qa, fontstyle='italic')
ax.text(-lim*0.65, -lim*0.85, 'Both downregulated\nwith aging', ha='center', va='center',
        fontsize=7, color=C_TEAL_D, alpha=qa, fontstyle='italic')
ax.text(-lim*0.65, lim*0.85, 'Kidney down,\nmuscle up', ha='center', va='center',
        fontsize=7, color=C_CORAL_D, alpha=qa, fontstyle='italic')
ax.text(lim*0.65, -lim*0.85, 'Kidney up,\nmuscle down', ha='center', va='center',
        fontsize=7, color=C_CORAL_D, alpha=qa, fontstyle='italic')

stats_text = (f'n = {len(rows)} genes\n'
              f'Pearson r = {r_val:.3f}\n'
              f'Same direction: {n_same}/98 (86%)\n'
              f'Opposite: {n_opp}/98 (14%)')
ax.text(0.03, 0.97, stats_text, transform=ax.transAxes,
        fontsize=7.5, va='top', ha='left', color=C_TEXT,
        bbox=dict(boxstyle='round,pad=0.4', facecolor=C_WHITE,
                  edgecolor=C_GRAY_D, linewidth=0.5, alpha=0.9))

ax.legend(loc='lower right', fontsize=8, frameon=True, framealpha=0.9,
          edgecolor=C_GRAY_D, fancybox=True, handlelength=1.2)

ax.set_title('Fig. 3.6B \u2014 Aging fold-change concordance for 98 shared rescued genes',
             fontsize=10.5, fontweight='bold', color=C_TEXT, pad=15)

fig.text(0.5, 0.01,
         'Each point = one gene rescued-to-normal in both kidney and muscle. '
         'log2FC from DESeq2 age vs. ctrl contrast.\n'
         'Green shading = concordant quadrants; red shading = discordant.',
         ha='center', fontsize=6.5, color=C_TEXT2)

plt.tight_layout(rect=[0, 0.04, 1, 1])
save_fig(fig, FILENAME)
print('Done \u2014 Fig 3.6B')

## Fig 3.6C — GOBP common rescued genes

In [ ]:
FILENAME = 'Fig_3_6C_GOBP_common_rescued_genes'

# ── Data ─────────────────────────────────────────────────────────
# ── Auto-load shared gene set ─────────────────────────────────────
with open(f"{DATA_BASE}/Kidney/1_DEGs_tables/kidney_rescued_to_normal.csv") as f:
    k_rtn_set = {r['Symbol'] for r in csv.DictReader(f) if r.get('Symbol','')}
with open(f"{DATA_BASE}/Muscle_old/1_DEGs_tables/muscle_rescued_to_normal.csv") as f:
    m_rtn_set = {r['Symbol'] for r in csv.DictReader(f) if r.get('Symbol','')}
shared_set = k_rtn_set & m_rtn_set
n_shared   = len(shared_set)

# ── Auto-load context GOBP from kidney enrichment ─────────────────
k_gobp_path = f"{DATA_BASE}/Kidney/2_Enrichment_bulk&sn/GOBP bulk/kidney_enrichment_results_rescued-to-normal.csv"
gobp_lookup = {}
with open(k_gobp_path) as f:
    for r in csv.DictReader(f):
        try:
            gobp_lookup[r['Description']] = {
                'padj':    float(r['p.adjust']),
                'n_total': int(r['GeneRatio'].split('/')[1]),
            }
        except (ValueError, IndexError):
            continue

term_gene_sets = {
    'Regulation of protein complex assembly': ['Ctnna1','Ube3a','Cul5','Arih1','Lrp6','Pcbp2','Sfmbt1','Zmym2'],
    'Golgi organization':                     ['Ssr3','Arcn1','Cog1','Gbf1','Rab4a','Golph3','Cert1'],
    'Response to peptide hormone stimulus':   ['Tgfb1','Lmna','Bax','Lifr','Pcbp2','Crkl','Mxi1'],
    'Golgi vesicle transport':                ['Ssr3','Arcn1','Cog1','Rab4a','Rab13','Gbf1'],
    'Endosomal transport':                    ['Ssr3','Rab4a','Rab22a','Laptm5','Tmem9','Hps1'],
    'Mitochondrion organization':             ['Ube3a','Huwe1','Cul5','Rnf126','Dcaf8','Ubxn4'],
    'Response to oxidative stress':           ['Bax','Maf','Mxi1','Trp53inp2','Lifr','Ninj1'],
    'regulation of protein phosphorylation':  ['Tnks2','Hipk3','Plekho1','Sfmbt1','Mbd5','Lrp6'],
    'Protein localization to nucleus':        ['Lmna','Pcbp2','Yy1','Sfmbt1','Zmym2','Mbd5'],
    'Protein localization to plasma membrane':['Cfl1','Arpc5l','Dbnl','Lrp6','Rab4a'],
}

context_gobp = []
for term, genes in term_gene_sets.items():
    n_common = sum(1 for g in genes if g in shared_set)
    info = gobp_lookup.get(term, {})
    padj    = info.get('padj',    1.0)
    n_total = info.get('n_total', 0)
    context_gobp.append((term, n_common, n_total, padj))

golgi_padj = gobp_lookup.get('Golgi organization', {}).get('padj', 0.00381)

# ── Auto-load functional groups ───────────────────────────────────
func_groups_raw = [
    ('Vesicle / Golgi transport',    ['Ssr3','Srpra','Arcn1','Golph3','Cog1','Rab4a','Rab13','Gbf1','Tmed7','Rab22a','Laptm5','Tmem9','Hps1','Slc11a1'], C_TEAL,   C_TEAL_D),
    ('Cell adhesion / cytoskeleton', ['Ctnna1','Cfl1','Arpc5l','Dbnl','Tuba1c','Lrp6','Map1b'],                                                           C_BLUE,   C_BLUE_D),
    ('Ubiquitin / proteostasis',     ['Ube3a','Huwe1','Cul5','Arih1','Rnf126','Dcaf8','Ubxn4'],                                                            C_AMBER,  C_AMBER_D),
    ('Nuclear / chromatin',          ['Lmna','Pcbp2','Sfmbt1','Yy1','Zmym2','Mbd5'],                                                                       C_PURPLE, C_PURP_D),
    ('Immune / inflammation',        ['Ly6e','Nrros','Hcls1','Pld4','Unc93b1','Ninj1'],                                                                    C_CORAL,  C_CORAL_D),
    ('Apoptosis / stress response',  ['Bax','Maf','Lifr','Mxi1','Trp53inp2'],                                                                              C_RED,    C_RED_D),
    ('TGF-beta / signalling',        ['Tgfb1','Grap','Prkacb','Crkl'],                                                                                     C_GREEN,  C_GREEN_D),
    ('RNA processing',               ['Pcnp','Cnbp','Luc7l2','Rtf1'],                                                                                      C_GRAY,   C_GRAY_D),
    ('Kinase / phosphatase',         ['Tnks2','Hipk3','Plekho1'],                                                                                          C_TEAL,   C_TEAL_D),
]
func_groups = []
for name, genes, fc, ec in func_groups_raw:
    present  = [g for g in genes if g in shared_set]
    gene_str = ', '.join(present[:4]) + ('...' if len(present) > 4 else '')
    if len(present) > 0:
        func_groups.append((name, len(present), fc, ec, gene_str))
categorized   = sum(g[1] for g in func_groups)
uncategorized = n_shared - categorized

# ── Figure ───────────────────────────────────────────────────────
fig = plt.figure(figsize=(14, 7.5))

# Two panels: left = context GOBP dot plot, right = functional grouping bars
ax_left  = fig.add_axes([0.05, 0.12, 0.42, 0.78])
ax_right = fig.add_axes([0.58, 0.12, 0.40, 0.78])

# ── LEFT: GOBP context dot plot ──────────────────────────────────
terms = [t[0] for t in context_gobp]
n_common = [t[1] for t in context_gobp]
n_total  = [t[2] for t in context_gobp]
padj     = [t[3] for t in context_gobp]

y_pos = np.arange(len(terms))[::-1]
neg_log_p = [-np.log10(p) for p in padj]

# Size = n_common genes, color = -log10(padj)
scatter = ax_left.scatter(
    n_common, y_pos,
    s=[n * 45 for n in n_common],
    c=neg_log_p,
    cmap='YlOrRd', vmin=3, vmax=8,
    edgecolors=C_TEXT2, linewidths=0.4,
    zorder=3, alpha=0.85
)

ax_left.set_yticks(y_pos)
ax_left.set_yticklabels(terms, fontsize=7.5)
ax_left.set_xlabel('Common rescued genes in term', fontsize=9, fontweight='bold')
ax_left.set_xlim(3.5, 9.5)
ax_left.spines['top'].set_visible(False)
ax_left.spines['right'].set_visible(False)
ax_left.spines['left'].set_linewidth(0.5)
ax_left.spines['bottom'].set_linewidth(0.5)
ax_left.tick_params(axis='y', length=0)
ax_left.grid(axis='x', alpha=0.15, lw=0.5)

# Highlight the formally significant term
golgi_idx = terms.index('Golgi organization')
golgi_y = y_pos[golgi_idx]
ax_left.annotate(f'Only term significant\nin {n_shared}-gene ORA (padj = {golgi_padj:.3f})',
                  xy=(7, golgi_y), xytext=(8.2, golgi_y + 1.2),
                  fontsize=6.5, color=C_TEXT2, fontstyle='italic',
                  arrowprops=dict(arrowstyle='->', color=C_TEXT2, lw=0.6),
                  zorder=5)

# Colorbar
cbar = fig.colorbar(scatter, ax=ax_left, shrink=0.4, aspect=15, pad=0.02)
cbar.set_label('-log10(padj)\n(kidney rescued-to-normal)', fontsize=7)
cbar.ax.tick_params(labelsize=6.5)

# Size legend
for n_leg in [5, 7]:
    ax_left.scatter([], [], s=n_leg*45, c='gray', alpha=0.5,
                     edgecolors=C_TEXT2, linewidths=0.4,
                     label=f'{n_leg} genes')
ax_left.legend(loc='lower right', fontsize=6.5, frameon=True, framealpha=0.9,
               edgecolor=C_GRAY_D, title='Count', title_fontsize=6.5,
               handletextpad=0.3, borderpad=0.4)

ax_left.set_title('GO:BP terms containing\ncommon rescued genes', fontsize=10,
                    fontweight='bold', color=C_TEXT, pad=10)

# ── RIGHT: Functional grouping horizontal bars ───────────────────
categories = [g[0] for g in func_groups]
counts     = [g[1] for g in func_groups]
fills      = [g[2] for g in func_groups]
edges      = [g[3] for g in func_groups]
gene_lists = [g[4] for g in func_groups]

y_pos_r = np.arange(len(categories))[::-1]

bars = ax_right.barh(y_pos_r, counts, height=0.65,
                      color=fills, edgecolor=edges, linewidth=0.6, zorder=3)

# Add gene lists inside/beside bars
for i, (yp, count, genes_txt, ec) in enumerate(zip(y_pos_r, counts, gene_lists, edges)):
    # Count label at end of bar
    ax_right.text(count + 0.3, yp, str(count), va='center', ha='left',
                  fontsize=8, fontweight='bold', color=ec)
    # Gene names to the right
    ax_right.text(count + 1.5, yp, genes_txt.replace('\n', ', '),
                  va='center', ha='left', fontsize=5.5, color=C_TEXT2,
                  fontstyle='italic')

ax_right.set_yticks(y_pos_r)
ax_right.set_yticklabels(categories, fontsize=7.5)
ax_right.set_xlabel(f'Number of genes (of {n_shared} common)', fontsize=9, fontweight='bold')
ax_right.set_xlim(0, 22)
ax_right.spines['top'].set_visible(False)
ax_right.spines['right'].set_visible(False)
ax_right.spines['left'].set_linewidth(0.5)
ax_right.spines['bottom'].set_linewidth(0.5)
ax_right.tick_params(axis='y', length=0)
ax_right.grid(axis='x', alpha=0.15, lw=0.5)

# Note about uncategorized
ax_right.text(15.5, 0.3, f'+ {uncategorized} uncategorised',
              fontsize=7, color=C_TEXT2, fontstyle='italic')

ax_right.set_title(f'Functional grouping of\n{n_shared} common rescued genes', fontsize=10,
                     fontweight='bold', color=C_TEXT, pad=10)

# ── Suptitle ─────────────────────────────────────────────────────
fig.suptitle('Fig. 3.6C \u2014 Biological functions of genes rescued in both kidney and muscle',
             fontsize=11, fontweight='bold', color=C_TEXT, y=0.99)

# ── Footnote ─────────────────────────────────────────────────────
fig.text(0.5, 0.02,
         'Left: Top 10 GO:BP terms from the kidney rescued-to-normal enrichment (enrichGO, padj < 0.05) that contain the most common rescued genes.\n'
         f'Formal ORA on the {n_shared} common genes alone yielded only one significant term (Golgi organization, padj = {golgi_padj:.3f}) due to small input set size.\n'
         'Right: Manual functional grouping based on gene ontology annotations and literature. Categories are not mutually exclusive.',
         ha='center', fontsize=6, color=C_TEXT2, linespacing=1.5)

save_fig(fig, FILENAME)
print('Done \u2014 Fig 3.6C')

## Fig 3.6D — Summary comparison table


In [ ]:
# ── Fig 3.6D — Summary comparison table ──────────────────────────
FILENAME = 'Fig_3_6D_summary_comparison_table'

# Redefine count_rows — defined in diagnostic cell, may not be in memory
def count_rows(filepath):
    try:
        with open(filepath) as f:
            return sum(1 for _ in csv.DictReader(f))
    except FileNotFoundError:
        print(f"Not found: {filepath}")
        return 0

# ── Font sizes ────────────────────────────────────────────────────
FS_CAPTION  = 11.5
FS_CAPTION2 = 9.5
FS_HEADER   = 11
FS_SECTION  = 10
FS_DATA     = 9.5
FS_FOOTNOTE = 7.5

# ── Auto-load rescue framework numbers ───────────────────────────
k_rtn_n    = count_rows(f"{DATA_BASE}/Kidney/1_DEGs_tables/kidney_rescued_to_normal.csv")
k_nonres_n = count_rows(f"{DATA_BASE}/Kidney/1_DEGs_tables/kidney_non_rescued_aged_DEGs.csv")
k_dr_n     = count_rows(f"{DATA_BASE}/Kidney/3_Effect_comparison_tables/kidney_intervention_impact_comparison_DR.csv")
k_sact_n   = count_rows(f"{DATA_BASE}/Kidney/3_Effect_comparison_tables/kidney_intervention_impact_comparison_sAct.csv")
k_combi_n  = count_rows(f"{DATA_BASE}/Kidney/3_Effect_comparison_tables/kidney_intervention_impact_comparison_only_rescued_in_combi.csv")
k_both_n   = k_rtn_n - k_dr_n - k_sact_n - k_combi_n

with open(f"{DATA_BASE}/Kidney/1_DEGs_tables/kidney_res_age_results_with_symbols.csv") as f:
    k_age_n = sum(1 for r in csv.DictReader(f)
                  if r.get('padj','') not in ('','NA') and float(r.get('padj',1)) < 0.05)
with open(f"{DATA_BASE}/Kidney/1_DEGs_tables/kidney_combined_rescued_genes_DE_info.csv") as f:
    k_rescued_n = sum(1 for _ in csv.DictReader(f))

m_rtn_n    = count_rows(f"{DATA_BASE}/Muscle_old/1_DEGs_tables/muscle_rescued_to_normal.csv")
m_nonres_n = count_rows(f"{DATA_BASE}/Muscle_old/1_DEGs_tables/muscle_non_rescued_aged_DEGs.csv")
m_dr_n     = count_rows(f"{DATA_BASE}/Muscle_old/3_Effect_comparison_tables/muscle_intervention_impact_comparison_DR.csv")
m_sact_n   = count_rows(f"{DATA_BASE}/Muscle_old/3_Effect_comparison_tables/muscle_intervention_impact_comparison_sAct.csv")
m_combi_n  = count_rows(f"{DATA_BASE}/Muscle_old/3_Effect_comparison_tables/muscle_intervention_impact_comparison_only_rescued_in_combi.csv")
m_both_n   = m_rtn_n - m_dr_n - m_sact_n - m_combi_n

with open(f"{DATA_BASE}/Muscle_old/1_DEGs_tables/muscle_res_age_results_with_symbols.csv") as f:
    m_age_n = sum(1 for r in csv.DictReader(f)
                  if r.get('padj','') not in ('','NA') and float(r.get('padj',1)) < 0.05)
with open(f"{DATA_BASE}/Muscle_old/1_DEGs_tables/muscle_combined_rescued_genes_DE_info.csv") as f:
    m_rescued_n = sum(1 for _ in csv.DictReader(f))

rows = [
    # Bulk RNA-seq
    (0, 'Bulk RNA-seq',                            '',                            '',                            True,  '#EEF4FB', False, None),
    (1, 'Genes tested',                            '22,109',                      '21,055',                      False, C_WHITE,  False, None),
    (1, 'Aging DEGs (padj < 0.05)',                f'{k_age_n:,}',                f'{m_age_n:,}',                False, C_WHITE,  False, 'kidney'),
    (1, 'Pipeline',                                'Salmon + DESeq2',             'FeatureCounts + DESeq2',      False, C_WHITE,  True,  None),
    (1, 'Reference genome',                        'GRCm39 / Ensembl 110',        'GRCm38 / GENCODE M20',        False, C_WHITE,  True,  None),
    (0, '',                                        '',                            '',                            False, C_WHITE,  False, None),
    # Rescue framework
    (0, 'Rescue framework',                        '',                            '',                            True,  '#EDF8F2', False, None),
    (1, 'Rescued by combi (Stage 1)',              f'{k_rescued_n:,} ({k_rescued_n/k_age_n*100:.0f}%)',  f'{m_rescued_n:,} ({m_rescued_n/m_age_n*100:.0f}%)',  False, C_WHITE, False, None),
    (1, 'Rescued-to-normal (Stage 2)',             f'{k_rtn_n:,}',                f'{m_rtn_n:,}',                False, C_WHITE,  False, None),
    (1, 'DR-exclusive',                            f'{k_dr_n:,} ({k_dr_n/k_rtn_n*100:.0f}%)',    f'{m_dr_n:,} ({m_dr_n/m_rtn_n*100:.0f}%)',    False, C_WHITE, False, 'muscle'),
    (1, 'sAct-exclusive',                          f'{k_sact_n:,} ({k_sact_n/k_rtn_n*100:.0f}%)',  f'{m_sact_n:,} ({m_sact_n/m_rtn_n*100:.0f}%)',  False, C_WHITE, False, None),
    (1, 'Combi-only (synergistic)',                f'{k_combi_n:,} ({k_combi_n/k_rtn_n*100:.0f}%)', f'{m_combi_n:,} ({m_combi_n/m_rtn_n*100:.0f}%)', False, C_WHITE, False, None),
    (1, 'Both DR + sAct',                          f'{k_both_n:,} ({k_both_n/k_rtn_n*100:.0f}%)',   f'{m_both_n:,} ({m_both_n/m_rtn_n*100:.0f}%)',   False, C_WHITE, False, 'kidney'),
    (1, 'Non-rescued aging DEGs',                  f'{k_nonres_n:,} ({k_nonres_n/k_age_n*100:.1f}%)', f'{m_nonres_n:,} ({m_nonres_n/m_age_n*100:.1f}%)', False, C_WHITE, True, None),
    (0, '',                                        '',                            '',                            False, C_WHITE,  False, None),
    (0, 'Top enriched pathways',                   '',                     '',                     True,  '#FDF8EE', False, None),
    (1, 'Aging signature',                         'Fatty acid metabolism', 'Striated muscle diff.', False, C_WHITE, False, None),
    (1, 'Rescued-to-normal',                       'Cellular respiration',  '42 GOBP terms',        False, C_WHITE,  False, None),
    (1, 'Non-rescued',                             'Mito gene expression,\nprotein folding', '1,100 GOBP terms', False, C_WHITE, False, None),
    (0, '',                                        '',                     '',                     False, C_WHITE,  False, None),
    (0, 'Single-nucleus RNA-seq',                  '',                     '',                     True,  '#F5EDF8', False, None),
    (1, 'Total nuclei',                            '58,504',               '24,419',               False, C_WHITE,  False, None),
    (1, 'Cell types annotated',                    '18',                   '15',                   False, C_WHITE,  False, None),
    (1, 'Reversal gene markers (hits)',            '986',                  '292',                  False, C_WHITE,  False, None),
    (1, 'Unique reversal genes as markers',        '354',                  '94',                   False, C_WHITE,  False, None),
    (1, 'Top rescue cell types',                   'PT-2, FIB, vSMC/MC,\nEC-2/3/4, IMM', 'Myh4+ myonuclei,\nMacrophages, NMJ', False, C_WHITE, False, None),
    (1, 'EC rescue contribution',                  'Major (6 subtypes)',    'Minimal (20 markers)', False, C_WHITE,  False, 'kidney'),
    (1, 'Key shared gene',                         'Tgfb1 (IMM, EC-2/3)',  'Tgfb1 (Macrophages)',  False, C_WHITE,  False, 'both'),
    (0, '',                                        '',                     '',                     False, C_WHITE,  False, None),
    (0, 'Cell\u2013cell communication (CellChat)',  '',                    '',                     True,  '#FDF2F2', False, None),
    (1, 'Pathways analysed',                       '13',                   '7',                    False, C_WHITE,  False, None),
    (1, 'Key signalling hubs',                     'VEGF, EGF, PDGF,\nSEMA3, EPHA/B', 'TGFb, ANGPTL,\nPTPRM, KIT', False, C_WHITE, False, None),
    (0, '',                                        '',                     '',                     False, C_WHITE,  False, None),
    (0, 'Cross-tissue integration',                '',                     '',                     True,  '#EEF0EE', False, None),
    (1, 'Common rescued-to-normal genes',          '98',                   '98',                   False, C_WHITE,  False, 'both'),
    (1, 'Direction concordance',                   '84/98 (86%)',          '84/98 (86%)',           False, C_WHITE,  False, 'both'),
    (1, 'Pearson r (aging LFC)',                   '0.751',                '0.751',                False, C_WHITE,  False, 'both'),
    (1, 'Overlap as % of tissue RTN set',          '5.1%',                 '19.2%',                False, C_WHITE,  False, 'muscle'),
]

fig, ax = plt.subplots(1, 1, figsize=(12, 13))
ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.axis('off')

tl, tr = 0.03, 0.97
tw = tr - tl
col_w = [0.40, 0.30, 0.30]
col_x = [tl]
for w in col_w[:-1]:
    col_x.append(col_x[-1] + w * tw)

rh = 0.024; rh_sec = 0.026; hdr_h = 0.028

ax.text(tl, 0.98, 'Fig. 3.6D.', fontsize=FS_CAPTION, fontweight='bold',
        color=C_TEXT, va='top')
ax.text(tl + 0.065, 0.98,
        'Side-by-side comparison of key results metrics for kidney and muscle datasets.',
        fontsize=FS_CAPTION, color=C_TEXT, va='top')
ax.text(tl, 0.96,
        'Highlights indicate tissue with the more notable value for that metric (green = emphasized).',
        fontsize=FS_CAPTION2, color=C_TEXT2, va='top')

y_top = 0.945
ax.plot([tl, tr], [y_top, y_top], color=C_TEXT, lw=1.5, clip_on=False)

headers = ['Metric', 'Kidney', 'Muscle']
aligns  = ['left', 'center', 'center']
pads    = [0.01, 0, 0]
for i, (lab, align, pad) in enumerate(zip(headers, aligns, pads)):
    hx = col_x[i] + pad if align == 'left' else col_x[i] + col_w[i]*tw/2
    ax.text(hx, y_top - hdr_h/2, lab, fontsize=FS_HEADER, fontweight='bold',
            color=C_TEXT, va='center', ha=align if align != 'center' else 'center')

ax.plot(col_x[1] + col_w[1]*tw/2 - 0.05, y_top - hdr_h/2, 'o',
        color=C_RED_D, markersize=6, transform=ax.transAxes, clip_on=False)
ax.plot(col_x[2] + col_w[2]*tw/2 - 0.04, y_top - hdr_h/2, 'o',
        color=C_BLUE_D, markersize=6, transform=ax.transAxes, clip_on=False)

y_hdr_bot = y_top - hdr_h
ax.plot([tl, tr], [y_hdr_bot, y_hdr_bot], color=C_TEXT, lw=0.8, clip_on=False)

y_cur = y_hdr_bot
highlight_fill = '#E8F5E9'

for indent, cat, kidney, muscle, is_sec, bg, is_gray, hl in rows:
    if cat == '' and kidney == '' and muscle == '':
        y_cur -= rh * 0.3; continue

    this_rh = rh_sec if is_sec else rh
    n_lines = max(kidney.count('\n'), muscle.count('\n'), cat.count('\n')) + 1
    if n_lines > 1:
        this_rh = rh * n_lines * 0.9

    if bg != C_WHITE:
        ax.add_patch(mpatches.Rectangle((tl, y_cur-this_rh), tw, this_rh,
                                         fc=bg, ec='none', zorder=0))
    if hl == 'kidney':
        ax.add_patch(mpatches.Rectangle((col_x[1], y_cur-this_rh), col_w[1]*tw, this_rh,
                                         fc=highlight_fill, ec='none', zorder=0))
    elif hl == 'muscle':
        ax.add_patch(mpatches.Rectangle((col_x[2], y_cur-this_rh), col_w[2]*tw, this_rh,
                                         fc=highlight_fill, ec='none', zorder=0))
    elif hl == 'both':
        ax.add_patch(mpatches.Rectangle((col_x[1], y_cur-this_rh), (col_w[1]+col_w[2])*tw, this_rh,
                                         fc=highlight_fill, ec='none', zorder=0))

    ax.plot([tl, tr], [y_cur-this_rh, y_cur-this_rh],
            color='#E5E5E5', lw=0.3, clip_on=False, zorder=1)

    tc = C_TEXT2 if is_gray else C_TEXT
    ax.text(col_x[0]+0.01+indent*0.02, y_cur-this_rh/2, cat,
            fontsize=FS_SECTION if is_sec else FS_DATA,
            fontweight='bold' if is_sec else 'normal',
            color=tc, va='center', ha='left', zorder=2, linespacing=1.3)
    if kidney:
        ax.text(col_x[1]+col_w[1]*tw/2, y_cur-this_rh/2, kidney,
                fontsize=FS_DATA, color=tc, va='center', ha='center',
                zorder=2, linespacing=1.3)
    if muscle:
        ax.text(col_x[2]+col_w[2]*tw/2, y_cur-this_rh/2, muscle,
                fontsize=FS_DATA, color=tc, va='center', ha='center',
                zorder=2, linespacing=1.3)
    y_cur -= this_rh

ax.plot([tl, tr], [y_cur, y_cur], color=C_TEXT, lw=1.5, clip_on=False)

fn_y = y_cur - 0.015
fns = [
    'Green highlighting indicates the tissue with the more prominent value for that metric.',
    '\u2020 Muscle non-rescued count requires verification (see Table 3.2A footnote).',
    'Percentages in rescue framework rows are relative to rescued-to-normal set (Stage 3) or aging DEGs (Stage 1).',
    'Pipeline note: cross-tissue DEG count comparisons should account for different processing pipelines (see Section 3.1.1).',
    'CellChat results are exploratory (n = 1 per condition). snRNA-seq module scores use bulk-derived gene sets.',
]
for fn in fns:
    ax.text(tl+0.01, fn_y, fn, fontsize=FS_FOOTNOTE, color=C_TEXT2, va='top')
    fn_y -= 0.020

plt.tight_layout(pad=0.3)
save_fig(fig, FILENAME)
print('Done \u2014 Fig 3.6D')

## Fig 3.6E ─ Integrative schematic summarizing the rescue model across tissues

In [ ]:
FILENAME = 'Fig_3_6E_integrative_rescue_model'

fig, ax = plt.subplots(figsize=(14, 16))
ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.axis('off')

# ── Helpers ──────────────────────────────────────────────────────
def rbox(cx, cy, w, h, fc, ec, title, sub=None, fs=9, fs_sub=6.5, lw=0.8):
    ax.add_patch(FancyBboxPatch((cx-w/2, cy-h/2), w, h, boxstyle="round,pad=0.012",
                                 facecolor=fc, edgecolor=ec, linewidth=lw,
                                 transform=ax.transAxes, zorder=2))
    if sub:
        ax.text(cx, cy+h*0.18, title, transform=ax.transAxes, ha='center', va='center',
                fontsize=fs, fontweight='bold', color=ec, zorder=3)
        ax.text(cx, cy-h*0.22, sub, transform=ax.transAxes, ha='center', va='center',
                fontsize=fs_sub, color=C_TEXT2, zorder=3, linespacing=1.3)
    else:
        ax.text(cx, cy, title, transform=ax.transAxes, ha='center', va='center',
                fontsize=fs, fontweight='bold', color=ec, zorder=3)

def arrow(x1, y1, x2, y2, color=C_TEXT2, lw=1.0, style='->', head='->'):
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                xycoords='axes fraction', textcoords='axes fraction',
                arrowprops=dict(arrowstyle=head, color=color, lw=lw,
                                shrinkA=2, shrinkB=2),
                zorder=1)

def bracket_text(x, y1, y2, text, side='right', color=C_TEXT2):
    """Draw a vertical bracket with text."""
    xoff = 0.008 if side == 'right' else -0.008
    ax.plot([x, x+xoff, x+xoff, x], [y1, y1, y2, y2],
            color=color, lw=0.6, transform=ax.transAxes, zorder=1, clip_on=False)
    ax.text(x + xoff*3, (y1+y2)/2, text, transform=ax.transAxes,
            ha='left' if side == 'right' else 'right', va='center',
            fontsize=6.5, color=color, fontstyle='italic')

# ══════════════════════════════════════════════════════════════════
# LAYER 1 — TOP: The aging model
# ══════════════════════════════════════════════════════════════════
ax.text(0.50, 0.975, 'Fig. 3.6E \u2014 Integrative model of transcriptional rescue',
        ha='center', fontsize=12, fontweight='bold', color=C_TEXT, transform=ax.transAxes)
ax.text(0.50, 0.96, 'Ercc1\u0394/\u2212 accelerated aging model: combined DR + sActRIIB intervention',
        ha='center', fontsize=9, color=C_TEXT2, transform=ax.transAxes)

# DNA damage / Ercc1 box
rbox(0.50, 0.925, 0.35, 0.035, C_GRAY, C_GRAY_D,
     'Ercc1\u0394/\u2212: impaired nucleotide excision repair',
     'Genomic instability \u2192 accelerated multisystem aging', fs=8.5, fs_sub=6)

# Arrow down to aging
arrow(0.50, 0.907, 0.50, 0.895, C_GRAY_D)

# Aging signature box
rbox(0.50, 0.878, 0.30, 0.030, C_RED, C_RED_D,
     'Aging transcriptional signature', fs=8.5)

# Split into two tissues
arrow(0.40, 0.863, 0.25, 0.843, C_RED_D, lw=0.8)
arrow(0.60, 0.863, 0.75, 0.843, C_RED_D, lw=0.8)

# ══════════════════════════════════════════════════════════════════
# LAYER 2 — TISSUE AGING PANELS
# ══════════════════════════════════════════════════════════════════
# Kidney aging panel
kx = 0.25
rbox(kx, 0.838, 0.28, 0.028, C_RED, C_RED_D,
     'Kidney: 7,499 aging DEGs', fs=8)

# Muscle aging panel
mx = 0.75
rbox(mx, 0.838, 0.28, 0.028, C_RED, C_RED_D,
     'Muscle: 1,739 aging DEGs', fs=8)

# Kidney aging pathways
k_aging_y = 0.793
rbox(kx, k_aging_y, 0.26, 0.042, C_RED, C_RED_D,
     'Aging pathways', 'Fatty acid metabolism, oxidative\nstress, amide metabolism', fs=7.5, fs_sub=5.5)

# Muscle aging pathways
rbox(mx, k_aging_y, 0.26, 0.042, C_RED, C_RED_D,
     'Aging pathways', 'Muscle differentiation,\ncontraction, development', fs=7.5, fs_sub=5.5)

arrow(kx, 0.814, kx, 0.814, C_RED_D, lw=0.5)
arrow(mx, 0.814, mx, 0.814, C_RED_D, lw=0.5)

# ══════════════════════════════════════════════════════════════════
# LAYER 3 — INTERVENTION (center)
# ══════════════════════════════════════════════════════════════════
intv_y = 0.735

# Large intervention box spanning center
ax.add_patch(FancyBboxPatch((0.15, intv_y - 0.025), 0.70, 0.05,
             boxstyle="round,pad=0.015", facecolor=C_GREEN, edgecolor=C_GREEN_D,
             linewidth=1.2, transform=ax.transAxes, zorder=2))
ax.text(0.50, intv_y + 0.008, 'Combined intervention: Dietary Restriction + sActRIIB',
        ha='center', fontsize=9.5, fontweight='bold', color=C_GREEN_D,
        transform=ax.transAxes, zorder=3)
ax.text(0.50, intv_y - 0.012,
        'DR: 30% caloric restriction from 9 weeks  |  sActRIIB: 10 mg/kg i.p. 2\u00d7/week from 8 weeks',
        ha='center', fontsize=6, color=C_TEXT2, transform=ax.transAxes, zorder=3)

# Arrows from aging panels down to intervention
arrow(kx, 0.772, 0.35, intv_y + 0.025, C_GREEN_D, lw=0.8, head='->')
arrow(mx, 0.772, 0.65, intv_y + 0.025, C_GREEN_D, lw=0.8, head='->')

# ══════════════════════════════════════════════════════════════════
# LAYER 4 — RESCUE OUTCOMES
# ══════════════════════════════════════════════════════════════════
rescue_y = 0.672

# Arrows from intervention to rescue
arrow(0.35, intv_y - 0.025, kx, rescue_y + 0.020, C_TEAL_D, lw=0.8)
arrow(0.65, intv_y - 0.025, mx, rescue_y + 0.020, C_TEAL_D, lw=0.8)

# Kidney rescue
rbox(kx, rescue_y, 0.26, 0.035, C_TEAL, C_TEAL_D,
     '1,961 rescued-to-normal',
     '62% of aging DEGs reversed by combi', fs=8, fs_sub=5.5)

# Muscle rescue
rbox(mx, rescue_y, 0.26, 0.035, C_TEAL, C_TEAL_D,
     '511 rescued-to-normal',
     '53% of aging DEGs reversed by combi', fs=8, fs_sub=5.5)

# ══════════════════════════════════════════════════════════════════
# LAYER 5 — DRIVER BREAKDOWN
# ══════════════════════════════════════════════════════════════════
driver_y = 0.61

# Kidney drivers — small stacked boxes
k_drivers = [
    ('DR-excl. 28%', C_GREEN, C_GREEN_D, 0.07),
    ('Both 62%',     C_CORAL, C_CORAL_D, 0.07),
    ('sAct 6%',      C_PURPLE, C_PURP_D, 0.04),
    ('Syn. 4%',      C_AMBER, C_AMBER_D, 0.04),
]
kd_x = 0.11
for label, fc, ec, w in k_drivers:
    rbox(kd_x + w/2, driver_y, w, 0.022, fc, ec, label, fs=6)
    kd_x += w + 0.005

# Muscle drivers
m_drivers = [
    ('DR-excl. 47%', C_GREEN, C_GREEN_D, 0.08),
    ('Both 45%',     C_CORAL, C_CORAL_D, 0.07),
    ('sAct 5%',      C_PURPLE, C_PURP_D, 0.035),
    ('Syn. 4%',      C_AMBER, C_AMBER_D, 0.035),
]
md_x = 0.61
for label, fc, ec, w in m_drivers:
    rbox(md_x + w/2, driver_y, w, 0.022, fc, ec, label, fs=6)
    md_x += w + 0.005

# Labels
ax.text(kx, driver_y + 0.025, 'Intervention drivers:', ha='center',
        fontsize=6.5, fontweight='bold', color=C_TEXT2, transform=ax.transAxes)
ax.text(mx, driver_y + 0.025, 'Intervention drivers:', ha='center',
        fontsize=6.5, fontweight='bold', color=C_TEXT2, transform=ax.transAxes)

# ══════════════════════════════════════════════════════════════════
# LAYER 6 — CELL-TYPE RESOLUTION
# ══════════════════════════════════════════════════════════════════
ct_y = 0.535

ax.text(0.50, 0.580, 'Cell-type resolution (snRNA-seq module score projection)',
        ha='center', fontsize=8, fontweight='bold', color=C_TEXT, transform=ax.transAxes)
ax.plot([0.10, 0.90], [0.573, 0.573], color='#E0E0E0', lw=0.5, transform=ax.transAxes)

# Kidney cell types
k_cells = [
    ('EC-2/3/4',  'Tgfb1, Flt1, B2m\nVascular rescue',     C_GREEN,  C_GREEN_D),
    ('FIB',       'Flt1, Prkd1\nECM remodelling',           C_TEAL,   C_TEAL_D),
    ('IMM',       'Tgfb1, Hcls1\nImmune rebalancing',       C_PURPLE, C_PURP_D),
    ('PT-2',      'Gstm1, Acox2\nMetabolic rescue',         C_TEAL,   C_TEAL_D),
    ('PODO',      'Prkd1, Plekho1\nGlomerular',             C_TEAL,   C_TEAL_D),
]

kcx_start = 0.06
kcx_step = 0.086
for i, (name, sub, fc, ec) in enumerate(k_cells):
    x = kcx_start + i * kcx_step + 0.04
    rbox(x, ct_y, 0.075, 0.048, fc, ec, name, sub, fs=7, fs_sub=4.5)

# Muscle cell types
m_cells = [
    ('Myh4+\nmyonuclei', 'Kcnq5, Wwox\nFibre maintenance',  C_TEAL,   C_TEAL_D),
    ('Macro-\nphages',    'Tgfb1, Hcls1\nImmune hub',        C_PURPLE, C_PURP_D),
    ('NMJ\nnuclei',      'Ryr3, Ptprm\nLong-gene rescue',   C_AMBER,  C_AMBER_D),
    ('Muscle\nstem',      'Ryr3, Pxdn\nRegeneration',        C_TEAL,   C_TEAL_D),
    ('Endo-\nthelial',   'Minimal\nresponse',                C_GRAY,   C_GRAY_D),
]

mcx_start = 0.54
for i, (name, sub, fc, ec) in enumerate(m_cells):
    x = mcx_start + i * kcx_step + 0.04
    rbox(x, ct_y, 0.075, 0.048, fc, ec, name, sub, fs=6.5, fs_sub=4.5)

# Tissue labels above cell type rows
ax.text(kx, ct_y + 0.038, 'Kidney cell types', ha='center', fontsize=8,
        fontweight='bold', color=C_RED_D, transform=ax.transAxes)
ax.text(mx, ct_y + 0.038, 'Muscle cell types', ha='center', fontsize=8,
        fontweight='bold', color=C_BLUE_D, transform=ax.transAxes)

# ══════════════════════════════════════════════════════════════════
# LAYER 7 — SHARED CORE (center)
# ══════════════════════════════════════════════════════════════════
shared_y = 0.445

# Large shared box
ax.add_patch(FancyBboxPatch((0.22, shared_y - 0.040), 0.56, 0.080,
             boxstyle="round,pad=0.015", facecolor=C_PURPLE, edgecolor=C_PURP_D,
             linewidth=1.0, alpha=0.5, transform=ax.transAxes, zorder=2))
ax.text(0.50, shared_y + 0.022, 'Conserved rescue core: 98 genes (r = 0.75, 86% concordant)',
        ha='center', fontsize=8.5, fontweight='bold', color=C_PURP_D,
        transform=ax.transAxes, zorder=3)

# Shared gene modules in a row
modules = [
    ('Tgfb1\nsignalling', C_CORAL, C_CORAL_D),
    ('Immune\nrebalancing', C_PURPLE, C_PURP_D),
    ('Golgi/vesicle\ntransport', C_TEAL, C_TEAL_D),
    ('Ubiquitin/\nproteostasis', C_AMBER, C_AMBER_D),
    ('Cell adhesion/\ncytoskeleton', C_BLUE, C_BLUE_D),
]

mod_w = 0.085
mod_start = 0.50 - (len(modules) * (mod_w + 0.008)) / 2 + mod_w/2
for i, (label, fc, ec) in enumerate(modules):
    x = mod_start + i * (mod_w + 0.008)
    rbox(x, shared_y - 0.012, mod_w, 0.032, fc, ec, label, fs=5.5)

# Arrows from cell types to shared core
# IMM (kidney, index 2) and Macrophages (muscle, index 1) — symmetrical about center
imm_x = kcx_start + 2 * kcx_step + 0.04     # kidney IMM x position
macro_x = mcx_start + 1 * kcx_step + 0.04    # muscle Macrophages x position
arrow(imm_x, ct_y - 0.024, 0.38, shared_y + 0.040, C_PURP_D, lw=0.6)
arrow(macro_x, ct_y - 0.024, 0.62, shared_y + 0.040, C_PURP_D, lw=0.6)

# ══════════════════════════════════════════════════════════════════
# LAYER 8 — TISSUE-SPECIFIC INSIGHTS
# ══════════════════════════════════════════════════════════════════
ins_y = 0.355

ax.plot([0.10, 0.90], [0.385, 0.385], color='#E0E0E0', lw=0.5, transform=ax.transAxes)
ax.text(0.50, 0.395, 'Tissue-specific rescue programmes',
        ha='center', fontsize=8, fontweight='bold', color=C_TEXT, transform=ax.transAxes)

# Kidney insights
k_ins = [
    'Vascular rescue: Flt1/VEGFR1 in 7 EC subtypes + FIB',
    'Broadly distributed: 23 cell types contribute',
    'Podocyte programmes: Golgi integrity (Prkd1)',
    'B2m senescence in IMM + EC-3',
    'ECM remodelling centred on FIB + vSMC/MC',
]

ax.add_patch(FancyBboxPatch((0.04, ins_y - 0.065), 0.42, 0.12,
             boxstyle="round,pad=0.012", facecolor=C_RED, edgecolor=C_RED_D,
             linewidth=0.6, alpha=0.3, transform=ax.transAxes, zorder=0))
ax.text(kx, ins_y + 0.045, 'Kidney', ha='center', fontsize=9,
        fontweight='bold', color=C_RED_D, transform=ax.transAxes)
for i, txt in enumerate(k_ins):
    ax.text(kx, ins_y + 0.02 - i*0.020, '\u2022 ' + txt, ha='center',
            fontsize=5.5, color=C_TEXT2, transform=ax.transAxes)

# Muscle insights
m_ins = [
    'NMJ long-gene vulnerability (Ryr3, Kcnq5, Wwox >500kb)',
    'Concentrated: top 3 cell types carry 73% of markers',
    'DR disproportionately dominant (47% vs 28%)',
    'ECs transcriptionally quiescent (contrast to kidney)',
    'Myonuclei fibre-type maintenance (Myh4+)',
]

ax.add_patch(FancyBboxPatch((0.54, ins_y - 0.065), 0.42, 0.12,
             boxstyle="round,pad=0.012", facecolor=C_BLUE, edgecolor=C_BLUE_D,
             linewidth=0.6, alpha=0.3, transform=ax.transAxes, zorder=0))
ax.text(mx, ins_y + 0.045, 'Muscle', ha='center', fontsize=9,
        fontweight='bold', color=C_BLUE_D, transform=ax.transAxes)
for i, txt in enumerate(m_ins):
    ax.text(mx, ins_y + 0.02 - i*0.020, '\u2022 ' + txt, ha='center',
            fontsize=5.5, color=C_TEXT2, transform=ax.transAxes)

# ══════════════════════════════════════════════════════════════════
# LAYER 9 — CELL-CELL COMMUNICATION
# ══════════════════════════════════════════════════════════════════
cc_y = 0.215

ax.plot([0.10, 0.90], [0.260, 0.260], color='#E0E0E0', lw=0.5, transform=ax.transAxes)
ax.text(0.50, 0.270, 'Cell\u2013cell communication (CellChat: exploratory)',
        ha='center', fontsize=8, fontweight='bold', color=C_TEXT, transform=ax.transAxes)

# Kidney CellChat
k_cc = [
    ('VEGF', 'Lost to vSMC/IMM;\nrestored by treatments'),
    ('EGF', 'Partial restoration\nby sAct, combi'),
    ('COLLAGEN', 'Silenced by aging;\nDR/combi restore'),
]
for i, (path, note) in enumerate(k_cc):
    x = 0.10 + i * 0.14 + 0.06
    rbox(x, cc_y, 0.11, 0.045, C_TEAL, C_TEAL_D, path, note, fs=7, fs_sub=4.5)

# Muscle CellChat
m_cc = [
    ('TGFb', 'Macrophage-driven;\nrescue hub'),
    ('ANGPTL', 'Cross-celltype\nsignalling'),
    ('PTPRM', 'NMJ-associated\nsignalling'),
]
for i, (path, note) in enumerate(m_cc):
    x = 0.56 + i * 0.14 + 0.04
    rbox(x, cc_y, 0.11, 0.045, C_TEAL, C_TEAL_D, path, note, fs=7, fs_sub=4.5)

# Inter-organ label
rbox(0.50, cc_y - 0.055, 0.30, 0.030, C_AMBER, C_AMBER_D,
     'Inter-organ: ANGPT, SEMA3, COLLAGEN signalling rescued', fs=6.5)

# ══════════════════════════════════════════════════════════════════
# LAYER 10 — NON-RESCUED / LIMITATIONS
# ══════════════════════════════════════════════════════════════════
nr_y = 0.105

ax.plot([0.10, 0.90], [0.135, 0.135], color='#E0E0E0', lw=0.5, transform=ax.transAxes)

rbox(0.50, nr_y, 0.70, 0.045, C_GRAY, C_GRAY_D,
     'Non-rescued processes: mitochondrial gene expression, protein folding / chaperone pathways',
     'Proteostasis may operate at protein activity level rather than transcription (see Discussion 4.4)',
     fs=7.5, fs_sub=5.5)

# ══════════════════════════════════════════════════════════════════
# BOTTOM — Key message
# ══════════════════════════════════════════════════════════════════
ax.add_patch(FancyBboxPatch((0.10, 0.025), 0.80, 0.045,
             boxstyle="round,pad=0.012", facecolor='#F0F7F0', edgecolor=C_GREEN_D,
             linewidth=1.0, transform=ax.transAxes, zorder=2))
ax.text(0.50, 0.055, 'Combined DR + sActRIIB rescues aging signatures through converging and tissue-specific programmes',
        ha='center', fontsize=8, fontweight='bold', color=C_GREEN_D, transform=ax.transAxes, zorder=3)
ax.text(0.50, 0.037,
        'Immune rebalancing (Tgfb1) is the shared hub | Kidney: vascular/stromal | Muscle: neuromuscular/NMJ | DR is the dominant driver in both',
        ha='center', fontsize=6, color=C_TEXT2, transform=ax.transAxes, zorder=3)

# ── Footnote ─────────────────────────────────────────────────────
ax.text(0.50, 0.008,
        'Bulk RNA-seq: n = 3/condition (statistically powered). snRNA-seq: n = 1/condition (exploratory cell-type localization). CellChat: exploratory.',
        ha='center', fontsize=5.5, color=C_TEXT2, transform=ax.transAxes)

plt.tight_layout(pad=0.3)
save_fig(fig, FILENAME)
print('Done \u2014 Fig 3.6E')

# Supplementary Tables (all GOBP enrichment CSVs)

In [ ]:
# ── Supplementary Tables ─────────────────────────────────────────
# Compile all GOBP enrichment CSVs into formatted xlsx files
FILENAME_S = 'Supplementary_Tables_GOBP_Enrichment'

import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

# ── Define all enrichment CSVs to compile ────────────────────────
K_GOBP = f"{DATA_BASE}/Kidney/2_Enrichment_bulk&sn/GOBP bulk"
M_GOBP = f"{DATA_BASE}/Muscle_old/2_Enrichment_tables/GOBP bulk"
K_CT = f"{DATA_BASE}/Kidney/2_Enrichment_bulk&sn/GOBP_per_celltype/rescued"
M_CT = f"{DATA_BASE}/Muscle_old/2_Enrichment_tables/GOBP_per_celltype/rescued"

sheets_bulk = [
    # (sheet_name, filepath, description)
    ('S3.4a_K_aging', f'{K_GOBP}/kidney_enrichment_results_aging.csv',
     'Kidney GOBP enrichment of aging DEGs (7,499 genes)'),
    ('S3.4b_M_aging', f'{M_GOBP}/muscle_enrichment_results_aging.csv',
     'Muscle GOBP enrichment of aging DEGs (1,739 genes)'),
    ('S3.4c_K_rescued', f'{K_GOBP}/kidney_enrichment_results_rescued-to-normal.csv',
     'Kidney GOBP enrichment of rescued-to-normal genes (1,961 genes)'),
    ('S3.4d_M_rescued', f'{M_GOBP}/muscle_enrichment_results_rescued_to_normal.csv',
     'Muscle GOBP enrichment of rescued-to-normal genes (511 genes)'),
    ('S3.4e_K_nonrescued', f'{K_GOBP}/kidney_enrichment_results_non_rescued_genes.csv',
     'Kidney GOBP enrichment of non-rescued genes (1,169 genes)'),
    ('S3.4f_M_nonrescued', f'{M_GOBP}/muscle_enrichment_results_non_rescued_genes.csv',
     'Muscle GOBP enrichment of non-rescued genes (300 genes)'),
    ('S3.4g_K_DR', f'{K_GOBP}/kidney_enrichment_results_rescued-to-normal__DR.csv',
     'Kidney GOBP enrichment of DR-exclusive rescued genes (546 genes)'),
    ('S3.4h_M_DR', f'{M_GOBP}/muscle_enrichment_results_rescued-to-normal__DR.csv',
     'Muscle GOBP enrichment of DR-exclusive rescued genes (241 genes)'),
    ('S3.4i_K_sAct', f'{K_GOBP}/kidney_enrichment_results_rescued-to-normal_sAct.csv',
     'Kidney GOBP enrichment of sAct-exclusive rescued genes (108 genes)'),
    ('S3.4j_M_sAct', f'{M_GOBP}/muscle_enrichment_results_rescued-to-normal_sAct.csv',
     'Muscle GOBP enrichment of sAct-exclusive rescued genes (23 genes)'),
    ('S3.4k_K_combi', f'{K_GOBP}/kidney_enrichment_results_rescued-to-normal_only_in_combi.csv',
     'Kidney GOBP enrichment of combi-only rescued genes (87 genes)'),
    ('S3.4l_M_combi', f'{M_GOBP}/muscle_enrichment_results_rescued-to-normal_only_in_combi.csv',
     'Muscle GOBP enrichment of combi-only rescued genes (18 genes)'),
]

# Per-celltype sheets (top cell types only)
import os
sheets_celltype = []

# Kidney per-celltype
for subdir in ['ECs', 'IMM', 'Podo', 'Rest']:
    dirpath = f'{K_CT}/{subdir}'
    if os.path.isdir(dirpath):
        for fname in sorted(os.listdir(dirpath)):
            if fname.endswith('.csv'):
                ct_name = fname.replace('gobp_rescued_', '').replace('.csv', '')
                sheet_name = f'S3.5_K_{ct_name}'[:31]  # xlsx max 31 chars
                sheets_celltype.append((
                    sheet_name,
                    f'{dirpath}/{fname}',
                    f'Kidney {ct_name}: GOBP enrichment of rescued gene markers'
                ))

# Muscle per-celltype
for subdir in ['ECs', 'IMM', 'NMJs', 'Rest']:
    dirpath = f'{M_CT}/{subdir}'
    if os.path.isdir(dirpath):
        for fname in sorted(os.listdir(dirpath)):
            if fname.endswith('.csv'):
                ct_name = fname.replace('gobp_rescued_', '').replace('.csv', '')
                sheet_name = f'S3.5_M_{ct_name}'[:31]
                sheets_celltype.append((
                    sheet_name,
                    f'{dirpath}/{fname}',
                    f'Muscle {ct_name}: GOBP enrichment of rescued gene markers'
                ))

# ── Styling ──────────────────────────────────────────────────────
header_font = Font(name='Arial', bold=True, size=10, color='FFFFFF')
header_fill = PatternFill(start_color='0F6E56', end_color='0F6E56', fill_type='solid')
desc_font = Font(name='Arial', italic=True, size=9, color='555555')
data_font = Font(name='Arial', size=9)
thin_border = Border(
    bottom=Side(style='thin', color='E0E0E0')
)

# Columns to keep (skip geneID which is very long)
keep_cols = ['ID', 'Description', 'GeneRatio', 'BgRatio', 'FoldEnrichment',
             'pvalue', 'p.adjust', 'qvalue', 'Count']
col_widths = {'ID': 15, 'Description': 45, 'GeneRatio': 12, 'BgRatio': 12,
              'FoldEnrichment': 15, 'pvalue': 14, 'p.adjust': 14, 'qvalue': 14, 'Count': 8}

def add_sheet(wb, sheet_name, csv_path, description):
    """Read a GOBP enrichment CSV and add as a formatted sheet."""
    try:
        with open(csv_path) as f:
            rows = list(csv.DictReader(f))
    except FileNotFoundError:
        print(f"  SKIP: {csv_path} not found")
        return

    if not rows:
        print(f"  SKIP: {sheet_name} — empty")
        return

    ws = wb.create_sheet(title=sheet_name)

    # Description row
    ws.merge_cells(start_row=1, start_column=1, end_row=1, end_column=len(keep_cols))
    ws.cell(row=1, column=1, value=description).font = desc_font

    # Term count
    ws.merge_cells(start_row=2, start_column=1, end_row=2, end_column=len(keep_cols))
    ws.cell(row=2, column=1, value=f'{len(rows)} significant GOBP terms (padj < 0.05)').font = desc_font

    # Headers
    available_cols = [c for c in keep_cols if c in rows[0]]
    for j, col in enumerate(available_cols, 1):
        cell = ws.cell(row=4, column=j, value=col)
        cell.font = header_font
        cell.fill = header_fill
        cell.alignment = Alignment(horizontal='center')

    # Data rows
    for i, row in enumerate(rows):
        for j, col in enumerate(available_cols, 1):
            val = row.get(col, '')
            # Convert numeric strings
            if col in ('pvalue', 'p.adjust', 'qvalue', 'FoldEnrichment'):
                try:
                    val = float(val)
                except (ValueError, TypeError):
                    pass
            elif col == 'Count':
                try:
                    val = int(val)
                except (ValueError, TypeError):
                    pass

            cell = ws.cell(row=5 + i, column=j, value=val)
            cell.font = data_font
            cell.border = thin_border

    # Column widths
    for j, col in enumerate(available_cols, 1):
        ws.column_dimensions[get_column_letter(j)].width = col_widths.get(col, 12)

    # Freeze header
    ws.freeze_panes = 'A5'

    print(f"  Added: {sheet_name:35s}  {len(rows)} terms")

# ── Build workbook ───────────────────────────────────────────────
wb = openpyxl.Workbook()

# Remove default sheet
wb.remove(wb.active)

# Index sheet
ws_idx = wb.create_sheet(title='Index')
ws_idx.cell(row=1, column=1, value='Supplementary Table S3.4/S3.5 — GOBP Enrichment Results').font = Font(bold=True, size=12)
ws_idx.cell(row=2, column=1, value='clusterProfiler enrichGO, padj < 0.05, GO:BP ontology').font = desc_font
ws_idx.cell(row=4, column=1, value='Sheet').font = Font(bold=True, size=10)
ws_idx.cell(row=4, column=2, value='Description').font = Font(bold=True, size=10)
ws_idx.cell(row=4, column=3, value='Terms').font = Font(bold=True, size=10)
ws_idx.column_dimensions['A'].width = 30
ws_idx.column_dimensions['B'].width = 60
ws_idx.column_dimensions['C'].width = 10

print("=== Building bulk enrichment sheets (S3.4) ===")
idx_row = 5
for sheet_name, csv_path, desc in sheets_bulk:
    add_sheet(wb, sheet_name, csv_path, desc)
    try:
        with open(csv_path) as f:
            n = sum(1 for _ in csv.DictReader(f))
    except FileNotFoundError:
        n = 0
    ws_idx.cell(row=idx_row, column=1, value=sheet_name).font = data_font
    ws_idx.cell(row=idx_row, column=2, value=desc).font = data_font
    ws_idx.cell(row=idx_row, column=3, value=n).font = data_font
    idx_row += 1

idx_row += 1
ws_idx.cell(row=idx_row, column=1, value='Per-cell-type enrichment (S3.5)').font = Font(bold=True, size=10)
idx_row += 1

print("\n=== Building per-celltype sheets (S3.5) ===")
for sheet_name, csv_path, desc in sheets_celltype:
    add_sheet(wb, sheet_name, csv_path, desc)
    try:
        with open(csv_path) as f:
            n = sum(1 for _ in csv.DictReader(f))
    except FileNotFoundError:
        n = 0
    ws_idx.cell(row=idx_row, column=1, value=sheet_name).font = data_font
    ws_idx.cell(row=idx_row, column=2, value=desc).font = data_font
    ws_idx.cell(row=idx_row, column=3, value=n).font = data_font
    idx_row += 1

# Save
out_path = f'{OUTPUT_DIR}/{FILENAME_S}.xlsx'
wb.save(out_path)
print(f'\nSaved: {out_path}')
print(f'Total sheets: {len(wb.sheetnames)} (1 index + {len(wb.sheetnames)-1} data)')

# DAVID Enrichment

## ── Supplementary DAVID Enrichment Tables (kidney, muscle v2) ────────────────────────

In [ ]:
def parse_david_cluster_csv_v2(filepath):
    """Parse DAVID FunctAnnotClusterReport — handles quoted CSV correctly."""
    rows = []
    current_cluster = None
    current_score   = None

    with open(filepath, encoding='utf-8', errors='replace') as f:
        content = f.read()

    reader = csvmod.reader(io.StringIO(content))
    for row in reader:
        if not row or len(row) < 3: continue
        field0 = row[0].strip()
        field1 = row[1].strip() if len(row) > 1 else ''

        if field0 == 'Cluster': continue

        try:
            cluster_num = int(field0)
            score       = float(field1)
            current_cluster = cluster_num
            current_score   = score
            if len(row) >= 11:
                rows.append({
                    'Cluster':          current_cluster,
                    'Enrichment Score': current_score,
                    'Category':         row[2] if len(row) > 2 else '',
                    'Term':             row[3] if len(row) > 3 else '',
                    'Count':            row[4] if len(row) > 4 else '',
                    'List Total':       row[5] if len(row) > 5 else '',
                    'Pop Hits':         row[6] if len(row) > 6 else '',
                    'Pop Total':        row[7] if len(row) > 7 else '',
                    '%':                row[8] if len(row) > 8 else '',
                    'P-Value':          row[9] if len(row) > 9 else '',
                    'Benjamini':        row[10] if len(row) > 10 else '',
                    'Fold Enrichment':  row[11] if len(row) > 11 else '',
                    'FDR':              row[13] if len(row) > 13 else '',
                    'Genes':            row[15] if len(row) > 15 else '',
                })
        except (ValueError, IndexError):
            continue

    df = pd.DataFrame(rows)
    print(f"  Parsed {len(df)} annotation terms across {df['Cluster'].nunique() if len(df) > 0 else 0} clusters")
    return df

In [ ]:
# ── Full DAVID supplementary table + Fig 3.2E ────────────────────
FILENAME_DAVID = 'Supplementary_Tables_DAVID_Functional_Enrichment'
DAVID_OUT = f"{DATA_BASE}/DAVID_functional_annotation/output"

# ── Files to parse ───────────────────────────────────────────────
david_files = {
    'K_aged':         f"{DAVID_OUT}/DAVIDFunctAnnotClusterReport_kidney_aged_lfc1_symbols_2026-05-13.csv",
    'K_rescued':      f"{DAVID_OUT}/DAVIDFunctAnnotClusterReport_kidney_rescued_to_normal_lfc1_symbols_2026-05-13.csv",
    'K_non_rescued':  f"{DAVID_OUT}/DAVIDFunctAnnotClusterReport_kidney_non_rescued_lfc1_symbols_2026-05-13.csv",
    'M_aged':         f"{DAVID_OUT}/DAVIDFunctAnnotClusterReport_muscle_v2_aged_lfc1_symbols_2026-05-13.csv",
    'M_rescued':      f"{DAVID_OUT}/DAVIDFunctAnnotClusterReport_muscle_v2_rescued_to_normal_lfc1_symbols_2026-05-13.csv",
    'M_non_rescued':  f"{DAVID_OUT}/DAVIDFunctAnnotClusterReport_muscle_v2_non_rescued_lfc1_symbols_2026-05-13.csv",
}

sheet_meta = {
    'K_aged':        ('S3.5_K_Aging_DAVID',       'Kidney: DAVID clustering of accelerated aging DEGs (|LFC| > 1, padj < 0.05)'),
    'K_rescued':     ('S3.5_K_Rescued_DAVID',      'Kidney: DAVID clustering of rescued-to-normal genes (|LFC| > 1)'),
    'K_non_rescued': ('S3.5_K_NonRescued_DAVID',   'Kidney: DAVID clustering of non-rescued aging DEGs (|LFC| > 1)'),
    'M_aged':        ('S3.5_M_Aging_DAVID',        'Muscle: DAVID clustering of accelerated aging DEGs (|LFC| > 1, padj < 0.05)'),
    'M_rescued':     ('S3.5_M_Rescued_DAVID',      'Muscle: DAVID clustering of rescued-to-normal genes (|LFC| > 1)'),
    'M_non_rescued': ('S3.5_M_NonRescued_DAVID',   'Muscle: DAVID clustering of non-rescued aging DEGs (|LFC| > 1)'),
}

# ── Parse all files ───────────────────────────────────────────────
parsed = {}
for key, fpath in david_files.items():
    print(f"\nParsing {key}...")
    parsed[key] = parse_david_cluster_csv_v2(fpath)

# ── Build supplementary workbook ──────────────────────────────────
header_font  = Font(name='Arial', bold=True, size=10, color='FFFFFF')
header_fill  = PatternFill(start_color='1E2761', end_color='1E2761', fill_type='solid')
kidney_fill  = PatternFill(start_color='EEF4FB', end_color='EEF4FB', fill_type='solid')
muscle_fill  = PatternFill(start_color='EDF8F2', end_color='EDF8F2', fill_type='solid')
cluster_font = Font(name='Arial', bold=True, size=9, color='1E2761')
desc_font    = Font(name='Arial', italic=True, size=9, color='555555')
data_font    = Font(name='Arial', size=9)
alt_fill     = PatternFill(start_color='F8FAFC', end_color='F8FAFC', fill_type='solid')
thin_border  = Border(bottom=Side(style='thin', color='E0E0E0'))

wb = openpyxl.Workbook()
wb.remove(wb.active)

# Index sheet
ws_idx = wb.create_sheet(title='Index')
ws_idx.cell(row=1, column=1,
    value='Supplementary Table S3.5 — DAVID Functional Annotation Clustering'
    ).font = Font(bold=True, size=12, name='Arial')
ws_idx.cell(row=2, column=1,
    value='Functional annotation clustering performed on gene subsets with |log2FC| > 1. '
          'Both up- and downregulated genes included. Direction not preserved in clustering.'
    ).font = desc_font
ws_idx.cell(row=3, column=1,
    value='Muscle data: final pipeline (STAR+Salmon/GRCm39/Ensembl 110). '
          'Kidney data: same pipeline. Both re-processed from raw FASTQs.'
    ).font = desc_font

for col, (hdr, w) in enumerate(zip(['Sheet','Description','Clusters','Terms'], [30,65,10,8]), 1):
    c = ws_idx.cell(row=5, column=col, value=hdr)
    c.font = Font(bold=True, size=10, name='Arial')
    ws_idx.column_dimensions[get_column_letter(col)].width = w

idx_row = 6
for key, (sheet_name, desc) in sheet_meta.items():
    df = parsed[key]
    tissue = 'K' if key.startswith('K') else 'M'

    ws = wb.create_sheet(title=sheet_name)
    section_fill = kidney_fill if tissue == 'K' else muscle_fill

    # Description rows
    ws.merge_cells(start_row=1, start_column=1, end_row=1, end_column=9)
    ws.cell(row=1, column=1, value=desc).font = desc_font
    ws.merge_cells(start_row=2, start_column=1, end_row=2, end_column=9)
    ws.cell(row=2, column=1,
        value=f'{len(df)} annotation terms across {df["Cluster"].nunique()} clusters. '
              f'Genes filtered to |LFC| > 1. Up- and downregulated genes pooled.'
    ).font = desc_font

    # Headers
    cols = ['Cluster', 'Enrichment Score', 'Category', 'Term', 'Count',
            'Fold Enrichment', 'P-Value', 'Benjamini', 'Genes']
    for j, col in enumerate(cols, 1):
        cell = ws.cell(row=4, column=j, value=col)
        cell.font = header_font
        cell.fill = header_fill
        cell.alignment = Alignment(horizontal='center', wrap_text=True)

    col_widths = [8, 14, 22, 45, 7, 14, 10, 10, 60]
    for j, w in enumerate(col_widths, 1):
        ws.column_dimensions[get_column_letter(j)].width = w

    # Data rows — shade by cluster
    cluster_fills = {}
    fill_toggle = True
    prev_cluster = None
    for i, (_, row) in enumerate(df.iterrows()):
        r = 5 + i
        cluster = row['Cluster']
        if cluster != prev_cluster:
            fill_toggle = not fill_toggle
            prev_cluster = cluster
        bg = section_fill if fill_toggle else alt_fill

        for j, col in enumerate(cols, 1):
            val = row.get(col, '')
            cell = ws.cell(row=r, column=j, value=val)
            cell.font = cluster_font if col == 'Cluster' else data_font
            cell.fill = bg
            cell.border = thin_border
            if col == 'Genes':
                cell.alignment = Alignment(wrap_text=False)

    ws.freeze_panes = 'A5'
    n_clusters = df['Cluster'].nunique()
    n_terms    = len(df)

    # Index entry
    ws_idx.cell(row=idx_row, column=1, value=sheet_name).font = data_font
    ws_idx.cell(row=idx_row, column=2, value=desc).font = data_font
    ws_idx.cell(row=idx_row, column=3, value=n_clusters).font = data_font
    ws_idx.cell(row=idx_row, column=4, value=n_terms).font = data_font
    idx_row += 1
    print(f"  Sheet '{sheet_name}': {n_clusters} clusters, {n_terms} terms")

out_path = f'{OUTPUT_DIR}/{FILENAME_DAVID}.xlsx'
wb.save(out_path)
print(f'\nSaved: {out_path}')

# Quick Checks

In [ ]:
# Map condition labels
COND_MAP = {'rgzj1': 'ctrl', 'rgzj2': 'age', 'rgzj3': 'sAct', 'rgzj4': 'DR', 'rgzj5': 'combi'}
# VERIFY THIS MAPPING IS CORRECT - check your Colab notebooks for the order!

fap = adata_m[adata_m.obs['celltype'] == 'FAPs']
fap_raw = fap.raw.to_adata() if fap.raw is not None else fap

genes = ['Col3a1', 'Col4a1', 'Mmp2', 'Timp2', 'Lox']

print(f"\nFAP cells per condition:")
for old, new in COND_MAP.items():
    n = (fap.obs['sample'] == old).sum()
    print(f"  {new} ({old}): {n}")

print(f"\nGene expression in FAPs:")
for old, new in COND_MAP.items():
    subset = fap_raw[fap_raw.obs['sample'] == old]
    print(f"\n  {new} (n={subset.n_obs}):")
    for gene in genes:
        if gene in subset.var_names:
            expr = subset[:, gene].X.toarray().flatten()
            print(f"    {gene}: mean={expr.mean():.3f}, detection={((expr>0).mean()*100):.1f}%")
        else:
            print(f"    {gene}: NOT FOUND")

print(f"\n--- Col3a1/Col4a1 ratio in muscle FAPs ---")
for old, new in COND_MAP.items():
    subset = fap_raw[fap_raw.obs['sample'] == old]
    c3 = subset[:, 'Col3a1'].X.toarray().flatten().mean() if 'Col3a1' in subset.var_names else 0
    c4 = subset[:, 'Col4a1'].X.toarray().flatten().mean() if 'Col4a1' in subset.var_names else 0
    if c4 > 0:
        print(f"  {new}: Col3a1={c3:.3f}, Col4a1={c4:.3f}, ratio={c3/c4:.3f}")
    else:
        print(f"  {new}: Col3a1={c3:.3f}, Col4a1={c4:.3f}, ratio=N/A")

print(f"\n--- Mmp2/Timp2 ratio in muscle FAPs ---")
for old, new in COND_MAP.items():
    subset = fap_raw[fap_raw.obs['sample'] == old]
    m2 = subset[:, 'Mmp2'].X.toarray().flatten().mean() if 'Mmp2' in subset.var_names else 0
    t2 = subset[:, 'Timp2'].X.toarray().flatten().mean() if 'Timp2' in subset.var_names else 0
    if t2 > 0:
        print(f"  {new}: Mmp2={m2:.3f}, Timp2={t2:.3f}, ratio={m2/t2:.3f}")
    else:
        print(f"  {new}: Mmp2={m2:.3f}, Timp2={t2:.3f}, ratio=N/A")

In [ ]:
CELLTYPE_COL = 'celltype'
CONDITION_COL = 'sample'

fib = adata[adata.obs[CELLTYPE_COL] == 'FIB']
fib_raw = fib.raw.to_adata() if fib.raw is not None else fib

genes = ['Col3a1', 'Col4a1', 'Mmp2', 'Timp2', 'Lox']
conditions = ['ctrl', 'age', 'DR', 'sAct', 'combi']

print(f"Fibroblast cells per condition:")
for cond in conditions:
    n = (fib.obs[CONDITION_COL] == cond).sum()
    print(f"  {cond}: {n}")

print(f"\nGene expression in FIB:")
for cond in conditions:
    subset = fib_raw[fib_raw.obs[CONDITION_COL] == cond]
    print(f"\n  {cond} (n={subset.n_obs}):")
    for gene in genes:
        if gene in subset.var_names:
            expr = subset[:, gene].X.toarray().flatten()
            print(f"    {gene}: mean={expr.mean():.3f}, detection={((expr>0).mean()*100):.1f}%")

print(f"\n--- Col3a1/Col4a1 ratio ---")
for cond in conditions:
    subset = fib_raw[fib_raw.obs[CONDITION_COL] == cond]
    c3 = subset[:, 'Col3a1'].X.toarray().flatten().mean()
    c4 = subset[:, 'Col4a1'].X.toarray().flatten().mean()
    print(f"  {cond}: Col3a1={c3:.3f}, Col4a1={c4:.3f}, ratio={c3/c4:.3f}")

print(f"\n--- Mmp2/Timp2 ratio ---")
for cond in conditions:
    subset = fib_raw[fib_raw.obs[CONDITION_COL] == cond]
    m2 = subset[:, 'Mmp2'].X.toarray().flatten().mean()
    t2 = subset[:, 'Timp2'].X.toarray().flatten().mean()
    print(f"  {cond}: Mmp2={m2:.3f}, Timp2={t2:.3f}, ratio={m2/t2:.3f}")

# ══ MUSCLE v2 (STAR+Salmon/GRCm39) ══

## ── Fig 3.1A v2 — Volcano (kidney + muscle v2 side by side) ───────────────────



In [ ]:
# ── Fig 3.1A v2 — Volcano (kidney + muscle v2 side by side) ─────
FILENAME_V2 = 'Fig_3_1A_volcano_aging_kidney_muscle_v2'

# Redefine to ensure correct 3-return version
def load_degs_volcano(filepath):
    lfcs, padjs, syms = [], [], []
    with open(filepath) as f:
        for r in csv.DictReader(f):
            p, l = r.get('padj','NA'), r.get('log2FoldChange','NA')
            if p in ('NA','','NaN') or l in ('NA','','NaN'): continue
            padj = max(float(p), 1e-310)
            lfcs.append(float(l)); padjs.append(padj); syms.append(r.get('Symbol',''))
    return np.array(lfcs), np.array(padjs), syms

k_lfc, k_padj, k_sym = load_degs_volcano(f"{DATA_BASE}/Kidney/1_DEGs_tables/kidney_res_age_results_with_symbols.csv")
m_lfc, m_padj, m_sym = load_degs_volcano(f"{MUSCLE_V2}/muscle_v2_res_age_results_with_symbols.csv")

print(f"Kidney: {len(k_lfc)} genes | Muscle v2: {len(m_lfc)} genes")

k_labels = pick_labels(k_lfc, k_padj, k_sym, 14, ['Tgfb1','Flt1','B2m','Gstm1','Prkd1'])
m_labels = pick_labels(m_lfc, m_padj, m_sym, 14, ['Tgfb1','Ryr3','Kcnq5','Wwox','Ptprm'])

fig, (ax_k, ax_m) = plt.subplots(1, 2, figsize=(13, 6.5))
plot_volcano(ax_k, k_lfc, k_padj, k_sym, 'Kidney', k_labels)
plot_volcano(ax_m, m_lfc, m_padj, m_sym, 'Muscle (v2, STAR+Salmon/GRCm39)', m_labels)

y_max = max(ax_k.get_ylim()[1], ax_m.get_ylim()[1])
ax_k.set_ylim(-2, y_max * 1.02)
ax_m.set_ylim(-2, y_max * 1.02)

fig.suptitle('Fig. 3.1A — Aging transcriptional signatures in kidney and muscle (v2)',
             fontsize=11, fontweight='bold', color=C_TEXT, y=0.99)

legend_elements = [
    mpatches.Patch(facecolor=C_RED_D, alpha=0.6, label='Upregulated (padj < 0.05)'),
    mpatches.Patch(facecolor=C_BLUE_D, alpha=0.6, label='Downregulated (padj < 0.05)'),
    mpatches.Patch(facecolor='#CCCCCC', alpha=0.4, label='Not significant'),
]
fig.legend(handles=legend_elements, loc='lower center', ncol=3, fontsize=8,
           frameon=False, bbox_to_anchor=(0.5, 0.02))
fig.text(0.5, -0.01,
         'Bulk RNA-seq, DESeq2 age vs. ctrl contrast. Dashed line = padj 0.05 threshold. '
         'Top genes labelled by significance × effect size.',
         ha='center', fontsize=6.5, color=C_TEXT2)
plt.tight_layout(rect=[0, 0.06, 1, 0.97])
save_fig(fig, FILENAME_V2)
print('Done — Fig 3.1A v2')

In [ ]:
# ── Volcano rescued highlighted — Kidney + Muscle v2 side by side ─
FILENAME_V2 = 'Fig_3_1A2_volcano_rescued_highlighted_kidney_muscle_v2'

def load_volcano_data(res_csv, rescued_csv):
    """Load all DEGs and mark rescued-to-normal genes."""
    rescued_syms = set()
    with open(rescued_csv) as f:
        for r in csv.DictReader(f):
            s = r.get('Symbol', '')
            if s:
                rescued_syms.add(s)

    syms, lfcs, padjs, colors_list, sizes = [], [], [], [], []
    with open(res_csv) as f:
        for r in csv.DictReader(f):
            p = r.get('padj', 'NA')
            l = r.get('log2FoldChange', 'NA')
            if p in ('NA', '', 'NaN') or l in ('NA', '', 'NaN'):
                continue
            sym  = r.get('Symbol', '') or ''
            lfc  = float(l)
            padj = max(float(p), 1e-310)
            sig  = padj < 0.05
            is_rescued = sym in rescued_syms and sig

            syms.append(sym)
            lfcs.append(lfc)
            padjs.append(padj)

            if is_rescued:
                colors_list.append(C_PURP_D)
                sizes.append(14)
            elif sig:
                colors_list.append('#AAAAAA')   # rescued = purple, all other sig = grey
                sizes.append(5)
            else:
                colors_list.append('#DDDDDD')   # NS = light grey
                sizes.append(2)

    return (np.array(syms), np.array(lfcs), np.array(padjs),
            np.array(colors_list), np.array(sizes),
            np.array([c == C_PURP_D for c in colors_list]), rescued_syms)

# Load kidney
k_syms, k_lfcs, k_padjs, k_cols, k_sizes, k_resc, k_resc_syms = load_volcano_data(
    f"{DATA_BASE}/Kidney/1_DEGs_tables/kidney_res_age_results_with_symbols.csv",
    f"{DATA_BASE}/Kidney/1_DEGs_tables/kidney_rescued_to_normal.csv"
)
# Load muscle v2
m_syms, m_lfcs, m_padjs, m_cols, m_sizes, m_resc, m_resc_syms = load_volcano_data(
    f"{MUSCLE_V2}/muscle_v2_res_age_results_with_symbols.csv",
    f"{MUSCLE_V2}/muscle_v2_rescued_to_normal.csv"
)

print(f"Kidney  — genes: {len(k_lfcs):,} | rescued: {k_resc.sum():,}")
print(f"Muscle v2 — genes: {len(m_lfcs):,} | rescued: {m_resc.sum():,}")

def plot_volcano_rescued(ax, syms, lfcs, padjs, cols, sizes, is_resc, title, n_labels=15):
    # Plot in z-order
    for mask, zo, alpha in [
        (cols == '#DDDDDD', 1, 0.3),   # NS
        (cols == '#AAAAAA', 2, 0.4),   # sig, not rescued
        (is_resc,           3, 0.95),  # rescued
    ]:
        if mask.sum() == 0:
            continue
        ax.scatter(lfcs[mask], -np.log10(padjs[mask]),
                   c=cols[mask], s=sizes[mask],
                   alpha=alpha, edgecolors='none',
                   rasterized=True, zorder=zo)

    # Reference lines
    ax.axhline(-np.log10(0.05), color=C_GRAY_D, lw=0.6, ls='--', zorder=0)
    ax.axvline(0,               color=C_GRAY_D, lw=0.4, zorder=0)

    # Top rescued labels
    resc_idx  = np.where(is_resc)[0]
    top_idx   = resc_idx[np.argsort(padjs[resc_idx])[:n_labels]]
    texts = []
    for i in top_idx:
        s = syms[i]
        if not s or s.startswith('ENSMUSG'):
            continue
        texts.append(ax.text(lfcs[i], -np.log10(padjs[i]), s,
                              fontsize=5.5, fontstyle='italic', fontweight='bold',
                              color=C_PURP_D, zorder=5))
    if texts:
        adjust_text(texts, ax=ax,
                    arrowprops=dict(arrowstyle='-', color=C_PURP_D, lw=0.4, alpha=0.5),
                    expand_points=(1.3, 1.3))

    # Count annotations
    n_sig   = int((padjs < 0.05).sum())
    n_up    = int(((padjs < 0.05) & (lfcs > 0)).sum())
    n_down  = int(((padjs < 0.05) & (lfcs < 0)).sum())
    n_resc  = int(is_resc.sum())
    ax.text(0.98, 0.97, f'{n_up:,} up',      transform=ax.transAxes, ha='right', va='top',
            fontsize=8.5, fontweight='bold', color='#888888')
    ax.text(0.02, 0.97, f'{n_down:,} down',  transform=ax.transAxes, ha='left',  va='top',
            fontsize=8.5, fontweight='bold', color='#888888')
    ax.text(0.98, 0.90, f'{n_resc:,} rescued', transform=ax.transAxes, ha='right', va='top',
            fontsize=8, fontweight='bold', color=C_PURP_D)
    ax.text(0.98, 0.04, 'padj = 0.05', transform=ax.transAxes,
            ha='right', va='bottom', fontsize=6, color=C_GRAY_D)

    ax.set_xlabel('log$_2$ fold change (age vs. ctrl)', fontsize=10, fontweight='bold')
    ax.set_ylabel('$-$log$_{10}$(padj)', fontsize=10, fontweight='bold')
    ax.set_title(title, fontsize=12, fontweight='bold', color=C_TEXT, pad=10)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.tick_params(labelsize=8)

# ── Figure ───────────────────────────────────────────────────────
fig, (ax_k, ax_m) = plt.subplots(1, 2, figsize=(14, 6.5))

plot_volcano_rescued(ax_k, k_syms, k_lfcs, k_padjs, k_cols, k_sizes, k_resc, 'Kidney')
plot_volcano_rescued(ax_m, m_syms, m_lfcs, m_padjs, m_cols, m_sizes, m_resc, 'Muscle')

# Match y-axes
y_max = max(ax_k.get_ylim()[1], ax_m.get_ylim()[1])
ax_k.set_ylim(-2, y_max * 1.02)
ax_m.set_ylim(-2, y_max * 1.02)

# Shared legend
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0],[0], marker='o', color='w', markerfacecolor=C_PURP_D,  markersize=9,  label='Rescued-to-normal'),
    Line2D([0],[0], marker='o', color='w', markerfacecolor='#AAAAAA', markersize=7,  label='Sig. DEG (not rescued)'),
    Line2D([0],[0], marker='o', color='w', markerfacecolor='#DDDDDD', markersize=6,  label='Not significant'),
]
fig.legend(handles=legend_elements, loc='lower center', ncol=3, fontsize=8,
           frameon=False, bbox_to_anchor=(0.5, 0.01))

fig.suptitle('Aging DEGs with rescued-to-normal genes highlighted — kidney vs muscle',
             fontsize=11, fontweight='bold', color=C_TEXT, y=1.0)
fig.text(0.5, -0.03,
         'Purple: rescued-to-normal genes (sign reversal + |LFC combi vs ctrl| < 0.5). '
         'Grey: significant DEGs not rescued. Top 15 rescued genes labelled by padj.',
         ha='center', fontsize=6.5, color=C_TEXT2)

plt.tight_layout(rect=[0, 0.06, 1, 0.98])
save_fig(fig, FILENAME_V2)
print(f'Done — {FILENAME_V2}')

## ── Fig 3.1B v2 — DEG counts bar chart ──────────────────────────

In [ ]:
# ── Fig 3.1B v2 — DEG counts bar chart ──────────────────────────
FILENAME_V2 = 'Fig_3_1B_DEG_counts_per_tissue_v2'

m_up_v2, m_down_v2, m_total_v2 = count_degs(f"{MUSCLE_V2}/muscle_v2_res_age_results_with_symbols.csv")
k_up, k_down, k_total = count_degs(f"{DATA_BASE}/Kidney/1_DEGs_tables/kidney_res_age_results_with_symbols.csv")
print(f"Kidney  — tested: {k_total:,}  up: {k_up:,}  down: {k_down:,}")
print(f"Muscle v2 — tested: {m_total_v2:,}  up: {m_up_v2:,}  down: {m_down_v2:,}")

tissues   = ['Kidney', 'Muscle (v2)']
up_counts = [k_up, m_up_v2]
dn_counts = [k_down, m_down_v2]

fig, ax = plt.subplots(figsize=(5, 5))
x = np.arange(len(tissues))
w = 0.35
ax.bar(x - w/2, up_counts, w, color=C_RED_D, alpha=0.75, label='Upregulated', edgecolor=C_RED_D, linewidth=0.5)
ax.bar(x + w/2, dn_counts, w, color=C_BLUE_D, alpha=0.75, label='Downregulated', edgecolor=C_BLUE_D, linewidth=0.5)
for i, (u, d) in enumerate(zip(up_counts, dn_counts)):
    ax.text(i - w/2, u + 80, f'{u:,}', ha='center', va='bottom', fontsize=8, fontweight='bold', color=C_RED_D)
    ax.text(i + w/2, d + 80, f'{d:,}', ha='center', va='bottom', fontsize=8, fontweight='bold', color=C_BLUE_D)
ax.set_xticks(x)
ax.set_xticklabels(tissues, fontsize=10, fontweight='bold')
ax.set_ylabel('Number of DEGs (padj < 0.05)', fontsize=10, fontweight='bold')
ax.set_title('Fig. 3.1B — Aging DEGs per tissue (v2)', fontsize=11, fontweight='bold', color=C_TEXT)
ax.legend(frameon=False, fontsize=9)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
save_fig(fig, FILENAME_V2)
print(f'Done — Fig 3.1B v2')

## ── Fig 3.1C/D v2 — GOBP aging enrichment (muscle panel) ────────


In [ ]:
# ── Fig 3.1C/D v2 — GOBP aging enrichment (muscle panel) ────────
FILENAME_V2 = 'Fig_3_1D_GOBP_aging_muscle_v2'

with open(f"{MUSCLE_V2}/muscle_v2_enrichment_results_aging.csv") as f:
    aging_terms_v2 = list(csv.DictReader(f))

FILENAME = 'Fig_3_1CD_GOBP_aging_kidney_muscle_v2'

# ── Load enrichment data ─────────────────────────────────────────
def load_enrich(filepath, n=18):
    """Load top n GOBP terms, removing near-duplicates."""
    with open(filepath) as f:
        rows = list(csv.DictReader(f))

    terms = []
    seen_words = []
    for r in rows:
        desc = r['Description']
        # Simple dedup: skip if >60% word overlap with an already-selected term
        words = set(desc.lower().split())
        overlap = any(len(words & s) / max(len(words), 1) > 0.6 for s in seen_words)
        if overlap and len(terms) > 3:
            continue

        num, denom = r['GeneRatio'].split('/')
        terms.append({
            'desc': desc,
            'count': int(num),
            'bg_count': int(denom),
            'gene_ratio': int(num) / int(denom),
            'padj': float(r['p.adjust']),
            'nlog10p': -np.log10(max(float(r['p.adjust']), 1e-310)),
        })
        seen_words.append(words)
        if len(terms) >= n:
            break
    return terms

k_terms = load_enrich(f"{DATA_BASE}/Kidney/2_Enrichment_bulk&sn/GOBP bulk/kidney_enrichment_results_aging.csv", 15)
m_terms = load_enrich(f"{MUSCLE_V2}/muscle_v2_enrichment_results_aging.csv", 15)

# ── Plot ─────────────────────────────────────────────────────────
fig, (ax_k, ax_m) = plt.subplots(1, 2, figsize=(14, 8),
                                  gridspec_kw={'width_ratios': [1.1, 1]})

def plot_dotplot(ax, terms, title, color_accent, total_degs):
    n = len(terms)
    y_pos = np.arange(n)[::-1]

    counts = [t['count'] for t in terms]
    nlog10p = [t['nlog10p'] for t in terms]
    gene_ratio = [t['gene_ratio'] for t in terms]
    descs = [t['desc'] for t in terms]

    # Wrap long descriptions
    descs_wrapped = []
    for d in descs:
        if len(d) > 45:
            d = textwrap.fill(d, width=40)
        descs_wrapped.append(d)

    # Dot size proportional to gene count
    min_count, max_count = min(counts), max(counts)
    sizes = [30 + (c - min_count) / max(max_count - min_count, 1) * 200 for c in counts]

    # Color by -log10(padj)
    scatter = ax.scatter(
        gene_ratio, y_pos,
        s=sizes, c=nlog10p,
        cmap='YlOrRd', vmin=min(nlog10p)*0.8, vmax=max(nlog10p)*1.05,
        edgecolors=color_accent, linewidths=0.4,
        zorder=3, alpha=0.85
    )

    ax.set_yticks(y_pos)
    ax.set_yticklabels(descs_wrapped, fontsize=10)
    ax.set_xlabel('Gene ratio', fontsize=9.5, fontweight='bold')
    ax.set_title(title, fontsize=11, fontweight='bold', color=C_TEXT, pad=10)

    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_linewidth(0.5)
    ax.spines['bottom'].set_linewidth(0.5)
    ax.tick_params(axis='y', length=0)
    ax.grid(axis='x', alpha=0.15, lw=0.5)

    # DEG count annotation
    ax.text(0.98, 0.02, f'{total_degs:,} aging DEGs\n{len(terms)} of {2757 if "Kidney" in title else 264} terms shown',
            transform=ax.transAxes, ha='right', va='bottom',
            fontsize=6.5, color=C_TEXT2, fontstyle='italic')

    return scatter

sc_k = plot_dotplot(ax_k, k_terms, 'Kidney — aging GOBP enrichment', C_RED_D, 7499)
sc_m = plot_dotplot(ax_m, m_terms, 'Muscle — aging GOBP enrichment', C_BLUE_D, 1739)

# Inset colorbars (inside plot area, bottom-left)
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
for ax_ref, sc in [(ax_k, sc_k), (ax_m, sc_m)]:
    cbax = inset_axes(ax_ref, width="30%", height="3%", loc='lower left',
                       bbox_to_anchor=(0.02, 0.02, 1, 1), bbox_transform=ax_ref.transAxes)
    cbar = fig.colorbar(sc, cax=cbax, orientation='horizontal')
    cbar.set_label('$-$log$_{10}$(padj)', fontsize=6, labelpad=2)
    cbar.ax.tick_params(labelsize=5)

# Size legend
for n_leg, lab in [(50, '50'), (150, '150'), (250, '250')]:
    sz = 30 + (n_leg - 20) / 250 * 200
    ax_m.scatter([], [], s=sz, c='gray', alpha=0.5, edgecolors=C_GRAY_D,
                  linewidths=0.4, label=f'{n_leg} genes')
ax_m.legend(loc='lower right', fontsize=6.5, frameon=True, framealpha=0.9,
            edgecolor='#E0E0E0', title='Gene count', title_fontsize=6.5,
            handletextpad=0.3, borderpad=0.5, labelspacing=0.8)

# Suptitle
fig.suptitle('Fig. 3.1C/D \u2014 GO Biological Process enrichment of aging DEGs',
             fontsize=11, fontweight='bold', color=C_TEXT, y=1.01)

# Footnote
fig.text(0.5, 0.005,
         'enrichGO (clusterProfiler), padj < 0.05. Near-duplicate terms removed for clarity.\n'
         'Kidney: 2,757 significant GOBP terms from 7,499 aging DEGs. Muscle: 264 terms from 1,739 DEGs.',
         ha='center', fontsize=6.5, color=C_TEXT2)

plt.tight_layout(rect=[0, 0.03, 1, 0.97])
save_fig(fig, FILENAME)
print('Done \u2014 Fig 3.1C/D v2')

## ── Fig 3.1C/D2, 3.2C2, 3.2D2 - Directional split GOBP

In [ ]:
# ── Fig 3.1C/D2 — Directional GOBP aging DEGs (improved layout) ──
FILENAME_AGE_DIR = 'Fig_3_1CD2_GOBP_aging_directional_split'

import pandas as pd

age_dir = f"{DATA_BASE}/GOBP_directional_split"
k_up = pd.read_csv(f"{age_dir}/Kidney_aging_UP_enrichGO_results.csv")
k_down = pd.read_csv(f"{age_dir}/Kidney_aging_DOWN_enrichGO_results.csv")
m_up = pd.read_csv(f"{age_dir}/Muscle_aging_UP_enrichGO_results.csv")
m_down = pd.read_csv(f"{age_dir}/Muscle_aging_DOWN_enrichGO_results.csv")

def get_top_terms(df, n=8):
    sig = df[df['p.adjust'] < 0.05].sort_values('p.adjust').head(n)
    return sig

def plot_single_direction(ax, df, n_terms, color, direction_label, tissue_label):
    """Plot one direction (UP or DOWN) as horizontal bars with dot overlay."""
    top = get_top_terms(df, n_terms)
    terms = top['Description'].tolist()[::-1]
    counts = top['Count'].tolist()[::-1]
    padj_vals = top['p.adjust'].tolist()[::-1]
    y_pos = range(len(terms))

    # Horizontal bars
    bars = ax.barh(list(y_pos), counts, height=0.6, color=color, alpha=0.3,
                    edgecolor=color, linewidth=0.5, zorder=2)

    # Dot overlay sized by -log10(padj)
    for i, (count, padj) in enumerate(zip(counts, padj_vals)):
        neg_log_p = -np.log10(max(padj, 1e-50))
        size = max(neg_log_p * 15, 20)
        ax.scatter(count, i, s=size, c=color, alpha=0.9,
                   edgecolors='white', linewidths=0.8, zorder=3)

    ax.set_yticks(list(y_pos))
    ax.set_yticklabels(terms, fontsize=12)
    ax.set_xlabel('Gene count', fontsize=9, fontweight='bold')
    ax.set_title(f'{tissue_label} — {direction_label}',
                 fontsize=13, fontweight='bold', color=color, pad=10)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_linewidth(0.5)
    ax.spines['bottom'].set_linewidth(0.5)
    ax.tick_params(axis='y', length=0)
    ax.grid(axis='x', alpha=0.15, lw=0.5)

    # Add term count annotation
    n_sig = len(df[df['p.adjust'] < 0.05])
    ax.text(0.95, 0.05, f'{n_sig} sig. terms', transform=ax.transAxes,
            ha='right', va='bottom', fontsize=8, color=color,
            fontweight='bold', fontstyle='italic')

# ── 2x2 layout: rows = tissue, columns = direction ───────────────
fig, axes = plt.subplots(2, 2, figsize=(22, 14))

plot_single_direction(axes[0, 0], k_up, 8, C_RED_D, '↑ Upregulated with aging', 'Kidney')
plot_single_direction(axes[0, 1], k_down, 8, C_BLUE_D, '↓ Downregulated with aging', 'Kidney')
plot_single_direction(axes[1, 0], m_up, 8, C_RED_D, '↑ Upregulated with aging', 'Muscle')
plot_single_direction(axes[1, 1], m_down, 8, C_BLUE_D, '↓ Downregulated with aging', 'Muscle')

# Tissue labels on left
fig.text(0.02, 0.72, 'KIDNEY', ha='center', va='center', fontsize=14,
         fontweight='bold', color=C_TEXT, rotation=90)
fig.text(0.02, 0.28, 'MUSCLE', ha='center', va='center', fontsize=14,
         fontweight='bold', color=C_TEXT, rotation=90)

fig.suptitle('GOBP enrichment of aging DEGs split by direction of expression change',
             fontsize=14, fontweight='bold', color=C_TEXT, y=0.98)
fig.text(0.5, 0.01,
         'Bar length = gene count. Dot size = -log10(padj). '
         'enrichGO (clusterProfiler, org.Mm.eg.db, BH-adjusted p < 0.05). Top 8 terms shown.',
         ha='center', fontsize=8, color=C_TEXT2, linespacing=1.4)

plt.tight_layout(rect=[0.04, 0.03, 1, 0.95])
plt.subplots_adjust(hspace=0.35, wspace=0.45)
save_fig(fig, FILENAME_AGE_DIR)
print('Done — Aging directional GOBP')


# ── Fig 3.2C2 — Directional GOBP rescued-to-normal ───────────────
FILENAME_RTN_DIR = 'Fig_3_2C2_GOBP_rescued_directional_split'

k_rtn_up = pd.read_csv(f"{age_dir}/Kidney_RTN_UP_enrichGO_results.csv")
k_rtn_down = pd.read_csv(f"{age_dir}/Kidney_RTN_DOWN_enrichGO_results.csv")
m_rtn_up = pd.read_csv(f"{age_dir}/Muscle_RTN_UP_enrichGO_results.csv")
m_rtn_down = pd.read_csv(f"{age_dir}/Muscle_RTN_DOWN_enrichGO_results.csv")

fig2, axes2 = plt.subplots(2, 2, figsize=(22, 14))

plot_single_direction(axes2[0, 0], k_rtn_up, 8, C_RED_D,
                       '↑ Originally upregulated (rescued down)', 'Kidney')
plot_single_direction(axes2[0, 1], k_rtn_down, 8, C_BLUE_D,
                       '↓ Originally downregulated (rescued up)', 'Kidney')
plot_single_direction(axes2[1, 0], m_rtn_up, 8, C_RED_D,
                       '↑ Originally upregulated (rescued down)', 'Muscle')
plot_single_direction(axes2[1, 1], m_rtn_down, 8, C_BLUE_D,
                       '↓ Originally downregulated (rescued up)', 'Muscle')

fig2.text(0.02, 0.72, 'KIDNEY', ha='center', va='center', fontsize=14,
          fontweight='bold', color=C_TEXT, rotation=90)
fig2.text(0.02, 0.28, 'MUSCLE', ha='center', va='center', fontsize=14,
          fontweight='bold', color=C_TEXT, rotation=90)

fig2.suptitle('GOBP enrichment of rescued-to-normal genes split by direction of aging change',
              fontsize=14, fontweight='bold', color=C_TEXT, y=0.98)
fig2.text(0.5, 0.01,
          'Bar length = gene count. Dot size = -log10(padj). '
          'enrichGO (clusterProfiler, org.Mm.eg.db, BH-adjusted p < 0.05). Top 8 terms shown.',
          ha='center', fontsize=8, color=C_TEXT2, linespacing=1.4)

plt.tight_layout(rect=[0.04, 0.03, 1, 0.95])
plt.subplots_adjust(hspace=0.35, wspace=0.45)
save_fig(fig2, FILENAME_RTN_DIR)
print('Done — RTN directional GOBP')


# ── Fig 3.2D2 — Directional GOBP non-rescued ─────────────────────
FILENAME_NR_DIR = 'Fig_3_2D2_GOBP_nonrescued_directional_split'

k_nr_up = pd.read_csv(f"{age_dir}/Kidney_nonrescued_UP_enrichGO_results.csv")
k_nr_down = pd.read_csv(f"{age_dir}/Kidney_nonrescued_DOWN_enrichGO_results.csv")
m_nr_up = pd.read_csv(f"{age_dir}/Muscle_nonrescued_UP_enrichGO_results.csv")
m_nr_down = pd.read_csv(f"{age_dir}/Muscle_nonrescued_DOWN_enrichGO_results.csv")

fig3, axes3 = plt.subplots(2, 2, figsize=(22, 14))

plot_single_direction(axes3[0, 0], k_nr_up, 8, C_RED_D,
                       '↑ Upregulated (not rescued)', 'Kidney')
plot_single_direction(axes3[0, 1], k_nr_down, 8, C_BLUE_D,
                       '↓ Downregulated (not rescued)', 'Kidney')
plot_single_direction(axes3[1, 0], m_nr_up, 8, C_RED_D,
                       '↑ Upregulated (not rescued)', 'Muscle')
plot_single_direction(axes3[1, 1], m_nr_down, 8, C_BLUE_D,
                       '↓ Downregulated (not rescued)', 'Muscle')

fig3.text(0.02, 0.72, 'KIDNEY', ha='center', va='center', fontsize=14,
          fontweight='bold', color=C_TEXT, rotation=90)
fig3.text(0.02, 0.28, 'MUSCLE', ha='center', va='center', fontsize=14,
          fontweight='bold', color=C_TEXT, rotation=90)

fig3.suptitle('GOBP enrichment of non-rescued aging DEGs split by direction',
              fontsize=14, fontweight='bold', color=C_TEXT, y=0.98)
fig3.text(0.5, 0.01,
          'Bar length = gene count. Dot size = -log10(padj). '
          'enrichGO (clusterProfiler, org.Mm.eg.db, BH-adjusted p < 0.05). Top 8 terms shown.',
          ha='center', fontsize=8, color=C_TEXT2, linespacing=1.4)

plt.tight_layout(rect=[0.04, 0.03, 1, 0.95])
plt.subplots_adjust(hspace=0.35, wspace=0.45)
save_fig(fig3, FILENAME_NR_DIR)
print('Done — Non-rescued directional GOBP')

## ── Fig 3.1F v2 — Muscle aging module score violin ──────────────

In [ ]:
# ── Fig 3.1F v2 — Muscle aging module score violin ──────────────
FILENAME_V2 = 'Fig_3_1F_violin_aging_module_muscle_v2'

def load_module_scores_v2(filepath):
    """Load v2 module scores — column is 'Aged1' or 'Aged'"""
    from collections import defaultdict
    ct_scores = defaultdict(list)
    with open(filepath) as f:
        for r in csv.DictReader(f):
            # v2 file uses 'Aged1' column
            val = r.get('Aged1') or r.get('Aged')
            if val and val not in ('NA', ''):
                try:
                    ct_scores[r['celltype']].append(float(val))
                except ValueError:
                    pass
    return ct_scores

MUSCLE_MODULE_CSV_V2 = f"{MUSCLE_V2}/muscle_v2_AgingGeneScores_perCell_seurat.csv"
m_scores_v2 = load_module_scores_v2(MUSCLE_MODULE_CSV_V2)

print(f"Cell types found: {list(m_scores_v2.keys())[:5]}")
print(f"Example scores: {list(m_scores_v2.values())[0][:3]}")

plot_violin(m_scores_v2, 'Fig. 3.1F — Muscle aging module scores by cell type',
            FILENAME_V2, figsize=(11, 6))

## ── Fig 3.1G v2 — Long-gene bias (muscle v2)

In [ ]:
# ══════════════════════════════════════════════════════════════════
# ── Fig 3.1G v2 — Long-gene bias (muscle v2) ─────────────────────
# ══════════════════════════════════════════════════════════════════
FILENAME_V2 = 'Fig_3_1G_long_gene_bias_aging_v2'

# gene_len dict must already exist from the original 3.1G cell — reuse it
# If not, re-run the BioMart fetch cell first

k_lfc, k_len, k_padj, k_sym = load_with_lengths(
    f"{DATA_BASE}/Kidney/1_DEGs_tables/kidney_res_age_results_with_symbols.csv", gene_len)
m_lfc_v2, m_len_v2, m_padj_v2, m_sym_v2 = load_with_lengths(
    f"{MUSCLE_V2}/muscle_v2_res_age_results_with_symbols.csv", gene_len)

print(f"Kidney: {len(k_lfc)} genes | Muscle v2: {len(m_lfc_v2)} genes")

bins_kb = [0, 25, 50, 100, 200, 500, 1000, 5000]
k_bins  = bin_analysis(k_len, k_lfc, k_padj, bins_kb)
m_bins_v2 = bin_analysis(m_len_v2, m_lfc_v2, m_padj_v2, bins_kb)

k_corr,   k_pval   = sp_stats.spearmanr(k_len,    np.abs(k_lfc))
m_corr_v2, m_pval_v2 = sp_stats.spearmanr(m_len_v2, np.abs(m_lfc_v2))
print(f"Kidney:    Spearman rho(length, |LFC|) = {k_corr:.3f},    p = {k_pval:.2e}")
print(f"Muscle v2: Spearman rho(length, |LFC|) = {m_corr_v2:.3f}, p = {m_pval_v2:.2e}")

fig, axes = plt.subplots(2, 2, figsize=(12, 9))
plot_tissue(axes[0, 0], axes[1, 0], k_bins,    'Kidney',         k_corr,    k_pval,    C_RED_D)
plot_tissue(axes[0, 1], axes[1, 1], m_bins_v2, 'Muscle',    m_corr_v2, m_pval_v2, C_BLUE_D)

fig.suptitle('Fig. 3.1G — Gene length bias in aging transcriptional dysregulation',
             fontsize=11, fontweight='bold', color=C_TEXT, y=1.01)
fig.text(0.5, -0.01,
         'Genes binned by genomic span (Ensembl). Top: mean |log2FC| per length bin.\n'
         'Bottom: fraction reaching significance per bin. Spearman correlation on unbinned data.\n'
         'Muscle: STAR+Salmon/GRCm39 pipeline (harmonised with kidney).',
         ha='center', fontsize=6.5, color=C_TEXT2, linespacing=1.4)
plt.tight_layout(rect=[0, 0.04, 1, 0.98])
save_fig(fig, FILENAME_V2)
print('Done — Fig 3.1G v2')

## ── Fig 3.1H v2 — Directional long-gene bias (muscle v2)

In [ ]:
# ══════════════════════════════════════════════════════════════════
# ── Fig 3.1H v2 — Directional long-gene bias (muscle v2) ─────────
# ══════════════════════════════════════════════════════════════════
FILENAME_V2 = 'Fig_3_1H_long_gene_directional_bias_v2'

# Uses m_lfc_v2, m_len_v2, m_padj_v2 from 3.1G v2 cell above
# k_lfc, k_len, k_padj unchanged

bins_kb = [0, 25, 50, 100, 200, 500, 1000, 5000]
k_dir     = directional_bin_analysis(k_len,    k_lfc,    k_padj,    bins_kb)
m_dir_v2  = directional_bin_analysis(m_len_v2, m_lfc_v2, m_padj_v2, bins_kb)

print("=== Kidney: fraction of sig DEGs that are DOWNREGULATED ===")
for b in k_dir:
    print(f"  {b['bin_label']:>10s} kb:  {b['frac_down']:.1%}  "
          f"({b['n_down']} down / {b['n_up']} up of {b['n_sig']} sig)")

print("\n=== Muscle v2: fraction of sig DEGs that are DOWNREGULATED ===")
for b in m_dir_v2:
    print(f"  {b['bin_label']:>10s} kb:  {b['frac_down']:.1%}  "
          f"({b['n_down']} down / {b['n_up']} up of {b['n_sig']} sig)")

k_sig_mask    = k_padj    < 0.05
m_sig_mask_v2 = m_padj_v2 < 0.05
k_dir_corr,    k_dir_p    = sp_stats.spearmanr(k_len[k_sig_mask],       k_lfc[k_sig_mask])
m_dir_corr_v2, m_dir_p_v2 = sp_stats.spearmanr(m_len_v2[m_sig_mask_v2], m_lfc_v2[m_sig_mask_v2])
print(f"\nKidney:    Spearman rho(length, signed LFC) = {k_dir_corr:.3f},    p = {k_dir_p:.2e}")
print(f"Muscle v2: Spearman rho(length, signed LFC) = {m_dir_corr_v2:.3f}, p = {m_dir_p_v2:.2e}")

fig, (ax_k, ax_m) = plt.subplots(1, 2, figsize=(13, 5.5))
plot_directional(ax_k, k_dir,    'Kidney',      k_dir_corr,    k_dir_p,    C_RED_D,  C_BLUE_D)
plot_directional(ax_m, m_dir_v2, 'Muscle', m_dir_corr_v2, m_dir_p_v2, C_RED_D,  C_BLUE_D)

fig.suptitle('Fig. 3.1H — Do long genes preferentially go DOWN with aging?',
             fontsize=11, fontweight='bold', color=C_TEXT, y=1.02)
fig.text(0.5, -0.02,
         'Stacked bars show fraction of sig aging DEGs (padj < 0.05) that are down- vs upregulated per length bin.\n'
         'Muscle: STAR+Salmon/GRCm39 (harmonised with kidney). Transcription stress signal should be preserved.',
         ha='center', fontsize=6.5, color=C_TEXT2, linespacing=1.4)
plt.tight_layout(rect=[0, 0.05, 1, 0.98])
save_fig(fig, FILENAME_V2)
print('Done — Fig 3.1H v2')

In [ ]:
# fresh kernel, after running through Fig 3.1H v2 (cell 118):
print("gene_len entries:", len(gene_len))
print("kidney matched:", len(k_lfc), "| muscle v2 matched:", len(m_lfc_v2))

ksig = k_padj < 0.05
msig = m_padj_v2 < 0.05
print("\n-- CONDITIONED on padj<0.05 (what Fig 3.1H reports) --")
print("kidney   signed-LFC rho:", sp_stats.spearmanr(k_len[ksig], k_lfc[ksig]), "n=", int(ksig.sum()))
print("muscle v2 signed-LFC rho:", sp_stats.spearmanr(m_len_v2[msig], m_lfc_v2[msig]), "n=", int(msig.sum()))

print("\n-- UNCONDITIONED, all matched genes --")
print("kidney   signed-LFC rho:", sp_stats.spearmanr(k_len, k_lfc))
print("muscle v2 signed-LFC rho:", sp_stats.spearmanr(m_len_v2, m_lfc_v2))

## ── Fig 3.1I v2 — Composition vs transcription stress (muscle v2) ─

In [ ]:
# ══════════════════════════════════════════════════════════════════
# ── Fig 3.1I v2 — Composition vs transcription stress (muscle v2) ─
# ══════════════════════════════════════════════════════════════════
FILENAME_V2 = 'Fig_3_1I_composition_vs_transcription_stress_v2'

# Kidney unchanged — reuse k_degs, k_markers, k_marker_genes from original cell
# Reload muscle with v2 DEGs and v2 markers

def load_degs_v2(filepath):
    degs = {}
    with open(filepath) as f:
        for r in csv.DictReader(f):
            if r.get('padj', 'NA') not in ('NA', '') and float(r['padj']) < 0.05:
                degs[r['Symbol']] = float(r['log2FoldChange'])
    return degs

m_degs_v2 = load_degs_v2(f"{MUSCLE_V2}/muscle_v2_res_age_results_with_symbols.csv")
m_markers_v2, m_marker_genes_v2 = load_markers(
    f"{MUSCLE_V2}/muscle_v2_markers_aged_genes_celltypes.csv")

print(f"Muscle v2 DEGs: {len(m_degs_v2)} | Muscle v2 marker genes: {len(m_marker_genes_v2)}")

m_cts_v2, m_ct_up_v2, m_ct_down_v2 = celltype_direction(m_markers_v2, m_degs_v2)

# ── Figure (same layout as original) ─────────────────────────────
fig = plt.figure(figsize=(15, 9))
gs = fig.add_gridspec(2, 2, height_ratios=[1, 1.2], hspace=0.35, wspace=0.5)

# ── Top left: Marker vs non-marker summary ────────────────────────
ax_sum = fig.add_subplot(gs[0, 0])

k_m_degs  = set(k_degs.keys())  & k_marker_genes
k_nm_degs = set(k_degs.keys())  - k_marker_genes
m_m_degs  = set(m_degs_v2.keys()) & m_marker_genes_v2
m_nm_degs = set(m_degs_v2.keys()) - m_marker_genes_v2

frac_up = [
    sum(1 for g in k_m_degs  if k_degs[g]    > 0) / max(len(k_m_degs),  1),
    sum(1 for g in k_nm_degs if k_degs[g]    > 0) / max(len(k_nm_degs), 1),
    sum(1 for g in m_m_degs  if m_degs_v2[g] > 0) / max(len(m_m_degs),  1),
    sum(1 for g in m_nm_degs if m_degs_v2[g] > 0) / max(len(m_nm_degs), 1),
]
frac_down = [1 - f for f in frac_up]
n_genes   = [len(k_m_degs), len(k_nm_degs), len(m_m_degs), len(m_nm_degs)]
categories = ['Kidney\nmarkers', 'Kidney\nnon-markers', 'Muscle (v2)\nmarkers', 'Muscle (v2)\nnon-markers']

x = np.arange(len(categories))
ax_sum.bar(x, frac_up,   color=C_RED_D,  alpha=0.6, edgecolor=C_RED_D,  linewidth=0.5, label='Upregulated')
ax_sum.bar(x, frac_down, bottom=frac_up, color=C_BLUE_D, alpha=0.6, edgecolor=C_BLUE_D, linewidth=0.5, label='Downregulated')
ax_sum.axhline(0.5, color=C_GRAY_D, lw=0.8, ls='--')
for i in range(len(categories)):
    ax_sum.text(i, frac_up[i] / 2,                   f'{frac_up[i]:.0%}\nup',
                ha='center', va='center', fontsize=7, color='white', fontweight='bold')
    ax_sum.text(i, frac_up[i] + frac_down[i] / 2,    f'{frac_down[i]:.0%}\ndown',
                ha='center', va='center', fontsize=7, color='white', fontweight='bold')
    ax_sum.text(i, 1.02, f'n={n_genes[i]:,}', ha='center', va='bottom', fontsize=6, color=C_TEXT2)
ax_sum.set_xticks(x)
ax_sum.set_xticklabels(categories, fontsize=8)
ax_sum.set_ylabel('Fraction of sig DEGs', fontsize=9, fontweight='bold')
ax_sum.set_title('Cell-type markers vs non-markers:\ndirectional bias', fontsize=10, fontweight='bold', color=C_TEXT, pad=10)
ax_sum.set_ylim(0, 1.1)
ax_sum.spines['top'].set_visible(False)
ax_sum.spines['right'].set_visible(False)
ax_sum.legend(loc='upper right', fontsize=7, frameon=True, framealpha=0.9, edgecolor='#E0E0E0')

# ── Top right: Interpretation schematic ──────────────────────────
ax_txt = fig.add_subplot(gs[0, 1])
ax_txt.axis('off')
ax_txt.set_title('Interpretation', fontsize=10, fontweight='bold', color=C_TEXT, pad=10)
notes = [
    (C_PURP_D, 'Cell-type markers biased UPWARD (63-79%)'),
    (C_PURP_D, '→ Expanding cell populations with aging'),
    ('', ''),
    (C_TEAL_D, 'In muscle: long genes go DOWN (Fig 3.1H)'),
    (C_TEAL_D, '→ Opposite to marker bias → confirms transcription stress'),
    (C_TEAL_D, '→ Signal robust despite composition push'),
    ('', ''),
    (C_AMBER_D, 'In kidney: long genes ~50/50 (no directional bias)'),
    (C_AMBER_D, '→ Larger composition shift (44 pp) may mask signal'),
    (C_AMBER_D, '→ Two opposing forces cancelling each other'),
]
txt_y = 0.88
for color, txt in notes:
    if txt == '':
        txt_y -= 0.03; continue
    ax_txt.text(0.05, txt_y, txt, fontsize=8, color=color if color else C_TEXT,
                fontweight='bold' if txt.startswith('→') else 'normal',
                transform=ax_txt.transAxes, va='top')
    txt_y -= 0.08

# ── Bottom left: Kidney cell types ───────────────────────────────
ax_k = fig.add_subplot(gs[1, 0])
k_cts_top = k_cts[:15]
y_k = np.arange(len(k_cts_top))[::-1]
ax_k.barh(y_k, [k_ct_up[ct]   for ct in k_cts_top], height=0.6,
          color=C_RED_D,  alpha=0.6, edgecolor=C_RED_D,  linewidth=0.5, label='Up')
ax_k.barh(y_k, [-k_ct_down[ct] for ct in k_cts_top], height=0.6,
          color=C_BLUE_D, alpha=0.6, edgecolor=C_BLUE_D, linewidth=0.5, label='Down')
for i, ct in enumerate(k_cts_top):
    total = k_ct_up[ct] + k_ct_down[ct]
    pct_up = k_ct_up[ct] / total * 100 if total > 0 else 0
    ax_k.text(max(k_ct_up[ct] for ct in k_cts_top) + 1, y_k[i],
              f'{pct_up:.0f}% up', va='center', ha='left', fontsize=6, color=C_RED_D)
ax_k.axvline(0, color=C_GRAY_D, lw=0.5)
ax_k.set_yticks(y_k); ax_k.set_yticklabels([clean_name(ct) for ct in k_cts_top], fontsize=7.5)
ax_k.set_xlabel('Number of marker DEGs', fontsize=9, fontweight='bold')
ax_k.set_title('Kidney — aging marker DEGs per cell type', fontsize=10, fontweight='bold', color=C_TEXT, pad=8)
ax_k.spines['top'].set_visible(False); ax_k.spines['right'].set_visible(False)
ax_k.tick_params(axis='y', length=0)
ax_k.set_xticklabels([str(abs(int(t))) for t in ax_k.get_xticks()])
ax_k.legend(loc='lower right', fontsize=7, frameon=True, framealpha=0.9, edgecolor='#E0E0E0')

# ── Bottom right: Muscle v2 cell types ───────────────────────────
ax_m = fig.add_subplot(gs[1, 1])
y_m = np.arange(len(m_cts_v2))[::-1]
ax_m.barh(y_m, [m_ct_up_v2[ct]    for ct in m_cts_v2], height=0.6,
          color=C_RED_D,  alpha=0.6, edgecolor=C_RED_D,  linewidth=0.5, label='Up')
ax_m.barh(y_m, [-m_ct_down_v2[ct] for ct in m_cts_v2], height=0.6,
          color=C_BLUE_D, alpha=0.6, edgecolor=C_BLUE_D, linewidth=0.5, label='Down')
max_up_v2 = max((m_ct_up_v2[ct] for ct in m_cts_v2), default=1)
for i, ct in enumerate(m_cts_v2):
    total = m_ct_up_v2[ct] + m_ct_down_v2[ct]
    pct_up = m_ct_up_v2[ct] / total * 100 if total > 0 else 0
    ax_m.text(max_up_v2 + 1, y_m[i], f'{pct_up:.0f}% up',
              va='center', ha='left', fontsize=6, color=C_RED_D)
ax_m.axvline(0, color=C_GRAY_D, lw=0.5)
ax_m.set_yticks(y_m); ax_m.set_yticklabels(m_cts_v2, fontsize=7)
ax_m.set_xlabel('Number of marker DEGs', fontsize=9, fontweight='bold')
ax_m.set_title('Muscle — aging marker DEGs per cell type', fontsize=10, fontweight='bold', color=C_TEXT, pad=8)
ax_m.spines['top'].set_visible(False); ax_m.spines['right'].set_visible(False)
ax_m.tick_params(axis='y', length=0)
ax_m.set_xticklabels([str(abs(int(t))) for t in ax_m.get_xticks()])
ax_m.legend(loc='lower right', fontsize=7, frameon=True, framealpha=0.9, edgecolor='#E0E0E0')

fig.suptitle('Fig. 3.1I — Cell-type composition shifts vs transcription stress',
             fontsize=11, fontweight='bold', color=C_TEXT, y=1.01)
fig.text(0.5, -0.01,
         'Muscle : STAR+Salmon/GRCm39 (harmonised with kidney). Marker genes from  FindAllMarkers.\n'
         'Top left: marker vs non-marker directional split. Bottom: per-cell-type direction of aging DEG markers.',
         ha='center', fontsize=6.5, color=C_TEXT2, linespacing=1.4)
plt.tight_layout(rect=[0, 0.04, 1, 0.98])
save_fig(fig, FILENAME_V2)
print('Done — Fig 3.1I v2')

## ── Fig 3.2A v2 — Rescue framework schematic ────────────────────

In [ ]:
# ── Fig 3.2A v2 — Rescue framework (muscle v2 values) ────────────
FILENAME_V2 = 'Fig_3_2A_rescue_framework_v2'

fig, ax = plt.subplots(1, 1, figsize=(8.5, 11))
ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.axis('off')

delta_text      = -0.01
delta_lvl0      = -0.02
delta_lvl1      = -0.04
delta_lvl3      = -0.06
delta_stage3    = -0.065

# ── Title ─────────────────────────────────────────────────────────
ax.text(0.50, 0.96, 'Fig. 3.2A', transform=ax.transAxes, ha='center', va='center',
        fontsize=11, fontweight='bold', color=C_TEXT)
ax.text(0.50, 0.95 + delta_text,
        'Direction-aware rescue framework: three-stage classification of aging DEGs',
        transform=ax.transAxes, ha='center', va='center', fontsize=8.5, color=C_TEXT2)

# ── Stage 0: Aging DEGs ───────────────────────────────────────────
rounded_box(ax, 0.24, 0.885 + delta_lvl0, 0.52, 0.045, C_RED, C_RED_D,
            'Aging DEGs (padj < 0.05)',
            'Kidney: 7,499  \u00b7  Muscle: 1,381')         # ← v2
arrow_down(ax, 0.50, 0.885 + delta_lvl0, 0.835 + delta_lvl0,
           'sign(LFC_aging) \u2260 sign(LFC_combi)?', text_x_offset=0.015)

# ── Stage 1 ───────────────────────────────────────────────────────
rounded_box(ax, 0.12, 0.80 + delta_lvl1, 0.42, 0.045, C_BLUE, C_BLUE_D,
            'Stage 1 \u2014 Rescued (sign reversal by combi)',
            'K: 4,645 (62%)  \u00b7  M: 719 (52%)')         # ← v2
rounded_box(ax, 0.58, 0.80 + delta_lvl1, 0.30, 0.045, C_GRAY, C_GRAY_D,
            'Not reversed by combi',
            'K: 2,854 (38%)  \u00b7  M: 662 (48%)')         # ← v2
arrow_right(ax, 0.54, 0.8225 + delta_lvl1, 0.58)
arrow_down(ax, 0.33, 0.80 + delta_lvl1, 0.755 + delta_lvl1 + delta_text,
           '|LFC combi vs ctrl| < 0.5?')

# ── Stage 2 ───────────────────────────────────────────────────────
rounded_box(ax, 0.08, 0.71 + delta_lvl3, 0.48, 0.045, C_TEAL, C_TEAL_D,
            'Stage 2 \u2014 Rescued-to-normal',
            'K: 1,961 (42% of rescued)  \u00b7  M: 491 (68%)')  # ← v2
rounded_box(ax, 0.60, 0.71 + delta_lvl3, 0.30, 0.045, C_GRAY, C_GRAY_D,
            'Rescued, not normalised',
            'K: 2,684 (58%)  \u00b7  M: 228 (32%)')         # ← v2
arrow_right(ax, 0.56, 0.7325 + delta_lvl3, 0.60)
arrow_down(ax, 0.32, 0.71 + delta_lvl3, 0.665 + delta_lvl3 + delta_text,
           'Which intervention drives the reversal?')

# ── Stage 3 ───────────────────────────────────────────────────────
ax.text(0.50, 0.65 + delta_stage3, 'Stage 3 \u2014 Intervention driver classification',
        transform=ax.transAxes, ha='center', va='center', fontsize=9, fontweight='bold', color=C_TEXT)
ax.text(0.50, 0.636 + delta_stage3, '(within rescued-to-normal set)',
        transform=ax.transAxes, ha='center', va='center', fontsize=7, color=C_TEXT2, fontstyle='italic')
ax.plot([0.13, 0.87], [0.625 + delta_stage3]*2,
        color=C_TEXT2, lw=0.6, transform=ax.transAxes, zorder=1)
for xp in [0.13, 0.37, 0.63, 0.87]:
    arrow_down(ax, xp, 0.625 + delta_stage3, 0.60 + delta_stage3)

bw2, bh2, by = 0.21, 0.065, 0.535 + delta_stage3
rounded_box(ax, 0.025, by, bw2, bh2, C_GREEN,  C_GREEN_D,  'DR-exclusive',
            'K: 546 (28%)  \u00b7  M: 211 (43%)', fontsize1=8.5)  # ← v2
rounded_box(ax, 0.260, by, bw2, bh2, C_PURPLE, C_PURP_D,   'sAct-exclusive',
            'K: 108 (6%)  \u00b7  M: 20 (4%)',   fontsize1=8.5)   # ← v2
rounded_box(ax, 0.505, by, bw2+0.015, bh2, C_AMBER, C_AMBER_D, 'Combination-only',
            'K: 87 (4%)  \u00b7  M: 14 (3%)',    fontsize1=8.5)   # ← v2
rounded_box(ax, 0.755, by, bw2, bh2, C_CORAL,  C_CORAL_D,  'Both DR + sAct',
            'K: 1,220 (62%)  \u00b7  M: 246 (50%)', fontsize1=8.5) # ← v2

ey = 0.505 + delta_stage3
for xp, txt in [(0.13,  'Only DR reverses\naging direction'),
                (0.365, 'Only sActRIIB reverses\naging direction'),
                (0.62,  'Neither alone reverses;\nsynergistic rescue'),
                (0.86,  'Both interventions\nindependently reverse')]:
    ax.text(xp, ey, txt, transform=ax.transAxes, ha='center', va='top',
            fontsize=6.5, color=C_TEXT2, linespacing=1.3)

ax.plot([0.73, 0.73], [0.80 + delta_stage3, 0.43 + delta_stage3],
        color=C_GRAY_D, lw=0.6, ls='--', transform=ax.transAxes, zorder=1)
ax.plot([0.73, 0.50], [0.43 + delta_stage3]*2,
        color=C_GRAY_D, lw=0.6, ls='--', transform=ax.transAxes, zorder=1)
arrow_down(ax, 0.50, 0.43 + delta_stage3, 0.415 + delta_stage3)

# ── Non-rescued ───────────────────────────────────────────────────
rounded_box(ax, 0.22, 0.37 + delta_stage3, 0.56, 0.045, C_GRAY, C_GRAY_D,
            'Non-rescued aging DEGs',
            'Not reversed by any intervention:  K: 1,169 (15.6%)  \u00b7  M: 241 (17.5%)')  # ← v2

# ── Definitions legend ────────────────────────────────────────────
leg_y = 0.27
ax.text(0.08, leg_y, 'Definitions', transform=ax.transAxes,
        fontsize=8, fontweight='bold', color=C_TEXT)
for i, (fc, ec, lab) in enumerate([
    (C_RED,    C_RED_D,    'Aging DEGs: padj < 0.05 and |log\u2082FC| > 0 in age vs. ctrl (DESeq2)'),
    (C_BLUE,   C_BLUE_D,   'Rescued: aging DEG where sign(LFC_aging) \u2260 sign(LFC_combi vs aging)'),
    (C_TEAL,   C_TEAL_D,   'Rescued-to-normal: rescued gene with |LFC_combi vs ctrl| < 0.5'),
    (C_GREEN,  C_GREEN_D,  'DR-exclusive: Restoration_DR > 0 and Restoration_sAct = 0'),
    (C_PURPLE, C_PURP_D,   'sAct-exclusive: Restoration_sAct > 0 and Restoration_DR = 0'),
    (C_AMBER,  C_AMBER_D,  'Combination-only: Restoration_DR = 0 and Restoration_sAct = 0 (synergistic)'),
    (C_CORAL,  C_CORAL_D,  'Both DR + sAct: Restoration_DR > 0 and Restoration_sAct > 0 (convergent)'),
]):
    yi = leg_y - 0.020 - i * 0.020
    ax.add_patch(FancyBboxPatch((0.08, yi-0.005), 0.015, 0.013,
                                 boxstyle="round,pad=0.001",
                                 facecolor=fc, edgecolor=ec, linewidth=0.5,
                                 transform=ax.transAxes, zorder=2))
    ax.text(0.105, yi+0.002, lab, transform=ax.transAxes,
            ha='left', va='center', fontsize=6.5, color=C_TEXT2)

# ── Footnote (pipeline note updated) ─────────────────────────────
ax.text(0.08, 0.115,
        'K = kidney, M = muscle. Percentages in Stages 1\u20132 are relative to the preceding category;\n'
        'percentages in Stage 3 are relative to the rescued-to-normal set.\n'
        'Both tissues processed with STAR+Salmon/GRCm39/DESeq2 (harmonised pipeline).',  # ← v2
        transform=ax.transAxes, ha='left', va='top',
        fontsize=6, color=C_TEXT2, linespacing=1.5)

plt.tight_layout(pad=0.5)
save_fig(fig, FILENAME_V2)
print('Done \u2014 Fig 3.2A v2')

## ── Fig 3.2B v2 — Sankey ─────────────────────────────────────────

In [ ]:
# ── Fig 3.2B v2 — Sankey ─────────────────────────────────────────
FILENAME_V2 = 'Fig_3_2B_sankey_rescue_v2'

kidney_data = dict(aging=7499, rescued=4645, not_reversed=2854, rtn=1961,
                   rescued_not_normal=2684, dr_excl=546, sact_excl=108, combi_only=87, both=1220)

muscle_data_v2 = dict(aging=1381, rescued=719, not_reversed=662, rtn=491,
                      rescued_not_normal=228, dr_excl=211, sact_excl=20, combi_only=14, both=246)

fig, (ax_k, ax_m) = plt.subplots(1, 2, figsize=(14, 7.5))
for a in [ax_k, ax_m]:
    a.set_xlim(-0.02, 1.08); a.set_ylim(-0.02, 1.02); a.axis('off')
draw_tissue(ax_k, 'Kidney', kidney_data)
draw_tissue(ax_m, 'Muscle', muscle_data_v2)

fig.suptitle('Fig. 3.2B — Rescue framework gene flow: aging DEGs to intervention drivers',
             fontsize=11, fontweight='bold', color=C_TEXT, y=0.99)

legend_items = [(C_RED, C_RED_D, 'Aging DEGs'), (C_BLUE, C_BLUE_D, 'Rescued (Stage 1)'),
                (C_TEAL, C_TEAL_D, 'Rescued-to-normal (Stage 2)'), (C_GREEN, C_GREEN_D, 'DR-exclusive'),
                (C_PURPLE, C_PURP_D, 'sAct-exclusive'), (C_AMBER, C_AMBER_D, 'Combi-only'),
                (C_CORAL, C_CORAL_D, 'Both DR+sAct'), (C_GRAY, C_GRAY_D, 'Not rescued / not normalised')]
handles = [mpatches.Patch(facecolor=fc, edgecolor=ec, linewidth=0.5, label=lab) for fc, ec, lab in legend_items]
fig.legend(handles=handles, loc='lower center', ncol=4, fontsize=7, frameon=False,
           bbox_to_anchor=(0.5, -0.01), handlelength=1.2, handleheight=0.8, columnspacing=1.2)

plt.tight_layout(rect=[0, 0.04, 1, 0.97])
save_fig(fig, FILENAME_V2)
print('Done — Fig 3.2B v2')

## ── Fig 3.2C/D v2 — GOBP rescued + non-rescued (muscle) ─────────

In [ ]:
def plot_dotplot(ax, terms, title, color_accent, total_degs):
    n = len(terms)
    y_pos = np.arange(n)[::-1]

    counts = [t['count'] for t in terms]
    nlog10p = [t['nlog10p'] for t in terms]
    gene_ratio = [t['gene_ratio'] for t in terms]
    descs = [t['desc'] for t in terms]

    # Wrap long descriptions
    descs_wrapped = []
    for d in descs:
        if len(d) > 45:
            d = textwrap.fill(d, width=40)
        descs_wrapped.append(d)

    # Dot size proportional to gene count
    min_count, max_count = min(counts), max(counts)
    sizes = [30 + (c - min_count) / max(max_count - min_count, 1) * 200 for c in counts]

    # Color by -log10(padj)
    scatter = ax.scatter(
        gene_ratio, y_pos,
        s=sizes, c=nlog10p,
        cmap='YlOrRd', vmin=min(nlog10p)*0.8, vmax=max(nlog10p)*1.05,
        edgecolors=color_accent, linewidths=0.4,
        zorder=3, alpha=0.85
    )

    ax.set_yticks(y_pos)
    ax.set_yticklabels(descs_wrapped, fontsize=10)
    ax.set_xlabel('Gene ratio', fontsize=9.5, fontweight='bold')
    ax.set_title(title, fontsize=11, fontweight='bold', color=C_TEXT, pad=10)

    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_linewidth(0.5)
    ax.spines['bottom'].set_linewidth(0.5)
    ax.tick_params(axis='y', length=0)
    ax.grid(axis='x', alpha=0.15, lw=0.5)

    # DEG count annotation
    ax.text(0.98, 0.02, f'{total_degs:,} aging DEGs\n{len(terms)} of {2757 if "Kidney" in title else 264} terms shown',
            transform=ax.transAxes, ha='right', va='bottom',
            fontsize=6.5, color=C_TEXT2, fontstyle='italic')

    return scatter

In [ ]:
# ── Fig 3.2C v2 — GOBP rescued-to-normal (muscle v2) ─────────────
FILENAME_V2 = 'Fig_3_2C_GOBP_rescued_to_normal_kidney_muscle_v2'

k_terms = load_enrich(
    f"{DATA_BASE}/Kidney/2_Enrichment_bulk&sn/GOBP bulk/kidney_enrichment_results_rescued-to-normal.csv", 15)
m_terms_v2 = load_enrich(
    f"{MUSCLE_V2}/muscle_v2_enrichment_results_rescued_to_normal.csv", 15)   # ← v2

print(f"Kidney: {len(k_terms)} terms | Muscle v2: {len(m_terms_v2)} terms")

fig, (ax_k, ax_m) = plt.subplots(1, 2, figsize=(14, 8),
                                   gridspec_kw={'width_ratios': [1.1, 1]})

def plot_dotplot(ax, terms, title, color_accent, n_rtn, total_terms):
    n = len(terms)
    y_pos = np.arange(n)[::-1]

    counts = [t['count'] for t in terms]
    nlog10p = [t['nlog10p'] for t in terms]
    gene_ratio = [t['gene_ratio'] for t in terms]
    descs = [textwrap.fill(d['desc'], width=42) if len(d['desc']) > 45 else d['desc'] for d in terms]

    min_c, max_c = min(counts), max(counts)
    sizes = [30 + (c - min_c) / max(max_c - min_c, 1) * 200 for c in counts]

    scatter = ax.scatter(
        gene_ratio, y_pos, s=sizes, c=nlog10p,
        cmap='YlGn', vmin=min(nlog10p)*0.8, vmax=max(nlog10p)*1.05,
        edgecolors=color_accent, linewidths=0.4, zorder=3, alpha=0.85
    )

    ax.set_yticks(y_pos)
    ax.set_yticklabels(descs, fontsize=9)
    ax.set_xlabel('Gene ratio', fontsize=9.5, fontweight='bold')
    ax.set_title(title, fontsize=11, fontweight='bold', color=C_TEXT, pad=10)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_linewidth(0.5)
    ax.spines['bottom'].set_linewidth(0.5)
    ax.tick_params(axis='y', length=0)
    ax.grid(axis='x', alpha=0.15, lw=0.5)

    ax.text(0.98, 0.02, f'{n_rtn:,} rescued-to-normal genes\n{len(terms)} of {total_terms} terms shown',
            transform=ax.transAxes, ha='right', va='bottom',
            fontsize=6.5, color=C_TEXT2, fontstyle='italic')

    # Inset colorbar
    cbax = inset_axes(ax, width="30%", height="3%", loc='lower left',
                       bbox_to_anchor=(0.02, 0.02, 1, 1), bbox_transform=ax.transAxes)
    cbar = fig.colorbar(scatter, cax=cbax, orientation='horizontal')
    cbar.set_label('$-$log$_{10}$(padj)', fontsize=6, labelpad=2)
    cbar.ax.tick_params(labelsize=5)

    return scatter

sc_k = plot_dotplot(ax_k, k_terms,    'Kidney — rescued-to-normal GOBP', C_TEAL_D, 1961, 703)
sc_m = plot_dotplot(ax_m, m_terms_v2, 'Muscle — rescued-to-normal GOBP', C_TEAL_D,
                    491,   # ← v2: 491 rescued-to-normal genes
                    17)    # ← v2: 17 significant terms

# Size legend
for n_leg, lab in [(5, '5'), (10, '10'), (15, '15')]:
    min_c = min(t['count'] for t in m_terms_v2) if m_terms_v2 else 1
    max_c = max(t['count'] for t in m_terms_v2) if m_terms_v2 else 1
    sz = 30 + (n_leg - min_c) / max(max_c - min_c, 1) * 200
    ax_m.scatter([], [], s=max(sz, 10), c='gray', alpha=0.5,
                  edgecolors=C_GRAY_D, linewidths=0.4, label=f'{n_leg} genes')
ax_m.legend(loc='lower right', fontsize=6.5, frameon=True, framealpha=0.9,
            edgecolor='#E0E0E0', title='Gene count', title_fontsize=6.5,
            handletextpad=0.3, borderpad=0.5, labelspacing=0.8)

fig.suptitle('Fig. 3.2C \u2014 GO Biological Process enrichment of rescued-to-normal genes',
             fontsize=11, fontweight='bold', color=C_TEXT, y=1.01)
fig.text(0.5, 0.005,
         'enrichGO (clusterProfiler), padj < 0.05. Near-duplicate terms removed.\n'
         'Kidney: 703 GOBP terms from 1,961 rescued-to-normal genes. '
         'Muscle v2: 17 terms from 491 genes (STAR+Salmon/GRCm39).',
         ha='center', fontsize=6.5, color=C_TEXT2)

plt.tight_layout(rect=[0, 0.03, 1, 0.97])
save_fig(fig, FILENAME_V2)
print('Done \u2014 Fig 3.2C v2')

In [ ]:

def plot_dotplot(ax, terms, title, color_accent, n_genes, total_terms):
    n = len(terms)
    y_pos = np.arange(n)[::-1]
    counts = [t['count'] for t in terms]
    nlog10p = [t['nlog10p'] for t in terms]
    gene_ratio = [t['gene_ratio'] for t in terms]
    descs = [textwrap.fill(d['desc'], width=42) if len(d['desc']) > 45 else d['desc'] for d in terms]

    min_c, max_c = min(counts), max(counts)
    sizes = [30 + (c - min_c) / max(max_c - min_c, 1) * 200 for c in counts]

    scatter = ax.scatter(
        gene_ratio, y_pos, s=sizes, c=nlog10p,
        cmap='OrRd', vmin=min(nlog10p)*0.8, vmax=max(nlog10p)*1.05,
        edgecolors=color_accent, linewidths=0.4, zorder=3, alpha=0.85
    )

    ax.set_yticks(y_pos)
    ax.set_yticklabels(descs, fontsize=9)
    ax.set_xlabel('Gene ratio', fontsize=9.5, fontweight='bold')
    ax.set_title(title, fontsize=11, fontweight='bold', color=C_TEXT, pad=10)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_linewidth(0.5)
    ax.spines['bottom'].set_linewidth(0.5)
    ax.tick_params(axis='y', length=0)
    ax.grid(axis='x', alpha=0.15, lw=0.5)

    ax.text(0.98, 0.02, f'{n_genes:,} non-rescued genes\n{len(terms)} of {total_terms} terms shown',
            transform=ax.transAxes, ha='right', va='bottom',
            fontsize=6.5, color=C_TEXT2, fontstyle='italic')

    cbax = inset_axes(ax, width="30%", height="3%", loc='lower left',
                       bbox_to_anchor=(0.02, 0.02, 1, 1), bbox_transform=ax.transAxes)
    cbar = fig.colorbar(scatter, cax=cbax, orientation='horizontal')
    cbar.set_label('$-$log$_{10}$(padj)', fontsize=6, labelpad=2)
    cbar.ax.tick_params(labelsize=5)

    return scatter

In [ ]:
# ── Fig 3.2D v2 — GOBP non-rescued (muscle v2) ───────────────────
FILENAME_V2 = 'Fig_3_2D_GOBP_non_rescued_kidney_muscle_v2'

k_terms = load_enrich(
    f"{DATA_BASE}/Kidney/2_Enrichment_bulk&sn/GOBP bulk/kidney_enrichment_results_non_rescued_genes.csv", 15)
m_terms_v2 = load_enrich(
    f"{MUSCLE_V2}/muscle_v2_enrichment_results_non_rescued_genes.csv", 15)  # ← v2

print(f"Kidney: {len(k_terms)} terms | Muscle v2: {len(m_terms_v2)} terms")

fig, (ax_k, ax_m) = plt.subplots(1, 2, figsize=(14, 8),
                                   gridspec_kw={'width_ratios': [1.1, 1]})

sc_k = plot_dotplot(ax_k, k_terms,    'Kidney — non-rescued GOBP',      C_RED_D, 1169, 194)
sc_m = plot_dotplot(ax_m, m_terms_v2, 'Muscle — non-rescued GOBP', C_RED_D,
                    241,   # ← v2: 241 non-rescued genes
                    45)    # ← v2: 45 significant terms

# Size legend — adjust bins for v2 gene counts
for n_leg in [5, 15, 30]:
    min_c = min(t['count'] for t in m_terms_v2) if m_terms_v2 else 1
    max_c = max(t['count'] for t in m_terms_v2) if m_terms_v2 else 1
    sz = 30 + (n_leg - min_c) / max(max_c - min_c, 1) * 200
    ax_m.scatter([], [], s=max(sz, 10), c='gray', alpha=0.5,
                  edgecolors=C_GRAY_D, linewidths=0.4, label=f'{n_leg} genes')
ax_m.legend(loc='lower right', fontsize=6.5, frameon=True, framealpha=0.9,
            edgecolor='#E0E0E0', title='Gene count', title_fontsize=6.5,
            handletextpad=0.3, borderpad=0.5, labelspacing=0.8)

fig.suptitle('Fig. 3.2D \u2014 GO Biological Process enrichment of non-rescued aging genes',
             fontsize=11, fontweight='bold', color=C_TEXT, y=1.01)
fig.text(0.5, 0.005,
         'enrichGO (clusterProfiler), padj < 0.05. Near-duplicate terms removed.\n'
         'Kidney: 194 GOBP terms from 1,169 non-rescued genes. '
         'Muscle v2: 45 terms from 241 genes (STAR+Salmon/GRCm39).',  # ← v2
         ha='center', fontsize=6.5, color=C_TEXT2)

plt.tight_layout(rect=[0, 0.03, 1, 0.97])
save_fig(fig, FILENAME_V2)
print('Done \u2014 Fig 3.2D v2')

## ── Fig 3.2E — side-by-side DAVID cluster bars ──

In [ ]:
# ── Fig 3.2E — DAVID cluster detail: rescued-to-normal ───────────
import matplotlib.gridspec as gridspec

def make_david_detail_fig(df_k, df_m, filename, suptitle, footnote,
                           top_n=10, figsize=(14, 7)):
    """
    Side-by-side horizontal bar charts showing top DAVID clusters
    for kidney (left) and muscle (right) separately.
    """

    def get_top_clusters(df, n):
        clusters = []
        for cluster_id, grp in df.groupby('Cluster'):
            score = float(grp['Enrichment Score'].iloc[0])
            go    = grp[grp['Category'].str.startswith('GOTERM')]
            best  = go.iloc[0] if len(go) > 0 else grp.iloc[0]
            term  = best['Term']
            count = int(best['Count'])
            label = term if len(term) <= 38 else term[:37] + '…'
            clusters.append({'label': label, 'score': score, 'count': count})
        df_c = pd.DataFrame(clusters).sort_values('score', ascending=False)
        return df_c.head(n).reset_index(drop=True)

    k_top = get_top_clusters(df_k, top_n)
    m_top = get_top_clusters(df_m, top_n)

    fig = plt.figure(figsize=figsize)
    gs  = gridspec.GridSpec(1, 2, figure=fig, wspace=0.5)
    ax_k = fig.add_subplot(gs[0])
    ax_m = fig.add_subplot(gs[1])

    def draw_panel(ax, df_top, color, color_d, title):
        n      = len(df_top)
        y      = np.arange(n)[::-1]
        scores = df_top['score'].values
        counts = df_top['count'].values
        labels = df_top['label'].values
        max_s  = max(scores) if max(scores) > 0 else 1

        ax.barh(y, scores, height=0.65,
                color=color, alpha=0.75,
                edgecolor=color_d, linewidth=0.6, zorder=3)

        for yp, score, count in zip(y, scores, counts):
            ax.text(score + max_s * 0.02, yp, f'{score:.2f}',
                    ha='left', va='center', fontsize=8,
                    fontweight='bold', color=color_d)
            ax.text(score + max_s * 0.18, yp, f'n={count}',
                    ha='left', va='center', fontsize=7,
                    color=C_TEXT2, fontstyle='italic')

        ax.set_yticks(y)
        ax.set_yticklabels(labels, fontsize=8.5)
        ax.set_xlim(0, max_s * 1.55)
        ax.set_ylim(-0.7, n - 0.2)
        ax.set_xlabel('Cluster enrichment score', fontsize=9, fontweight='bold')
        ax.set_title(title, fontsize=11, fontweight='bold', color=C_TEXT, pad=10)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.spines['left'].set_linewidth(0.5)
        ax.spines['bottom'].set_linewidth(0.5)
        ax.tick_params(axis='y', length=0)
        ax.grid(axis='x', alpha=0.12, lw=0.5, zorder=0)

    draw_panel(ax_k, k_top, C_CORAL, C_RED_D, 'Kidney')
    draw_panel(ax_m, m_top, C_BLUE,  C_BLUE_D, 'Muscle')

    fig.suptitle(suptitle, fontsize=12, fontweight='bold', color=C_TEXT, y=1.01)
    fig.text(0.5, -0.02, footnote, ha='center', fontsize=7,
             color=C_TEXT2, linespacing=1.4)

    plt.tight_layout(rect=[0, 0.02, 1, 0.98])
    save_fig(fig, filename)
    print(f'Done — {filename}')
    return fig

# ── Call for Fig 3.2E ─────────────────────────────────────────────
make_david_detail_fig(
    parsed['K_rescued'], parsed['M_rescued'],
    filename = 'Fig_3_2E_DAVID_clusters_rescued_detail',
    suptitle = 'Fig. 3.2E — DAVID functional annotation clusters: rescued-to-normal genes',
    footnote = ('Top 10 DAVID annotation clusters per tissue ranked by enrichment score. '
                'n = genes from input list (|LFC| > 1) mapping to cluster. '
                'Kidney: 135 input genes; Muscle: 64 input genes. '
                'Cross-tissue module comparison shown in Fig. 3.2F. '
                'Full cluster details in Supp. Table S3.5. '
                'Both tissues: STAR+Salmon/GRCm39/Ensembl 110.'),
    top_n=10, figsize=(15, 7),
)

## ── Fig 3.2F — Butterfly: rescued-to-normal modules (from DAVID) ──

In [ ]:
# ── Fig 3.2F — Butterfly: rescued-to-normal modules (from DAVID) ──
import matplotlib.patches as mpatches

def make_butterfly_fig(modules, rep_genes_dict, k_set, m_set,
                       filename, suptitle, footnote,
                       figsize=(13,8), xlim=(-35,42)):

    mod_names = [m[0] for m in modules]
    k_counts  = [sum(1 for g in genes if g in k_set) for _, genes in modules]
    m_counts  = [sum(1 for g in genes if g in m_set) for _, genes in modules]
    shared    = [sum(1 for g in genes if g in k_set and g in m_set) for _, genes in modules]
    k_only    = [k - s for k, s in zip(k_counts, shared)]
    m_only    = [m - s for m, s in zip(m_counts, shared)]

    print(f"\nModule counts for {filename}:")
    for i, name in enumerate(mod_names):
        print(f"  {name.replace(chr(10),' '):<42} K={k_counts[i]:3d}  S={shared[i]:2d}  M={m_counts[i]:3d}")

    order     = sorted(range(len(modules)), key=lambda i: k_counts[i]+m_counts[i], reverse=True)
    mod_names = [mod_names[i] for i in order]
    k_only    = [k_only[i]    for i in order]
    m_only    = [m_only[i]    for i in order]
    shared    = [shared[i]    for i in order]
    k_counts  = [k_counts[i]  for i in order]
    m_counts  = [m_counts[i]  for i in order]

    n     = len(mod_names)
    y     = np.arange(n)[::-1]
    bar_h = 0.62

    fig, ax = plt.subplots(figsize=figsize)

    for i in range(n):
        yp = y[i]
        if k_only[i] > 0:
            ax.barh(yp, -k_only[i], height=bar_h, left=-shared[i]/2,
                    color=C_RED_D, alpha=0.75, edgecolor=C_RED_D, linewidth=0.5, zorder=3)
        if shared[i] > 0:
            ax.barh(yp, shared[i], height=bar_h, left=-shared[i]/2,
                    color=C_PURP_D, alpha=0.65, edgecolor=C_PURP_D, linewidth=0.5, zorder=3)
        if m_only[i] > 0:
            ax.barh(yp, m_only[i], height=bar_h, left=shared[i]/2,
                    color=C_BLUE_D, alpha=0.75, edgecolor=C_BLUE_D, linewidth=0.5, zorder=3)

        left_edge  = -k_only[i] - shared[i]/2
        right_edge =  m_only[i] + shared[i]/2

        if k_counts[i] > 0:
            ax.text(left_edge - 0.4, yp, str(k_counts[i]),
                    ha='right', va='center', fontsize=9,
                    fontweight='bold', color=C_RED_D)
        if m_counts[i] > 0:
            ax.text(right_edge + 0.4, yp, str(m_counts[i]),
                    ha='left', va='center', fontsize=9,
                    fontweight='bold', color=C_BLUE_D)
        if shared[i] > 0:
            ax.text(0, yp, str(shared[i]), ha='center', va='center',
                    fontsize=7.5, fontweight='bold', color='white', zorder=5)

        name = mod_names[i]
        if name in rep_genes_dict:
            ax.text(right_edge + 1.2, yp, rep_genes_dict[name],
                    ha='left', va='center', fontsize=6.5,
                    color=C_TEXT2, fontstyle='italic')

    ax.set_yticks(y)
    ax.set_yticklabels(mod_names, fontsize=9)
    ax.set_xlabel('Number of genes in module', fontsize=10, fontweight='bold')
    ax.axvline(0, color=C_GRAY_D, lw=0.8, zorder=5)
    ax.set_xlim(xlim[0], xlim[1])
    ax.set_ylim(-0.7, n - 0.2)

    mid_k = xlim[0] / 2
    mid_m = xlim[1] / 2
    ax.text(mid_k, n - 0.05, '← Kidney', ha='center', va='bottom',
            fontsize=10.5, fontweight='bold', color=C_RED_D)
    ax.text(mid_m, n - 0.05, 'Muscle →', ha='center', va='bottom',
            fontsize=10.5, fontweight='bold', color=C_BLUE_D)

    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_linewidth(0.5)
    ax.spines['bottom'].set_linewidth(0.5)
    ax.tick_params(axis='y', length=0)
    ax.grid(axis='x', alpha=0.12, lw=0.5, zorder=0)
    ax.set_xticklabels([str(abs(int(t))) for t in ax.get_xticks()])

    legend_items = [
        mpatches.Patch(facecolor=C_RED_D,  alpha=0.75, label='Kidney-only'),
        mpatches.Patch(facecolor=C_PURP_D, alpha=0.65, label='Both tissues'),
        mpatches.Patch(facecolor=C_BLUE_D, alpha=0.75, label='Muscle-only'),
    ]
    ax.legend(handles=legend_items, loc='lower right', fontsize=8.5,
              frameon=True, framealpha=0.9, edgecolor='#E0E0E0', ncol=3)

    fig.suptitle(suptitle, fontsize=12, fontweight='bold', color=C_TEXT, y=0.99)
    fig.text(0.5, 0.01, footnote, ha='center', fontsize=7,
             color=C_TEXT2, linespacing=1.4)

    plt.tight_layout(rect=[0, 0.04, 1, 0.97])
    save_fig(fig, filename)
    print(f'Done — {filename}')
    return fig

FILENAME = 'Fig_3_2F_DAVID_modules_rescued_butterfly'

# ── Reload RTN sets lowercase ─────────────────────────────────────
with open(f"{DATA_BASE}/Kidney/1_DEGs_tables/kidney_rescued_to_normal.csv") as f:
    k_rtn = {r['Symbol'].lower() for r in csv.DictReader(f) if r.get('Symbol','')}
with open(f"{MUSCLE_V2}/muscle_v2_rescued_to_normal.csv") as f:
    m_rtn = {r['Symbol'].lower() for r in csv.DictReader(f) if r.get('Symbol','')}

# ── Modules defined from actual DAVID cluster gene lists ──────────
# Each module pools genes from semantically related clusters
# across both tissues — genes drawn directly from DAVID output above

rescued_modules = [
    # Shared theme: immune/inflammatory
    ('Immune /\ninflammatory',
     # K clusters 5+7+19: immune system process, carbohydrate binding, inflammatory
     # M cluster 3: SH3 domain (hcls1 — immune)
     ['cd74','cd86','clec4a1','cx3cr1','cxcr3','fcrl2','lat','il7','hck',
      'sting1','apaf1','cracr2a','dok3','itk','tcf7','lst1','irag2','ripk3',
      'hcls1','nfkb2']),

    # Shared theme: ECM/matrix
    ('ECM /\nmatrix organisation',
     # K clusters 1+8: extracellular matrix, elastic fiber assembly
     # M cluster 7: extracellular matrix
     ['col3a1','postn','tnn','emilin1','mfap4','clec3b','ccn4','srpx2',
      'sbspon','cpxm1','maf','tcf7',
      'cilp','coch','igfbp2','kazald1','otor','tgfb1','tgfa','tnfrsf22']),

    # Shared theme: apoptosis/damage
    ('Apoptosis /\ndamage response',
     # K cluster 5+6: apaf1, ripk3, gsdmd
     # M: birc5 (anti-apoptotic)
     ['apaf1','gsdmd','ripk3','htra3','maf',
      'birc5']),

    # Shared theme: TGF-β/developmental/signalling
    ('TGF-β /\ndevelopmental',
     # K cluster 6: ephb2, tyro3, hck, dyrk3
     # M clusters 6+8: tgfb1, fez1, hes6, robo2, lbh
     ['ephb2','tyro3','hck','dyrk3','cracr2a','dok3','itk',
      'tgfb1','fez1','hes6','robo2','lbh','map1b']),

    # Kidney-specific: peroxisomal/lipid
    ('Peroxisomal /\nlipid metabolism',
     # K cluster 2+9+10: peroxisome, oxidoreductase, androgen biosynthetic
     ['acox2','hao2','acot4','acnat2','gstk1','pxmp2','lrat','rdh16',
      'hsd17b2','etnppl','flvcr2','cryl1','gnmt','aldob','acnat2',
      'osbpl7','pld4','rdh16f2','prodh2']),

    # Kidney-specific: cell surface/receptor
    ('Cell surface /\nreceptor signalling',
     # K clusters 4+6+11+15: cell surface, RTK signalling, Fibronectin, Ig
     ['ache','ano4','clec2h','clec2i','selplg','slc4a4','slc6a19',
      'slitrk4','sdk1','lsamp','negr1','ly6c2','trbc2','ighg3',
      'igkv8-24','cd52','susd1','pcdhgc4']),

    # Kidney-specific: ER membrane
    ('ER membrane /\nprotein processing',
     # K cluster 13
     ['flvcr2','irag2','osbpl7','pigz','sting1','tyro3']),

    # Muscle-specific: microtubule/cytoskeleton
    ('Microtubule /\ncytoskeleton',
     # M clusters 1+4: microtubule, microtubule binding
     ['map1b','stmn1','birc5','arhgef2','gtse1','ckap2l','dlgap5',
      'fez1','tekt1','eef1a1','actc1','igfn1','pgm5','carmil3']),

    # Muscle-specific: cell cycle/mitosis
    ('Cell cycle /\nmitotic spindle',
     # M clusters 2+5+10: mitotic spindle, cell division, centromeric
     ['birc5','ccnb2','dlgap5','spc25','arhgef2','stmn1','cenpa']),

    # Muscle-specific: nervous system development
    ('Nervous system /\ndevelopment',
     # M cluster 8
     ['fez1','hcls1','hes6','kazald1','lbh','map1b','robo2','stmn1',
      'crybg3','nfkb2']),
]

rescued_rep_genes = {
    'Immune /\ninflammatory':           'Cd74, Cxcr3, Lat, Sting1, Hcls1',
    'ECM /\nmatrix organisation':       'Col3a1, Postn, Tnn, Tgfb1, Cilp',
    'Apoptosis /\ndamage response':     'Apaf1, Gsdmd, Ripk3, Birc5',
    'TGF-β /\ndevelopmental':           'Tgfb1, Ephb2, Robo2, Hes6',
    'Peroxisomal /\nlipid metabolism':  'Acox2, Hao2, Acot4, Pxmp2',
    'Cell surface /\nreceptor signalling': 'Sdk1, Lsamp, Negr1, Ly6c2',
    'ER membrane /\nprotein processing': 'Flvcr2, Osbpl7, Sting1',
    'Microtubule /\ncytoskeleton':      'Map1b, Stmn1, Birc5, Arhgef2',
    'Cell cycle /\nmitotic spindle':    'Ccnb2, Birc5, Dlgap5, Spc25',
    'Nervous system /\ndevelopment':    'Fez1, Robo2, Hes6, Lbh',
}

# ── Generate figure using make_butterfly_fig ──────────────────────
make_butterfly_fig(
    rescued_modules, rescued_rep_genes,
    k_rtn, m_rtn,
    FILENAME,
    suptitle='Fig. 3.2F — Functional modules among rescued-to-normal genes: kidney vs muscle',
    footnote=('Modules defined from DAVID functional annotation clustering of rescued-to-normal genes (|LFC| > 1). '
              'Gene lists drawn directly from DAVID cluster members. '
              'Purple = genes rescued in both tissues. Bold numbers = total per tissue. '
              'Italic = representative genes. Both tissues: STAR+Salmon/GRCm39/Ensembl 110.'),
    figsize=(14, 9), xlim=(-25, 30),
)

## ── Table 3.2A v2 — Summary table (muscle column updated) ───────

In [ ]:
# ── Table 3.2A v2 ─────────────────────────────────────────────────
FILENAME_V2 = 'Table_3_2A_rescue_summary_v2'

rows_v2 = [
    (0, 'Genes tested',                               '22,109',                        '18,569',       True,  C_BG_ALT,  False),  # ← v2
    (0, 'Aging DEGs (padj < 0.05)',                    '7,499',                         '1,381',        True,  C_BG_ALT,  False),  # ← v2
    (1, 'Upregulated',                                 '3,704 (49.4%)',                 '722 (52.3%)',   False, C_WHITE,   False),  # ← v2
    (1, 'Downregulated',                               '3,795 (50.6%)',                 '659 (47.7%)',   False, C_WHITE,   False),  # ← v2
    (0, '',                                            '',                              '',              False, C_WHITE,   False),
    (0, 'Stage 1 \u2014 Rescued by combined intervention', '',                         '',              True,  '#EEF4FB', False),
    (1, 'Sign reversal (combi vs. aging)',              '4,645 (61.9% of aging DEGs)',   '719 (52.1%)',   False, C_WHITE,   False),  # ← v2
    (1, 'Not reversed by combined intervention',        '2,854 (38.1%)',                 '662 (47.9%)',   False, C_WHITE,   True),   # ← v2
    (0, '',                                            '',                              '',              False, C_WHITE,   False),
    (0, 'Stage 2 \u2014 Rescued-to-normal',           '',                              '',              True,  '#EDF8F2', False),
    (1, 'Rescued + restored (|LFC combi vs ctrl| < 0.5)', '1,961 (42.2% of rescued)',  '491 (68.3%)',   False, C_WHITE,   False),  # ← v2
    (1, 'Rescued but not normalised',                   '2,684 (57.8%)',                 '228 (31.7%)',   False, C_WHITE,   True),   # ← v2
    (0, '',                                            '',                              '',              False, C_WHITE,   False),
    (0, 'Stage 3 \u2014 Intervention driver classification', '',                       '',              True,  '#FDF8EE', False),
    (0, '(within rescued-to-normal set)',               '',                              '',              False, C_WHITE,   True),
    (1, 'DR-exclusive',                                '546 (27.8%)',                   '211 (43.0%)',   False, C_WHITE,   False),  # ← v2
    (1, 'sActRIIB-exclusive',                          '108 (5.5%)',                    '20 (4.1%)',     False, C_WHITE,   False),  # ← v2
    (1, 'Combination-only (synergistic)',               '87 (4.4%)',                     '14 (2.9%)',     False, C_WHITE,   False),  # ← v2
    (1, 'Both DR + sActRIIB (convergent)',              '1,220 (62.2%)',                 '246 (50.1%)',   False, C_WHITE,   False),  # ← v2
    (0, '',                                            '',                              '',              False, C_WHITE,   False),
    (0, 'Non-rescued aging DEGs',                      '',                              '',              True,  '#FDF2F2', False),
    (1, 'Not rescued by any intervention',             '1,169 (15.6% of aging DEGs)',   '241 (17.5%)',   False, C_WHITE,   False),  # ← v2
    (1, 'Rescued by DR/sAct but not by combi',         '1,685 (22.5%)',                 '\u2014',        False, C_WHITE,   True),
]

fig, ax = plt.subplots(1, 1, figsize=(10, 8.5))
ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.axis('off')

table_left, table_right = 0.04, 0.96
table_width = table_right - table_left
col_widths = [0.48, 0.26, 0.26]
col_xs = [table_left]
for w in col_widths[:-1]:
    col_xs.append(col_xs[-1] + w * table_width)
row_height, header_height = 0.028, 0.032
y_top = 0.89

# Caption
ax.text(table_left, 0.96, 'Table 3.2A.', fontsize=9, fontweight='bold', color=C_TEXT, va='top')
ax.text(table_left + 0.065, 0.96,
        'Summary of rescue framework categories for kidney and muscle bulk RNA-seq datasets.',
        fontsize=8.5, color=C_TEXT, va='top')
ax.text(table_left, 0.935,
        'The rescue framework is applied in three sequential stages: (1) identification of aging DEGs showing\n'
        'directional reversal under the combined intervention, (2) filtering to genes restored to near-control\n'
        'expression levels, and (3) classification by individual intervention driver.',
        fontsize=7.5, color=C_TEXT2, va='top', linespacing=1.4)

# Header
ax.plot([table_left, table_right], [y_top, y_top], color=C_TEXT, lw=1.5, clip_on=False)
for i, (lab, align, pad) in enumerate(zip(
        ['Category', 'Kidney', 'Muscle'], ['left', 'right', 'right'], [0.01, -0.01, -0.01])):
    hx = col_xs[i] + pad if align == 'left' else col_xs[i] + col_widths[i] * table_width + pad
    ax.text(hx, y_top - header_height/2, lab, fontsize=8.5, fontweight='bold',
            color=C_TEXT, va='center', ha=align)
y_hdr_bot = y_top - header_height
ax.plot([table_left, table_right], [y_hdr_bot, y_hdr_bot], color=C_TEXT, lw=0.8, clip_on=False)

# Data rows (identical loop to original)
y_cur = y_hdr_bot
for indent, cat, kidney, muscle, is_sec, bg, is_gray in rows_v2:
    if cat == '' and kidney == '' and muscle == '':
        y_cur -= row_height * 0.4; continue
    rh = row_height * 1.05 if is_sec else row_height
    if bg != C_WHITE:
        ax.add_patch(mpatches.Rectangle((table_left, y_cur - rh), table_width, rh,
                                         fc=bg, ec='none', zorder=0))
    ax.plot([table_left, table_right], [y_cur - rh, y_cur - rh],
            color='#E5E5E5', lw=0.3, clip_on=False, zorder=1)
    tc = C_TEXT2 if is_gray else C_TEXT
    cat_x = col_xs[0] + 0.01 + indent * 0.025
    cat_fs = 6.5 if '(within' in cat else (8 if is_sec else 7.5)
    cat_sty = 'italic' if '(within' in cat else 'normal'
    ax.text(cat_x, y_cur - rh/2, cat,
            fontsize=cat_fs, fontweight='bold' if is_sec else 'normal',
            fontstyle=cat_sty, color=tc, va='center', ha='left', zorder=2)
    if kidney:
        ax.text(col_xs[1] + col_widths[1]*table_width - 0.01, y_cur - rh/2, kidney,
                fontsize=7.5, color=tc, va='center', ha='right', zorder=2)
    if muscle:
        ax.text(col_xs[2] + col_widths[2]*table_width - 0.01, y_cur - rh/2, muscle,
                fontsize=7.5, color=tc, va='center', ha='right', zorder=2)
    y_cur -= rh
ax.plot([table_left, table_right], [y_cur, y_cur], color=C_TEXT, lw=1.5, clip_on=False)

# Footnotes (pipeline note updated)
fn_y = y_cur - 0.025
for fn in [
    'Aging DEGs defined as padj < 0.05 and |log\u2082FC| > 0 (age vs. ctrl contrast, DESeq2). Rescue defined as sign(LFC_aging) \u2260 sign(LFC_intervention).',
    'Rescued-to-normal requires additionally |LFC combi vs. ctrl| < 0.5. Percentages in Stages 1\u20132 relative to preceding category; Stage 3 relative to rescued-to-normal set.',
    'Both tissues processed with identical pipeline: STAR v2.7.10a + Salmon / GRCm39 / Ensembl 110 / DESeq2 + lfcShrink (ashr).',  # ← v2
]:
    ax.text(table_left + 0.01, fn_y, fn, fontsize=6, color=C_TEXT2, va='top')
    fn_y -= 0.025

plt.tight_layout(pad=0.3)
save_fig(fig, FILENAME_V2)
print('Done \u2014 Table 3.2A v2')

## ── Fig 3.3A v2 — Driver proportions (muscle only) ──────────────

In [ ]:
# ── Fig 3.3A v2 — Driver proportions (muscle v2) ─────────────────
FILENAME_V2 = 'Fig_3_3A_driver_proportions_kidney_muscle_v2'

# Redefine in case overwritten by earlier cells
categories = ['Exclusive_DR', 'Exclusive_sAct', 'Only_Active_in_Combi', 'Other']
cat_colors = [C_GREEN_D, C_PURP_D, C_AMBER_D, C_GRAY_D]
cat_labels = ['DR-exclusive', 'sAct-exclusive', 'Combi-only', 'Both DR & sAct']

k_data    = load_proportions(f"{DATA_BASE}/Kidney/3_Effect_comparison_tables/gene_category_proportions.csv")
m_data_v2 = load_proportions(f"{MUSCLE_V2}/muscle_v2_gene_category_proportions.csv")

k_total    = sum(k_data[c]['count']    for c in categories)
m_total_v2 = sum(m_data_v2[c]['count'] for c in categories)

fig = plt.figure(figsize=(13, 6))
gs = fig.add_gridspec(1, 3, width_ratios=[1, 1, 1.3], wspace=0.3)

ax_k   = fig.add_subplot(gs[0])
ax_m   = fig.add_subplot(gs[1])
ax_bar = fig.add_subplot(gs[2])

plot_donut(ax_k, k_data,    k_total,    'Kidney')
plot_donut(ax_m, m_data_v2, m_total_v2, 'Muscle (v2)')  # ← v2

# ── Stacked bar ──────────────────────────────────────────────────
y_pos = [1.2, 0.4]
bar_h = 0.5

for idx, (tissue, data, total) in enumerate([
    ('Kidney',      k_data,    k_total),
    ('Muscle (v2)', m_data_v2, m_total_v2),   # ← v2
]):
    left = 0
    for i, cat in enumerate(categories):
        pct = data[cat]['prop'] * 100
        ax_bar.barh(y_pos[idx], pct, left=left, height=bar_h,
                     color=cat_colors[i], edgecolor='white', linewidth=0.8)
        if pct > 8:
            ax_bar.text(left + pct/2, y_pos[idx], f'{pct:.0f}%',
                        ha='center', va='center', fontsize=8,
                        fontweight='bold', color='white')
        left += pct

ax_bar.set_yticks(y_pos)
ax_bar.set_yticklabels(['Kidney', 'Muscle'], fontsize=10, fontweight='bold')
ax_bar.set_xlabel('% of rescued-to-normal genes', fontsize=9, fontweight='bold')
ax_bar.set_xlim(0, 100)
ax_bar.spines['top'].set_visible(False)
ax_bar.spines['right'].set_visible(False)
ax_bar.spines['left'].set_visible(False)
ax_bar.tick_params(axis='y', length=0)
ax_bar.set_title('Direct comparison', fontsize=11, fontweight='bold', color=C_TEXT, pad=15)

# Key finding annotation updated with v2 muscle %
m_dr_pct = m_data_v2['Exclusive_DR']['prop'] * 100
ax_bar.text(50, -0.35,
            f'DR dominates both tissues:\n28% kidney, {m_dr_pct:.0f}% muscle',
            ha='center', va='top', fontsize=8, color=C_GREEN_D,
            fontweight='bold', fontstyle='italic')

# Legend
legend_items = [mpatches.Patch(facecolor=cat_colors[i], label=cat_labels[i])
                for i in range(len(categories))]
fig.legend(handles=legend_items, loc='lower center', ncol=4, fontsize=8,
           frameon=True, framealpha=0.9, edgecolor='#E0E0E0',
           bbox_to_anchor=(0.5, -0.02))

fig.suptitle('Fig. 3.3A \u2014 Intervention driver proportions among rescued-to-normal genes',
             fontsize=11, fontweight='bold', color=C_TEXT, y=1.01)
fig.text(0.5, -0.07,
         'DR-exclusive = reversed only by DR. sAct-exclusive = reversed only by sAct.\n'
         'Combi-only = reversed only when both combined. Both = reversed by DR and sAct individually.\n'
         f'Kidney: 1,961 rescued-to-normal genes. Muscle v2: {m_total_v2} genes (STAR+Salmon/GRCm39).',
         ha='center', fontsize=6.5, color=C_TEXT2, linespacing=1.4)

plt.tight_layout(rect=[0, 0.05, 1, 0.97])
save_fig(fig, FILENAME_V2)
print('Done \u2014 Fig 3.3A v2')

## ── Table 3.3A v2 — Top genes per driver ────────────────────────


In [ ]:
# ── Table 3.3A v2 — Top genes per driver ────────────────────────
FILENAME_V2 = 'Table_3_3A_top_genes_per_driver_v2'

# Redefine the 2-argument version locally (Cell 56 overwrote it)
def load_driver_genes_table(base, tissue):
    cats = {}
    for cat, suffix, rest_col, intv_col in [
        ('DR-exclusive', f'{tissue}_intervention_impact_comparison_DR.csv', 'Restoration_DR', 'LFC_DR'),
        ('sAct-exclusive', f'{tissue}_intervention_impact_comparison_sAct.csv', 'Restoration_sAct', 'LFC_sAct'),
        ('Combi-only', f'{tissue}_intervention_impact_comparison_only_rescued_in_combi.csv', 'Restoration_Combined', 'LFC_Combined'),
    ]:
        try:
            with open(f'{base}/{suffix}') as f:
                rows = list(csv.DictReader(f))
            rows = [r for r in rows if r.get('Symbol', '') and
                    not r['Symbol'].startswith('ENSMUSG') and r['Symbol'] != 'NA']
            rows.sort(key=lambda r: abs(float(r['LFC_Age'])), reverse=True)
            cats[cat] = rows[:5]
        except FileNotFoundError:
            print(f"Missing: {base}/{suffix}")
            cats[cat] = []
    return cats

KIDNEY_BASE = f"{DATA_BASE}/Kidney/3_Effect_comparison_tables"

k_cats = load_driver_genes_table(KIDNEY_BASE, 'kidney')
m_cats_v2 = load_driver_genes_table(MUSCLE_V2, 'muscle_v2')

fig, (ax_k, ax_m) = plt.subplots(2, 1, figsize=(13, 10))
draw_table(ax_k, k_cats, 'Kidney — top genes per driver category', 'kidney')
draw_table(ax_m, m_cats_v2, 'Muscle (v2) — top genes per driver category', 'muscle_v2')

fig.suptitle('Table 3.3A — Top rescued genes by intervention driver (v2)',
             fontsize=12, fontweight='bold', color=C_TEXT, y=1.01)
plt.tight_layout()
save_fig(fig, FILENAME_V2)
print('Done — Table 3.3A v2')

## ── Fig 3.3B v2 — Restoration scores ────────────────────────────

In [ ]:
# ── Fig 3.3B v2 — Restoration scores (muscle v2) ─────────────────
FILENAME_V2 = 'Fig_3_3B_restoration_scores_kidney_muscle_v2'

KIDNEY_IMPACT_CSV = f"{DATA_BASE}/Kidney/3_Effect_comparison_tables/kidney_intervention_impact_comparison.csv"
MUSCLE_IMPACT_CSV_V2 = f"{MUSCLE_V2}/muscle_v2_intervention_impact_comparison.csv"  # ← v2

k_rest    = load_restoration(KIDNEY_IMPACT_CSV)
m_rest_v2 = load_restoration(MUSCLE_IMPACT_CSV_V2)

for label, data in [('Kidney', k_rest), ('Muscle', m_rest_v2)]:
    print(f"\n{label}:")
    for intv in ['DR', 'sAct', 'Combi']:
        vals = data[intv]
        print(f"  {intv:6s}: median={np.median(vals):.3f}, mean={np.mean(vals):.3f}, "
              f">1 (full reversal): {(vals > 1).sum()} ({(vals > 1).mean()*100:.0f}%)")

fig, (ax_k, ax_m) = plt.subplots(1, 2, figsize=(11, 6), sharey=True)

plot_restoration(ax_k, k_rest,    'Kidney',      1961)
plot_restoration(ax_m, m_rest_v2, 'Muscle', 491)  # ← v2

ax_k.set_ylabel('Restoration score', fontsize=10, fontweight='bold')
ax_k.set_ylim(-0.5, 2.5)

fig.suptitle('Fig. 3.3B \u2014 Restoration scores by intervention: kidney vs muscle',
             fontsize=11, fontweight='bold', color=C_TEXT, y=1.01)
fig.text(0.5, -0.02,
         'Restoration score = intervention LFC / |aging LFC|. Score of 1 = full reversal of aging change.\n'
         'Score > 1 = overcorrection. Score < 0 = worsening. Dashed line = full reversal threshold.\n'
         'Computed for rescued-to-normal genes only. Muscle v2: STAR+Salmon/GRCm39.',
         ha='center', fontsize=6.5, color=C_TEXT2, linespacing=1.4)

plt.tight_layout(rect=[0, 0.04, 1, 0.97])
save_fig(fig, FILENAME_V2)
print('Done \u2014 Fig 3.3B v2')

## ── Fig 3.3C/D v2 — DR-exclusive GOBP ───────────────────────────

In [ ]:
# ── Fig 3.3CD v2 — GOBP intervention drivers (muscle v2) ─────────
FILENAME_V2 = 'Fig_3_3CD_GOBP_intervention_drivers_v2'

# Redefine plot_dotplot with correct green colormap
def plot_dotplot(ax, terms, title, color_accent, n_genes, total_terms):
    n = len(terms)
    y_pos = np.arange(n)[::-1]
    counts = [t['count'] for t in terms]
    nlog10p = [t['nlog10p'] for t in terms]
    gene_ratio = [t['gene_ratio'] for t in terms]
    descs = [textwrap.fill(d['desc'], width=38) if len(d['desc']) > 40 else d['desc'] for d in terms]

    min_c, max_c = min(counts), max(counts)
    sizes = [30 + (c - min_c) / max(max_c - min_c, 1) * 180 for c in counts]

    scatter = ax.scatter(gene_ratio, y_pos, s=sizes, c=nlog10p,
                          cmap='YlGn',  # ← green palette
                          vmin=min(nlog10p)*0.8, vmax=max(nlog10p)*1.05,
                          edgecolors=color_accent, linewidths=0.4, zorder=3, alpha=0.85)

    ax.set_yticks(y_pos)
    ax.set_yticklabels(descs, fontsize=9)
    ax.set_xlabel('Gene ratio', fontsize=9, fontweight='bold')
    ax.set_title(title, fontsize=10, fontweight='bold', color=C_TEXT, pad=10)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_linewidth(0.5)
    ax.spines['bottom'].set_linewidth(0.5)
    ax.tick_params(axis='y', length=0)
    ax.grid(axis='x', alpha=0.15, lw=0.5)

    ax.text(0.98, 0.02, f'{n_genes} DR-excl. genes\n{len(terms)} of {total_terms} terms',
            transform=ax.transAxes, ha='right', va='bottom',
            fontsize=6, color=C_TEXT2, fontstyle='italic')

    cbax = inset_axes(ax, width="30%", height="3%", loc='lower left',
                       bbox_to_anchor=(0.02, 0.02, 1, 1), bbox_transform=ax.transAxes)
    cbar = fig.colorbar(scatter, cax=cbax, orientation='horizontal')
    cbar.set_label('$-$log$_{10}$(padj)', fontsize=5.5, labelpad=2)
    cbar.ax.tick_params(labelsize=5)

k_dr = load_enrich(
    f"{DATA_BASE}/Kidney/2_Enrichment_bulk&sn/GOBP bulk/kidney_enrichment_results_rescued-to-normal__DR.csv", 12)
m_dr_v2 = load_enrich(
    f"{MUSCLE_V2}/muscle_v2_enrichment_results_rescued-to-normal__DR.csv", 12)


# v2 counts for summary panel
k_sact_n=5;   k_sact_sig=0;   k_combi_n=8;   k_combi_sig=7
m_sact_n_v2=20; m_combi_n_v2=14  # ← v2

fig = plt.figure(figsize=(15, 8))
gs = fig.add_gridspec(1, 3, width_ratios=[1.1, 1.1, 0.6], wspace=0.55)
ax_k   = fig.add_subplot(gs[0])
ax_m   = fig.add_subplot(gs[1])
ax_sum = fig.add_subplot(gs[2])

plot_dotplot(ax_k, k_dr,    'Kidney — DR-exclusive GOBP', C_GREEN_D, k_dr_genes, k_dr_terms)    # ← auto
plot_dotplot(ax_m, m_dr_v2, 'Muscle — DR-exclusive GOBP', C_GREEN_D, m_dr_genes, m_dr_terms)  # ← auto




# ── Summary panel ─────────────────────────────────────────────────
ax_sum.set_xlim(0, 1); ax_sum.set_ylim(0, 1); ax_sum.axis('off')
ax_sum.set_title('Other drivers:\nenriched GOBP terms', fontsize=9, fontweight='bold',
                  color=C_TEXT, pad=10)
ax_sum.text(0.50, 0.95, 'Kidney',       ha='center', fontsize=8, fontweight='bold', color=C_RED_D)
ax_sum.text(0.85, 0.95, 'Muscle (v2)',  ha='center', fontsize=8, fontweight='bold', color=C_BLUE_D)

categories_sum = [                                                                          # ← all auto
    ('DR-exclusive',   C_GREEN,  C_GREEN_D, k_dr_genes,    k_dr_terms,    m_dr_genes,    m_dr_terms),
    ('sAct-exclusive', C_PURPLE, C_PURP_D,  k_sact_genes,  k_sact_terms,  m_sact_genes,  m_sact_terms),
    ('Combi-only',     C_AMBER,  C_AMBER_D, k_combi_genes, k_combi_terms, m_combi_genes, m_combi_terms),
]

y_start = 0.85
row_h   = 0.07

# ── Fig 3.3CD v2 — GOBP intervention drivers (muscle v2) ─────────
FILENAME_V2 = 'Fig_3_3CD_GOBP_intervention_drivers_v2'

# Redefine plot_dotplot with correct green colormap
def plot_dotplot(ax, terms, title, color_accent, n_genes, total_terms):
    n = len(terms)
    y_pos = np.arange(n)[::-1]
    counts = [t['count'] for t in terms]
    nlog10p = [t['nlog10p'] for t in terms]
    gene_ratio = [t['gene_ratio'] for t in terms]
    descs = [textwrap.fill(d['desc'], width=38) if len(d['desc']) > 40 else d['desc'] for d in terms]

    min_c, max_c = min(counts), max(counts)
    sizes = [30 + (c - min_c) / max(max_c - min_c, 1) * 180 for c in counts]

    scatter = ax.scatter(gene_ratio, y_pos, s=sizes, c=nlog10p,
                          cmap='YlGn',  # ← green palette
                          vmin=min(nlog10p)*0.8, vmax=max(nlog10p)*1.05,
                          edgecolors=color_accent, linewidths=0.4, zorder=3, alpha=0.85)

    ax.set_yticks(y_pos)
    ax.set_yticklabels(descs, fontsize=9)
    ax.set_xlabel('Gene ratio', fontsize=9, fontweight='bold')
    ax.set_title(title, fontsize=10, fontweight='bold', color=C_TEXT, pad=10)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_linewidth(0.5)
    ax.spines['bottom'].set_linewidth(0.5)
    ax.tick_params(axis='y', length=0)
    ax.grid(axis='x', alpha=0.15, lw=0.5)

    ax.text(0.98, 0.02, f'{n_genes} DR-excl. genes\n{len(terms)} of {total_terms} terms',
            transform=ax.transAxes, ha='right', va='bottom',
            fontsize=6, color=C_TEXT2, fontstyle='italic')

    cbax = inset_axes(ax, width="30%", height="3%", loc='lower left',
                       bbox_to_anchor=(0.02, 0.02, 1, 1), bbox_transform=ax.transAxes)
    cbar = fig.colorbar(scatter, cax=cbax, orientation='horizontal')
    cbar.set_label('$-$log$_{10}$(padj)', fontsize=5.5, labelpad=2)
    cbar.ax.tick_params(labelsize=5)

k_dr = load_enrich(
    f"{DATA_BASE}/Kidney/2_Enrichment_bulk&sn/GOBP bulk/kidney_enrichment_results_rescued-to-normal__DR.csv", 12)
m_dr_v2 = load_enrich(
    f"{MUSCLE_V2}/muscle_v2_enrichment_results_rescued-to-normal__DR.csv", 12)


# v2 counts for summary panel
k_sact_n=5;   k_sact_sig=0;   k_combi_n=8;   k_combi_sig=7
m_sact_n_v2=20; m_combi_n_v2=14  # ← v2

fig = plt.figure(figsize=(15, 8))
gs = fig.add_gridspec(1, 3, width_ratios=[1.1, 1.1, 0.6], wspace=0.55)
ax_k   = fig.add_subplot(gs[0])
ax_m   = fig.add_subplot(gs[1])
ax_sum = fig.add_subplot(gs[2])

plot_dotplot(ax_k, k_dr,    'Kidney — DR-exclusive GOBP', C_GREEN_D, k_dr_genes, k_dr_terms)    # ← auto
plot_dotplot(ax_m, m_dr_v2, 'Muscle — DR-exclusive GOBP', C_GREEN_D, m_dr_genes, m_dr_terms)  # ← auto




# ── Summary panel ─────────────────────────────────────────────────
ax_sum.set_xlim(0, 1); ax_sum.set_ylim(0, 1); ax_sum.axis('off')
ax_sum.set_title('Other drivers:\nenriched GOBP terms', fontsize=9, fontweight='bold',
                  color=C_TEXT, pad=10)
ax_sum.text(0.50, 0.95, 'Kidney',       ha='center', fontsize=8, fontweight='bold', color=C_RED_D)
ax_sum.text(0.85, 0.95, 'Muscle',  ha='center', fontsize=8, fontweight='bold', color=C_BLUE_D)

categories_sum = [                                                                          # ← all auto
    ('DR-exclusive',   C_GREEN,  C_GREEN_D, k_dr_genes,    k_dr_terms,    m_dr_genes,    m_dr_terms),
    ('sAct-exclusive', C_PURPLE, C_PURP_D,  k_sact_genes,  k_sact_terms,  m_sact_genes,  m_sact_terms),
    ('Combi-only',     C_AMBER,  C_AMBER_D, k_combi_genes, k_combi_terms, m_combi_genes, m_combi_terms),
]

y_start = 0.85
row_h   = 0.07

for i, (label, fc, ec, k_genes, k_terms, m_genes, m_terms) in enumerate(categories_sum):
    y = y_start - i * (row_h * 3 + 0.02)
    ax_sum.add_patch(FancyBboxPatch((0.02, y - row_h*0.6), 0.96, row_h*2.8,
                     boxstyle="round,pad=0.02", facecolor=fc, edgecolor=ec,
                     linewidth=0.6, alpha=0.3, transform=ax_sum.transAxes, zorder=0))
    ax_sum.text(0.10, y + row_h, label, ha='left', va='center',
                fontsize=8, fontweight='bold', color=ec, transform=ax_sum.transAxes)

    # Kidney stats
    ax_sum.text(0.50, y + row_h*0.3, f'{k_genes} genes', ha='center', va='center',
                fontsize=7, color=C_TEXT, transform=ax_sum.transAxes)
    ax_sum.text(0.50, y - row_h*0.3,
                f'{k_terms} sig. terms' if k_terms > 0 else '0 sig. terms',
                ha='center', va='center',
                fontsize=6.5, color=C_TEXT2, transform=ax_sum.transAxes)

    # Muscle stats
    ax_sum.text(0.85, y + row_h*0.3, f'{m_genes} genes', ha='center', va='center',
                fontsize=7, color=C_TEXT, transform=ax_sum.transAxes)
    ax_sum.text(0.85, y - row_h*0.3,
                f'{m_terms} sig. terms' if m_terms > 0 else '0 sig. terms',
                ha='center', va='center',
                fontsize=6.5, color=C_TEXT2, transform=ax_sum.transAxes)

# Key insight box
ax_sum.add_patch(FancyBboxPatch((0.02, 0.02), 0.96, 0.15,
                 boxstyle="round,pad=0.015", facecolor='#F8F7F5',
                 edgecolor=C_GRAY_D, linewidth=0.6, transform=ax_sum.transAxes, zorder=2))
ax_sum.text(0.50, 0.12, 'Key finding', ha='center', fontsize=7, fontweight='bold',
            color=C_TEXT, transform=ax_sum.transAxes, zorder=3)
ax_sum.text(0.50, 0.06,
            'DR-exclusive: strong pathway\nenrichment in both tissues (105 v2).\n'
            'sAct/combi-only: too few genes\nfor robust pathway-level signal.',
            ha='center', va='center', fontsize=6, color=C_TEXT2,
            transform=ax_sum.transAxes, zorder=3, linespacing=1.4)

fig.suptitle('Fig. 3.3C/D — Pathway specificity of intervention drivers',
             fontsize=11, fontweight='bold', color=C_TEXT, y=1.01)
fig.text(0.37, -0.01,
         'Left/center: Top DR-exclusive GOBP terms (enrichGO, padj < 0.05). Right: summary of all driver categories.\n'
         'Muscle: STAR+Salmon/GRCm39 (211 DR-exclusive, 20 sAct-exclusive, 14 combi-only genes).',
         ha='center', fontsize=6.5, color=C_TEXT2, linespacing=1.4)

plt.tight_layout(rect=[0, 0.03, 1, 0.97])
save_fig(fig, FILENAME_V2)
print('Done — Fig 3.3C/D v2')

# Key insight box
ax_sum.add_patch(FancyBboxPatch((0.02, 0.02), 0.96, 0.15,
                 boxstyle="round,pad=0.015", facecolor='#F8F7F5',
                 edgecolor=C_GRAY_D, linewidth=0.6, transform=ax_sum.transAxes, zorder=2))
ax_sum.text(0.50, 0.12, 'Key finding', ha='center', fontsize=7, fontweight='bold',
            color=C_TEXT, transform=ax_sum.transAxes, zorder=3)
ax_sum.text(0.50, 0.06,
            'DR-exclusive: strong pathway\nenrichment in both tissues (105 v2).\n'
            'sAct/combi-only: too few genes\nfor robust pathway-level signal.',
            ha='center', va='center', fontsize=6, color=C_TEXT2,
            transform=ax_sum.transAxes, zorder=3, linespacing=1.4)

fig.suptitle('Fig. 3.3C/D — Pathway specificity of intervention drivers',
             fontsize=11, fontweight='bold', color=C_TEXT, y=1.01)
fig.text(0.37, -0.01,
         'Left/center: Top DR-exclusive GOBP terms (enrichGO, padj < 0.05). Right: summary of all driver categories.\n'
         'Muscle v2: STAR+Salmon/GRCm39 (211 DR-exclusive, 20 sAct-exclusive, 14 combi-only genes).',
         ha='center', fontsize=6.5, color=C_TEXT2, linespacing=1.4)

plt.tight_layout(rect=[0, 0.03, 1, 0.97])
save_fig(fig, FILENAME_V2)
print('Done — Fig 3.3C/D v2')

## ── Fig 3.4(A/B)/E/F v2 — Module score violins ──────────────────────


In [ ]:
# ── Fig 3.4E/F v2 — Muscle module score violins (muscle v2 only) ─

MUSCLE_ALL_CSV_V2 = f"{MUSCLE_V2}/muscle_v2_module_scores_all.csv"  # ← v2

figures_muscle_v2 = [
    ('Reversal_norm', '3.4E', 'rescued-to-normal',      'Fig_3_4E_violin_rescued_module'),
    ('Reversal_DR',   '3.4F', 'DR-exclusive rescued',   'Fig_3_4F1_violin_DR_module'),
    ('Reversal_sAct', '3.4F', 'sAct-exclusive rescued', 'Fig_3_4F2_violin_sAct_module'),
]

try:
    all_scores_v2 = load_all_scores(MUSCLE_ALL_CSV_V2)
    print(f"Muscle v2: loaded score types: {list(all_scores_v2.keys())}")
except FileNotFoundError:
    print(f"Muscle v2 CSV not found at {MUSCLE_ALL_CSV_V2}")
    all_scores_v2 = {}

for score_col, fig_id, title_suffix, fname_base in figures_muscle_v2:
    if score_col not in all_scores_v2 or not all_scores_v2[score_col]:
        print(f"  Skipping {score_col} — no data")
        continue
    title = f'Fig. {fig_id} — Muscle {title_suffix} module scores by cell type'
    plot_violin(all_scores_v2[score_col], title, f'{fname_base}_muscle_v2', figsize=(11, 6))

## ── Fig 3.4C v2 — Reversal marker heatmap ───────────────────────


In [ ]:
# ── Fig 3.4C v2 — Reversal marker heatmap (muscle v2) ────────────
FILENAME_V2 = 'Fig_3_4C_reversal_marker_celltype_heatmap_v2'

# Redefine colours — overwritten by other cells in this session
cat_colors = [C_GREEN_D, C_CORAL_D, C_PURP_D, C_AMBER_D]
cat_fills  = [C_GREEN,   C_CORAL,   C_PURPLE,  C_AMBER]

# Redefine build_matrix — overwritten by CellChat cell in this session
def build_matrix(marker_csv, driver_sets):
    categories = ['DR-exclusive', 'Both DR+sAct', 'sAct-exclusive', 'Combi-only']
    ct_counts = defaultdict(lambda: {c: 0 for c in categories})
    with open(marker_csv) as f:
        for row in csv.DictReader(f):
            gene, ct = row['gene'], row['cluster']
            for cat in categories:
                if gene in driver_sets[cat]:
                    ct_counts[ct][cat] += 1
                    break
    totals = {ct: sum(d.values()) for ct, d in ct_counts.items()}
    sorted_cts = sorted(totals, key=totals.get, reverse=True)
    matrix = [[ct_counts[ct][c] for c in categories] for ct in sorted_cts]
    return sorted_cts, categories, matrix, totals

# Redefine draw_heatmap — overwritten in this session
def draw_heatmap(ax, cell_types, matrix, totals, title, max_val=None):
    n_ct = len(cell_types)
    n_cat = len(cats)
    if max_val is None:
        max_val = max(max(row) for row in matrix) if matrix else 1
    for i, ct in enumerate(cell_types):
        y = n_ct - 1 - i
        for j, cat in enumerate(cats):
            val = matrix[i][j]
            intensity = val / max_val if max_val > 0 else 0
            r1, g1, b1 = mcolors.to_rgb(C_WHITE)
            r2, g2, b2 = mcolors.to_rgb(cat_fills[j])
            fc = (r1+(r2-r1)*intensity, g1+(g2-g1)*intensity, b1+(b2-b1)*intensity)
            rect = mpatches.FancyBboxPatch((j, y), 0.92, 0.85, boxstyle="round,pad=0.02",
                                            facecolor=fc, edgecolor='#E0E0E0', linewidth=0.3, zorder=2)
            ax.add_patch(rect)
            if val > 0:
                txt_color = cat_colors[j] if intensity > 0.3 else C_TEXT2
                ax.text(j+0.46, y+0.42, str(val), ha='center', va='center', fontsize=7.5,
                        fontweight='bold' if val >= 10 else 'normal', color=txt_color, zorder=3)
        ax.text(-0.15, y+0.42, ct, ha='right', va='center', fontsize=7.5, color=C_TEXT)
        ax.text(n_cat+0.15, y+0.42, str(totals[ct]), ha='left', va='center', fontsize=7.5,
                fontweight='bold', color=C_TEXT2)

    header_y = n_ct + 0.5
    for j, cat in enumerate(cats):
        ax.text(j+0.46, header_y, cat, ha='center', va='bottom', fontsize=7.5,
                fontweight='bold', color=cat_colors[j], rotation=35)
    ax.text(n_cat+0.15, header_y, 'Total', ha='left', va='bottom', fontsize=7.5,
            fontweight='bold', color=C_TEXT2, rotation=35)
    ax.text(n_cat/2, n_ct + 3.5, title, ha='center', va='center', fontsize=12,
            fontweight='bold', color=C_TEXT)
    ax.set_xlim(-0.3, n_cat+0.8)
    ax.set_ylim(-0.5, n_ct + 4.5)
    ax.set_aspect('equal')
    ax.axis('off')

# Kidney unchanged
kidney_drivers = load_driver_genes(
    f"{DATA_BASE}/Kidney/3_Effect_comparison_tables/kidney_intervention_impact_comparison")

# Muscle v2 — updated driver gene files
muscle_drivers_v2 = load_driver_genes(
    f"{MUSCLE_V2}/muscle_v2_intervention_impact_comparison")               # ← v2

k_cts, cats, k_mat, k_totals = build_matrix(
    f"{DATA_BASE}/Kidney/4_Aged-Rescued_genes_celltype_breakdown/kidney_markers_reversal_genes_celltypes.csv",
    kidney_drivers)
m_cts_v2, _, m_mat_v2, m_totals_v2 = build_matrix(
    f"{MUSCLE_V2}/muscle_v2_markers_reversal_genes_celltypes.csv",         # ← v2
    muscle_drivers_v2)

fig, (ax_k, ax_m) = plt.subplots(1, 2, figsize=(14, 9),
                                   gridspec_kw={'width_ratios': [1.3, 1]})

global_max = max(
    max(max(row) for row in k_mat)   if k_mat   else 1,
    max(max(row) for row in m_mat_v2) if m_mat_v2 else 1
)
draw_heatmap(ax_k, k_cts,    k_mat,    k_totals,    'Kidney',      max_val=global_max)
draw_heatmap(ax_m, m_cts_v2, m_mat_v2, m_totals_v2, 'Muscle', max_val=global_max)  # ← v2

fig.suptitle('Fig. 3.4C \u2014 Reversal gene markers by cell type and intervention driver',
             fontsize=11, fontweight='bold', color=C_TEXT, y=0.99)

legend_items = [(C_GREEN, C_GREEN_D, 'DR-exclusive'), (C_CORAL, C_CORAL_D, 'Both DR+sAct'),
                (C_PURPLE, C_PURP_D, 'sAct-exclusive'), (C_AMBER, C_AMBER_D, 'Combi-only')]
handles = [mpatches.Patch(facecolor=fc, edgecolor=ec, linewidth=0.5, label=lab)
           for fc, ec, lab in legend_items]
fig.legend(handles=handles, loc='lower center', ncol=4, fontsize=8, frameon=False,
           bbox_to_anchor=(0.5, 0.005), handlelength=1.5, handleheight=0.9, columnspacing=2.0)

fig.text(0.5, 0.035,
         'Values = number of rescued-to-normal genes that are also cell-type markers (FindAllMarkers, padj < 0.05).\n'
         'Color intensity scales with count. Muscle: STAR+Salmon/GRCm39 driver gene lists.',
         ha='center', va='bottom', fontsize=7, color=C_TEXT2)

plt.tight_layout(rect=[0, 0.065, 1, 0.97])
save_fig(fig, FILENAME_V2)
print('Done \u2014 Fig 3.4C v2')

## ── Fig 3.4H v2 — GOBP per cell type muscle ─────────────────────

In [ ]:
# ── Fig 3.4H v2 — GOBP per cell type muscle (v2 only) ────────────
FILENAME_M_V2 = 'Fig_3_4H_GOBP_per_celltype_muscle_v2'

m_base_v2 = f"{MUSCLE_V2}/GOBP_per_celltype/rescued"

# Alternative: 2-row layout for 8 panels
def plot_ct_enrichment_2row(celltypes_files, title, filename, cols=4):
    n_ct  = len(celltypes_files)
    rows  = int(np.ceil(n_ct / cols))
    fig, axes = plt.subplots(rows, cols,
                              figsize=(cols * 5.5, rows * 7.5),
                              gridspec_kw={'wspace': 0.6, 'hspace': 0.5})
    axes_flat = axes.flatten()

    for idx, (ct_label, filepath) in enumerate(celltypes_files):
        ax = axes_flat[idx]
        terms = load_enrich_ct(filepath, n=8)
        if not terms:
            ax.text(0.5, 0.5, 'No significant\nterms', ha='center', va='center',
                    transform=ax.transAxes, fontsize=9, color=C_TEXT2)
            ax.set_title(clean_name(ct_label), fontsize=10, fontweight='bold', color=C_TEXT)
            ax.axis('off')
            continue

        n = len(terms)
        y_pos      = np.arange(n)[::-1]
        counts     = [t['count']       for t in terms]
        nlog10p    = [t['nlog10p']     for t in terms]
        gene_ratio = [t['gene_ratio']  for t in terms]
        descs      = [textwrap.fill(t['desc'], width=22) for t in terms]

        min_c, max_c = min(counts), max(counts)
        sizes = [30 + (c - min_c) / max(max_c - min_c, 1) * 150 for c in counts]

        scatter = ax.scatter(gene_ratio, y_pos, s=sizes, c=nlog10p,
                              cmap='YlGn', vmin=min(nlog10p)*0.8, vmax=max(nlog10p)*1.05,
                              edgecolors=C_TEAL_D, linewidths=0.3, zorder=3, alpha=0.85)

        ax.set_yticks(y_pos)
        ax.set_yticklabels(descs, fontsize=10)
        ax.set_xlabel('Gene ratio', fontsize=8, fontweight='bold')
        ax.set_title(clean_name(ct_label), fontsize=12, fontweight='bold', color=C_TEXT, pad=10)
        ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
        ax.spines['left'].set_linewidth(0.5); ax.spines['bottom'].set_linewidth(0.5)
        ax.tick_params(axis='y', length=0)
        ax.grid(axis='x', alpha=0.15, lw=0.5)
        ax.text(0.98, 0.02, f'{len(terms)} terms shown',
                transform=ax.transAxes, ha='right', va='bottom',
                fontsize=5.5, color=C_TEXT2, fontstyle='italic')

        cbax = inset_axes(ax, width="30%", height="3%", loc='lower left',
                           bbox_to_anchor=(0.02, 0.02, 1, 1), bbox_transform=ax.transAxes)
        cbar = fig.colorbar(scatter, cax=cbax, orientation='horizontal')
        cbar.set_label('$-$log$_{10}$(padj)', fontsize=5, labelpad=2)
        cbar.ax.tick_params(labelsize=4)

    # Hide empty axes
    for idx in range(n_ct, len(axes_flat)):
        axes_flat[idx].axis('off')

    fig.suptitle(title, fontsize=11, fontweight='bold', color=C_TEXT, y=1.01)
    fig.text(0.5, -0.01,
             'Top GOBP terms per cell type (enrichGO on rescued-to-normal genes overlapping cell-type markers).\n'
             'Near-duplicate terms removed. snRNA-seq marker overlap: exploratory.',
             ha='center', fontsize=8, color=C_TEXT2, linespacing=1.4)

    plt.tight_layout(rect=[0, 0.03, 1, 0.98])
    save_fig(fig, filename)
    print(f'Done — {filename}')


m_celltypes_v2 = [
    ('Macrophages',          f'{m_base_v2}/gobp_rescued_Macrophages.csv'),
    ('NMJ nuclei',           f'{m_base_v2}/gobp_rescued_Myonuclei (neuromuscular junction nuclei).csv'),
    ('Myonuclei (Myh4+)',    f'{m_base_v2}/gobp_rescued_Myonuclei (Myh4+_).csv'),
    ('MTJ nuclei',           f'{m_base_v2}/gobp_rescued_Myonuclei (myotendinous junction nuclei).csv'),
    ('Diff. myocytes',       f'{m_base_v2}/gobp_rescued_Myonuclei (differentiating myocytes_).csv'),
    ('Endothelial cells',    f'{m_base_v2}/gobp_rescued_Endothelial cells.csv'),
    ('Muscle stem cell',     f'{m_base_v2}/gobp_rescued_Muscle stem cell.csv'),
    ('Tenocytes',            f'{m_base_v2}/gobp_rescued_Tenocytes.csv'),
]

# Use 2-row layout for 8 panels
plot_ct_enrichment_2row(m_celltypes_v2,
    'Fig. 3.4H — Muscle (v2): GOBP enrichment of rescued genes per cell type',
    FILENAME_M_V2, cols=4)

## ── Fig 3.4I v2 — Cross-tissue rescue cell types (muscle v2) ─────

In [ ]:
# ── Fig 3.4I v2 — Cross-tissue rescue cell types (muscle v2) ─────
FILENAME_V2 = 'Fig_3_4I_cross_tissue_rescue_celltypes_v2'

fig, ax = plt.subplots(figsize=(13, 10))
ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.axis('off')

# ── Helpers ──────────────────────────────────────────────────────
def rbox(cx, cy, w, h, fc, ec, title, sub=None, fs=10, fs_sub=7.5, lw=0.8):
    ax.add_patch(FancyBboxPatch((cx-w/2, cy-h/2), w, h, boxstyle="round,pad=0.012",
                                 facecolor=fc, edgecolor=ec, linewidth=lw,
                                 transform=ax.transAxes, zorder=2))
    if sub:
        ax.text(cx, cy+0.009, title, transform=ax.transAxes, ha='center', va='center',
                fontsize=fs, fontweight='bold', color=ec, zorder=3)
        ax.text(cx, cy-0.009, sub, transform=ax.transAxes, ha='center', va='center',
                fontsize=fs_sub, color=C_TEXT, zorder=3)           # ← black
    else:
        ax.text(cx, cy, title, transform=ax.transAxes, ha='center', va='center',
                fontsize=fs, fontweight='bold', color=ec, zorder=3)

def hline(x1, y1, x2, y2, color, lw=0.7, alpha=0.5):
    ax.plot([x1, x2], [y1, y2], color=color, lw=lw, alpha=alpha,
            transform=ax.transAxes, zorder=1)

kx = 0.15;  mx = 0.85;  cx = 0.50
bw = 0.22;  bh = 0.042; step = 0.058

# ── Title ─────────────────────────────────────────────────────────
ax.text(0.50, 0.97, 'Fig. 3.4I \u2014 Cross-tissue comparison of rescue-driving cell types',
        ha='center', fontsize=13, fontweight='bold', color=C_TEXT, transform=ax.transAxes)

# ── Tissue headers — v2 muscle numbers ────────────────────────────
rbox(kx, 0.91, 0.26, 0.050, C_RED,  C_RED_D,  'KIDNEY',
     '7,499 aging DEGs \u2192 1,961 rescued-to-normal', fs=13, fs_sub=8)
rbox(mx, 0.91, 0.26, 0.050, C_BLUE, C_BLUE_D, 'MUSCLE',
     '1,381 aging DEGs \u2192 491 rescued-to-normal',   fs=13, fs_sub=8)  # ← v2

# ── Kidney cell types — unchanged ─────────────────────────────────
k_cells = [
    ('PT-2 (119)',     'Gstm1, Acox2, Rdh16',       C_TEAL,   C_TEAL_D),
    ('FIB (101)',      'Flt1, Prkd1, ECM genes',     C_TEAL,   C_TEAL_D),
    ('vSMC/MC (85)',   'Prkd1, vascular tone',       C_TEAL,   C_TEAL_D),
    ('EC-2/3/4 (122)', 'Tgfb1, Flt1, B2m',          C_GREEN,  C_GREEN_D),
    ('IMM (51)',        'Tgfb1, Hcls1, Pld4, Nrros', C_PURPLE, C_PURP_D),
    ('PODO (43)',       'Prkd1, Plekho1',             C_TEAL,   C_TEAL_D),
]
ky_start = 0.84
k_ys = []
for i, (name, genes, fc, ec) in enumerate(k_cells):
    y = ky_start - i * step
    k_ys.append(y)
    rbox(kx, y, bw, bh, fc, ec, name, genes)

# ── Muscle cell types — auto-loaded v2 counts ─────────────────────
from collections import Counter
marker_counts_v2 = Counter()
with open(f"{MUSCLE_V2}/muscle_v2_markers_reversal_genes_celltypes.csv") as f:
    for r in csv.DictReader(f):
        marker_counts_v2[r['cluster']] += 1

def mc(name, ct_key, genes, fc, ec):
    n = marker_counts_v2.get(ct_key, 0)
    return (f'{name} ({n})', genes, fc, ec)

m_cells = [
    mc('Myonuclei Myh4+', 'Myonuclei (Myh4+?)',                        'Kcnq5, Wwox',              C_TEAL,   C_TEAL_D),
    mc('Macrophages',      'Macrophages',                               'Tgfb1, Hcls1, Pld4',       C_PURPLE, C_PURP_D),
    mc('Myonuclei diff.',  'Myonuclei (differentiating myocytes?)',     'Kcnq5, regeneration',       C_TEAL,   C_TEAL_D),
    mc('NMJ nuclei',       'Myonuclei (neuromuscular junction nuclei)', 'Ryr3, Kcnq5, Ptprm, Wwox', C_AMBER,  C_AMBER_D),
    mc('Muscle stem',      'Muscle stem cell',                          'Ryr3, Pxdn',                C_TEAL,   C_TEAL_D),
    mc('Endothelial',      'Endothelial cells',                         'Minimal response',          C_GRAY,   C_GRAY_D),
]
m_ys = []
for i, (name, genes, fc, ec) in enumerate(m_cells):
    y = ky_start - i * step
    m_ys.append(y)
    rbox(mx, y, bw, bh, fc, ec, name, genes)

# ── Center: shared themes — DR % v2 ───────────────────────────────
shared = [
    ('Tgfb1 signalling',     'Top upstream regulator in both tissues', C_CORAL,  C_CORAL_D, 0.80),
    ('Immune rebalancing',   'Hcls1, Pld4, Nrros, Ly6e shared',       C_PURPLE, C_PURP_D,  0.73),
    ('12 shared marker genes','Reversal markers in both tissues',      C_AMBER,  C_AMBER_D, 0.66),
    ('DR as dominant driver', 'K: 28% | M: 43% DR-exclusive',         C_GREEN,  C_GREEN_D, 0.59),  # ← v2
]
for label, sub, fc, ec, y in shared:
    rbox(cx, y, 0.22, 0.045, fc, ec, label, sub, fs=9, fs_sub=7)

# ── Connectors ─────────────────────────────────────────────────────
hline(kx+bw/2, k_ys[3], cx-0.11, 0.80, C_CORAL_D)
hline(kx+bw/2, k_ys[4], cx-0.11, 0.80, C_CORAL_D)
hline(mx-bw/2, m_ys[1], cx+0.11, 0.80, C_CORAL_D)
hline(kx+bw/2, k_ys[4], cx-0.11, 0.73, C_PURP_D)
hline(mx-bw/2, m_ys[1], cx+0.11, 0.73, C_PURP_D)

# ── Divider ────────────────────────────────────────────────────────
ax.plot([0.05, 0.95], [0.50, 0.50], color='#E0E0E0', lw=0.5,
        transform=ax.transAxes, zorder=0)

# ── Tissue-specific highlights ─────────────────────────────────────
hl_y = 0.47
ax.text(kx, hl_y, 'Kidney-specific rescue features', ha='center',
        fontsize=10, fontweight='bold', color=C_RED_D, transform=ax.transAxes)
k_hl = [
    'EC-driven vascular rescue: Flt1 in 7 EC subtypes + FIB',
    'Broadly distributed: 23 cell types contribute markers',
    'Podocyte-specific: Prkd1 (Golgi), Plekho1',
    'B2m (senescence marker) in IMM + EC-3',
    'ECM remodelling centred on FIB + vSMC/MC',
]
for i, t in enumerate(k_hl):
    ax.text(kx, hl_y-0.03-i*0.028, '\u2022  '+t, ha='center',
            fontsize=8, color=C_TEXT, transform=ax.transAxes)     # ← black

ax.text(mx, hl_y, 'Muscle-specific rescue features', ha='center',
        fontsize=10, fontweight='bold', color=C_BLUE_D, transform=ax.transAxes)
m_hl = [
    'NMJ long-gene vulnerability: Ryr3, Kcnq5, Wwox (>500 kb)',
    'Concentrated: top 3 cell types carry 73% of markers',
    'Myonuclei fibre-type maintenance (Myh4+)',
    'ECs transcriptionally quiescent (contrast to kidney)',
    'DR disproportionately dominant (43% vs 28%)',               # ← v2
]
for i, t in enumerate(m_hl):
    ax.text(mx, hl_y-0.03-i*0.028, '\u2022  '+t, ha='center',
            fontsize=8, color=C_TEXT, transform=ax.transAxes)     # ← black

# ── Key finding box ────────────────────────────────────────────────
fy = 0.155
ax.add_patch(FancyBboxPatch((0.12, fy-0.04), 0.76, 0.08,
             boxstyle="round,pad=0.015", facecolor='#F8F7F5',
             edgecolor=C_GRAY_D, linewidth=0.8,
             transform=ax.transAxes, zorder=2))
ax.text(0.50, fy+0.02, 'Key finding', ha='center', fontsize=10,
        fontweight='bold', color=C_TEXT, transform=ax.transAxes, zorder=3)
ax.text(0.50, fy-0.01,
        'Immune cells (IMM/macrophages) and Tgfb1 signalling are rescue hubs in both tissues.\n'
        'Kidney rescue is vascular/stromal (ECs, FIB); muscle rescue is neuromuscular (NMJ, myonuclei).\n'
        'ECs are major rescue contributors in kidney but transcriptionally silent in muscle.',
        ha='center', va='center', fontsize=8, color=C_TEXT,        # ← black
        transform=ax.transAxes, zorder=3, linespacing=1.5)

# ── Footnote ───────────────────────────────────────────────────────
ax.text(0.50, 0.05,
        'Marker counts from FindAllMarkers (Seurat) intersected with bulk rescued-to-normal gene sets.\n'
        'Numbers in parentheses = reversal gene markers per cell type. '
        'Muscle v2: STAR+Salmon/GRCm39. EC-2/3/4 counts combined for kidney.',
        ha='center', fontsize=7.5, color=C_TEXT, transform=ax.transAxes)  # ← black

plt.tight_layout(pad=0.5)
save_fig(fig, FILENAME_V2)
print('Done \u2014 Fig 3.4I v2')

## ── Fig 3.5A/D v2 — CellChat interaction overview ───────────────


In [ ]:
# ── Fig 3.5A/D v2 — CellChat interaction overview (muscle v2) ────
FILENAME_V2 = 'Fig_3_5AD_cellchat_interaction_overview_v2'

# Kidney unchanged
k_stats = load_lr_stats(f"{DATA_BASE}/Kidney/5_Cellchat_LR_analysis")

# Muscle v2
m_stats_v2 = load_lr_stats(f"{MUSCLE_V2}", suffix='_v2')            # ← v2

print("Kidney:")
for c in conditions:
    s = k_stats[c]
    print(f"  {c:6s}: {s['n_interactions']:>5d} interactions, prob={s['total_prob']:.1f}, "
          f"{s['n_pairs']} pairs, {s['n_pathways']} pathways")

print("\nMuscle v2:")
for c in conditions:
    s = m_stats_v2[c]
    print(f"  {c:6s}: {s['n_interactions']:>5d} interactions, prob={s['total_prob']:.1f}, "
          f"{s['n_pairs']} pairs, {s['n_pathways']} pathways")

fig, axes = plt.subplots(2, 2, figsize=(13, 9),
                          gridspec_kw={'hspace': 0.35, 'wspace': 0.3})

plot_overview(axes[0, 0], axes[0, 1], k_stats,    'Kidney')
plot_overview(axes[1, 0], axes[1, 1], m_stats_v2, 'Muscle')    # ← v2

fig.suptitle('Fig. 3.5A/D — CellChat interaction overview: kidney and muscle',
             fontsize=11, fontweight='bold', color=C_TEXT, y=1.01)
fig.text(0.5, -0.01,
         'LR interactions filtered to v2 reversal genes (STAR+Salmon/GRCm39 rescued-to-normal gene set).\n'
         'Left: number of significant LR interactions per condition. Right: summed interaction probability (strength).\n'
         'CellChat analysis: exploratory (n=1 per condition).',
         ha='center', fontsize=6.5, color=C_TEXT2, linespacing=1.4)

plt.tight_layout(rect=[0, 0.04, 1, 0.98])
save_fig(fig, FILENAME_V2)
print('Done — Fig 3.5A/D v2')

## ── Fig 3.5C v2 — CellChat pathway heatmap ──────────────────────


In [ ]:
# ── Fig 3.5C v2 — CellChat pathway heatmap (muscle v2) ───────────
FILENAME_V2 = 'Fig_3_5C_cellchat_pathway_heatmap_v2'

# Redefine both — overwritten by Fig 3.4C cell
def build_matrix(pathways, prob_dict):
    mat = []
    for pw in pathways:
        row = [prob_dict[pw].get(c, 0) for c in conditions]
        mat.append(row)
    return np.array(mat)

def draw_heatmap(ax, pathways, mat_raw, mat_norm, title, n_interactions):
    n_pw   = len(pathways)
    n_cond = len(conditions)
    cmap   = mcolors.LinearSegmentedColormap.from_list('w2teal',
             ['#FFFFFF', '#D0EDE2', '#6BBFB0', '#0F6E56', '#0A4A3A'])
    im = ax.imshow(mat_norm, cmap=cmap, aspect='auto', vmin=0, vmax=1,
                    interpolation='nearest')
    for i in range(n_pw):
        for j in range(n_cond):
            val = mat_raw[i, j]
            txt = f'{val:.1f}' if val >= 1 else (f'{val:.2f}' if val >= 0.01 else '')
            intensity = mat_norm[i, j]
            color = 'white' if intensity > 0.6 else C_TEXT
            ax.text(j, i, txt, ha='center', va='center', fontsize=6,
                    color=color, fontweight='bold' if intensity > 0.5 else 'normal')
    ax.set_xticks(range(n_cond))
    ax.set_xticklabels(cond_labels, fontsize=8, fontweight='bold')
    ax.xaxis.set_ticks_position('top')
    ax.xaxis.set_label_position('top')
    ax.set_yticks(range(n_pw))
    ax.set_yticklabels(pathways, fontsize=8)
    ax.set_title(title, fontsize=11, fontweight='bold', color=C_TEXT, pad=12)
    ax.tick_params(axis='y', length=0)
    ax.tick_params(axis='x', length=0)
    for i in range(n_pw + 1):
        ax.axhline(i - 0.5, color='white', lw=1.5)
    for j in range(n_cond + 1):
        ax.axvline(j - 0.5, color='white', lw=1.5)
    for i in range(n_pw):
        ctrl_val = mat_raw[i, 0]
        age_val  = mat_raw[i, 1]
        if ctrl_val > 0 or age_val > 0:
            if age_val > ctrl_val * 1.3:
                ax.text(n_cond - 0.3, i, '\u2191', ha='left', va='center',
                        fontsize=8, color=C_RED_D, fontweight='bold')
            elif age_val < ctrl_val * 0.7:
                ax.text(n_cond - 0.3, i, '\u2193', ha='left', va='center',
                        fontsize=8, color=C_BLUE_D, fontweight='bold')
    return im

def load_pathway_probs_v2(base_path, suffix=''):
    """Load pathway probabilities with optional filename suffix."""
    pathway_prob  = defaultdict(lambda: defaultdict(float))
    pathway_count = defaultdict(lambda: defaultdict(int))
    for cond in conditions:
        fpath = f"{base_path}/{cond}_lr_interactions_reversal_genes{suffix}.csv"
        with open(fpath) as f:
            for r in csv.DictReader(f):
                pw   = r['pathway_name']
                prob = float(r['prob'])
                pathway_prob[pw][cond]  += prob
                pathway_count[pw][cond] += 1
    return pathway_prob, pathway_count

# Kidney unchanged
k_prob, k_count = load_pathway_probs_v2(
    f"{DATA_BASE}/Kidney/5_Cellchat_LR_analysis")

# Muscle v2
m_prob_v2, m_count_v2 = load_pathway_probs_v2(
    f"{MUSCLE_V2}", suffix='_v2')                                    # ← v2

k_pathways    = sorted(k_prob.keys(),
                        key=lambda pw: sum(k_prob.get(pw, {}).values()), reverse=True)
m_pathways_v2 = sorted(m_prob_v2.keys(),
                        key=lambda pw: sum(m_prob_v2[pw].values()), reverse=True)

print(f"Kidney pathways: {k_pathways}")
print(f"Muscle v2 pathways: {m_pathways_v2}")

k_mat         = build_matrix(k_pathways,    k_prob)
m_mat_v2      = build_matrix(m_pathways_v2, m_prob_v2)
k_norm        = row_normalise(k_mat)
m_norm_v2     = row_normalise(m_mat_v2)

fig, (ax_k, ax_m) = plt.subplots(1, 2, figsize=(12, 8),
                                   gridspec_kw={'width_ratios': [1.3, 1], 'wspace': 0.4})

im_k = draw_heatmap(ax_k, k_pathways,    k_mat,    k_norm,
                     'Kidney — CellChat pathways', None)
im_m = draw_heatmap(ax_m, m_pathways_v2, m_mat_v2, m_norm_v2,
                     'Muscle — CellChat pathways', None)        # ← v2

cbar = fig.colorbar(im_k, ax=[ax_k, ax_m], shrink=0.3, aspect=12,
                     pad=0.02, location='bottom')
cbar.set_label('Relative interaction probability (row-normalised)', fontsize=8)
cbar.ax.tick_params(labelsize=7)

fig.suptitle('Fig. 3.5C \u2014 CellChat pathway activity across conditions (reversal genes)',
             fontsize=11, fontweight='bold', color=C_TEXT, y=0.99)
fig.text(0.5, 0.02,
         'Values = summed LR interaction probability per pathway across all cell-type pairs.\n'
         'Colour = row-normalised (brightest = condition with highest activity for that pathway).\n'
         'Muscle v2: STAR+Salmon/GRCm39. Arrows: \u2191 = increased with aging; \u2193 = decreased. CellChat: exploratory (n=1/condition).',
         ha='center', fontsize=6.5, color=C_TEXT2, linespacing=1.4)

plt.tight_layout(rect=[0, 0.09, 1, 0.97])
save_fig(fig, FILENAME_V2)
print('Done \u2014 Fig 3.5C v2')

## ── Fig 3.5E v2 — Muscle pathway communication ──────────────────


In [ ]:
# ── Fig 3.5E v2 — Muscle pathway communication dot matrix ────────
m_lr_v2 = load_all_lr(f"{MUSCLE_V2}", suffix='_v2')

m_pathways_v2 = sorted({r.get('pathway_name', '')
                          for cond in conditions
                          for r in m_lr_v2[cond]
                          if r.get('pathway_name', '')})
print(f"v2 pathways detected: {m_pathways_v2}")

n_cols_v2 = 2
n_rows_v2 = int(np.ceil(len(m_pathways_v2) / n_cols_v2))

fig_m, axes_m = plt.subplots(n_rows_v2, n_cols_v2,
                               figsize=(n_cols_v2 * 7, n_rows_v2 * 7),  # ← wider
                               gridspec_kw={'hspace': 0.5, 'wspace': 0.8})  # ← more wspace

# Handle single row or single panel safely
if n_rows_v2 == 1 and n_cols_v2 == 1:
    axes_m_flat = [axes_m]
elif n_rows_v2 == 1:
    axes_m_flat = list(axes_m)
else:
    axes_m_flat = axes_m.flatten().tolist()

for i, pw in enumerate(m_pathways_v2):
    plot_pathway_heatmap(axes_m_flat[i], m_lr_v2, pw, pw, max_pairs=12)

for i in range(len(m_pathways_v2), len(axes_m_flat)):
    axes_m_flat[i].axis('off')
ax.set_xlim(-0.3, n_cond - 0.7)
fig_m.suptitle('Fig. 3.5E (v2) — Muscle: cell-type communication per pathway across conditions',
               fontsize=11, fontweight='bold', color=C_TEXT, y=1.01)
fig_m.text(0.5, -0.01,
           'Dot size and colour = summed LR interaction probability for each source\u2192target pair.\n'
           'Top cell-type pairs shown per pathway. CellChat on v2 reversal genes (STAR+Salmon/GRCm39), exploratory (n=1/condition).',
           ha='center', fontsize=8, color=C_TEXT, linespacing=1.4)

plt.tight_layout(rect=[0, 0.04, 1, 0.98])
save_fig(fig_m, 'Fig_3_5E_muscle_pathway_communication_v2')
print('Done — Fig 3.5E v2 (muscle)')

## ── Fig 3.5E v2 — Muscle circular networks ──────────────────────


In [ ]:
# ── Fig 3.5E v2 — Muscle circular networks ───────────────────────

# Load v2 LR files
m_lr_v2 = {}
for cond in conditions:
    fpath = f"{MUSCLE_V2}/{cond}_lr_interactions_reversal_genes_v2.csv"
    with open(fpath) as f:
        m_lr_v2[cond] = list(csv.DictReader(f))
    print(f"{cond}: {len(m_lr_v2[cond])} LR interactions")

# Cell types and colors from v2 data
m_celltypes_v2 = sorted(
    set(r['source'] for cond in m_lr_v2 for r in m_lr_v2[cond]) |
    set(r['target'] for cond in m_lr_v2 for r in m_lr_v2[cond])
)
m_ct_colors_v2 = {}
m_palette = ['#6BA3C7','#7E94C0','#9B8EC2','#B68CB5','#C98BA3',
             '#D4A19A','#BDB5A0','#A0C4A8','#D4C5A0','#8FB8A0',
             '#A89BC2','#C4A882','#9CB8D4','#B0A8C0']
for i, ct in enumerate(m_celltypes_v2):
    m_ct_colors_v2[ct] = m_palette[i % len(m_palette)]

print(f"\nCell types in v2: {len(m_celltypes_v2)}")

# ── Overall circular networks ─────────────────────────────────────
FILENAME_ME_V2 = 'Fig_3_5E_muscle_network_circles_v2'

fig_m, axes_m = plt.subplots(1, 5, figsize=(34, 8))
for ci, cond in enumerate(conditions):
    mat = build_adjacency_general(m_lr_v2[cond], m_celltypes_v2)
    draw_network_single(axes_m[ci], mat, m_celltypes_v2, cond_labels[ci],
                         m_ct_colors_v2, label_size=FS_LABEL)

fig_m.suptitle('Muscle (v2) intra-tissue communication networks across conditions',
               fontsize=FS_SUPTITLE, fontweight='bold', color=C_TEXT, y=0.985)
fig_m.text(0.5, 0.93, 'v2 reversal genes (STAR+Salmon/GRCm39)',
           ha='center', fontsize=FS_FOOTNOTE+1, fontweight='bold', color=C_TEXT2)
fig_m.text(0.5, -0.01,
           'Arrows show LR interaction probability between muscle cell types (v2 reversal gene set).\n'
           'Arrow thickness and opacity proportional to probability. Arrow colour = source cell type.\n'
           'CellChat: exploratory (n=1/condition).',
           ha='center', fontsize=FS_FOOTNOTE, color=C_TEXT, linespacing=1.4)
plt.tight_layout(rect=[0, 0.04, 1, 0.95])
save_fig(fig_m, FILENAME_ME_V2)
print('Done — Fig 3.5E muscle circles v2')

# ── Per-pathway circular networks — auto-detect v2 pathways ──────
FILENAME_ME2_V2 = 'Fig_3_5E_muscle_network_by_pathway_v2'

m_pathways_v2 = sorted({r.get('pathway_name', '')
                          for cond in conditions
                          for r in m_lr_v2[cond]
                          if r.get('pathway_name', '')})
print(f"v2 pathways: {m_pathways_v2}")

fig_mp, axes_mp = plt.subplots(len(m_pathways_v2), 5,
                                 figsize=(34, len(m_pathways_v2) * 6.5))
if len(m_pathways_v2) == 1:
    axes_mp = np.array(axes_mp).reshape(1, -1)

for pi, pathway in enumerate(m_pathways_v2):
    all_mats = [build_adjacency_pathway(m_lr_v2[c], m_celltypes_v2, pathway)
                for c in conditions]
    global_max = max(m.max() for m in all_mats) if max(m.max() for m in all_mats) > 0 else 1

    for ci, cond in enumerate(conditions):
        ax = axes_mp[pi, ci]
        mat = all_mats[ci]
        n = len(m_celltypes_v2)
        angles = np.linspace(np.pi/2, -3*np.pi/2, n, endpoint=False)
        node_x = [np.cos(a) for a in angles]
        node_y = [np.sin(a) for a in angles]

        prob_threshold = 0.005
        for i in range(n):
            for j in range(n):
                if mat[i, j] < prob_threshold: continue
                width = 0.3 + (mat[i, j] / global_max) * 4
                alpha = 0.15 + (mat[i, j] / global_max) * 0.55
                color = m_ct_colors_v2.get(m_celltypes_v2[i], C_GRAY_D)
                ax.add_patch(FancyArrowPatch(
                    (node_x[i], node_y[i]), (node_x[j], node_y[j]),
                    connectionstyle="arc3,rad=0.25", arrowstyle='->',
                    mutation_scale=9, color=color, alpha=alpha,
                    linewidth=width, zorder=1))

        for i, ct in enumerate(m_celltypes_v2):
            color = m_ct_colors_v2.get(ct, C_GRAY_D)
            activity = mat[i,:].sum() + mat[:,i].sum()
            node_size = 170 + (activity / max(global_max, 0.01)) * 450
            ax.scatter(node_x[i], node_y[i], s=node_size, c=color,
                       edgecolors='white', linewidths=0.8, zorder=3)
            if ci == 0:
                short = short_name(ct)
                lx, ly = node_x[i]*1.42, node_y[i]*1.42
                ha = 'right' if node_x[i] < -0.1 else ('left' if node_x[i] > 0.1 else 'center')
                ax.text(lx, ly, short, ha=ha, va='center', fontsize=FS_LABEL,
                        fontweight='bold', color=color)

        total = mat.sum()
        n_int = (mat > prob_threshold).sum()
        ax.text(0, -1.72, f'{n_int} int.\nΣ={total:.2f}',
                ha='center', fontsize=FS_STATS, fontweight='bold', color=C_TEXT)

        if pi == 0:
            ax.set_title(cond_labels[ci], fontsize=FS_TITLE,
                         fontweight='bold', color=C_TEXT, pad=10)
        if ci == 0:
            ax.text(-2.3, 0, pathway, ha='center', va='center',
                    fontsize=FS_PATHWAY, fontweight='bold', color=C_TEAL_D, rotation=90)

        ax.set_xlim(-1.9, 1.9); ax.set_ylim(-1.8, 1.6)
        ax.set_aspect('equal'); ax.axis('off')

fig_mp.suptitle('Muscle (v2) communication networks by pathway and condition',
                fontsize=FS_SUPTITLE, fontweight='bold', color=C_TEXT, y=0.985)
fig_mp.text(0.5, 0.955, 'v2 reversal genes (STAR+Salmon/GRCm39)',
            ha='center', fontsize=FS_FOOTNOTE+1, fontweight='bold', color=C_TEXT2)
fig_mp.text(0.5, -0.01,
            'Each panel = one pathway × one condition. Node size = pathway activity.\n'
            'Arrow width = interaction probability. Scaling consistent within each pathway row.\n'
            'CellChat: exploratory (n=1/condition).',
            ha='center', fontsize=FS_FOOTNOTE, color=C_TEXT, linespacing=1.4)
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
save_fig(fig_mp, FILENAME_ME2_V2)
print('Done — Fig 3.5E muscle pathway circles v2')

## ── Fig 3.6A v2 — Venn rescued-to-normal (muscle v2) ─────────────

In [ ]:
# ── Fig 3.6A v2 — Venn rescued-to-normal (muscle v2) ─────────────
FILENAME_V2 = 'Fig_3_6A_venn_rescued_to_normal_v2'

# ── Auto-load counts from v2 files ───────────────────────────────
with open(f"{DATA_BASE}/Kidney/1_DEGs_tables/kidney_rescued_to_normal.csv") as f:
    k_syms = [r['Symbol'] for r in csv.DictReader(f) if r.get('Symbol','') and r['Symbol'] != 'NA']
with open(f"{MUSCLE_V2}/muscle_v2_rescued_to_normal.csv") as f:
    m_syms = [r['Symbol'] for r in csv.DictReader(f) if r.get('Symbol','') and r['Symbol'] != 'NA']

k_set   = set(k_syms)
m_set   = set(m_syms)
shared_set = k_set & m_set
shared  = len(shared_set)
k_total = len(k_set)
m_total = len(m_set)
k_only  = k_total - shared
m_only  = m_total - shared

print(f"Kidney: {k_total} | Muscle: {m_total} | Shared: {shared}")
print(f"Shared: {shared/k_total*100:.1f}% of kidney, {shared/m_total*100:.1f}% of muscle")

# Check direction concordance
with open(f"{DATA_BASE}/Kidney/1_DEGs_tables/kidney_rescued_to_normal.csv") as f:
    k_lfc_dict = {r['Symbol']: float(r['log2FoldChange'])
                  for r in csv.DictReader(f) if r.get('Symbol','')}
with open(f"{MUSCLE_V2}/muscle_v2_rescued_to_normal.csv") as f:
    m_lfc_dict = {r['Symbol']: float(r['log2FoldChange'])
                  for r in csv.DictReader(f) if r.get('Symbol','')}

same_dir = sum(1 for g in shared_set
               if g in k_lfc_dict and g in m_lfc_dict
               and np.sign(k_lfc_dict[g]) == np.sign(m_lfc_dict[g]))
print(f"Same direction: {same_dir}/{shared} ({same_dir/shared*100:.0f}%)")

# Notable shared genes — check which of original 12 are still shared
original_notable = ['Tgfb1','Bax','Lmna','Ctnna1','Maf','Pcbp2',
                     'Ssr3','Tnks2','Ly6e','Pxdn','Cert1','Mxi1']
notable_shared = [g for g in original_notable if g in shared_set]
# Fill to 12 with other shared genes if needed
if len(notable_shared) < 12:
    extras = [g for g in sorted(shared_set) if g not in notable_shared]
    notable_shared += extras[:12 - len(notable_shared)]
notable_shared = notable_shared[:12]
print(f"Notable shared genes: {notable_shared}")

k_pct = shared / k_total * 100
m_pct = shared / m_total * 100

# ── Plot (identical structure to original) ────────────────────────
fig, ax = plt.subplots(figsize=(10, 8))
ax.set_xlim(-5, 5); ax.set_ylim(-4, 5)
ax.set_aspect('equal'); ax.axis('off')

ax.text(0, 4.7, 'Fig. 3.6A', ha='center', va='center',
        fontsize=12, fontweight='bold', color=C_TEXT)
ax.text(0, 4.3, 'Rescued-to-normal genes shared between kidney and muscle',
        ha='center', va='center', fontsize=9.5, color=C_TEXT2)

kidney_e = Ellipse((-0.8, 0.4), width=5.0, height=5.0, angle=0,
                    facecolor=C_RED, edgecolor=C_RED_D, linewidth=1.2,
                    alpha=0.45, zorder=1)
ax.add_patch(kidney_e)
muscle_e = Ellipse((1.5, 0.4), width=3.4, height=3.4, angle=0,
                    facecolor=C_BLUE, edgecolor=C_BLUE_D, linewidth=1.2,
                    alpha=0.45, zorder=1)
ax.add_patch(muscle_e)

ax.text(-2.8, 3.2, 'Kidney',           ha='center', fontsize=14, fontweight='bold', color=C_RED_D)
ax.text(-2.8, 2.8, 'rescued-to-normal', ha='center', fontsize=9,  color=C_RED_D)
ax.text( 3.2, 3.2, 'Muscle',      ha='center', fontsize=14, fontweight='bold', color=C_BLUE_D)
ax.text( 3.2, 2.8, 'rescued-to-normal', ha='center', fontsize=9,  color=C_BLUE_D)

ax.text(-2.5,  0.6, f'{k_only:,}',    ha='center', fontsize=28, fontweight='bold', color=C_RED_D,  zorder=5)
ax.text(-2.5,  0.05,'kidney-only',     ha='center', fontsize=9,  fontweight='bold', color=C_RED_D,  zorder=5)
ax.text( 2.4,  0.6, f'{m_only}',      ha='center', fontsize=28, fontweight='bold', color=C_BLUE_D, zorder=5)
ax.text( 2.4,  0.05,'muscle-only',    ha='center', fontsize=9,  fontweight='bold', color=C_BLUE_D, zorder=5)
ax.text( 0.7,  0.6, f'{shared}',      ha='center', fontsize=28, fontweight='bold', color=C_PURP_D, zorder=5)
ax.text( 0.7,  0.05,'shared',         ha='center', fontsize=9,  fontweight='bold', color=C_PURP_D, zorder=5)
ax.text( 0.7, -0.3, f'({same_dir/shared*100:.0f}% same direction)',
         ha='center', fontsize=7.5, color=C_TEXT2, zorder=5)

# Notable shared genes box
box_y = -2.2
ax.add_patch(FancyBboxPatch((-3.2, box_y-0.7), 6.4, 1.4,
             boxstyle="round,pad=0.15", facecolor='#F8F7F5',
             edgecolor=C_GRAY_D, linewidth=0.6, zorder=2))
ax.text(0, box_y+0.45, f'Notable shared rescued genes ({len(notable_shared)} of {shared})',
        ha='center', fontsize=8.5, fontweight='bold', color=C_TEXT, zorder=3)
for i, gene in enumerate(notable_shared):
    row = i // 4; col = i % 4
    gx = -2.2 + col * 1.5; gy = box_y + 0.05 - row * 0.35
    ax.add_patch(FancyBboxPatch((gx-0.55, gy-0.12), 1.1, 0.28,
                 boxstyle="round,pad=0.04", facecolor=C_PURPLE,
                 edgecolor=C_PURP_D, linewidth=0.4, zorder=3))
    ax.text(gx, gy, gene, ha='center', fontsize=7, fontweight='bold',
            color=C_PURP_D, fontstyle='italic', zorder=4)

ax.text(-2.5, -0.3, f'({k_total:,} total)', ha='center', fontsize=7.5, color=C_TEXT2)
ax.text( 2.4, -0.3, f'({m_total} total)',   ha='center', fontsize=7.5, color=C_TEXT2)

ax.text(0, -3.6,
        f'Overlap: {shared} genes = {k_pct:.1f}% of kidney rescued-to-normal, '
        f'{m_pct:.1f}% of muscle rescued-to-normal.\n'
        f'{same_dir/shared*100:.0f}% ({same_dir}/{shared}) show the same direction of aging change in both tissues.\n'
        'Gene symbols mapped via org.Mm.eg.db. Muscle: STAR+Salmon/GRCm39.',
        ha='center', fontsize=6.5, color=C_TEXT2, linespacing=1.5)

plt.tight_layout(pad=0.5)
save_fig(fig, FILENAME_V2)
print('Done \u2014 Fig 3.6A v2')

## ── Fig 3.6B v2 — Scatter aging LFC kidney vs muscle (v2) ────────



In [ ]:
# ── Fig 3.6B v2 — Scatter aging LFC kidney vs muscle (v2) ────────
FILENAME_V2 = 'Fig_3_6B_scatter_aging_LFC_kidney_vs_muscle_v2'

# ── Auto-load shared genes from v2 files ─────────────────────────
with open(f"{DATA_BASE}/Kidney/1_DEGs_tables/kidney_rescued_to_normal.csv") as f:
    k_rtn = {r['Symbol']: float(r['log2FoldChange'])
             for r in csv.DictReader(f) if r.get('Symbol','')}

with open(f"{MUSCLE_V2}/muscle_v2_rescued_to_normal.csv") as f:
    m_rtn = {r['Symbol']: float(r['log2FoldChange'])
             for r in csv.DictReader(f) if r.get('Symbol','')}

shared_genes = sorted(set(k_rtn.keys()) & set(m_rtn.keys()))
print(f"Shared rescued-to-normal genes: {len(shared_genes)}")

symbols = shared_genes
k_lfc   = np.array([k_rtn[g] for g in shared_genes])
m_lfc   = np.array([m_rtn[g] for g in shared_genes])

same_dir = np.array([(k > 0) == (m > 0) for k, m in zip(k_lfc, m_lfc)])
n_same   = int(same_dir.sum())
n_opp    = len(same_dir) - n_same
r_val    = float(np.corrcoef(k_lfc, m_lfc)[0, 1])
n_total  = len(shared_genes)

print(f"Same direction: {n_same}/{n_total} ({n_same/n_total*100:.0f}%)")
print(f"Pearson r: {r_val:.3f}")

# Keep same label genes — filter to those still in shared set
genes_to_label = {
    'Tgfb1', 'Bax', 'Lmna', 'Ctnna1', 'Maf', 'Pld4', 'Tmem158',
    'Ly6e', 'Nrros', 'Hcls1', 'Tppp3', 'Pxdn', 'Grap',
    'Tbc1d1', 'Prcp', 'Man2b1', 'Pcbp2', 'Ssr3',
} & set(shared_genes)

label_offsets = {
    'Tgfb1': (8, 12),  'Bax': (10, -14),  'Lmna': (8, 12),
    'Ctnna1': (10,-12),'Maf': (-10,-10),   'Pld4': (8, -10),
    'Tmem158': (8, 8), 'Ly6e': (-14, -8),  'Nrros': (10, -8),
    'Hcls1': (-14,10), 'Tppp3': (-12, 8),  'Pxdn': (-12, 12),
    'Grap': (8, 12),   'Tbc1d1': (8, -10), 'Prcp': (-10, 8),
    'Man2b1':(-10,-10),'Pcbp2': (10, -10), 'Ssr3': (-12,-12),
}

# ── Plot ──────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 8))
lim = max(abs(k_lfc).max(), abs(m_lfc).max()) * 1.15
ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim)

ax.fill_between([0, lim],  [0,0],    [lim,lim],   color=C_TEAL,  alpha=0.08, zorder=0)
ax.fill_between([-lim,0],  [-lim,-lim],[0,0],      color=C_TEAL,  alpha=0.08, zorder=0)
ax.fill_between([-lim,0],  [0,0],    [lim,lim],   color=C_CORAL, alpha=0.06, zorder=0)
ax.fill_between([0, lim],  [-lim,-lim],[0,0],      color=C_CORAL, alpha=0.06, zorder=0)

ax.axhline(0, color=C_GRAY_D, lw=0.5, ls='--', zorder=1)
ax.axvline(0, color=C_GRAY_D, lw=0.5, ls='--', zorder=1)
ax.plot([-lim,lim],[-lim,lim], color=C_GRAY_D, lw=0.8, ls=':', alpha=0.5, zorder=1)

ax.scatter(k_lfc[same_dir],  m_lfc[same_dir],  c=C_TEAL_D,  s=40, alpha=0.7,
           edgecolors='white', linewidths=0.3, zorder=3, label=f'Same direction (n={n_same})')
ax.scatter(k_lfc[~same_dir], m_lfc[~same_dir], c=C_CORAL_D, s=40, alpha=0.7,
           edgecolors='white', linewidths=0.3, zorder=3, label=f'Opposite direction (n={n_opp})')

for i, sym in enumerate(symbols):
    if sym in genes_to_label:
        color  = C_TEAL_D if same_dir[i] else C_CORAL_D
        offset = label_offsets.get(sym, (8, 8))
        ax.annotate(sym, xy=(k_lfc[i], m_lfc[i]), xytext=offset,
                    textcoords='offset points', fontsize=7, fontstyle='italic',
                    fontweight='bold', color=color,
                    arrowprops=dict(arrowstyle='-', color=color, lw=0.5, alpha=0.6), zorder=5)

ax.set_xlabel('log$_2$FC aging (kidney)',       fontsize=11, fontweight='bold', color=C_TEXT)
ax.set_ylabel('log$_2$FC aging (muscle)',    fontsize=11, fontweight='bold', color=C_TEXT)
ax.tick_params(labelsize=9)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
ax.spines['left'].set_linewidth(0.5); ax.spines['bottom'].set_linewidth(0.5)

qa = 0.55
ax.text( lim*0.65,  lim*0.85, 'Both upregulated\nwith aging',   ha='center', fontsize=7, color=C_TEAL_D,  alpha=qa, fontstyle='italic')
ax.text(-lim*0.65, -lim*0.85, 'Both downregulated\nwith aging', ha='center', fontsize=7, color=C_TEAL_D,  alpha=qa, fontstyle='italic')
ax.text(-lim*0.65,  lim*0.85, 'Kidney down,\nmuscle up',        ha='center', fontsize=7, color=C_CORAL_D, alpha=qa, fontstyle='italic')
ax.text( lim*0.65, -lim*0.85, 'Kidney up,\nmuscle down',        ha='center', fontsize=7, color=C_CORAL_D, alpha=qa, fontstyle='italic')

stats_text = (f'n = {n_total} genes\n'
              f'Pearson r = {r_val:.3f}\n'
              f'Same direction: {n_same}/{n_total} ({n_same/n_total*100:.0f}%)\n'
              f'Opposite: {n_opp}/{n_total} ({n_opp/n_total*100:.0f}%)')
ax.text(0.03, 0.97, stats_text, transform=ax.transAxes,
        fontsize=7.5, va='top', ha='left', color=C_TEXT,
        bbox=dict(boxstyle='round,pad=0.4', facecolor=C_WHITE,
                  edgecolor=C_GRAY_D, linewidth=0.5, alpha=0.9))

ax.legend(loc='lower right', fontsize=8, frameon=True, framealpha=0.9,
          edgecolor=C_GRAY_D, fancybox=True, handlelength=1.2)

ax.set_title(f'Fig. 3.6B \u2014 Aging fold-change concordance for {n_total} shared rescued genes',
             fontsize=10.5, fontweight='bold', color=C_TEXT, pad=15)
fig.text(0.5, 0.01,
         'Each point = one gene rescued-to-normal in both kidney and muscle. '
         'log2FC from DESeq2 age vs. ctrl.\n'
         'Green shading = concordant quadrants; red shading = discordant. '
         'Muscle v2: STAR+Salmon/GRCm39.',
         ha='center', fontsize=6.5, color=C_TEXT2)

plt.tight_layout(rect=[0, 0.04, 1, 1])
save_fig(fig, FILENAME_V2)
print('Done \u2014 Fig 3.6B v2')

## ── Fig 3.6C v2 — GOBP common rescued genes (v2) ─────────────────

In [ ]:
# Reload shared genes
with open(f"{DATA_BASE}/Kidney/1_DEGs_tables/kidney_rescued_to_normal.csv") as f:
    k_rtn_v2 = {r['Symbol'] for r in csv.DictReader(f) if r.get('Symbol','') and r['Symbol'] != 'NA'}
with open(f"{MUSCLE_V2}/muscle_v2_rescued_to_normal.csv") as f:
    m_rtn_v2 = {r['Symbol'] for r in csv.DictReader(f) if r.get('Symbol','') and r['Symbol'] != 'NA'}

shared_v2 = k_rtn_v2 & m_rtn_v2
valid_shared = sorted([g for g in shared_v2 if g != 'NA' and g.strip() != ''])
print(f"Valid shared genes: {len(valid_shared)}")

# Then run the enrichment
enr = gp.enrichr(
    gene_list=valid_shared,
    gene_sets=['GO_Biological_Process_2023'],
    organism='mouse',
    outdir=None,
    no_plot=True,
)

print(f"Total terms: {len(enr.results)}")
sig = enr.results[enr.results['Adjusted P-value'] < 0.05]
print(f"Significant (padj<0.05): {len(sig)}")

print("\nTop 15 terms:")
for _, r in enr.results.head(15).iterrows():
    print(f"  {r['Term']} (padj={r['Adjusted P-value']:.4f}, genes={r['Genes']})")

In [ ]:
aaaaaa# ── Fig 3.6C v2 — GOBP of common rescued genes (single panel) ────
FILENAME = 'Fig_3_6C_GOBP_common_rescued_genes_v2'

import gseapy as gp

# Load v2 shared genes
with open(f"{DATA_BASE}/Kidney/1_DEGs_tables/kidney_rescued_to_normal.csv") as f:
    k_rtn_v2 = {r['Symbol'] for r in csv.DictReader(f) if r.get('Symbol','') and r['Symbol'] != 'NA'}
with open(f"{MUSCLE_V2}/muscle_v2_rescued_to_normal.csv") as f:
    m_rtn_v2 = {r['Symbol'] for r in csv.DictReader(f) if r.get('Symbol','') and r['Symbol'] != 'NA'}
shared_v2 = k_rtn_v2 & m_rtn_v2
valid_shared = sorted([g for g in shared_v2 if g != 'NA' and g.strip() != ''])
n_shared = len(valid_shared)

# Run direct ORA
enr = gp.enrichr(
    gene_list=valid_shared,
    gene_sets=['GO_Biological_Process_2023'],
    organism='mouse',
    outdir=None,
    no_plot=True,
)

# Top 12 terms
top_terms = enr.results.head(12).copy()
top_terms['clean_term'] = top_terms['Term'].apply(lambda x: x.split('(GO:')[0].strip())
top_terms['gene_list'] = top_terms['Genes'].apply(lambda x: [g.strip() for g in x.split(';')])
top_terms['n_genes'] = top_terms['gene_list'].apply(len)

# Single panel figure
fig, ax = plt.subplots(figsize=(10, 7))

terms     = top_terms['clean_term'].tolist()
n_genes   = top_terms['n_genes'].tolist()
padj_vals = top_terms['Adjusted P-value'].tolist()
y_pos     = np.arange(len(terms))[::-1]
neg_log_p = [-np.log10(max(p, 1e-10)) for p in padj_vals]

scatter = ax.scatter(n_genes, y_pos,
    s=[max(n*55, 35) for n in n_genes],
    c=neg_log_p, cmap='YlOrRd', vmin=0, vmax=3,
    edgecolors=C_TEXT2, linewidths=0.4, zorder=3, alpha=0.85)

# Mark significant terms
for i, (yp, padj) in enumerate(zip(y_pos, padj_vals)):
    if padj < 0.05:
        ax.scatter(n_genes[i], yp, s=n_genes[i]*55,
                   facecolors='none', edgecolors=C_RED_D,
                   linewidths=1.5, zorder=4)

# Gene labels for significant terms
for i, (yp, padj) in enumerate(zip(y_pos, padj_vals)):
    if padj < 0.05:
        gene_str = ', '.join(top_terms['gene_list'].iloc[i])
        ax.text(n_genes[i] + 0.4, yp, gene_str,
                va='center', ha='left', fontsize=6.5,
                color=C_TEXT2, fontstyle='italic')

ax.set_yticks(y_pos)
ax.set_yticklabels(terms, fontsize=9)
ax.set_xlabel('Genes in term', fontsize=10, fontweight='bold')
ax.set_xlim(0.5, max(n_genes) + 6)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_linewidth(0.5)
ax.spines['bottom'].set_linewidth(0.5)
ax.tick_params(axis='y', length=0)
ax.grid(axis='x', alpha=0.15, lw=0.5)

cbar = fig.colorbar(scatter, ax=ax, shrink=0.4, aspect=15, pad=0.02)
cbar.set_label('-log10(padj)', fontsize=8)
cbar.ax.tick_params(labelsize=7)

ax.scatter([], [], s=100, facecolors='none', edgecolors=C_RED_D,
           linewidths=1.5, label='padj < 0.05')
for n_leg in [3, 6]:
    ax.scatter([], [], s=n_leg*55, c='gray', alpha=0.5,
               edgecolors=C_TEXT2, linewidths=0.4, label=f'{n_leg} genes')
ax.legend(loc='lower right', fontsize=7.5, frameon=True, framealpha=0.9,
          edgecolor=C_GRAY_D, title='Legend', title_fontsize=7.5,
          handletextpad=0.3, borderpad=0.4)

ax.set_title(f'GOBP enrichment of {n_shared} common rescued genes',
             fontsize=11, fontweight='bold', color=C_TEXT, pad=12)

fig.suptitle('Fig. 3.6C — Pathway enrichment of genes rescued in both kidney and muscle',
             fontsize=12, fontweight='bold', color=C_TEXT, y=0.99)

fig.text(0.5, 0.02,
         f'Over-representation analysis (Enrichr, GO_Biological_Process_2023) of {n_shared} common rescued-to-normal genes.\n'
         f'Red circles = significant terms (padj < 0.05). Dot size = number of genes. Colour = -log10(padj).\n'
         f'Muscle v2: STAR+Salmon/GRCm39/Ensembl 110.',
         ha='center', fontsize=7, color=C_TEXT2, linespacing=1.5)

save_fig(fig, FILENAME)
print(f'Done — Fig 3.6C v2 ({n_shared} shared genes)')

## ── Fig 3.6D v2 — Summary comparison table (muscle v2) ───────────

In [ ]:
# ── Fig 3.6D v2 — Summary comparison table (muscle v2) ───────────
FILENAME_V2 = 'Fig_3_6D_summary_comparison_table_v2'

# Auto-load cross-tissue stats from v2
with open(f"{DATA_BASE}/Kidney/1_DEGs_tables/kidney_rescued_to_normal.csv") as f:
    k_rtn_set = {r['Symbol'] for r in csv.DictReader(f) if r.get('Symbol','')}
with open(f"{MUSCLE_V2}/muscle_v2_rescued_to_normal.csv") as f:
    m_rtn_set = {r['Symbol'] for r in csv.DictReader(f) if r.get('Symbol','')}

shared_v2     = k_rtn_set & m_rtn_set
n_shared      = len(shared_v2)
k_overlap_pct = f'{n_shared/len(k_rtn_set)*100:.1f}%'
m_overlap_pct = f'{n_shared/len(m_rtn_set)*100:.1f}%'

with open(f"{DATA_BASE}/Kidney/1_DEGs_tables/kidney_rescued_to_normal.csv") as f:
    k_lfc_d = {r['Symbol']: float(r['log2FoldChange'])
               for r in csv.DictReader(f) if r.get('Symbol','')}
with open(f"{MUSCLE_V2}/muscle_v2_rescued_to_normal.csv") as f:
    m_lfc_d = {r['Symbol']: float(r['log2FoldChange'])
               for r in csv.DictReader(f) if r.get('Symbol','')}
same_dir = sum(1 for g in shared_v2
               if g in k_lfc_d and g in m_lfc_d
               and np.sign(k_lfc_d[g]) == np.sign(m_lfc_d[g]))
r_val = float(np.corrcoef(
    [k_lfc_d[g] for g in shared_v2 if g in k_lfc_d and g in m_lfc_d],
    [m_lfc_d[g] for g in shared_v2 if g in k_lfc_d and g in m_lfc_d]
)[0, 1])

print(f"Shared: {n_shared} | Same dir: {same_dir}/{n_shared} ({same_dir/n_shared*100:.0f}%) | r={r_val:.3f}")

# ── Auto-load rescue framework numbers ───────────────────────────
k_rtn_n      = count_rows(f"{DATA_BASE}/Kidney/1_DEGs_tables/kidney_rescued_to_normal.csv")
k_nonres_n   = count_rows(f"{DATA_BASE}/Kidney/1_DEGs_tables/kidney_non_rescued_aged_DEGs.csv")
k_dr_n       = count_rows(f"{DATA_BASE}/Kidney/3_Effect_comparison_tables/kidney_intervention_impact_comparison_DR.csv")
k_sact_n     = count_rows(f"{DATA_BASE}/Kidney/3_Effect_comparison_tables/kidney_intervention_impact_comparison_sAct.csv")
k_combi_n    = count_rows(f"{DATA_BASE}/Kidney/3_Effect_comparison_tables/kidney_intervention_impact_comparison_only_rescued_in_combi.csv")
k_both_n     = k_rtn_n - k_dr_n - k_sact_n - k_combi_n
with open(f"{DATA_BASE}/Kidney/1_DEGs_tables/kidney_res_age_results_with_symbols.csv") as f:
    k_age_n = sum(1 for r in csv.DictReader(f)
                  if r.get('padj','') not in ('','NA') and float(r.get('padj',1)) < 0.05)
with open(f"{DATA_BASE}/Kidney/1_DEGs_tables/kidney_combined_rescued_genes_DE_info.csv") as f:
    k_rescued_n = sum(1 for _ in csv.DictReader(f))

m_rtn_n_v2    = count_rows(f"{MUSCLE_V2}/muscle_v2_rescued_to_normal.csv")
m_nonres_n_v2 = count_rows(f"{MUSCLE_V2}/muscle_v2_non_rescued_aged_DEGs.csv")
m_dr_n_v2     = count_rows(f"{MUSCLE_V2}/muscle_v2_intervention_impact_comparison_DR.csv")
m_sact_n_v2   = count_rows(f"{MUSCLE_V2}/muscle_v2_intervention_impact_comparison_sAct.csv")
m_combi_n_v2  = count_rows(f"{MUSCLE_V2}/muscle_v2_intervention_impact_comparison_only_rescued_in_combi.csv")
m_both_n_v2   = m_rtn_n_v2 - m_dr_n_v2 - m_sact_n_v2 - m_combi_n_v2
with open(f"{MUSCLE_V2}/muscle_v2_res_age_results_with_symbols.csv") as f:
    m_age_n_v2 = sum(1 for r in csv.DictReader(f)
                     if r.get('padj','') not in ('','NA') and float(r.get('padj',1)) < 0.05)
with open(f"{MUSCLE_V2}/muscle_v2_combined_rescued_genes_DE_info.csv") as f:
    m_rescued_n_v2 = sum(1 for _ in csv.DictReader(f))

# Also auto-load genes tested
with open(f"{DATA_BASE}/Kidney/1_DEGs_tables/kidney_res_age_results_with_symbols.csv") as f:
    k_genes_tested = sum(1 for _ in csv.DictReader(f))
with open(f"{MUSCLE_V2}/muscle_v2_res_age_results_with_symbols.csv") as f:
    m_genes_tested_v2 = sum(1 for _ in csv.DictReader(f))

rows_v2 = [
    # Bulk RNA-seq
    (0, 'Bulk RNA-seq',              '',                       '',                        True,  '#EEF4FB', False, None),
    (1, 'Genes tested',              f'{k_genes_tested:,}',    f'{m_genes_tested_v2:,}',  False, C_WHITE,   False, None),
    (1, 'Aging DEGs (padj < 0.05)', f'{k_age_n:,}',           f'{m_age_n_v2:,}',         False, C_WHITE,   False, 'kidney'),
    (1, 'Pipeline',                  'STAR + Salmon + DESeq2', 'STAR + Salmon + DESeq2',  False, C_WHITE,   False, None),
    (1, 'Reference genome',          'GRCm39 / Ensembl 110',  'GRCm39 / Ensembl 110',    False, C_WHITE,   False, None),
    (0, '',                          '',                       '',                        False, C_WHITE,   False, None),
    # Rescue framework
    (0, 'Rescue framework',          '',                       '',                        True,  '#EDF8F2', False, None),
    (1, 'Rescued by combi (Stage 1)',    f'{k_rescued_n:,} ({k_rescued_n/k_age_n*100:.0f}%)',       f'{m_rescued_n_v2:,} ({m_rescued_n_v2/m_age_n_v2*100:.0f}%)',      False, C_WHITE, False, None),
    (1, 'Rescued-to-normal (Stage 2)',   f'{k_rtn_n:,}',                                             f'{m_rtn_n_v2:,}',                                                 False, C_WHITE, False, None),
    (1, 'DR-exclusive',                  f'{k_dr_n:,} ({k_dr_n/k_rtn_n*100:.0f}%)',                 f'{m_dr_n_v2:,} ({m_dr_n_v2/m_rtn_n_v2*100:.0f}%)',               False, C_WHITE, False, 'muscle'),
    (1, 'sAct-exclusive',                f'{k_sact_n:,} ({k_sact_n/k_rtn_n*100:.0f}%)',             f'{m_sact_n_v2:,} ({m_sact_n_v2/m_rtn_n_v2*100:.0f}%)',           False, C_WHITE, False, None),
    (1, 'Combi-only (synergistic)',      f'{k_combi_n:,} ({k_combi_n/k_rtn_n*100:.0f}%)',           f'{m_combi_n_v2:,} ({m_combi_n_v2/m_rtn_n_v2*100:.0f}%)',         False, C_WHITE, False, None),
    (1, 'Both DR + sAct',                f'{k_both_n:,} ({k_both_n/k_rtn_n*100:.0f}%)',             f'{m_both_n_v2:,} ({m_both_n_v2/m_rtn_n_v2*100:.0f}%)',           False, C_WHITE, False, 'kidney'),
    (1, 'Non-rescued aging DEGs',        f'{k_nonres_n:,} ({k_nonres_n/k_age_n*100:.1f}%)',         f'{m_nonres_n_v2:,} ({m_nonres_n_v2/m_age_n_v2*100:.1f}%)',       False, C_WHITE, True,  None),
    (0, '',                          '',                       '',                        False, C_WHITE,   False, None),
    (0, 'Top enriched pathways',                  '',                      '',                     True,  '#FDF8EE', False, None),
    (1, 'Aging signature',                        'Fatty acid metabolism', 'Striated muscle\ncontraction',  False, C_WHITE, False, None),
    (1, 'Rescued-to-normal',                      'Cellular respiration\n(703 terms)',  'Supramolecular fibre\norganisation (17 terms)',  False, C_WHITE, False, None),
    (1, 'Non-rescued',                            'Mito gene expression,\nprotein folding\n(195 terms)', 'Ion transport,\nnitric oxide signalling\n(45 terms)', False, C_WHITE, False, None),
    (0, '',                                       '',                      '',                     False, C_WHITE,  False, None),
    (0, 'Single-nucleus RNA-seq',                 '',                      '',                     True,  '#F5EDF8', False, None),
    (1, 'Total nuclei',                           '56,686',                '24,419',               False, C_WHITE,  False, None),
    (1, 'Cell types annotated',                   '25',                    '14',                   False, C_WHITE,  False, None),
    (1, 'Reversal gene markers (hits)',           '986',                   '251',                  False, C_WHITE,  False, None),
    (1, 'Unique reversal genes as markers',       '354',                   '87',                   False, C_WHITE,  False, None),
    (1, 'Top rescue cell types',                  'PT-2, FIB, vSMC/MC,\nEC-2/3/4, IMM', 'Myh4+ myonuclei,\nMacrophages, diff. myo', False, C_WHITE, False, None),
    (1, 'EC rescue contribution',                 'Major (6 subtypes)',    'Minimal (15 markers)', False, C_WHITE,  False, 'kidney'),
    (1, 'Key shared gene',                        'Tgfb1 (IMM, EC-2/3)',  'Tgfb1 (Macrophages)',  False, C_WHITE,  False, 'both'),
    (0, '',                                       '',                      '',                     False, C_WHITE,  False, None),
    (0, 'Cell\u2013cell communication (CellChat)', '',                     '',                     True,  '#FDF2F2', False, None),
    (1, 'Pathways detected',                      '13',                    '5',                    False, C_WHITE,  False, None),
    (1, 'Key signalling hubs',                    'VEGF, EGF, PDGF,\nSEMA3, EPHA/B', 'PTPRM, APP,\nPROS, TGFb, NECTIN',  False, C_WHITE, False, None),
    (0, '',                                       '',                      '',                     False, C_WHITE,  False, None),
    (0, 'Cross-tissue integration',               '',                      '',                     True,  '#EEF0EE', False, None),
    (1, 'Common rescued-to-normal genes',         str(n_shared),           str(n_shared),          False, C_WHITE,  False, 'both'),
    (1, 'Direction consensus',                    f'{same_dir}/{n_shared} ({same_dir/n_shared*100:.0f}%)', f'{same_dir}/{n_shared} ({same_dir/n_shared*100:.0f}%)', False, C_WHITE, False, 'both'),
    (1, 'Pearson r (aging LFC)',                  f'{r_val:.3f}',          f'{r_val:.3f}',         False, C_WHITE,  False, 'both'),
    (1, 'Overlap as % of tissue RTN set',         k_overlap_pct,           m_overlap_pct,          False, C_WHITE,  False, 'muscle'),
]

# ── Font sizes ────────────────────────────────────────────────────
FS_CAPTION  = 11.5   # caption title
FS_CAPTION2 = 9.5    # caption subtitle
FS_HEADER   = 11     # column headers
FS_SECTION  = 10     # section row labels
FS_DATA     = 9.5    # data row text
FS_FOOTNOTE = 7.5    # footnotes

# ── Plot ──────────────────────────────────────────────────────────
fig, ax = plt.subplots(1, 1, figsize=(12, 13))  # ← slightly larger canvas
ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.axis('off')

tl, tr = 0.03, 0.97
tw = tr - tl
col_w = [0.40, 0.30, 0.30]
col_x = [tl]
for w in col_w[:-1]:
    col_x.append(col_x[-1] + w * tw)

rh = 0.024; rh_sec = 0.026; hdr_h = 0.028  # ← taller rows

ax.text(tl, 0.98, 'Fig. 3.6D.', fontsize=FS_CAPTION, fontweight='bold',
        color=C_TEXT, va='top')
ax.text(tl + 0.065, 0.98,
        'Side-by-side comparison of key results metrics for kidney and muscle datasets.',
        fontsize=FS_CAPTION, color=C_TEXT, va='top')
ax.text(tl, 0.96,
        'Muscle v2: STAR+Salmon/GRCm39 — harmonised with kidney pipeline.',
        fontsize=FS_CAPTION2, color=C_TEXT2, va='top')

y_top = 0.945
ax.plot([tl, tr], [y_top, y_top], color=C_TEXT, lw=1.5, clip_on=False)

headers = ['Metric', 'Kidney', 'Muscle']
aligns  = ['left', 'center', 'center']
pads    = [0.01, 0, 0]
for i, (lab, align, pad) in enumerate(zip(headers, aligns, pads)):
    hx = col_x[i] + pad if align == 'left' else col_x[i] + col_w[i]*tw/2
    ax.text(hx, y_top - hdr_h/2, lab, fontsize=FS_HEADER, fontweight='bold',
            color=C_TEXT, va='center', ha=align if align != 'center' else 'center')

ax.plot(col_x[1] + col_w[1]*tw/2 - 0.05, y_top - hdr_h/2, 'o',
        color=C_RED_D, markersize=6, transform=ax.transAxes, clip_on=False)
ax.plot(col_x[2] + col_w[2]*tw/2 - 0.04, y_top - hdr_h/2, 'o',
        color=C_BLUE_D, markersize=6, transform=ax.transAxes, clip_on=False)

y_hdr_bot = y_top - hdr_h
ax.plot([tl, tr], [y_hdr_bot, y_hdr_bot], color=C_TEXT, lw=0.8, clip_on=False)

y_cur = y_hdr_bot
highlight_fill = '#E8F5E9'

for indent, cat, kidney, muscle, is_sec, bg, is_gray, hl in rows_v2:
    if cat == '' and kidney == '' and muscle == '':
        y_cur -= rh * 0.3; continue

    this_rh = rh_sec if is_sec else rh
    n_lines = max(kidney.count('\n'), muscle.count('\n'), cat.count('\n')) + 1
    if n_lines > 1:
        this_rh = rh * n_lines * 0.9

    if bg != C_WHITE:
        ax.add_patch(mpatches.Rectangle((tl, y_cur-this_rh), tw, this_rh,
                                         fc=bg, ec='none', zorder=0))
    if hl == 'kidney':
        ax.add_patch(mpatches.Rectangle((col_x[1], y_cur-this_rh), col_w[1]*tw, this_rh,
                                         fc=highlight_fill, ec='none', zorder=0))
    elif hl == 'muscle':
        ax.add_patch(mpatches.Rectangle((col_x[2], y_cur-this_rh), col_w[2]*tw, this_rh,
                                         fc=highlight_fill, ec='none', zorder=0))
    elif hl == 'both':
        ax.add_patch(mpatches.Rectangle((col_x[1], y_cur-this_rh), (col_w[1]+col_w[2])*tw, this_rh,
                                         fc=highlight_fill, ec='none', zorder=0))

    ax.plot([tl, tr], [y_cur-this_rh, y_cur-this_rh],
            color='#E5E5E5', lw=0.3, clip_on=False, zorder=1)

    tc = C_TEXT2 if is_gray else C_TEXT
    ax.text(col_x[0]+0.01+indent*0.02, y_cur-this_rh/2, cat,
            fontsize=FS_SECTION if is_sec else FS_DATA,
            fontweight='bold' if is_sec else 'normal',
            color=tc, va='center', ha='left', zorder=2, linespacing=1.3)
    if kidney:
        ax.text(col_x[1]+col_w[1]*tw/2, y_cur-this_rh/2, kidney,
                fontsize=FS_DATA, color=tc, va='center', ha='center',
                zorder=2, linespacing=1.3)
    if muscle:
        ax.text(col_x[2]+col_w[2]*tw/2, y_cur-this_rh/2, muscle,
                fontsize=FS_DATA, color=tc, va='center', ha='center',
                zorder=2, linespacing=1.3)
    y_cur -= this_rh

ax.plot([tl, tr], [y_cur, y_cur], color=C_TEXT, lw=1.5, clip_on=False)

fn_y = y_cur - 0.015
fns = [
    'Green highlighting indicates the tissue with the more prominent value for that metric.',
    'Both tissues processed with STAR+Salmon/GRCm39/DESeq2+lfcShrink(ashr) — fully harmonised pipeline.',
    'Percentages in rescue framework relative to rescued-to-normal set (Stage 3) or aging DEGs (Stage 1).',
    'CellChat results are exploratory (n=1/condition). snRNA-seq module scores use bulk-derived gene sets.',
]
for fn in fns:
    ax.text(tl+0.01, fn_y, fn, fontsize=FS_FOOTNOTE, color=C_TEXT2, va='top')
    fn_y -= 0.020

plt.tight_layout(pad=0.3)
save_fig(fig, FILENAME_V2)
print('Done \u2014 Fig 3.6D v2')

## ── Fig 3.6E v2 — Integrative rescue model (muscle v2) ───────────


In [ ]:
# ── Fig 3.6E v2 — Integrative rescue model (muscle v2) ───────────
FILENAME_V2 = 'Fig_3_6E_integrative_rescue_model_v2'

fig, ax = plt.subplots(figsize=(14, 16))
ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.axis('off')

# Redefine helpers — arrow overwritten by CellChat cell
def rbox(cx, cy, w, h, fc, ec, title, sub=None, fs=9, fs_sub=6.5, lw=0.8):
    ax.add_patch(FancyBboxPatch((cx-w/2, cy-h/2), w, h, boxstyle="round,pad=0.012",
                                 facecolor=fc, edgecolor=ec, linewidth=lw,
                                 transform=ax.transAxes, zorder=2))
    if sub:
        ax.text(cx, cy+h*0.18, title, transform=ax.transAxes, ha='center', va='center',
                fontsize=fs, fontweight='bold', color=ec, zorder=3)
        ax.text(cx, cy-h*0.22, sub, transform=ax.transAxes, ha='center', va='center',
                fontsize=fs_sub, color=C_TEXT2, zorder=3, linespacing=1.3)
    else:
        ax.text(cx, cy, title, transform=ax.transAxes, ha='center', va='center',
                fontsize=fs, fontweight='bold', color=ec, zorder=3)

def arrow(x1, y1, x2, y2, color=C_TEXT2, lw=1.0, style='->', head='->'):
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                xycoords='axes fraction', textcoords='axes fraction',
                arrowprops=dict(arrowstyle=head, color=color, lw=lw,
                                shrinkA=2, shrinkB=2),
                zorder=1)

fig, ax = plt.subplots(figsize=(14, 16))
ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.axis('off')

# Helpers unchanged — rbox, arrow, bracket_text same as original

# ══ LAYER 1 — Title ══════════════════════════════════════════════
ax.text(0.50, 0.975, 'Fig. 3.6E \u2014 Integrative model of transcriptional rescue',
        ha='center', fontsize=12, fontweight='bold', color=C_TEXT, transform=ax.transAxes)
ax.text(0.50, 0.96, 'Ercc1\u0394/\u2212 accelerated aging model: combined DR + sActRIIB intervention',
        ha='center', fontsize=9, color=C_TEXT2, transform=ax.transAxes)

rbox(0.50, 0.925, 0.35, 0.035, C_GRAY, C_GRAY_D,
     'Ercc1\u0394/\u2212: impaired nucleotide excision repair',
     'Genomic instability \u2192 accelerated multisystem aging', fs=8.5, fs_sub=6)
arrow(0.50, 0.907, 0.50, 0.895, C_GRAY_D)
rbox(0.50, 0.878, 0.30, 0.030, C_RED, C_RED_D,
     'Aging transcriptional signature', fs=8.5)
arrow(0.40, 0.863, 0.25, 0.843, C_RED_D, lw=0.8)
arrow(0.60, 0.863, 0.75, 0.843, C_RED_D, lw=0.8)

# ══ LAYER 2 — Tissue aging ════════════════════════════════════════
kx = 0.25; mx = 0.75
rbox(kx, 0.838, 0.28, 0.028, C_RED, C_RED_D, 'Kidney: 7,499 aging DEGs', fs=8)
rbox(mx, 0.838, 0.28, 0.028, C_RED, C_RED_D, 'Muscle: 1,381 aging DEGs', fs=8)  # ← v2

k_aging_y = 0.793
rbox(kx, k_aging_y, 0.26, 0.042, C_RED, C_RED_D,
     'Aging pathways', 'Fatty acid metabolism, oxidative\nstress, amide metabolism', fs=7.5, fs_sub=5.5)
rbox(mx, k_aging_y, 0.26, 0.042, C_RED, C_RED_D,
     'Aging pathways', 'Muscle differentiation,\ncontraction, development', fs=7.5, fs_sub=5.5)

# ══ LAYER 3 — Intervention ════════════════════════════════════════
intv_y = 0.735
ax.add_patch(FancyBboxPatch((0.15, intv_y-0.025), 0.70, 0.05,
             boxstyle="round,pad=0.015", facecolor=C_GREEN, edgecolor=C_GREEN_D,
             linewidth=1.2, transform=ax.transAxes, zorder=2))
ax.text(0.50, intv_y+0.008, 'Combined intervention: Dietary Restriction + sActRIIB',
        ha='center', fontsize=9.5, fontweight='bold', color=C_GREEN_D,
        transform=ax.transAxes, zorder=3)
ax.text(0.50, intv_y-0.012,
        'DR: 30% caloric restriction from 9 weeks  |  sActRIIB: 10 mg/kg i.p. 2\u00d7/week from 8 weeks',
        ha='center', fontsize=6, color=C_TEXT2, transform=ax.transAxes, zorder=3)
arrow(kx, 0.772, 0.35, intv_y+0.025, C_GREEN_D, lw=0.8)
arrow(mx, 0.772, 0.65, intv_y+0.025, C_GREEN_D, lw=0.8)

# ══ LAYER 4 — Rescue outcomes ═════════════════════════════════════
rescue_y = 0.672
arrow(0.35, intv_y-0.025, kx, rescue_y+0.020, C_TEAL_D, lw=0.8)
arrow(0.65, intv_y-0.025, mx, rescue_y+0.020, C_TEAL_D, lw=0.8)

rbox(kx, rescue_y, 0.26, 0.035, C_TEAL, C_TEAL_D,
     '1,961 rescued-to-normal',
     '62% of aging DEGs reversed by combi', fs=8, fs_sub=5.5)
rbox(mx, rescue_y, 0.26, 0.035, C_TEAL, C_TEAL_D,
     '491 rescued-to-normal',                               # ← v2
     '52% of aging DEGs reversed by combi', fs=8, fs_sub=5.5)  # ← v2

# ══ LAYER 5 — Driver breakdown ════════════════════════════════════
driver_y = 0.61
ax.text(kx, driver_y+0.025, 'Intervention drivers:', ha='center',
        fontsize=6.5, fontweight='bold', color=C_TEXT2, transform=ax.transAxes)
ax.text(mx, driver_y+0.025, 'Intervention drivers:', ha='center',
        fontsize=6.5, fontweight='bold', color=C_TEXT2, transform=ax.transAxes)

k_drivers = [
    ('DR-excl. 28%', C_GREEN,  C_GREEN_D, 0.07),
    ('Both 62%',     C_CORAL,  C_CORAL_D, 0.07),
    ('sAct 6%',      C_PURPLE, C_PURP_D,  0.04),
    ('Syn. 4%',      C_AMBER,  C_AMBER_D, 0.04),
]
kd_x = 0.11
for label, fc, ec, w in k_drivers:
    rbox(kd_x+w/2, driver_y, w, 0.022, fc, ec, label, fs=6)
    kd_x += w + 0.005

m_drivers = [                                               # ← v2 percentages
    ('DR-excl. 43%', C_GREEN,  C_GREEN_D, 0.08),
    ('Both 50%',     C_CORAL,  C_CORAL_D, 0.07),
    ('sAct 4%',      C_PURPLE, C_PURP_D,  0.035),
    ('Syn. 3%',      C_AMBER,  C_AMBER_D, 0.035),
]
md_x = 0.61
for label, fc, ec, w in m_drivers:
    rbox(md_x+w/2, driver_y, w, 0.022, fc, ec, label, fs=6)
    md_x += w + 0.005

# ══ LAYER 6 — Cell-type resolution ═══════════════════════════════
ct_y = 0.535
ax.text(0.50, 0.580, 'Cell-type resolution (snRNA-seq module score projection)',
        ha='center', fontsize=8, fontweight='bold', color=C_TEXT, transform=ax.transAxes)
ax.plot([0.10, 0.90], [0.573, 0.573], color='#E0E0E0', lw=0.5, transform=ax.transAxes)

ax.text(kx, ct_y+0.038, 'Kidney cell types', ha='center', fontsize=8,
        fontweight='bold', color=C_RED_D, transform=ax.transAxes)
ax.text(mx, ct_y+0.038, 'Muscle cell types', ha='center', fontsize=8,
        fontweight='bold', color=C_BLUE_D, transform=ax.transAxes)

k_cells = [
    ('EC-2/3/4', 'Tgfb1, Flt1, B2m\nVascular rescue',  C_GREEN,  C_GREEN_D),
    ('FIB',      'Flt1, Prkd1\nECM remodelling',        C_TEAL,   C_TEAL_D),
    ('IMM',      'Tgfb1, Hcls1\nImmune rebalancing',    C_PURPLE, C_PURP_D),
    ('PT-2',     'Gstm1, Acox2\nMetabolic rescue',      C_TEAL,   C_TEAL_D),
    ('PODO',     'Prkd1, Plekho1\nGlomerular',          C_TEAL,   C_TEAL_D),
]
kcx_start = 0.06; kcx_step = 0.086
for i, (name, sub, fc, ec) in enumerate(k_cells):
    rbox(kcx_start+i*kcx_step+0.04, ct_y, 0.075, 0.048, fc, ec, name, sub, fs=7, fs_sub=4.5)

m_cells = [
    ('Myh4+\nmyonuclei', 'Kcnq5, Wwox\nFibre maintenance', C_TEAL,   C_TEAL_D),
    ('Macro-\nphages',    'Tgfb1, Hcls1\nImmune hub',       C_PURPLE, C_PURP_D),
    ('NMJ\nnuclei',      'Ryr3, Ptprm\nLong-gene rescue',  C_AMBER,  C_AMBER_D),
    ('Muscle\nstem',     'Ryr3, Pxdn\nRegeneration',        C_TEAL,   C_TEAL_D),
    ('Endo-\nthelial',   'Minimal\nresponse',               C_GRAY,   C_GRAY_D),
]
mcx_start = 0.54
for i, (name, sub, fc, ec) in enumerate(m_cells):
    rbox(mcx_start+i*kcx_step+0.04, ct_y, 0.075, 0.048, fc, ec, name, sub, fs=6.5, fs_sub=4.5)

imm_x   = kcx_start + 2*kcx_step + 0.04
macro_x = mcx_start + 1*kcx_step + 0.04
arrow(imm_x,   ct_y-0.024, 0.38, 0.445+0.040, C_PURP_D, lw=0.6)
arrow(macro_x, ct_y-0.024, 0.62, 0.445+0.040, C_PURP_D, lw=0.6)

# ══ LAYER 7 — Shared core ═════════════════════════════════════════
shared_y = 0.445
ax.add_patch(FancyBboxPatch((0.22, shared_y-0.040), 0.56, 0.080,
             boxstyle="round,pad=0.015", facecolor=C_PURPLE, edgecolor=C_PURP_D,
             linewidth=1.0, alpha=0.5, transform=ax.transAxes, zorder=2))
ax.text(0.50, shared_y+0.022,
        'Conserved rescue core: 81 genes (r = {:.2f}, {}% concordant)'.format(  # ← v2 auto
            r_val if 'r_val' in dir() else 0.75,
            int(same_dir/n_shared*100) if 'same_dir' in dir() else 86),
        ha='center', fontsize=8.5, fontweight='bold', color=C_PURP_D,
        transform=ax.transAxes, zorder=3)

modules = [
    ('Tgfb1\nsignalling',     C_CORAL,  C_CORAL_D),
    ('Immune\nrebalancing',   C_PURPLE, C_PURP_D),
    ('Golgi/vesicle\ntransport', C_TEAL, C_TEAL_D),
    ('Ubiquitin/\nproteostasis', C_AMBER, C_AMBER_D),
    ('Cell adhesion/\ncytoskeleton', C_BLUE, C_BLUE_D),
]
mod_w = 0.085
mod_start = 0.50 - (len(modules)*(mod_w+0.008))/2 + mod_w/2
for i, (label, fc, ec) in enumerate(modules):
    rbox(mod_start+i*(mod_w+0.008), shared_y-0.012, mod_w, 0.032, fc, ec, label, fs=5.5)

# ══ LAYER 8 — Tissue-specific insights ═══════════════════════════
ins_y = 0.355
ax.plot([0.10, 0.90], [0.385, 0.385], color='#E0E0E0', lw=0.5, transform=ax.transAxes)
ax.text(0.50, 0.395, 'Tissue-specific rescue programmes',
        ha='center', fontsize=8, fontweight='bold', color=C_TEXT, transform=ax.transAxes)

k_ins = [
    'Vascular rescue: Flt1/VEGFR1 in 7 EC subtypes + FIB',
    'Broadly distributed: 23 cell types contribute',
    'Podocyte programmes: Golgi integrity (Prkd1)',
    'B2m senescence in IMM + EC-3',
    'ECM remodelling centred on FIB + vSMC/MC',
]
ax.add_patch(FancyBboxPatch((0.04, ins_y-0.065), 0.42, 0.12,
             boxstyle="round,pad=0.012", facecolor=C_RED, edgecolor=C_RED_D,
             linewidth=0.6, alpha=0.3, transform=ax.transAxes, zorder=0))
ax.text(kx, ins_y+0.045, 'Kidney', ha='center', fontsize=9,
        fontweight='bold', color=C_RED_D, transform=ax.transAxes)
for i, txt in enumerate(k_ins):
    ax.text(kx, ins_y+0.02-i*0.020, '\u2022 '+txt, ha='center',
            fontsize=5.5, color=C_TEXT2, transform=ax.transAxes)

m_ins = [
    'NMJ long-gene vulnerability (Ryr3, Kcnq5, Wwox >500kb)',
    'Concentrated: top 3 cell types carry 73% of markers',
    'DR disproportionately dominant (43% vs 28%)',           # ← v2
    'ECs transcriptionally quiescent (contrast to kidney)',
    'Myonuclei fibre-type maintenance (Myh4+)',
]
ax.add_patch(FancyBboxPatch((0.54, ins_y-0.065), 0.42, 0.12,
             boxstyle="round,pad=0.012", facecolor=C_BLUE, edgecolor=C_BLUE_D,
             linewidth=0.6, alpha=0.3, transform=ax.transAxes, zorder=0))
ax.text(mx, ins_y+0.045, 'Muscle', ha='center', fontsize=9,
        fontweight='bold', color=C_BLUE_D, transform=ax.transAxes)
for i, txt in enumerate(m_ins):
    ax.text(mx, ins_y+0.02-i*0.020, '\u2022 '+txt, ha='center',
            fontsize=5.5, color=C_TEXT2, transform=ax.transAxes)

# ══ LAYER 9 — CellChat ════════════════════════════════════════════
cc_y = 0.215
ax.plot([0.10, 0.90], [0.260, 0.260], color='#E0E0E0', lw=0.5, transform=ax.transAxes)
ax.text(0.50, 0.270, 'Cell\u2013cell communication (CellChat: exploratory)',
        ha='center', fontsize=8, fontweight='bold', color=C_TEXT, transform=ax.transAxes)

k_cc = [
    ('VEGF',     'Lost to vSMC/IMM;\nrestored by treatments'),
    ('EGF',      'Partial restoration\nby sAct, combi'),
    ('COLLAGEN', 'Silenced by aging;\nDR/combi restore'),
]
for i, (path, note) in enumerate(k_cc):
    rbox(0.10+i*0.14+0.06, cc_y, 0.11, 0.045, C_TEAL, C_TEAL_D, path, note, fs=7, fs_sub=4.5)

m_cc = [                                                    # ← v2 pathways
    ('PROS',  'Macrophage-driven;\nrescue hub'),
    ('PTPRM', 'NMJ-associated\nsignalling'),
    ('TGFb',  'DR-specific\nactivation'),
]
for i, (path, note) in enumerate(m_cc):
    rbox(0.56+i*0.14+0.04, cc_y, 0.11, 0.045, C_TEAL, C_TEAL_D, path, note, fs=7, fs_sub=4.5)

rbox(0.50, cc_y-0.055, 0.30, 0.030, C_AMBER, C_AMBER_D,
     'Inter-organ: ANGPT, SEMA3, COLLAGEN signalling rescued', fs=6.5)

# ══ LAYER 10 — Non-rescued ════════════════════════════════════════
nr_y = 0.105
ax.plot([0.10, 0.90], [0.135, 0.135], color='#E0E0E0', lw=0.5, transform=ax.transAxes)
rbox(0.50, nr_y, 0.70, 0.045, C_GRAY, C_GRAY_D,
     'Non-rescued processes: mitochondrial gene expression, protein folding / chaperone pathways',
     'Proteostasis may operate at protein activity level rather than transcription (see Discussion 4.4)',
     fs=7.5, fs_sub=5.5)

# ══ Bottom key message ════════════════════════════════════════════
ax.add_patch(FancyBboxPatch((0.10, 0.025), 0.80, 0.045,
             boxstyle="round,pad=0.012", facecolor='#F0F7F0', edgecolor=C_GREEN_D,
             linewidth=1.0, transform=ax.transAxes, zorder=2))
ax.text(0.50, 0.055,
        'Combined DR + sActRIIB rescues aging signatures through converging and tissue-specific programmes',
        ha='center', fontsize=8, fontweight='bold', color=C_GREEN_D,
        transform=ax.transAxes, zorder=3)
ax.text(0.50, 0.037,
        'Immune rebalancing (Tgfb1) is the shared hub | Kidney: vascular/stromal | '
        'Muscle: neuromuscular/NMJ | DR is the dominant driver in both',
        ha='center', fontsize=6, color=C_TEXT2, transform=ax.transAxes, zorder=3)

ax.text(0.50, 0.008,
        'Bulk RNA-seq: n=3/condition (statistically powered). snRNA-seq: n=1/condition (exploratory). '
        'Muscle: STAR+Salmon/GRCm39.',
        ha='center', fontsize=5.5, color=C_TEXT2, transform=ax.transAxes)

plt.tight_layout(pad=0.3)
save_fig(fig, FILENAME_V2)
print('Done \u2014 Fig 3.6E v2')

## Supplementary Fig S3.1 v2 — Per-cell-type GOBP aging enrichment


In [ ]:
# ── Muscle v2 aged GOBP (NO subfolders) ─────────────────────────
M_AGED_V2 = f"{DATA_BASE}/Muscle_v2/GOBP_per_celltype/aged"

m_aged_files_v2 = []

if os.path.isdir(M_AGED_V2):
    for fname in sorted(os.listdir(M_AGED_V2)):
        if fname.endswith('.csv'):

            ct = fname.replace('gobp_aged_', '').replace('GOBP_aged_', '').replace('.csv', '')

            ct_short = (
                ct.replace('Myonuclei__neuromuscular_junction_nuclei_', 'NMJ nuclei')
                  .replace('Myonuclei__myotendinous_junction_nuclei_', 'MTJ nuclei')
                  .replace('Myonuclei__differentiating_myocytes__', 'Diff. myocytes')
                  .replace('Myonuclei__Myh4__', 'Myh4+ myonuclei')
                  .replace('Myonuclei__Myh1__', 'Myh1+ myonuclei')
                  .replace('Muscle_stem_cell', 'Muscle stem cells')
                  .replace('Immune_cells_', 'Immune cells')
                  .replace('_', ' ')
            )

            m_aged_files_v2.append((ct_short, f'{M_AGED_V2}/{fname}'))

print(f"Muscle: {len(m_aged_files_v2)} cell types with aged GOBP")

build_supp_aged_figure(
    m_aged_files_v2,
    'Muscle',
    'Supp_Fig_S3_1b_muscle_aged_GOBP_per_celltype_v2',  # ← NEW NAME
    cols=4
)

## Supp Fig S3.4a/b -DAVID modules

In [ ]:
# ── Supp Fig S3.4a — DAVID modules: accelerated aging DEGs ───────
FILENAME_AGED = 'Supp_Fig_S3_4a_DAVID_modules_aged'

# Gene sets already loaded above (k_aged_set, m_aged_set — lowercase)

aged_modules = [
    ('Glycoprotein /\nN-glycosylation',
     ['ace','ache','abca3','abcb1a','abcb1b','abcc1','abcc2','abcc3',
      'abcc6','aadac','adam32','adam33','adam5','adam8','adamts12',
      'adamts15','col3a1','fn1','l1cam','mrc1']),

    ('Extracellular region /\nsecreted proteins',
     ['ace','ache','adam32','adam33','adam5','adam8','adamts12','adamts15',
      'col3a1','fn1','l1cam','tgfb2','vcam1','thbs1','postn','tnn',
      'c1qa','c1qb','c1qc','ccl2','ccl5','cxcl13']),

    ('Membrane /\ntransmembrane',
     ['abcb1a','abcb1b','abcc1','abcc2','abcc3','abcc6','slc1a1',
      'slc1a3','slc22a6','slc22a2','slco1a1','slco2a1','tlr4','tlr2',
      'actc1','actn2','myl4','myl2','tnnt2','tnnt1']),

    ('Innate immune /\nantiviral response',
     ['adgre1','apobec3','bst2','c1qa','c1qb','c1qc','axl',
      'irf7','isg15','oasl1','oasl2','oas1a','oas1b','mx1','mx2',
      'ifitm1','ifitm3','sting1','hck','fcgr1','fcgr3']),

    ('Solute /\norganic anion transport',
     ['abcb1a','abcb1b','abcc1','abcc2','abcc3','abcc6',
      'slc22a6','slc22a2','slc22a12','slco1a1','slco2a1','slco3a1',
      'slco4c1','slco4a1','sfxn3','mfsd4a','mfsd4b1','flvcr1','flvcr2']),

    ('Endoplasmic reticulum /\nprotein processing',
     ['edem1','erlin1','p4htm','p4ha1','lrat','rdh16','rdh12',
      'hsd17b2','sting1','pigz','tmem43','tmco3','golm1','golm2']),

    ('Sarcomere /\nmuscle contraction',
     ['actc1','actn2','ampd3','ankrd1','ankrd2','atp1a3','col24a1',
      'col4a1','csrp3','fhl1','lmod2','myl4','myl2','myh3','myh8',
      'tnnc1','tnni1','tnnt1','tnnt2','myom3','myoz2','csrp3']),

    ('Actin binding /\ncytoskeleton',
     ['actn2','avil','csrp3','fscn1','hcls1','lmod2','map1b',
      'micall2','myh3','myh8','myo1a','shroom3','aif1l']),

    ('Myogenesis /\nmuscle development',
     ['col19a1','csrp3','fhl1','mustn1','myf6','mymk','mymx','myog',
      'musk','smtnl1','mb','tnni1']),
]

aged_rep_genes = {
    'Glycoprotein /\nN-glycosylation':          'Ace, Abcb1a, Adam33, Col3a1, Fn1',
    'Extracellular region /\nsecreted proteins': 'Tgfb2, Vcam1, Postn, Ccl2, C1qa',
    'Membrane /\ntransmembrane':                 'Abcc1, Slc22a6, Tlr4, Actc1, Tnnt2',
    'Innate immune /\nantiviral response':       'Irf7, Isg15, Mx1, Bst2, Sting1',
    'Solute /\norganic anion transport':         'Slc22a6, Slco1a1, Abcc1, Flvcr1',
    'Endoplasmic reticulum /\nprotein processing':'Edem1, Lrat, Rdh16, Golm1',
    'Sarcomere /\nmuscle contraction':           'Actc1, Myh3, Tnnc1, Tnnt2, Myl4',
    'Actin binding /\ncytoskeleton':             'Actn2, Map1b, Fscn1, Myh3',
    'Myogenesis /\nmuscle development':          'Myog, Myf6, Mymx, Mymk, Csrp3',
}

# Aged sets (lowercase, |LFC| > 1, padj < 0.05)
with open(f"{DATA_BASE}/Kidney/1_DEGs_tables/kidney_res_age_results_with_symbols.csv") as f:
    k_aged_set = {r['Symbol'].lower() for r in csv.DictReader(f)
                  if r.get('Symbol','') and r.get('padj','') not in ('','NA')
                  and float(r.get('padj',1)) < 0.05
                  and abs(float(r.get('log2FoldChange',0))) > 1}

with open(f"{MUSCLE_V2}/muscle_v2_res_age_results_with_symbols.csv") as f:
    m_aged_set = {r['Symbol'].lower() for r in csv.DictReader(f)
                  if r.get('Symbol','') and r.get('padj','') not in ('','NA')
                  and float(r.get('padj',1)) < 0.05
                  and abs(float(r.get('log2FoldChange',0))) > 1}

# Non-rescued sets (lowercase)
with open(f"{DATA_BASE}/Kidney/1_DEGs_tables/kidney_non_rescued_aged_DEGs.csv") as f:
    k_nonr_set = {r['Symbol'].lower() for r in csv.DictReader(f) if r.get('Symbol','')}
with open(f"{MUSCLE_V2}/muscle_v2_non_rescued_aged_DEGs.csv") as f:
    m_nonr_set = {r['Symbol'].lower() for r in csv.DictReader(f) if r.get('Symbol','')}

print(f"k_aged_set: {len(k_aged_set)} | m_aged_set: {len(m_aged_set)}")
print(f"k_nonr_set: {len(k_nonr_set)} | m_nonr_set: {len(m_nonr_set)}")

# ── Draw aged figure ──────────────────────────────────────────────
make_butterfly_fig(
    aged_modules, aged_rep_genes,
    k_aged_set, m_aged_set,
    FILENAME_AGED,
    suptitle='Supp. Fig. S3.4a — DAVID functional modules: accelerated aging DEGs (|LFC| > 1)',
    footnote=('Modules defined from DAVID functional annotation clustering of accelerated aging DEGs (padj < 0.05, |LFC| > 1). '
              'Purple = genes present in both tissues. Bold numbers = total per tissue. '
              'Both tissues: STAR+Salmon/GRCm39/Ensembl 110.'),
    figsize=(14, 8), xlim=(-30, 38),
)

# ── Supp Fig S3.4b — DAVID modules: non-rescued genes ────────────
FILENAME_NONR = 'Supp_Fig_S3_4b_DAVID_modules_non_rescued'

nonr_modules = [
    ('Solute / organic\nanion transport',
     ['abcb1a','add2','ano9','atp11a','best3','catsper4','cd36',
      'cdh17','clca3a1','clic6','slc22a6','slc22a2','slc22a12',
      'slc22a7','slco1a1','slco2a1','slco3a1','slco4c1',
      'slc15a1','slc17a2','slc17a3','slc39a8','scn7a']),

    ('Extracellular /\nsecreted proteins',
     ['ace','adamts12','adamts4','ager','alad','angptl3','angptl8',
      'apoc2','c1qtnf12','c1qtnf3','c3','wnt2','wnt16','fgf10',
      'apoc1','dkk3','lgals3','msln','orm2','serpina3m','serpina3n']),

    ('Peroxisome /\nlipid metabolism',
     ['agps','amacr','crot','dhrs4','ephx2','far1','idi1','mpv17l',
      'nudt19','pecr','acot4','cyp2e1','gsta3','lipg','ampd3']),

    ('Glycoprotein /\nER membrane',
     ['abcb1a','ace','adamts12','adamts4','ager','aldh1b1','angptl3',
      'angptl8','ano9','apoc2','arsg','slc22a6','slc22a2','cdh17',
      'chodl','musk','slc2a1','serpina3m','serpina3n','msln','orm2']),

    ('Lysosome /\nprotein degradation',
     ['hexb','hexa','ctss','ctsk','ctsc','ctse','arsa','arsg',
      'lamp1','gba','npc1','laptm5']),

    ('Protein kinase /\nsignalling',
     ['musk','irak2','plk2','mknk2','slc7a5','slc43a2','slc2a1']),

    ('Transcriptional\nregulation',
     ['dkk3','musk','myog','trp63','zim1','e2f1','nfkb2']),
]

nonr_rep_genes = {
    'Solute / organic\nanion transport':   'Slc22a6, Slco1a1, Abcb1a, Cdh17',
    'Extracellular /\nsecreted proteins':  'C3, Angptl3, Wnt2, Apoc1, Dkk3',
    'Peroxisome /\nlipid metabolism':      'Amacr, Ephx2, Acot4, Gsta3',
    'Glycoprotein /\nER membrane':         'Ace, Adamts4, Ager, Msln',
    'Lysosome /\nprotein degradation':     'Hexb, Ctss, Ctsk, Arsa',
    'Protein kinase /\nsignalling':        'Musk, Irak2, Plk2, Mknk2',
    'Transcriptional\nregulation':         'Myog, Trp63, Dkk3, Nfkb2',
}

make_butterfly_fig(
    nonr_modules, nonr_rep_genes,
    k_nonr_set, m_nonr_set,
    FILENAME_NONR,
    suptitle='Supp. Fig. S3.4b — DAVID functional modules: non-rescued aging DEGs (|LFC| > 1)',
    footnote=('Modules defined from DAVID functional annotation clustering of non-rescued aging DEGs (|LFC| > 1). '
              'Note: muscle non-rescued clusters show low enrichment scores due to small gene set size (n=37). '
              'Purple = genes present in both tissues. Both tissues: STAR+Salmon/GRCm39/Ensembl 110.'),
    figsize=(13, 7.5), xlim=(-25, 32),
)

## Supp Fig S3.4c/d — DAVID clusters: aged and non-rescued ───────────

In [ ]:
# ── Supp Fig S3.4c — DAVID cluster detail: accelerated aging DEGs ─
make_david_detail_fig(
    parsed['K_aged'], parsed['M_aged'],
    filename = 'Supp_Fig_S3_4c_DAVID_clusters_aged',
    suptitle = 'Supp. Fig. S3.4c — Top DAVID annotation clusters: accelerated aging DEGs (|LFC| > 1)',
    footnote = ('Top 10 clusters per tissue ranked by enrichment score. '
                'n = genes from input list (padj < 0.05, |LFC| > 1) mapping to cluster. '
                'Kidney: 3,033 input genes; Muscle: 334 input genes. '
                'Cross-tissue module comparison shown in Supp. Fig. S3.4a. '
                'Both tissues: STAR+Salmon/GRCm39/Ensembl 110.'),
    top_n=10, figsize=(15, 7),
)
# ── Supp Fig S3.4d — DAVID cluster detail: non-rescued ───────────
make_david_detail_fig(
    parsed['K_non_rescued'], parsed['M_non_rescued'],
    filename = 'Supp_Fig_S3_4d_DAVID_clusters_non_rescued',
    suptitle = 'Supp. Fig. S3.4d — Top DAVID annotation clusters: non-rescued aging DEGs (|LFC| > 1)',
    footnote = ('Top 10 clusters per tissue ranked by enrichment score. '
                'Note: muscle non-rescued input set is small (n=37 genes at |LFC|>1), '
                'resulting in low enrichment scores. '
                'Kidney: 431 genes input. Both tissues: STAR+Salmon/GRCm39/Ensembl 110.'),
    top_n=10, figsize=(15, 7),
)

## Supplementary Tables v2 (all GOBP enrichment CSVs)

In [ ]:
# ── Supplementary Tables v2 ──────────────────────────────────────
# Same as original but with v2 muscle enrichment files
FILENAME_S_V2 = 'Supplementary_Tables_GOBP_Enrichment_v2'

K_GOBP = f"{DATA_BASE}/Kidney/2_Enrichment_bulk&sn/GOBP bulk"
M_GOBP_V2 = f"{MUSCLE_V2}"                                          # ← v2
K_CT = f"{DATA_BASE}/Kidney/2_Enrichment_bulk&sn/GOBP_per_celltype/rescued"
M_CT_V2 = f"{MUSCLE_V2}/GOBP_per_celltype/rescued"                  # ← v2 (needs server export first)

sheets_bulk_v2 = [
    ('S3.4a_K_aging',     f'{K_GOBP}/kidney_enrichment_results_aging.csv',
     'Kidney GOBP enrichment of aging DEGs (7,499 genes)'),
    ('S3.4b_M_aging',     f'{M_GOBP_V2}/muscle_v2_enrichment_results_aging.csv',
     'Muscle GOBP enrichment of aging DEGs (1,381 genes) — STAR+Salmon/GRCm39'),        # ← v2
    ('S3.4c_K_rescued',   f'{K_GOBP}/kidney_enrichment_results_rescued-to-normal.csv',
     'Kidney GOBP enrichment of rescued-to-normal genes (1,961 genes)'),
    ('S3.4d_M_rescued',   f'{M_GOBP_V2}/muscle_v2_enrichment_results_rescued_to_normal.csv',
     'Muscle GOBP enrichment of rescued-to-normal genes (491 genes) — STAR+Salmon/GRCm39'),  # ← v2
    ('S3.4e_K_nonrescued',f'{K_GOBP}/kidney_enrichment_results_non_rescued_genes.csv',
     'Kidney GOBP enrichment of non-rescued genes (1,169 genes)'),
    ('S3.4f_M_nonrescued',f'{M_GOBP_V2}/muscle_v2_enrichment_results_non_rescued_genes.csv',
     'Muscle GOBP enrichment of non-rescued genes (241 genes) — STAR+Salmon/GRCm39'),   # ← v2
    ('S3.4g_K_DR',        f'{K_GOBP}/kidney_enrichment_results_rescued-to-normal__DR.csv',
     'Kidney GOBP enrichment of DR-exclusive rescued genes (546 genes)'),
    ('S3.4h_M_DR',        f'{M_GOBP_V2}/muscle_v2_enrichment_results_rescued-to-normal__DR.csv',
     'Muscle GOBP enrichment of DR-exclusive rescued genes (211 genes) — STAR+Salmon/GRCm39'),  # ← v2
    ('S3.4i_K_sAct',      f'{K_GOBP}/kidney_enrichment_results_rescued-to-normal_sAct.csv',
     'Kidney GOBP enrichment of sAct-exclusive rescued genes (108 genes)'),
    ('S3.4j_M_sAct',      f'{M_GOBP_V2}/muscle_v2_enrichment_results_rescued-to-normal_sAct.csv',
     'Muscle GOBP enrichment of sAct-exclusive rescued genes (20 genes) — STAR+Salmon/GRCm39'),  # ← v2
    ('S3.4k_K_combi',     f'{K_GOBP}/kidney_enrichment_results_rescued-to-normal_only_in_combi.csv',
     'Kidney GOBP enrichment of combi-only rescued genes (87 genes)'),
    ('S3.4l_M_combi',     f'{M_GOBP_V2}/muscle_v2_enrichment_results_rescued-to-normal_only_in_combi.csv',
     'Muscle GOBP enrichment of combi-only rescued genes (14 genes) — STAR+Salmon/GRCm39'),  # ← v2
]

# Per-celltype sheets
sheets_celltype_v2 = []

# Kidney per-celltype — unchanged
for subdir in ['ECs', 'IMM', 'Podo', 'Rest']:
    dirpath = f'{K_CT}/{subdir}'
    if os.path.isdir(dirpath):
        for fname in sorted(os.listdir(dirpath)):
            if fname.endswith('.csv'):
                ct_name = fname.replace('gobp_rescued_', '').replace('.csv', '')
                sheets_celltype_v2.append((
                    f'S3.5_K_{ct_name}'[:31],
                    f'{dirpath}/{fname}',
                    f'Kidney {ct_name}: GOBP enrichment of rescued gene markers'
                ))

# Muscle per-celltype v2
# Check if v2 per-celltype files exist (need server export first)
m_ct_v2_exists = os.path.isdir(M_CT_V2)
print(f"Muscle v2 per-celltype GOBP dir exists: {m_ct_v2_exists}")

if m_ct_v2_exists:
    for subdir in ['ECs', 'IMM', 'NMJs', 'Rest']:
        dirpath = f'{M_CT_V2}/{subdir}'
        if os.path.isdir(dirpath):
            for fname in sorted(os.listdir(dirpath)):
                if fname.endswith('.csv'):
                    ct_name = fname.replace('gobp_rescued_', '').replace('.csv', '')
                    sheets_celltype_v2.append((
                        f'S3.5_Mv2_{ct_name}'[:31],
                        f'{dirpath}/{fname}',
                        f'Muscle v2 {ct_name}: GOBP enrichment of rescued gene markers (STAR+Salmon/GRCm39)'
                    ))
else:
    print("  Muscle v2 per-celltype GOBP not yet generated — skipping those sheets.")
    print("  Run the R per-celltype enrichment script on the server first,")
    print("  then re-run this cell to include them.")
    # Fall back to original muscle per-celltype
    for subdir in ['ECs', 'IMM', 'NMJs', 'Rest']:
        dirpath = f"{DATA_BASE}/Muscle_old/2_Enrichment_tables/GOBP_per_celltype/rescued/{subdir}"
        if os.path.isdir(dirpath):
            for fname in sorted(os.listdir(dirpath)):
                if fname.endswith('.csv'):
                    ct_name = fname.replace('gobp_rescued_', '').replace('.csv', '')
                    sheets_celltype_v2.append((
                        f'S3.5_M_{ct_name}'[:31],
                        f'{dirpath}/{fname}',
                        f'Muscle {ct_name}: GOBP enrichment of rescued gene markers (original pipeline)'
                    ))

# ── Build workbook (same styling as original) ────────────────────
wb_v2 = openpyxl.Workbook()
wb_v2.remove(wb_v2.active)

# Index sheet
ws_idx = wb_v2.create_sheet(title='Index')
ws_idx.cell(row=1, column=1,
    value='Supplementary Table S3.4/S3.5 — GOBP Enrichment Results (v2: STAR+Salmon/GRCm39)'
    ).font = Font(bold=True, size=12)
ws_idx.cell(row=2, column=1,
    value='clusterProfiler enrichGO, padj < 0.05, GO:BP ontology. '
          'Muscle re-processed with harmonised STAR+Salmon/GRCm39 pipeline (identical to kidney).'
    ).font = desc_font
ws_idx.cell(row=4, column=1, value='Sheet').font = Font(bold=True, size=10)
ws_idx.cell(row=4, column=2, value='Description').font = Font(bold=True, size=10)
ws_idx.cell(row=4, column=3, value='Terms').font = Font(bold=True, size=10)
ws_idx.column_dimensions['A'].width = 30
ws_idx.column_dimensions['B'].width = 65
ws_idx.column_dimensions['C'].width = 10

# Redefine col_widths in case it was overwritten in this session
col_widths = {'ID': 15, 'Description': 45, 'GeneRatio': 12, 'BgRatio': 12,
              'FoldEnrichment': 15, 'pvalue': 14, 'p.adjust': 14, 'qvalue': 14, 'Count': 8}

print("=== Building bulk enrichment sheets (S3.4) ===")
idx_row = 5
for sheet_name, csv_path, desc in sheets_bulk_v2:
    add_sheet(wb_v2, sheet_name, csv_path, desc)
    try:
        with open(csv_path) as f:
            n = sum(1 for _ in csv.DictReader(f))
    except FileNotFoundError:
        n = 0
    ws_idx.cell(row=idx_row, column=1, value=sheet_name).font = data_font
    ws_idx.cell(row=idx_row, column=2, value=desc).font = data_font
    ws_idx.cell(row=idx_row, column=3, value=n).font = data_font
    idx_row += 1

idx_row += 1
ws_idx.cell(row=idx_row, column=1,
    value='Per-cell-type enrichment (S3.5)').font = Font(bold=True, size=10)
idx_row += 1

print("\n=== Building per-celltype sheets (S3.5) ===")
for sheet_name, csv_path, desc in sheets_celltype_v2:
    add_sheet(wb_v2, sheet_name, csv_path, desc)
    try:
        with open(csv_path) as f:
            n = sum(1 for _ in csv.DictReader(f))
    except FileNotFoundError:
        n = 0
    ws_idx.cell(row=idx_row, column=1, value=sheet_name).font = data_font
    ws_idx.cell(row=idx_row, column=2, value=desc).font = data_font
    ws_idx.cell(row=idx_row, column=3, value=n).font = data_font
    idx_row += 1

out_path = f'{OUTPUT_DIR}/{FILENAME_S_V2}.xlsx'
wb_v2.save(out_path)
print(f'\nSaved: {out_path}')
print(f'Total sheets: {len(wb_v2.sheetnames)} (1 index + {len(wb_v2.sheetnames)-1} data)')

## SUPPLEMENTARY FIGURES FOR DISCUSSION

In [ ]:
"""
============================================================
SUPPLEMENTARY FIGURES FOR DISCUSSION
============================================================
Add these code blocks to results_plots_clean.ipynb after all existing figures.
They use data from the thesis_supp_analysis CSVs generated on the server.
All figures follow existing style conventions.

PREREQUISITE: Upload the following CSVs from your server to Google Drive
and set SUPP_DIR to point to them:
  - 01_PT_marker_expression.csv
  - 02_kidney_IMM_marker_expression.csv
  - 03_kidney_FIB_collagen_ratio.csv
  - 04_kidney_EC_marker_expression.csv
  - 09_MMP_TIMP_expression.csv
  - 10_kidney_FIB_MMP2_TIMP2_ratio.csv
============================================================
"""

# ── Path to uploaded supplementary CSVs ──────────────────────────
SUPP_DIR = f"{DATA_BASE}/Thesis_suppl_analysis"
# Adjust this path to wherever you put the server output CSVs on Drive


# ============================================================
# Supp Fig S4.1 — PT-1 vs PT-2 damage characterization
# ============================================================
FILENAME_PT = 'Supp_Fig_S4_1_PT_damage_characterization'

import pandas as pd

pt_expr = pd.read_csv(f'{SUPP_DIR}/01_PT_marker_expression.csv')

damage_genes = ['Havcr1', 'Vcam1', 'Lcn2', 'Dcdc2a', 'Tpm1', 'Sox9']
healthy_genes = ['Slc34a1', 'Lrp2', 'Slc5a2', 'Slc22a6', 'Slc22a8']
all_genes = [g for g in damage_genes + healthy_genes if g in pt_expr.columns]

cond_order = ['ctrl', 'age', 'DR', 'sAct', 'combi']
cond_labels = ['Ctrl', 'Age', 'DR', 'sAct', 'Combi']

fig, axes = plt.subplots(1, 2, figsize=(18, 7), sharey=False)

for panel_idx, (ct, ax) in enumerate(zip(['PT-1', 'PT-2'], axes)):
    ct_data = pt_expr[pt_expr['celltype'] == ct].set_index('condition').reindex(cond_order)

    # Separate damage and healthy
    damage_avail = [g for g in damage_genes if g in ct_data.columns]
    healthy_avail = [g for g in healthy_genes if g in ct_data.columns]

    x = np.arange(len(cond_order))
    width = 0.08
    n_genes = len(all_genes)

    for i, gene in enumerate(all_genes):
        if gene not in ct_data.columns:
            continue
        vals = ct_data[gene].astype(float).values
        color = C_RED_D if gene in damage_genes else C_BLUE_D
        alpha = 0.5 + 0.5 * (i % 3) / 3
        offset = (i - n_genes/2) * width
        bars = ax.bar(x + offset, vals, width * 0.9, label=gene if panel_idx == 0 else '',
                      color=color, alpha=alpha, edgecolor='white', linewidth=0.3)

    ax.set_xticks(x)
    ax.set_xticklabels(cond_labels, fontsize=11, fontweight='bold')
    ax.set_ylabel('Mean expression', fontsize=11, fontweight='bold')
    ax.set_title(f'{clean_name(ct)}', fontsize=14, fontweight='bold', color=C_TEXT)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    # Add n_cells annotation
    for j, cond in enumerate(cond_order):
        n = ct_data.loc[cond, 'n_cells'] if cond in ct_data.index else 0
        ax.text(j, ax.get_ylim()[0] - 0.05 * (ax.get_ylim()[1] - ax.get_ylim()[0]),
                f'n={int(float(n)):,}', ha='center', fontsize=7, color=C_TEXT2)

axes[0].legend(fontsize=7, ncol=2, loc='upper right', framealpha=0.8,
               title='Genes', title_fontsize=8)

fig.suptitle('PT-1 vs PT-2: damage and healthy proximal tubule markers across conditions',
             fontsize=FS_CAPTION + 2, fontweight='bold', color=C_TEXT, y=1.02)
fig.text(0.5, -0.03,
         'Red bars = damage markers (Havcr1/KIM-1, Vcam1, Lcn2, Dcdc2a, Tpm1, Sox9). '
         'Blue bars = healthy PT markers (Slc34a1, Lrp2, Slc5a2, Slc22a6, Slc22a8).\n'
         'Mean expression from snRNA-seq (exploratory, n=1/condition).',
         ha='center', fontsize=FS_CAPTION - 1, color=C_TEXT2, wrap=True)

plt.tight_layout()
save_fig(fig, FILENAME_PT)
print(f'Done — {FILENAME_PT}')


# ============================================================
# Supp Fig S4.2 — Collagen ratio + MMP/TIMP ratio in FIB
# ============================================================
FILENAME_ECM = 'Supp_Fig_S4_2_collagen_MMP_TIMP_ratios'

col_ratio = pd.read_csv(f'{SUPP_DIR}/03_kidney_FIB_collagen_ratio.csv')
mmp_ratio = pd.read_csv(f'{SUPP_DIR}/10_kidney_FIB_MMP2_TIMP2_ratio.csv')

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Panel A: Col3a1/Col4a1 ratio
ax = axes[0]
x = np.arange(len(cond_order))
ratio_vals = col_ratio.set_index('condition').reindex(cond_order)['ratio_3to4'].astype(float).values
colors_bar = [C_TEAL_D if c == 'ctrl' else C_RED_D if c == 'age' else C_BLUE_D for c in cond_order]

bars = ax.bar(x, ratio_vals, color=colors_bar, edgecolor='white', linewidth=0.8, width=0.6)
ax.axhline(y=ratio_vals[0], color=C_GRAY_D, linestyle='--', linewidth=0.8, alpha=0.6)
ax.set_xticks(x)
ax.set_xticklabels(cond_labels, fontsize=11, fontweight='bold')
ax.set_ylabel('Col3a1 / Col4a1 ratio', fontsize=11, fontweight='bold')
ax.set_title('A. Interstitial / basement membrane\ncollagen ratio in fibroblasts',
             fontsize=12, fontweight='bold', color=C_TEXT)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

for i, (v, bar) in enumerate(zip(ratio_vals, bars)):
    ax.text(bar.get_x() + bar.get_width()/2, v + 0.05, f'{v:.2f}',
            ha='center', fontsize=9, fontweight='bold', color=C_TEXT)

# Add n_cells
for i, cond in enumerate(cond_order):
    n = col_ratio.set_index('condition').loc[cond, 'n_cells']
    ax.text(i, -0.15, f'n={int(n)}', ha='center', fontsize=7, color=C_TEXT2)

# Panel B: MMP2/TIMP2 ratio
ax = axes[1]
mmp_vals = mmp_ratio.set_index('condition').reindex(cond_order)['ratio_MMP2_TIMP2'].astype(float).values

bars = ax.bar(x, mmp_vals, color=colors_bar, edgecolor='white', linewidth=0.8, width=0.6)
ax.axhline(y=mmp_vals[0], color=C_GRAY_D, linestyle='--', linewidth=0.8, alpha=0.6)
ax.set_xticks(x)
ax.set_xticklabels(cond_labels, fontsize=11, fontweight='bold')
ax.set_ylabel('MMP2 / TIMP2 ratio', fontsize=11, fontweight='bold')
ax.set_title('B. Matrix degradation / inhibition\nratio in fibroblasts',
             fontsize=12, fontweight='bold', color=C_TEXT)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

for i, (v, bar) in enumerate(zip(mmp_vals, bars)):
    ax.text(bar.get_x() + bar.get_width()/2, v + 0.02, f'{v:.2f}',
            ha='center', fontsize=9, fontweight='bold', color=C_TEXT)

for i, cond in enumerate(cond_order):
    n = mmp_ratio.set_index('condition').loc[cond, 'n_cells']
    ax.text(i, -0.06, f'n={int(n)}', ha='center', fontsize=7, color=C_TEXT2)

fig.suptitle('ECM remodelling ratios in kidney fibroblasts across conditions',
             fontsize=FS_CAPTION + 2, fontweight='bold', color=C_TEXT, y=1.02)
fig.text(0.5, -0.04,
         'A. Col3a1 (interstitial collagen) / Col4a1 (basement membrane collagen) ratio. '
         'Values > ctrl indicate fibrotic shift; combi shows synergistic reversal below ctrl.\n'
         'B. MMP2 (gelatinase) / TIMP2 (inhibitor) ratio. '
         'Values > ctrl indicate active matrix degradation.\n'
         'Dashed line = ctrl baseline. Exploratory snRNA-seq data (n=1/condition).',
         ha='center', fontsize=FS_CAPTION - 1, color=C_TEXT2, wrap=True)

plt.tight_layout()
save_fig(fig, FILENAME_ECM)
print(f'Done — {FILENAME_ECM}')


# ============================================================
# Supp Fig S4.3 — Immune subtype markers across conditions
# ============================================================
FILENAME_IMM = 'Supp_Fig_S4_3_immune_subtype_markers'

imm_expr = pd.read_csv(f'{SUPP_DIR}/02_kidney_IMM_marker_expression.csv')

# Define marker categories and available genes
marker_categories = {
    'Macrophage': ['Cd68', 'Adgre1', 'Csf1r'],
    'B cell': ['Ms4a1'],
    'Dendritic': ['Cd80', 'Clec9a'],
    'Monocyte': ['Cd14', 'Fcgr3'],
    'Mast cell': ['Kit'],
    'MHC-II': ['H2-Aa', 'H2-Ab1', 'H2-Eb1'],
}

# Filter to available columns
avail_cats = {}
for cat, genes in marker_categories.items():
    avail = [g for g in genes if g in imm_expr.columns]
    if avail:
        avail_cats[cat] = avail

all_markers = [g for genes in avail_cats.values() for g in genes]
imm_data = imm_expr.set_index('condition').reindex(cond_order)[all_markers].astype(float)

fig, ax = plt.subplots(figsize=(14, 5))

# Grouped bar chart
n_conds = len(cond_order)
n_genes = len(all_markers)
x = np.arange(n_genes)
width = 0.15

cond_colors = {
    'ctrl': C_TEAL_D, 'age': C_RED_D, 'DR': '#E8A86B',
    'sAct': C_PURP_D, 'combi': C_BLUE_D
}

for i, cond in enumerate(cond_order):
    vals = imm_data.loc[cond].values
    offset = (i - n_conds/2 + 0.5) * width
    ax.bar(x + offset, vals, width * 0.9, label=cond_labels[i],
           color=cond_colors[cond], edgecolor='white', linewidth=0.3)

ax.set_xticks(x)
ax.set_xticklabels(all_markers, fontsize=9, rotation=45, ha='right')
ax.set_ylabel('Mean expression in kidney IMM', fontsize=11, fontweight='bold')
ax.legend(fontsize=9, ncol=5, loc='upper right', framealpha=0.8)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# Add category brackets
cat_starts = {}
pos = 0
for cat, genes in avail_cats.items():
    cat_starts[cat] = (pos, pos + len(genes) - 1)
    pos += len(genes)

ymin = ax.get_ylim()[0]
for cat, (start, end) in cat_starts.items():
    mid = (start + end) / 2
    ax.text(mid, ax.get_ylim()[1] * 1.05, cat, ha='center', fontsize=8,
            fontweight='bold', color=C_TEXT2)

fig.suptitle('Kidney IMM cluster: immune subtype marker expression across conditions',
             fontsize=FS_CAPTION + 2, fontweight='bold', color=C_TEXT, y=1.02)
fig.text(0.5, -0.06,
         'Grouped bars show mean expression of immune lineage markers in the kidney IMM cluster per condition.\n'
         'MHC-II markers (H2-Aa, H2-Ab1, H2-Eb1) show strong expression in ctrl that decreases with aging and combi.\n'
         'Macrophage identity markers (Adgre1, Csf1r) remain stable across conditions.\n'
         'Exploratory snRNA-seq data (n=1/condition).',
         ha='center', fontsize=FS_CAPTION - 1, color=C_TEXT2, wrap=True)

plt.tight_layout()
save_fig(fig, FILENAME_IMM)
print(f'Done — {FILENAME_IMM}')


# ============================================================
# Supp Fig S4.4 — ECM markers across cell types and conditions
# ============================================================
FILENAME_ECM_CT = 'Supp_Fig_S4_4_ECM_markers_by_celltype'

# This figure uses the MMP/TIMP expression CSV which has per-celltype data
mmp_expr = pd.read_csv(f'{SUPP_DIR}/09_MMP_TIMP_expression.csv')

# Also load the collagen data from Part 1 ECM markers
# We need to recreate this from the server output or use the ECM dotplot
# For now, create a combined heatmap from both CSVs

ecm_genes_mmp = [g for g in ['Mmp2', 'Mmp14', 'Timp2', 'Serpine1', 'Lox', 'Loxl2', 'Ccn2']
                 if g in mmp_expr.columns]

cell_types_ecm = ['FIB', 'vSMC/MC', 'PODO', 'PT-1', 'PT-2', 'IMM']

fig, ax = plt.subplots(figsize=(14, 10))

# Build heatmap matrix: rows = celltype_condition, cols = genes
rows = []
row_labels = []
for ct in cell_types_ecm:
    for cond in cond_order:
        mask = (mmp_expr['celltype'] == ct) & (mmp_expr['condition'] == cond)
        row_data = mmp_expr[mask]
        if len(row_data) > 0:
            vals = [float(row_data[g].values[0]) if g in row_data.columns else 0 for g in ecm_genes_mmp]
            rows.append(vals)
            row_labels.append(f'{clean_name(ct)}_{cond}')

matrix = np.array(rows)

# Normalize per column for better visualization
matrix_norm = np.zeros_like(matrix)
for j in range(matrix.shape[1]):
    col_max = matrix[:, j].max()
    if col_max > 0:
        matrix_norm[:, j] = matrix[:, j] / col_max

im = ax.imshow(matrix_norm, aspect='auto', cmap='Reds', interpolation='nearest')
ax.set_xticks(range(len(ecm_genes_mmp)))
ax.set_xticklabels(ecm_genes_mmp, fontsize=10, rotation=45, ha='right')
ax.set_yticks(range(len(row_labels)))
ax.set_yticklabels(row_labels, fontsize=8)

# Add gridlines between cell types
for i in range(1, len(cell_types_ecm)):
    ax.axhline(y=i * len(cond_order) - 0.5, color='black', linewidth=0.8)

cbar = plt.colorbar(im, ax=ax, shrink=0.6, pad=0.02)
cbar.set_label('Normalized expression', fontsize=9)

fig.suptitle('ECM remodelling markers across kidney cell types and conditions',
             fontsize=FS_CAPTION + 2, fontweight='bold', color=C_TEXT, y=1.01)
fig.text(0.5, -0.02,
         'Heatmap of MMP/TIMP and ECM remodelling gene expression, column-normalized.\n'
         'Mmp2 increase in FIB with aging and Ccn2 (CTGF) increase in vSMC/MC with combi are visible.\n'
         'Exploratory snRNA-seq data (n=1/condition).',
         ha='center', fontsize=FS_CAPTION - 1, color=C_TEXT2, wrap=True)

plt.tight_layout()
save_fig(fig, FILENAME_ECM_CT)
print(f'Done — {FILENAME_ECM_CT}')


print("\n=== All supplementary Discussion figures generated ===")

## Supp Fig S4.5 PDGF receptors both tissues

In [ ]:
# ── PDGF receptor DotPlot — both tissues side by side ────────

SN_DIR = "/content/drive/MyDrive/IMSB/projects/Hubers lab/Aging_DEG_Enrichment_Cellchat/fies/sn"


KIDNEY_CONDITION_MAP = {
    'sg18': 'ctrl', 'sg20': 'age', 'sg24': 'sAct', 'sg26': 'DR', 'sg28': 'combi'
}
MUSCLE_CONDITION_MAP = {
    'rgzj1': 'ctrl', 'rgzj2': 'age', 'rgzj3': 'sAct', 'rgzj4': 'DR', 'rgzj5': 'combi'
}

# ── PDGF receptors — Kidney ──────────────────────────────────
receptor_genes = ['Pdgfra', 'Pdgfrb']

sc.pl.dotplot(k_adata, var_names=receptor_genes, groupby=k_ct_col,
              standard_scale='var', show=False, figsize=(6, 6))
plt.title('Kidney: PDGF receptors by cell type', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}Supp_Fig_S4_5a_PDGF_receptors_kidney.png', dpi=150, bbox_inches='tight')
plt.show()

# ── PDGF receptors — Muscle ──────────────────────────────────
sc.pl.dotplot(m_adata, var_names=receptor_genes, groupby=m_ct_col,
              standard_scale='var', show=False, figsize=(6, 6))
plt.title('Muscle: PDGF receptors by cell type', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}Supp_Fig_S4_5b_PDGF_receptors_muscle.png', dpi=150, bbox_inches='tight')
plt.show()

print("Saved both panels")

In [ ]:
warnings.filterwarnings('ignore')
plt.close('all')

SN_DIR = "/content/drive/MyDrive/IMSB/projects/Hubers lab/Aging_DEG_Enrichment_Cellchat/files/sn"

KIDNEY_CONDITION_MAP = {
    'sg18': 'ctrl', 'sg20': 'age', 'sg24': 'sAct', 'sg26': 'DR', 'sg28': 'combi'
}
MUSCLE_CONDITION_MAP = {
    'rgzj1': 'ctrl', 'rgzj2': 'age', 'rgzj3': 'sAct', 'rgzj4': 'DR', 'rgzj5': 'combi'
}

print("Loading kidney...")
k_adata = sc.read_h5ad(f"{SN_DIR}/annotated-aging-soupxed.h5ad")
cond_col = 'sample' if 'sample' in k_adata.obs.columns else 'orig.ident' if 'orig.ident' in k_adata.obs.columns else k_adata.obs.columns[0]
k_adata.obs['condition'] = k_adata.obs[cond_col].astype(str).map(KIDNEY_CONDITION_MAP).fillna(k_adata.obs[cond_col].astype(str))
k_ct_col = 'mixed_celltype' if 'mixed_celltype' in k_adata.obs.columns else 'celltype'
print(f"Kidney loaded: {k_adata.shape}")

print("Loading muscle...")
m_adata = sc.read_h5ad(f"{SN_DIR}/annotated-muscle-soupxed.h5ad")
m_cond_col = 'sample' if 'sample' in m_adata.obs.columns else 'orig.ident' if 'orig.ident' in m_adata.obs.columns else m_adata.obs.columns[0]
m_adata.obs['condition'] = m_adata.obs[m_cond_col].astype(str).map(MUSCLE_CONDITION_MAP).fillna(m_adata.obs[m_cond_col].astype(str))
m_ct_col = 'mixed_celltype' if 'mixed_celltype' in m_adata.obs.columns else 'celltype'
print(f"Muscle loaded: {m_adata.shape}")

# ── PDGF receptors — Kidney ──────────────────────────────
receptor_genes = ['Pdgfra', 'Pdgfrb']

sc.pl.dotplot(k_adata, var_names=receptor_genes, groupby=k_ct_col,
              standard_scale='var', figsize=(6, 6), show=False)
plt.suptitle('Kidney: PDGF receptors by cell type', fontsize=12, fontweight='bold')
plt.savefig(f'{OUTPUT_DIR}/Supp_Fig_S4_5a_PDGF_receptors_kidney.png', dpi=150, bbox_inches='tight')
plt.show()
plt.close('all')

# ── PDGF receptors — Muscle ──────────────────────────────
sc.pl.dotplot(m_adata, var_names=receptor_genes, groupby=m_ct_col,
              standard_scale='var', figsize=(6, 6), show=False)
plt.suptitle('Muscle: PDGF receptors by cell type', fontsize=12, fontweight='bold')
plt.savefig(f'{OUTPUT_DIR}/Supp_Fig_S4_5b_PDGF_receptors_muscle.png', dpi=150, bbox_inches='tight')
plt.show()
plt.close('all')

print(f"Done — both panels saved in '{OUTPUT_DIR}'")

## Supp Fig S4.6 TGFB1 expression in immune cells across conditions ────────

In [ ]:
# ── TGFB1 expression in immune cells across conditions ────────

warnings.filterwarnings('ignore')

cond_order = ['ctrl', 'age', 'DR', 'sAct', 'combi']

# Kidney IMM
k_imm = k_adata[k_adata.obs[k_ct_col] == 'IMM'].copy()
k_imm.obs['condition'] = pd.Categorical(k_imm.obs['condition'], categories=cond_order, ordered=True)

# Muscle Macrophages
m_mac = m_adata[m_adata.obs[m_ct_col] == 'Macrophages'].copy()
m_mac.obs['condition'] = pd.Categorical(m_mac.obs['condition'], categories=cond_order, ordered=True)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Kidney
sc.pl.violin(k_imm, keys='Tgfb1', groupby='condition', order=cond_order,
             ax=axes[0], show=False, stripplot=True, jitter=0.3, size=1)
axes[0].set_title('Kidney IMM: Tgfb1', fontsize=12, fontweight='bold')
axes[0].set_xlabel('')
axes[0].set_ylabel('Expression')

# Muscle
sc.pl.violin(m_mac, keys='Tgfb1', groupby='condition', order=cond_order,
             ax=axes[1], show=False, stripplot=True, jitter=0.3, size=1)
axes[1].set_title('Muscle Macrophages: Tgfb1', fontsize=12, fontweight='bold')
axes[1].set_xlabel('')
axes[1].set_ylabel('')

fig.suptitle('Tgfb1 expression in immune cells across conditions (both tissues)',
             fontsize=13, fontweight='bold', y=1.03)
fig.text(0.5, -0.03, 'Exploratory snRNA-seq data (n=1/condition).',
         ha='center', fontsize=9, color='#555555')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/Supp_Fig_S4_6_Tgfb1_immune_both_tissues.png', dpi=150, bbox_inches='tight')
plt.savefig(f'{OUTPUT_DIR}/Supp_Fig_S4_6_Tgfb1_immune_both_tissues.pdf', bbox_inches='tight')
plt.show()
print("Saved: Supp_Fig_S4_6_Tgfb1_immune_both_tissues")


## Supp Fig S4.7b muscle_Mac_identity_vs_MHCII

In [ ]:
# ── DotPlot: immune markers in kidney IMM + muscle Macrophages per condition ──

cond_order = ['ctrl', 'age', 'DR', 'sAct', 'combi']
markers_dict = {
    'Macrophage': ['Adgre1', 'Csf1r', 'Cd68'],
    'MHC-II': ['H2-Aa', 'H2-Ab1', 'H2-Eb1']
}

# ── Kidney IMM ──
k_imm = k_adata[k_adata.obs[k_ct_col] == 'IMM'].copy()
k_imm.obs['condition'] = pd.Categorical(k_imm.obs['condition'], categories=cond_order, ordered=True)

k_markers = {}
for cat, genes in markers_dict.items():
    avail = [g for g in genes if g in k_imm.var_names]
    if avail:
        k_markers[cat] = avail

sc.pl.dotplot(k_imm, var_names=k_markers, groupby='condition',
              categories_order=cond_order, standard_scale='var',
              figsize=(8, 4), show=False)
plt.savefig(f'{OUTPUT_DIR_DIS}/Supp_Fig_S4_7a_kidney_IMM_identity_vs_MHCII.png',
            dpi=150, bbox_inches='tight')
plt.show()
plt.close('all')

# ── Muscle Macrophages ──
m_mac = m_adata[m_adata.obs[m_ct_col] == 'Macrophages'].copy()
m_mac.obs['condition'] = pd.Categorical(m_mac.obs['condition'], categories=cond_order, ordered=True)

m_markers = {}
for cat, genes in markers_dict.items():
    avail = [g for g in genes if g in m_mac.var_names]
    if avail:
        m_markers[cat] = avail

sc.pl.dotplot(m_mac, var_names=m_markers, groupby='condition',
              categories_order=cond_order, standard_scale='var',
              figsize=(8, 4), show=False)
plt.savefig(f'{OUTPUT_DIR_DIS}/Supp_Fig_S4_7b_muscle_Mac_identity_vs_MHCII.png',
            dpi=150, bbox_inches='tight')
plt.show()
plt.close('all')

print(f"Saved both panels to {OUTPUT_DIR_DIS}")

In [ ]:
# ── Check rescued genes from collaborator's list across conditions in kidney IMM ──
cond_order = ['ctrl', 'age', 'DR', 'sAct', 'combi']
collab_genes = ['B2m', 'Prkd1', 'Gstm1', 'Ccn4', 'Sp100', 'Marcks', 'Mndal', 'Stk10']
avail = [g for g in collab_genes if g in k_adata.var_names]

# DotPlot across all kidney cell types
sc.pl.dotplot(k_adata, var_names=avail, groupby=k_ct_col,
              standard_scale='var', figsize=(12, 8), show=False)
plt.savefig(f'{OUTPUT_DIR_DIS}/Supp_Fig_S4_8_collaborator_genes_all_celltypes.png',
            dpi=150, bbox_inches='tight')
plt.show()
plt.close('all')

# DotPlot in IMM + FIB + PT-2 per condition (the key cell types)
for ct in ['IMM', 'FIB', 'PT-2']:
    subset = k_adata[k_adata.obs[k_ct_col] == ct].copy()
    subset.obs['condition'] = pd.Categorical(subset.obs['condition'],
                                              categories=cond_order, ordered=True)
    ct_avail = [g for g in avail if g in subset.var_names]
    sc.pl.dotplot(subset, var_names=ct_avail, groupby='condition',
                  categories_order=cond_order, standard_scale='var',
                  figsize=(10, 4), show=False)
    plt.savefig(f'{OUTPUT_DIR_DIS}/Supp_Fig_S4_8_{ct}_collaborator_genes.png',
                dpi=150, bbox_inches='tight')
    plt.show()
    plt.close('all')

print(f"Saved to {OUTPUT_DIR_DIS}")

## ── Pipeline comparison summary figure ───────────────────────────

In [ ]:
# ── Pipeline comparison summary figure ───────────────────────────
FILENAME_V2 = 'Fig_pipeline_comparison_muscle'

fig, ax = plt.subplots(figsize=(10, 7))
ax.axis('off')

metrics = [
    ('Genes tested', 21055, 1381+662+241),  # approximate
    ('Aging DEGs', 1739, 1381),
    ('Rescued (Stage 1)', 926, 719),
    ('Rescued-to-normal', 511, 491),
    ('DR-exclusive', 241, 211),
    ('sAct-exclusive', 23, 20),
    ('Combi-only', 18, 14),
    ('Both DR + sAct', 229, 246),
    ('Non-rescued', 300, 241),
]

rows = [[m, str(old), str(new), f'{new/old*100:.0f}%' if old > 0 else '—']
        for m, old, new in metrics]

table = ax.table(cellText=rows,
                  colLabels=['Metric', 'Original\n(FeatureCounts/GRCm38)',
                            'v2\n(STAR+Salmon/GRCm39)', 'Ratio'],
                  cellLoc='center', loc='center',
                  colColours=[C_GRAY, C_RED, C_TEAL, C_AMBER])
table.auto_set_font_size(False)
table.set_fontsize(9)
table.scale(1, 1.4)

for (i, j), cell in table.get_celld().items():
    cell.set_edgecolor('#E0E0E0')
    if i == 0:
        cell.set_text_props(fontweight='bold', color='white')

ax.set_title('Pipeline comparison: muscle bulk RNA-seq',
             fontsize=12, fontweight='bold', color=C_TEXT, pad=20)
fig.text(0.5, 0.02,
         'Original: pre-processed FeatureCounts counts from GEO (GRCm38/GENCODE M20).\n'
         'v2: re-processed from raw FASTQs with STAR+Salmon/GRCm39, matching kidney pipeline.\n'
         'Mapping rates: 88.3–90.9%. Gene overlap: 82% of v2 DEGs confirmed by original.',
         ha='center', fontsize=7, color=C_TEXT2, linespacing=1.4)
plt.tight_layout()
save_fig(fig, FILENAME_V2)

# Vermeij long genes replication

In [ ]:
# ── Replicate Vermeij et al. long-gene analysis using existing pybiomart approach ──
import pandas as pd
import numpy as np
from pybiomart import Dataset
from scipy import stats as sp_stats

# Fetch gene lengths (same as Fig 3.1G)
print("Fetching gene lengths from Ensembl BioMart...")
dataset = Dataset(name='mmusculus_gene_ensembl', host='http://www.ensembl.org')
result = dataset.query(attributes=['ensembl_gene_id', 'external_gene_name',
                                    'start_position', 'end_position'])
result.columns = ['ensembl_id', 'symbol', 'start', 'end']
result['length'] = result['end'] - result['start']
gene_len = dict(zip(result['ensembl_id'], result['length']))
print(f"Got {len(gene_len)} gene lengths")

# Load DESeq2 results
KIDNEY_RES = f"{DATA_BASE}/Kidney/1_DEGs_tables/kidney_res_age_results_with_symbols.csv"
MUSCLE_RES = f"{MUSCLE_V2}/muscle_v2_res_age_results_with_symbols.csv"

k_res = pd.read_csv(KIDNEY_RES, index_col=0)
m_res = pd.read_csv(MUSCLE_RES, index_col=0)

for label, res in [("KIDNEY", k_res), ("MUSCLE v2", m_res)]:
    res = res.copy()
    res['gene_id'] = res.index.str.replace(r'\.\d+$', '', regex=True)
    res['length'] = res['gene_id'].map(gene_len)
    merged = res.dropna(subset=['length', 'log2FoldChange'])
    merged = merged[merged['baseMean'] > 1]
    merged_sorted = merged.sort_values('length', ascending=False)

    # Vermeij method 1: top 500 longest expressed genes
    top500 = merged_sorted.head(500)
    n_up = (top500['log2FoldChange'] > 0).sum()
    n_down = (top500['log2FoldChange'] < 0).sum()
    mean_lfc = top500['log2FoldChange'].mean()

    print(f"\n{'='*60}")
    print(f"{label} — VERMEIJ TOP 500 LONGEST GENES")
    print(f"{'='*60}")
    print(f"Expressed genes: {len(merged)}")
    print(f"  Upregulated: {n_up} ({n_up/500*100:.1f}%)")
    print(f"  Downregulated: {n_down} ({n_down/500*100:.1f}%)")
    print(f"  Mean log2FC: {mean_lfc:.4f}")

    # Vermeij method 2: up:down ratio across bins of 500 genes
    print(f"\nUp:Down ratio across gene length bins (longest first):")
    n_bins = min(15, len(merged_sorted) // 500)
    for i in range(n_bins):
        b = merged_sorted.iloc[i*500:(i+1)*500]
        bu = (b['log2FoldChange'] > 0).sum()
        bd = (b['log2FoldChange'] < 0).sum()
        ratio = bu / max(bd, 1)
        print(f"  Bin {i+1} ({b['length'].max()/1e6:.1f}-{b['length'].min()/1e6:.1f}Mb): up/down={ratio:.2f} ({bu}/{bd})")

# Fig. Established Aging Biomarkers — Rescued vs Non-Rescued

In [ ]:
# ── Figure: Established Aging Biomarkers — Rescued vs Non-Rescued ──
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Define the biomarker data from our results
data = {
    'Gene': ['Tgfb1', 'Bax', 'Lmna', 'B2m', 'Igf1r', 'Insr',
             'Eda2r', 'Cdkn2a', 'Cdkn1a', 'Gdf15', 'Kl', 'Sirt3',
             'Mtor', 'Foxo1', 'Sod1', 'Sod2', 'Cat', 'Nfkb1'],
    'Kidney_LFC': [0.533, 0.579, 0.504, 0.685, -0.538, -0.751,
                   4.042, 4.606, 3.378, 1.319, -0.825, -1.038,
                   -1.145, -0.845, -1.312, -0.561, -1.611, 0.651],
    'Muscle_LFC': [1.368, 0.657, 0.672, 0.169, 0.034, -0.098,
                   1.786, -0.013, 1.164, 1.970, -0.301, 0.112,
                   0.261, 0.086, 0.542, 0.123, 0.594, 0.111],
    'Rescued': ['Rescued', 'Rescued', 'Rescued', 'Rescued', 'Rescued', 'Rescued',
                'Not rescued', 'Not rescued', 'Not rescued', 'Not rescued',
                'Not rescued', 'Not rescued', 'Not rescued', 'Not rescued',
                'Not rescued', 'Not rescued', 'Not rescued', 'Not rescued'],
    'Category': ['Signalling', 'Apoptosis', 'Nuclear integrity', 'Senescence/MHC',
                 'Nutrient sensing', 'Nutrient sensing',
                 'Senescence', 'Senescence', 'Senescence', 'Stress/aging',
                 'Longevity', 'Mitochondrial', 'Nutrient sensing', 'Longevity',
                 'Antioxidant', 'Antioxidant', 'Antioxidant', 'Inflammation']
}

df = pd.DataFrame(data)

# Sort: rescued first, then by absolute kidney LFC
df['sort_key'] = df['Rescued'].map({'Rescued': 0, 'Not rescued': 1})
df = df.sort_values(['sort_key', 'Kidney_LFC'], ascending=[True, False]).reset_index(drop=True)

# Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 8), sharey=True)

colors = {'Rescued': '#2ecc71', 'Not rescued': '#e74c3c'}
y_pos = range(len(df))

# Kidney
bars1 = ax1.barh(y_pos, df['Kidney_LFC'], color=[colors[r] for r in df['Rescued']],
                  edgecolor='white', linewidth=0.5, height=0.7)
ax1.set_yticks(y_pos)
ax1.set_yticklabels([f"{row['Gene']}  ({row['Category']})" for _, row in df.iterrows()], fontsize=10)
ax1.set_xlabel('log₂FC (age vs ctrl)', fontsize=12)
ax1.set_title('Kidney', fontsize=14, fontweight='bold')
ax1.axvline(x=0, color='black', linewidth=0.8, linestyle='-')
ax1.axvline(x=0.5, color='gray', linewidth=0.5, linestyle='--', alpha=0.5)
ax1.axvline(x=-0.5, color='gray', linewidth=0.5, linestyle='--', alpha=0.5)
ax1.invert_yaxis()

# Add separator line between rescued and non-rescued
sep_idx = df[df['Rescued'] == 'Not rescued'].index[0] - 0.5
ax1.axhline(y=sep_idx, color='black', linewidth=1.5, linestyle='--', alpha=0.7)

# Muscle
bars2 = ax2.barh(y_pos, df['Muscle_LFC'], color=[colors[r] for r in df['Rescued']],
                  edgecolor='white', linewidth=0.5, height=0.7)
ax2.set_xlabel('log₂FC (age vs ctrl)', fontsize=12)
ax2.set_title('Muscle', fontsize=14, fontweight='bold')
ax2.axvline(x=0, color='black', linewidth=0.8, linestyle='-')
ax2.axvline(x=0.5, color='gray', linewidth=0.5, linestyle='--', alpha=0.5)
ax2.axvline(x=-0.5, color='gray', linewidth=0.5, linestyle='--', alpha=0.5)
ax2.axhline(y=sep_idx, color='black', linewidth=1.5, linestyle='--', alpha=0.7)

# Legend
rescued_patch = mpatches.Patch(color='#2ecc71', label='Rescued by combined intervention')
not_rescued_patch = mpatches.Patch(color='#e74c3c', label='Not rescued (therapeutic limit)')
fig.legend(handles=[rescued_patch, not_rescued_patch], loc='lower center',
           ncol=2, fontsize=11, bbox_to_anchor=(0.5, -0.02))

# Labels
ax1.text(0.02, 0.98, 'RESCUED', transform=ax1.transAxes, fontsize=9,
         fontweight='bold', color='#2ecc71', va='top')
ax1.text(0.02, 1 - (sep_idx + 1.5) / len(df), 'NOT RESCUED', transform=ax1.transAxes,
         fontsize=9, fontweight='bold', color='#e74c3c', va='top')

plt.suptitle('Established Aging Biomarkers: Rescued vs Non-Rescued\nby Combined DR + sActRIIB Intervention',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()

# Save
FILENAME = 'Supp_Fig_aging_biomarkers_rescued_vs_nonrescued'
plt.savefig(f"{OUTPUT_DIR_DIS}/{FILENAME}.pdf", bbox_inches='tight', dpi=300)
plt.savefig(f"{OUTPUT_DIR_DIS}/{FILENAME}.png", bbox_inches='tight', dpi=300)
plt.show()
print(f"Saved: {OUTPUT_DIR_DIS}/{FILENAME}.pdf")

# Zip & Download all v2 and corresponding

In [ ]:
# import os
# import zipfile
# from google.colab import files

# zip_path = '/content/Muscle_v2_figures_for_review.zip'

# # ── Only include figures that have both an original and a v2 version ──
# all_files = os.listdir(OUTPUT_DIR)

# # Find all v2 filenames
# v2_files = [f for f in all_files if '_v2.' in f]

# # For each v2 file, find the corresponding original (remove _v2 from name)
# pairs = []
# for v2f in sorted(v2_files):
#     orig_f = v2f.replace('_v2.', '.')
#     pairs.append(v2f)
#     if orig_f in all_files:
#         pairs.append(orig_f)
#     else:
#         print(f"No original found for: {v2f} (looked for {orig_f})")

# pairs = sorted(set(pairs))
# print(f"\nFiles to include: {len(pairs)}")
# for f in pairs:
#     tag = ' ← v2' if '_v2.' in f else '   original'
#     print(f"  {tag}  {f}")

# # ── Build zip with PNG / PDF subfolders ───────────────────────────
# with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
#     for fname in pairs:
#         fpath = os.path.join(OUTPUT_DIR, fname)
#         if not os.path.exists(fpath):
#             print(f"  MISSING: {fname}")
#             continue
#         ext = fname.rsplit('.', 1)[-1].upper()
#         zf.write(fpath, f'{ext}/{fname}')

# size_mb = os.path.getsize(zip_path) / 1e6
# print(f"\nZip created: {zip_path}  ({size_mb:.1f} MB)")

# files.download(zip_path)
# print("Download started!")

In [ ]:
import zipfile
from google.colab import files

zip_path = '/content/Muscle_v2_gene_lists.zip'
gene_list_files = [
    f"{MUSCLE_V2}/muscle_v2_deg_age.csv",
    f"{MUSCLE_V2}/muscle_v2_rescued_to_normal.csv",
    f"{MUSCLE_V2}/muscle_v2_intervention_impact_comparison_DR.csv",
    f"{MUSCLE_V2}/muscle_v2_intervention_impact_comparison_sAct.csv",
    f"{MUSCLE_V2}/muscle_v2_intervention_impact_comparison_only_rescued_in_combi.csv",
    f"{MUSCLE_V2}/muscle_v2_non_rescued_aged_DEGs.csv",
]

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fpath in gene_list_files:
        if os.path.exists(fpath):
            zf.write(fpath, os.path.basename(fpath))
            print(f"Added: {os.path.basename(fpath)}")
        else:
            print(f"NOT FOUND: {fpath}")

files.download(zip_path)
print("Download started!")